<div style="background: linear-gradient(135deg, #e9f2fb 0%, #f7fbff 100%); padding: 18px 20px; border: 1px solid #d5e3f0; border-radius: 10px; text-align: center;">
  <div style="font-size: 32px; font-weight: 700; color: #1f3b57; letter-spacing: 0.2px;">NX-414 Brain-like Computation and Intelligence</div>
</div>


<div style="text-align: center; font-size: 21px; font-weight: 600; color: #36536b; margin-top: 6px;">Project Notebook — Spring 2026</div>


<div style="text-align: center; color: #4f6478; font-size: 16px; margin-bottom: 10px;">Brain–Model Alignment Across Neural Recording Modalities</div>
<div style="text-align: center; color: #6b7280; font-size: 13px;">Prepared by: Abdulkadir Gokce</div>

---


# Group Information

Fill in this section at the top of your notebook and report.

- **Group member 1:** Anastasios Papapanagiotou, 414820, anastasios.papapanagiotou@epfl.ch
- **Group member 2:** Simon Seyfert, 408179, simon.seyfert@epfl.ch  
- **Group member 3:** Evan Massonnet, 346642, evan.massonnet@epfl.ch  

---

# What You Must Submit

Submit the following files:

1. **One Jupyter notebook** containing your full analysis.
2. **Any supporting Python scripts** needed to run the notebook.
3. **Figures that are part of your notebook answers** should be embedded and rendered in notebook Markdown.
4. **One PDF report** of **up to 2 pages**, **excluding references**, with **no appendix**.
5. **One zip archive** named exactly as:

```text
nx414_{SCIPER1}_{SCIPER2}_{SCIPER3}.zip
```

If your group has fewer than three members, reduce the number of `_SCIPER` fields accordingly.

## Submission Rules

- **Clear all notebook outputs before submission.**
- If outputs are not cleared, we will clear them ourselves and grade the cleaned notebook.
- Submit only the code required to reproduce your results.
- **Do not submit model weights.**
- **Do not submit CSV files or other large derived result dumps.**
- Keep the archive lightweight and reproducible.
- For the **final notebook**, any figure you want to present as part of your scientific argument should be **embedded in Markdown with accompanying interpretation**, rather than left as a raw cell output with no explanation.

Failure to follow these instructions may reduce your final grade.

## Use of LLMs

You may use LLM-based tools to help you write code, debug, or improve explanations. However, you remain fully responsible for the **correctness**, **quality**, and **clarity** of everything you submit, including both the notebook and the report.

In particular:

- check that any generated code actually runs and does what you claim it does,
- verify that any scientific statement or interpretation is correct,
- make sure the final writing sounds like a clear academic report written for this course,
- avoid vague, overly polished, or context-free text,
- avoid fancy wording or unnecessarily complex sentences that do not add clarity.

If you use an LLM, revise the output so that your submission reads naturally, is specific to your actual results, and does not look like generic generated text.

Failure to do so may result in **a point deduction**.

## Expected scope

Because this project spans roughly **three weeks** and counts for **30% of the final course grade**, the expected output is closer to a **compact course project** than to a one-week homework notebook. Your submission should therefore read like a small empirical study: it should be clearly structured, contain short written interpretations throughout, compare alternatives systematically, and end with a coherent synthesis of your main findings.

A strong notebook will not only run end-to-end, but will also explain **why** each analysis is being performed, what each metric is meant to capture, and what the results imply about the strengths and limitations of the models and datasets.

At the same time, **some parts of the project are intentionally left a bit loose**. This is by design: beyond implementing the required core analyses, you are expected to make reasonable scientific choices, justify them clearly, and show some ingenuity in how you explore the data and compare models.

## Suggested 3-week pacing

Use the notebook structure below to organize your work over the three weeks.

- **Week 1:** complete **Section 0** and **Section 1**. Understand the datasets, verify stimulus matching, inspect the processed responses, and start the visualization and reliability analyses.
- **Week 2:** complete the required analyses in **Section 2**. In this section, you must complete both the representational and predictive parts of the project. Begin **Section 3** by brainstorming possible extensions and sketching out a plan for your chosen extension.
- **Week 3:** complete **Section 3**, polish the notebook, select the strongest figures, and write the 2-page report.


---

# 0. Introduction and Setup

## 0.1 Project goal

In this project, you will study how neural responses from different recording modalities align with features extracted from two vision models. More specifically, you will work in the standard **brain–model alignment** setting: a model processes an image, a candidate internal layer is selected, and that representation is compared to measured neural responses using representational and predictive metrics.

The notebook is organized around four sections:

- **Section 0:** introduction, setup, and understanding the provided resources.
- **Section 1:** dataset inspection, visualization, and noise ceiling estimation.
- **Section 2:** brain–model alignment through both representational metrics and predictive linear models.
- **Section 3:** an open-ended extension beyond the baseline pipeline.

## 0.2 Why task-optimized models?

Task-optimized neural networks are among the most useful current **in-silico models of sensory cortex**. The central idea is simple: instead of hand-designing a model to mimic every biological detail, we optimize a model to perform a meaningful visual task and then ask whether its internal representations resemble those found in the brain. This approach has been highly influential because models trained to solve vision tasks often develop representations that predict activity along the visual hierarchy surprisingly well.

These models are useful scientifically because they provide **testable computational hypotheses**. If a model layer predicts neural responses well, that does not mean the brain literally implements the same mechanism, but it does suggest that the layer may encode information in a similar format or at a similar level of abstraction. Brain–model alignment is therefore a way to ask not just whether a model is accurate on a task, but whether it organizes visual information in a brain-relevant way.

## 0.3 Why compare multiple modalities?

A single recording modality gives only a partial view of neural computation. In this project, you will work with **electrophysiology, EEG, and fMRI**, which differ in temporal resolution, spatial resolution, and what exactly is measured. Looking across modalities helps you see which conclusions are robust and which depend on the measurement scale.

## 0.4 Learning goals

By the end of this project, you should be able to:

- inspect and summarize neural datasets from multiple modalities,
- visualize neural signals and data quality,
- implement and compare **two noise ceiling estimators**,
- implement **RSA** and **unbiased linear CKA**,
- fit **linear encoding models** from model features to neural responses,
- compare alignment across **models, layers, ROIs, and metrics**,
- interpret what each alignment metric captures,
- design and evaluate one meaningful extension beyond the baseline pipeline.

A strong submission should therefore demonstrate both **technical correctness** and **scientific reasoning**: beyond obtaining scores, you should be able to explain why a dataset is noisy, why one layer may outperform another, and why representational and predictive metrics sometimes disagree.

## 0.5 Provided data

All main data are stored in `/shared/NX-414/data/`.

### Background: processed data derivatives

The files provided for this project are **not raw neural recordings**. They are already processed, analysis-ready derivatives produced with modality-appropriate pipelines. This is important scientifically: many of your results will depend not only on the model features, but also on preprocessing choices such as repetition averaging, denoising, response-window selection, voxel/channel filtering, and how reliability is estimated.

We performed the preprocessing for you because these pipelines often require substantial modality-specific expertise, time, and compute.

At a high level, the datasets used here were prepared as follows:

- **TVSD (macaque electrophysiology)** — normalized multi-unit responses from ventral-stream areas. Responses were z-scored within session, firing rates were averaged in an analysis window centered on each site’s response peak, low-reliability channels were excluded, and repeated test responses were averaged for evaluation.
- **THINGS-EEG2 (human EEG)** — source EEG responses resampled to **100 Hz**. Noise ceilings were computed per subject, channel, and time point, and repetitions were averaged within train and test splits.
- **NSD (human fMRI)** — **b3 single-trial beta estimates** in `func1pt8mm` space, derived using voxel-wise HRF fitting, GLMdenoise, and ridge regression. Analyses are restricted to ROI-defined visually responsive voxels, low-reliability voxels are filtered out, and responses are averaged across available repetitions.

You are **not** expected to re-run the full preprocessing pipelines. You **are** expected to understand what kinds of neural quantities you are analyzing, what has already been averaged or denoised, and how these choices affect interpretation.

### Main neural datasets

- **`tvsd.h5`** — macaque electrophysiology from **2 monkeys**, with **22,248 train** and **100 test** stimuli, covering **V1, V4, and IT**.
- **`things_eeg2.h5`** — human EEG from **10 subjects**, with **16,540 train** and **200 test** stimuli, with region groupings such as **occipital**, **parietal**, **temporal**, **frontal**, **central**, **occipital_parietal**, and **whole_brain**.
- **`nsd_func1pt8mm_individualROIs.h5`** — human fMRI from **8 subjects**, with roughly **9,000 train** and **1,000 test** stimuli per subject, across multiple visual ROIs.

### Additional files

- `things_eeg2-test_reps.h5`  
  EEG test responses **with repetitions and without averaging**.  
  Use this file to implement and compare **two noise ceiling estimators**.

- `nsd-subj01-ncsnr-{lh,rh}.mgh`  
  Surface-based NSD reliability values for `subj-01` on **fsaverage**.  
  Use these to visualize cortical reliability and convert **ncsnr** into **noise ceiling**.

### Neural response shapes

- **TVSD:** `(n_stimuli, n_units)`
- **EEG2:** `(n_stimuli, n_channels, n_timepoints)`
- **NSD:** `(n_stimuli, n_voxels)`

For EEG, the time axis contains **80 time points** sampled at **100 Hz**, covering **0.0 s to 0.8 s**.

### Noise ceilings

Noise ceilings are stored per target:

- per neuron for **TVSD**,
- per channel × time point for **EEG2**,
- per voxel for **NSD**.

They are stored as **percent reliability**.  
To convert them to the range `[0, 1]`, divide by `100`.

In this project, the provided noise ceilings are mainly intended for **predictive metrics** such as **Pearson correlation** and **explained variance**. They reflect the reliability of the neural responses and therefore define an upper bound on how well any model can predict those responses.

When you compute predictive metrics, you should apply a noise ceiling correction to account for this upper bound. The standard idea is simple: divide the raw predictive score by the corresponding noise ceiling value for that target.

The provided noise ceilings are defined for **explained variance**. If you want to apply the same logic to **Pearson correlation**, first convert the explained-variance ceiling into a correlation ceiling by taking the element-wise square root, and then divide the raw correlation by that quantity. For a more detailed discussion of noise ceiling correction, see van Bree et al. (2025).

You may therefore report both **raw** and **noise-ceiling-corrected** predictive scores where appropriate.


For example, if the provided ceiling is an explained-variance reliability estimate, you can compute a noise-corrected Pearson correlation as:

```python
r_nc = r / np.sqrt(ev_ceiling)
```

where `r` is the raw Pearson correlation and `ev_ceiling` is the explained-variance ceiling expressed on the range `[0, 1]`.

By contrast, **RSA** and **CKA** should typically be reported as **raw values** in this project. Noise ceilings for representational similarity metrics require a different methodology and are **not** provided here.


Do **not** apply this correction directly to **RSA** or **CKA**.

### Stimulus identifiers

- **TVSD / EEG2:** byte strings such as `b'aardvark/aardvark_01b.jpg'`
- **NSD:** integer stimulus IDs

## 0.6 Model features

All extracted features are stored in `/shared/NX-414/extracted_features/`.

The feature files contain **internal activations extracted from multiple candidate layers** while the models process the same images shown in the neural experiments. You can think of each layer as a representation matrix of shape roughly **stimuli × features**. These layer-wise representations are what you will compare to the brain using RSA, CKA, and encoding models.

Feature extractions across models were made tractable by projecting activations to **30,000 dimensions** using a random projection. The provided feature files follow that same idea. In practice, this means you can treat the feature vectors as compact surrogates for the original activations while still performing meaningful alignment analyses.

### Model A: `adv_resnet152_imagenet_full_ffgsm_eps-1_alpha-125-ep10_seed-0`

- Architecture: **ResNet-152**
- Pretraining: ImageNet + adversarial fine-tuning
- Available feature files:
  - `things_stimuli.h5`
  - `nsd_stimuli.h5`

### Model B: `Qwen3-VL-2B-Instruct`

- Architecture: **vision-language transformer**
- Available feature files:
  - `things_stimuli.h5`
  - `nsd_stimuli.h5`

For both models, layers have been projected to **30,000 dimensions** using random projections.

### Feature extraction note

For this project, feature extraction has already been done for you. Your job is therefore not to run the vision models on raw images, but to understand **which layer** to use, how to match feature rows to neural stimuli, and what different layers reveal about representational hierarchy.

### Important note for NSD

For NSD, the feature files contain features for **all 73,000 images**, but each subject saw only a subset (~9,000).  
You must therefore select feature rows using the subject-specific NSD stimulus IDs.

## 0.7 Matching features to neural responses

You must match neural responses and features through the stimulus IDs.

### THINGS-based datasets: TVSD and EEG2

For these datasets, both neural IDs and feature IDs are byte strings. Matching should therefore be exact.

### NSD

For NSD, both sides use integer IDs. The feature file contains all 73,000 stimuli, while each subject has only a subset. Use the subject-specific NSD stimulus IDs to select the corresponding feature rows.

### Recommended procedure

```python
feat_ids = feature_file['ids'][:]
id_to_feat_idx = {id_: i for i, id_ in enumerate(feat_ids)}
feat_idx = np.array([id_to_feat_idx[x] for x in neural_ids])
```

To make HDF5 reads efficient:

1. build the index array,
2. sort indices before reading,
3. load only the required rows,
4. restore the original order.

## 0.8 General analysis rules

### Train/test discipline

Do **not** use the test split for model selection or hyperparameter tuning. Use a validation split or cross-validation within the training data.

### EEG targets

For EEG, you may either:

- flatten the targets to `(n_stimuli, n_channels * n_timepoints)`, or
- fit separate models per channel × time point.

Either choice is acceptable, but you must explain your decision clearly.

### Predictive metrics

Report the following predictive metrics where appropriate:

- `pearsonr`
- `pearsonr_nc`
- `explained_variance`
- `explained_variance_nc`

The provided EEG signals are not filtered like the other datasets. As a result, some low-reliability channels or time points can produce unstable predictive scores. When averaging predictive metrics over EEG targets, apply an on-the-fly filter such as **noise ceiling < 0.1**.

### Representational metrics

Also report:

- `RSA`
- `CKA`
- `encoding-RSA`
- `encoding-CKA`

Encoding RSA/CKA is a hybrid metric where you compute RSA or CKA between the predicted neural responses (from the linear encoding model) and the actual neural responses, instead of just comparing model activation directly. This can help you understand whether the linear encoding model captures the representational geometry of the neural data, beyond just predicting individual response values. You can refer to Conwell et al. (2024) for more details on this approach. 

Use only the test split for computing representational metrics.

Do **not** noise-correct RSA or CKA using the predictive-metric procedure.

## 0.9 Setup and data loading

Your notebook should begin with a short introduction, clear imports, utility functions, and a brief verification that the provided files are correctly organized and matched.

**Section 0 is required but not graded separately.** It is treated as setup for the rest of the project. Missing or incorrect setup may reduce scores in later sections if it affects correctness or reproducibility.

<div style="background:#eef5fb; border-left:4px solid #4c78a8; padding:8px 12px; border-radius:6px; font-weight:700; color:#26445e;">What you must do</div>

- Import the required packages.
- Define utility functions.
- Load metadata for all datasets.
- Inspect the structure of each `.h5` file.
- Load feature metadata for both models.
- Verify that feature IDs and neural stimulus IDs match.

<div style="background:#f3f6fa; border-left:4px solid #7a93ac; padding:8px 12px; border-radius:6px; font-weight:700; color:#32475b;">Required deliverables</div>

You must include all of the following:

1. **Dataset and feature overview:** one compact table or printed summary covering all neural datasets and both feature sets.
2. **Stimulus-matching verification:** explicit checks or assertions showing that stimulus matching works for THINGS-based datasets and for NSD.
3. **Short structural summary:** a short written note describing the main structural differences across datasets.

<div style="background:#eef8f4; border-left:4px solid #5b9a7a; padding:8px 12px; border-radius:6px; font-weight:700; color:#285943;">Questions you should answer</div>

- How many stimuli are available in each dataset?
- What is the shape of the neural response tensor in each dataset?
- Which datasets contain subjects, ROIs, repetitions, channels, or time points?
- What are the feature dimensionalities across layers?

In [ ]:
%load_ext autoreload
%autoreload 2

import torch
import torch.nn as nn
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from scipy.stats import spearmanr, pearsonr
import re
import os, gc, pickle
from collections import defaultdict
from typing import Literal
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from tqdm.auto import tqdm
import h5py
import nibabel as nib
from nilearn import plotting as nlplt
from nilearn import surface
from nilearn import datasets
import nibabel as nib
from sklearn.linear_model import Ridge # TODO: be careful with sklearn
from sklearn import metrics
from pathlib import Path
from itertools import combinations
import torch
import ipywidgets as widgets
from IPython.display import display
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')
# helper functions
from utils import *

# Reproducibility
SEED = 42
np.random.seed(SEED)

print("All imports OK")

In [ ]:
DATA_DIR    = Path("/shared/NX-414/data")
FEAT_DIR    = Path("/shared/NX-414/extracted_features")

# --- Neural data files ---
TVSD_PATH      = DATA_DIR / "tvsd.h5"
EEG_PATH       = DATA_DIR / "things_eeg2.h5"
EEG_REPS_PATH  = DATA_DIR / "things_eeg2-test_reps.h5"
NSD_PATH       = DATA_DIR / "nsd_func1pt8mm_individualROIs.h5"
NSD_NCSNR_LH   = DATA_DIR / "nsd-subj01-ncsnr-lh.mgh"
NSD_NCSNR_RH   = DATA_DIR / "nsd-subj01-ncsnr-rh.mgh"

# --- Model feature directories ---
MODEL_A_NAME = "adv_resnet152_imagenet_full_ffgsm_eps-1_alpha-125-ep10_seed-0"
MODEL_B_NAME = "Qwen3-VL-2B-Instruct"

FEAT_A_THINGS = FEAT_DIR / MODEL_A_NAME / "things_stimuli.h5"
FEAT_A_NSD    = FEAT_DIR / MODEL_A_NAME / "nsd_stimuli.h5"
FEAT_B_THINGS = FEAT_DIR / MODEL_B_NAME / "things_stimuli.h5"
FEAT_B_NSD    = FEAT_DIR / MODEL_B_NAME / "nsd_stimuli.h5"

# Sanity check — all files should exist
all_paths = {
    "tvsd":          TVSD_PATH,
    "eeg":           EEG_PATH,
    "eeg_reps":      EEG_REPS_PATH,
    "nsd":           NSD_PATH,
    "nsd_ncsnr_lh":  NSD_NCSNR_LH,
    "nsd_ncsnr_rh":  NSD_NCSNR_RH,
    "feat_A_things": FEAT_A_THINGS,
    "feat_A_nsd":    FEAT_A_NSD,
    "feat_B_things": FEAT_B_THINGS,
    "feat_B_nsd":    FEAT_B_NSD,
}

fsaverage = datasets.fetch_surf_fsaverage(mesh='fsaverage')

for name, p in all_paths.items():
    status = "✓" if p.exists() else "✗ MISSING"
    print(f"  {status}  {name}: {p}")

In [ ]:
# Qwen3-VL
qwen_ids = {}
for path, fname in [(FEAT_B_THINGS, "things_stimuli.h5"), (FEAT_B_NSD, "nsd_stimuli.h5")]:
    with h5py.File(path, 'r') as f:
        print(f"File {fname} contains {f['ids'].shape[0]} ids.")
        print(f'Contains {len(f["features"])} layers')
        key = next(iter(f['features']))
        print(f"Each feature set has shape: {f['features/'+key].shape}")
        print('-'*50)
        qwen_ids[fname] = f['ids'][:]

# Adv-ResNet152
resnet_ids = {}
for path, fname in [(FEAT_A_THINGS, "things_stimuli.h5"), (FEAT_A_NSD, "nsd_stimuli.h5")]:
    with h5py.File(path, 'r') as f:
        print(f"File {fname} contains {f['ids'].shape[0]} ids.")
        print(f'Contains {len(f["features"])} layers')
        key = next(iter(f['features']))
        print(f"Each feature set has shape: {f['features/'+key].shape}")
        print('-'*50)
        resnet_ids[fname] = f['ids'][:]

In [ ]:
# THINGS-based datasets (TVSD and EEG2)
tvsd_feat_idx = build_id_index(FEAT_A_THINGS)

with h5py.File(TVSD_PATH, "r") as f:
    tvsd_train_ids = f["train/stimulus_ids"][:]
    tvsd_test_ids  = f["test/stimulus_ids"][:]

missing_train = [x for x in tvsd_train_ids if x not in tvsd_feat_idx]
missing_test  = [x for x in tvsd_test_ids  if x not in tvsd_feat_idx]
assert len(missing_train) == 0, f"TVSD train: {len(missing_train)} IDs missing"
assert len(missing_test)  == 0, f"TVSD test:  {len(missing_test)} IDs missing"
print(f"TVSD ✓  train={len(tvsd_train_ids):,}  test={len(tvsd_test_ids):,}  all matched")

with h5py.File(EEG_PATH, "r") as f:
    eeg_train_ids = f["train/stimulus_ids"][:]
    eeg_test_ids  = f["test/stimulus_ids"][:]

missing_train = [x for x in eeg_train_ids if x not in tvsd_feat_idx]
missing_test  = [x for x in eeg_test_ids  if x not in tvsd_feat_idx]
assert len(missing_train) == 0, f"EEG train: {len(missing_train)} IDs missing"
assert len(missing_test)  == 0, f"EEG test:  {len(missing_test)} IDs missing"
print(f"EEG2 ✓  train={len(eeg_train_ids):,}  test={len(eeg_test_ids):,}  all matched")

# NSD
with h5py.File(NSD_PATH, "r") as f:
    nsd_ids_dict = {subj: f[f"train/stimulus_ids/{subj}"][:] 
                    for subj in f["train/stimulus_ids"].keys()}

nsd_ids_set = set(np.concatenate(list(nsd_ids_dict.values())))
assert set(qwen_ids['nsd_stimuli.h5']) >= nsd_ids_set and \
       (qwen_ids['nsd_stimuli.h5'] == resnet_ids['nsd_stimuli.h5']).all()
print('NSD ✓  feature ids contain all NSD ids found in the neural data')

things_ids_set = set(np.concatenate([tvsd_train_ids, tvsd_test_ids,
                                      eeg_train_ids,  eeg_test_ids]))
assert set(qwen_ids['things_stimuli.h5']) >= things_ids_set and \
       (qwen_ids['things_stimuli.h5'] == resnet_ids['things_stimuli.h5']).all()
print('THINGS ✓  feature ids contain all TVSD and EEG2 ids found in the neural data')

#### Structural summary

The three neural datasets differ in several key dimensions:

| Aspect | TVSD | EEG2 | NSD |
|---|---|---|---|
| Species | macaque | human | human |
| Modality | electrophysiology (MUA) | EEG | fMRI (betas) |
| Subjects | 2 monkeys | 10 subjects | 8 subjects |
| Response shape | (stimuli, units) | (stimuli, channels, timepoints) | (stimuli, voxels) |
| Temporal info | single scalar per unit | 80 time-points (100 Hz, 0–0.8 s) | single scalar per voxel |
| Spatial info | V1, V4, IT | channel groups (occipital, parietal, …) | individual ROIs |
| Train stimuli | 22,248 | 16,540 | 9,000 per subject |
| Test stimuli | 100 | 200 | 1,000 per subject |
| ID type | byte string | byte string | integer |


The two feature sets provide model activations for the same stimuli:

| Aspect | Model A (Adv-ResNet152) | Model B (Qwen3-VL-2B) |
|---|---|---|
| Architecture | CNN (ResNet-152) | Vision-language transformer |
| Training | ImageNet + adversarial fine-tuning | Multimodal instruction tuning |
| Stimulus sets | THINGS (22k), NSD (73k) | THINGS (22k), NSD (73k) |
| Feature dim per layer | 30,000 (random projection) | 30,000 (random projection) |
| ID type (THINGS) | byte string | byte string |
| ID type (NSD) | integer | integer |

<div style="background:#fff4c2; border:1px solid #c89b1f; border-left:6px solid #9a6f00; padding:10px 12px; border-radius:6px; margin-top:8px; margin-bottom:4px; color:#241a00; line-height:1.45;"><strong style="color:#5c4300;">Answer box 0</strong><br>

**TVSD** <a href="#ref-papale2025" title="Papale et al. (2025) — An extensive dataset of spiking activity to reveal the syntax of the ventral stream">[1]</a> contains responses shaped as *(stimuli × units)*. It has the most stimuli
(22k train) but no temporal axis. 

**THINGS-EEG2** <a href="#gifford2022" title="Gifford et al. (2022) — A large and rich EEG dataset for modeling human visual object recognition">[2]</a> provides human EEG from 10 subjects
with a tensor *(stimuli × channels × 80 time-points)*, making it the
most structurally complex dataset because every analysis can be resolved over both space
(channel groups) and time. 

**NSD** <a href="#allen2022" title="Allen et al. (2022) — A massive 7T fMRI dataset to bridge cognitive neuroscience and artificial intelligence">[3] </a>supplies human fMRI beta weights from 8 subjects shaped
as *(stimuli × voxels)*, but each subject only saw 9k images, a subset of the 73k total.
On the model side, 

**Model A** (adversarial ResNet-152 <a href="#wong2020fast" title="He et al. (2016) — Deep residual learning for image recognition">[4]</a>) is a CNN whose hierarchical
layers map naturally onto the ventral-stream hierarchy. 

**Model B** (Qwen3-VL-2B <a href="#bai2025" title="Bai et al. (2025) — Qwen3-VL technical report">[5]</a>) is a vision-language transformer whose layers may capture more semantic and multimodal
structure. 

Both provide layer activations projected to 30,000 dimensions. TVSD and EEG2 use images 
from the THINGS database <a href="#hebart2019" title="Hebart et al. (2019) — THINGS: A database of 1,854 object concepts and more than 26,000 naturalistic object images">[6]</a> — a curated set of 1,854 everyday object concepts — and are 
matched to model features via byte-string IDs. NSD uses a subset of 73k images drawn from 
MS-COCO <a href="#lin2014" title="Lin et al. (2014) — Microsoft COCO: Common objects in context">[7]</a>, matched via integer IDs, with each subject having seen a different subset of 
approximately 9k images.
</div>

---

# 1. Inspection, Visualization, and Noise Ceiling Estimates

This section is about understanding the data before doing model comparison. By the end of it, you should have a clear sense of how each modality is organized, which signals appear reliable, and how the provided reliability estimates relate to your own computations.

## 1.1 Inspect the datasets

<div style="background:#eef5fb; border-left:4px solid #4c78a8; padding:8px 12px; border-radius:6px; font-weight:700; color:#26445e;">What you must do</div>

- Inspect the content and axis meaning of TVSD, EEG2, and NSD.
- Identify where subject IDs, ROI labels, time axes, stimulus IDs, and noise ceilings are stored.
- State clearly what each axis means for each response array.

<div style="background:#f3f6fa; border-left:4px solid #7a93ac; padding:8px 12px; border-radius:6px; font-weight:700; color:#32475b;">Required deliverables</div>

You must include all of the following:

1. **TVSD structure table**
2. **EEG2 structure table**
3. **NSD structure table**

Each table must list the array name, array shape, and the meaning of each axis.

#### TVSD dataset structure

In [ ]:
print("=" * 55)
print("TVSD (macaque electrophysiology)")
print("=" * 55)
inspect_h5(TVSD_PATH, max_depth = 4) # adjusted to 4 to confirm shapes from "Neural response shapes" section

### TVSD dataset structure

|Array name | Array Shape | Axis Meaning |
|---|---|---|
| noise_ceilings/monkeyF/IT | (241,) | noise ceiling value per neuron |
| noise_ceilings/monkeyF/V1 | (462,) | noise ceiling value per neuron |
| noise_ceilings/monkeyF/V4 | (139,) | noise ceiling value per neuron |
| noise_ceilings/monkeyN/IT | (178,) | noise ceiling value per neuron |
| noise_ceilings/monkeyN/V1 | (437,) | noise ceiling value per neuron |
| noise_ceilings/monkeyN/V4 | (247,) | noise ceiling value per neuron |
| test/neural_data/monkeyF/IT | (100, 241) | (stimuli, units) |
| test/neural_data/monkeyF/V1 | (100, 462) | (stimuli, units) |
| test/neural_data/monkeyF/V4 | (100, 139) | (stimuli, units) |
| test/neural_data/monkeyN/IT | (100, 178) | (stimuli, units) |
| test/neural_data/monkeyN/V1 | (100, 437) | (stimuli, units) |
| test/neural_data/monkeyN/V4 | (100, 247) | (stimuli, units) |
| train/neural_data/monkeyF/IT | (22248, 241) | (stimuli, units) |
| train/neural_data/monkeyF/V1 | (22248, 462) | (stimuli, units) |
| train/neural_data/monkeyF/V4 | (22248, 139) | (stimuli, units) |
| train/neural_data/monkeyN/IT | (22248, 178) | (stimuli, units) |
| train/neural_data/monkeyN/V1 | (22248, 437) | (stimuli, units) |
| train/neural_data/monkeyN/V4 | (22248, 247) | (stimuli, units) |

The dataset is organized by monkey (monkeyF, monkeyN) and brain region (V1, V4, IT). 
Neural responses are 2D arrays of shape (stimuli x units), with rows corresponding 
positionally to stimulus IDs. Noise ceilings are stored per unit as 1D arrays matching 
the second axis of the neural data. The dataset comes pre-split into 22,248 training 
and 100 test stimuli. We use this split as provided — resplitting would require 
recomputing noise ceilings and would break comparability with published results using 
the same dataset. Cross-validation is performed within the training set only, ensuring 
no stimulus overlap between folds.


In [ ]:
#Check whether reponses are averaged repetitions or the same stimulus_ids appear multiple times per participant
with h5py.File(TVSD_PATH, "r") as f:
    train_ids = f["train/stimulus_ids"][:]

n_total = len(train_ids)
n_unique = len(set(train_ids))
print(f"Total: {n_total}, Unique: {n_unique}")

#### THINGS-EEG (human EEG) dataset structure

In [ ]:
print("\n" + "=" * 55)
print("THINGS-EEG2 (human EEG)")
print("=" * 55)
inspect_h5(EEG_PATH, max_depth = 4) # again depth = 4 for more details

In [ ]:
# Check whether reponses are averaged repetitions or the same stimulus_ids appear multiple times per participant
with h5py.File(EEG_PATH, "r") as f:
    train_ids = f["train/stimulus_ids"][:]

n_total = len(train_ids)
n_unique = len(set(train_ids))
print(f"Total: {n_total}, Unique: {n_unique}")

### THINGS-EEG2 dataset structure

Stimulus IDs serve as a shared key between neural recordings and model feature files. 
Since model features may only be extracted for a subset of available images, we cannot 
rely on row indices alone and must match explicitly through IDs. This is especially 
critical for NSD, where the feature file contains activations for all 73,000 images 
but each subject only saw roughly 9,000. For EEG2, stimulus IDs are shared across all 
10 subjects and all subjects saw the exact same stimuli. Training responses are already 
averaged across the 4 repetitions per stimulus.

| Array name | Array Shape | Axis Meaning |
|---|---|---|
| noise_ceilings/sub-X/central | (14,80) | (n_channels, n_timepoints) |
| noise_ceilings/sub-X/frontal | (21,80) | (n_channels, n_timepoints) |
| noise_ceilings/sub-X/occipital | (3,80) | (n_channels, n_timepoints) |
| noise_ceilings/sub-X/occipital_parietal | (17,80) | (n_channels, n_timepoints) |
| noise_ceilings/sub-X/parietal | (14,80) | (n_channels, n_timepoints) |
| noise_ceilings/sub-X/temporal | (6,80) | (n_channels, n_timepoints) |
| noise_ceilings/sub-X/whole_brain | (63,80) | (n_channels, n_timepoints) |
| test/neural_data/sub-X/central | (200,14,80) | (n_stimuli, n_channels, n_timepoints) |
| test/neural_data/sub-X/frontal | (200,21,80) | (n_stimuli, n_channels, n_timepoints) |
| test/neural_data/sub-X/occipital | (200,3,80) | (n_stimuli, n_channels, n_timepoints) |
| test/neural_data/sub-X/occipital_parietal | (200,17,80) | (n_stimuli, n_channels, n_timepoints) |
| test/neural_data/sub-X/parietal | (200,14,80) | (n_stimuli, n_channels, n_timepoints) |
| test/neural_data/sub-X/temporal | (200,6,80) | (n_stimuli, n_channels, n_timepoints) |
| test/neural_data/sub-X/whole_brain | (200,63,80) | (n_stimuli, n_channels, n_timepoints) |
| train/neural_data/sub-X/central | (16540,14,80) | (n_stimuli, n_channels, n_timepoints) |
| train/neural_data/sub-X/frontal | (16540,21,80) | (n_stimuli, n_channels, n_timepoints) |
| train/neural_data/sub-X/occipital | (16540,3,80) | (n_stimuli, n_channels, n_timepoints) |
| train/neural_data/sub-X/occipital_parietal | (16540,17,80) | (n_stimuli, n_channels, n_timepoints) |
| train/neural_data/sub-X/parietal | (16540,14,80) | (n_stimuli, n_channels, n_timepoints) |
| train/neural_data/sub-X/temporal | (16540,6,80) | (n_stimuli, n_channels, n_timepoints) |
| train/neural_data/sub-X/whole_brain | (16540,63,80) | (n_stimuli, n_channels, n_timepoints) |

The dataset is organized by subject (sub-01 through sub-10) and brain region grouping. 
Neural responses are 3D arrays of shape (stimuli x channels x timepoints), where 80 
timepoints cover 0–0.8s at 100Hz. Noise ceilings are 2D arrays of shape (channels x 
timepoints), reflecting that reliability varies across both space and time. The dataset 
is pre-split into 16,540 training and 200 test stimuli; we use this split as provided 
for the same reasons as TVSD. 

#### NSD (human fMRI) dataset

In [ ]:
print("\n" + "=" * 55)
print("NSD (human fMRI)")
print("=" * 55)
inspect_h5(NSD_PATH, max_depth=3)

In [ ]:
with h5py.File(NSD_PATH, "r") as f:
    mask_nc = f["roi_labels_nc/subj01/V1v"][:]
    nc      = f["noise_ceilings/subj01/V1v"][:]

print(f"roi_labels_nc True count: {mask_nc.sum()}")
print(f"noise_ceilings length:    {len(nc)}")

### NSD dataset structure

Unlike EEG2, each NSD subject has their own stimulus IDs since different subjects saw 
different subsets of the 73,000 COCO images. Neural responses are 2D arrays of shape 
(stimuli x voxels) per subject per ROI, with rows corresponding positionally to that 
subject's stimulus IDs. Not all subjects completed the same number of sessions, so the 
number of stimuli varies across subjects. Stimulus IDs are integers (COCO image IDs) 
and must be used explicitly to select the correct rows from the feature file — we 
cannot rely on row indices. Some ROIs may have zero voxels for certain subjects and 
are skipped when encountered.

| Array name | Array Shape | Axis Meaning |
|---|---|---|
| noise_ceiling_full/subj01 | (81, 104, 83) | (x, y, z) — 3D brain volume dimensions |
| noise_ceilings/subj01/EBA | (2078,) | (n_voxels,) |
| noise_ceilings/subj01/FBA-1 | (339,) | (n_voxels,) |
| noise_ceilings/subj01/FBA-2 | (255,) | (n_voxels,) |
| noise_ceilings/subj01/FFA-1 | (423,) | (n_voxels,) |
| noise_ceilings/subj01/FFA-2 | (173,) | (n_voxels,) |
| noise_ceilings/subj01/OFA | (283,) | (n_voxels,) |
| noise_ceilings/subj01/OPA | (1353,) | (n_voxels,) |
| noise_ceilings/subj01/OWFA | (369,) | (n_voxels,) |
| noise_ceilings/subj01/PPA | (604,) | (n_voxels,) |
| noise_ceilings/subj01/RSC | (362,) | (n_voxels,) |
| noise_ceilings/subj01/V1d | (727,) | (n_voxels,) |
| noise_ceilings/subj01/V1v | (570,) | (n_voxels,) |
| noise_ceilings/subj01/V2d | (557,) | (n_voxels,) |
| noise_ceilings/subj01/V2v | (725,) | (n_voxels,) |
| noise_ceilings/subj01/V3d | (505,) | (n_voxels,) |
| noise_ceilings/subj01/V3v | (554,) | (n_voxels,) |
| noise_ceilings/subj01/VWFA-1 | (666,) | (n_voxels,) |
| noise_ceilings/subj01/VWFA-2 | (281,) | (n_voxels,) |
| noise_ceilings/subj01/early | (39352,) | (n_voxels,) |
| noise_ceilings/subj01/hV4 | (578,) | (n_voxels,) |
| noise_ceilings/subj01/lateral | (2161,) | (n_voxels,) |
| noise_ceilings/subj01/midlateral | (735,) | (n_voxels,) |
| noise_ceilings/subj01/midparietal | (692,) | (n_voxels,) |
| noise_ceilings/subj01/midventral | (814,) | (n_voxels,) |
| noise_ceilings/subj01/nsdgeneral | (12687,) | (n_voxels,) |
| noise_ceilings/subj01/parietal | (1260,) | (n_voxels,) |
| noise_ceilings/subj01/ventral | (3027,) | (n_voxels,) |
| noise_ceilings/subj01/whole_brain | (36776,) | (n_voxels,) |

Noise ceilings are stored as 1D arrays corresponding exactly to the True positions in 
the roi_labels_nc masks — without this mapping you cannot know which voxels correspond 
to which entries in the 1D arrays. We verified this: the length of 
noise_ceilings/subj01/V1v matches exactly the number of True values in 
roi_labels_nc/subj01/V1v.

<div style="background:#fff4c2; border:1px solid #c89b1f; border-left:6px solid #9a6f00; padding:10px 12px; border-radius:6px; margin-top:8px; margin-bottom:4px; color:#241a00; line-height:1.45;"><strong style="color:#5c4300;">Answer box 1.1</strong><br>Explain the main differences between the three modalities in terms of what is being measured and how the data are organized.

The three datasets differ fundamentally in what they measure and how they are organized.

**TVSD** records multi-unit spiking activity (MUA) from macaque visual cortex using chronically 
implanted Utah arrays — 1,024 microelectrodes spread across V1, V4, and IT in two animals <a href="#ref-papale2025" title="Papale et al. (2025) — An extensive dataset of spiking activity to reveal the syntax of the ventral stream">[1]</a>.
MUA aggregates spiking from multiple neurons near each electrode tip, sampled at 30 kHz. This 
gives exceptional temporal resolution: response windows of 25–125ms for V1, 50–150ms for V4, 
and 75–175ms for IT already reflect the known latency gradient along the visual hierarchy. 
Within the implanted regions, spatial resolution is on the order of hundreds of micrometers. 
The data are organized as 2D arrays (n_stimuli × n_units) per monkey per region, with no time 
axis since responses are already averaged within the analysis window.

**THINGS-EEG2** records scalp EEG from 10 human subjects using non-invasive electrode caps <a href="#gifford2022" title="Gifford et al. (2022) — A large and rich EEG dataset for modeling human visual object recognition">[2]</a>. 
EEG provides full cortical coverage but poor spatial resolution due to volume conduction: the 
electrical signal spreads through skull and tissue before reaching the electrodes, blurring the 
spatial origin of the signal. Deep brain structures contribute minimally. Temporal resolution is 
good — the raw signal was sampled at 1 kHz and downsampled to 100 Hz for the provided data, 
though the 100 Hz is a preprocessing choice rather than a hardware limit. The data are organized 
as 3D arrays (n_stimuli × n_channels × n_timepoints), preserving the full temporal dynamics. 
Regions are defined anatomically by source location (occipital, parietal, temporal, frontal) 
rather than by functional visual area as in TVSD.

**NSD** records BOLD fMRI responses from 8 human subjects at 7T with 1.8mm voxel resolution <a href="#allen2022" title="Allen et al. (2022) — A massive 7T fMRI dataset to bridge cognitive neuroscience and artificial intelligence">[3]</a>. 
BOLD is a hemodynamic signal reflecting changes in the ratio of oxygenated to deoxygenated 
hemoglobin that follow neural activity with a substantial delay. The hemodynamic response peaks 
roughly 5–6 seconds after the neural event <a href="#liao2002" title="Liao et al. (2002) — Estimating the delay of the fMRI response">[8]</a>, and the brief initial dip in oxygenated 
hemoglobin that immediately follows neural firing is generally too small to reliably detect, 
meaning the measurable signal is dominated by the delayed overshoot of oxygenated blood flow. 
The 7T field strength was chosen specifically to maximize SNR over standard 3T scanners. Data 
are organized as 2D arrays (n_stimuli × n_voxels) per subject per ROI, where ROIs correspond 
to well-characterized functional visual areas (V1, V2, V3, hV4, FFA, PPA, etc.). Each subject 
has a different number of voxels per ROI due to individual brain anatomy, and stimulus IDs 
differ per subject since each saw a different subset of the 73,000 COCO images.

In summary, TVSD offers the highest temporal and spatial precision but is invasive, limited to 
macaques, and covers only a few targeted regions. EEG2 captures full-brain temporal dynamics 
non-invasively in humans but with poor spatial resolution. NSD provides the richest spatial 
picture of the human visual system at the cost of temporal resolution and an indirect 
hemodynamic signal. Together, the three modalities offer complementary views of the visual 
system that no single recording technique could provide alone.

**Data organization:**

TVSD is organized by monkey (monkeyF, monkeyN) and region (V1, V4, IT). Neural responses are 
stored as 2D arrays of shape `(n_stimuli, n_units)`, where the stimulus axis is aligned 
positionally to the stimulus ID array of shape `(n_stimuli,)`, meaning row $i$ of the neural 
data corresponds to stimulus $i$ of the ID array. Noise ceilings are stored as 1D arrays of 
shape `(n_units,)`, matching the second axis of the neural data exactly. The dataset is 
pre-split into 22,248 training and 100 test stimuli, shared across both monkeys.

THINGS-EEG2 is organized by subject (sub-01 through sub-10) and anatomical region (occipital, 
parietal, temporal, frontal, central, occipital_parietal, whole_brain). Neural responses are 
stored as 3D arrays of shape `(n_stimuli, n_channels, n_timepoints)`, where n_timepoints = 80 
covering 0–0.8s at 100 Hz. Noise ceilings are stored as 2D arrays of shape 
`(n_channels, n_timepoints)`, reflecting that reliability varies across both space and time. 
Two sets of noise ceilings are provided: `noise_ceilings/` computed from the 80 test 
repetitions and `noise_ceilings_train/` computed from the 4 training repetitions. Training 
responses are already averaged across repetitions (confirmed by all 16,540 training stimulus 
IDs being unique). The stimulus ID arrays of shape `(n_stimuli,)` are shared across all subjects 
since every subject saw the same images in the same order. The dataset is pre-split into 16,540 
training and 200 test stimuli.

NSD is organized by subject (subj01 through subj08) and ROI. Neural responses are stored as 2D 
arrays of shape `(n_stimuli, n_voxels)` per subject per ROI, positionally aligned to that 
subject's stimulus ID array. The number of stimuli and voxels varies across subjects due to 
differences in the number of completed sessions and individual brain anatomy. Two spatial 
structures are important here. `roi_labels/` stores 3D boolean masks of the full brain volume 
marking the anatomical extent of each ROI. `roi_labels_nc/` stores a refined version of this 
mask containing only the voxels that pass a reliability criterion. These are exactly the 
voxels included in the 1D neural data and noise ceiling arrays, confirmed by matching counts. 
Without `roi_labels_nc/` it is not possible to map the 1D arrays back to brain space. Noise 
ceilings are stored as 1D arrays of shape `(n_voxels,)` per subject per ROI, matching the 
second axis of the neural data. Stimulus IDs are integers (COCO image IDs) and differ per 
subject since each subject saw a different subset of the full 73,000-image pool.

</div>

---

## 1.2 Visualize EEG signals

<div style="background:#eef5fb; border-left:4px solid #4c78a8; padding:8px 12px; border-radius:6px; font-weight:700; color:#26445e;">What you must do</div>

- Plot example EEG responses for several stimuli and channels.
- Plot average responses over time for at least one subject and one ROI.
- Visualize the provided EEG noise ceilings over channels and time.

<div style="background:#f3f6fa; border-left:4px solid #7a93ac; padding:8px 12px; border-radius:6px; font-weight:700; color:#32475b;">Required deliverables</div>

You must include all of the following:

1. **One plot of example EEG time courses** for several stimuli and channels.
2. **One heatmap over channels × time** for at least one subject and one ROI.
3. **One summary plot of the provided EEG noise ceilings**.
4. **One short written interpretation** in Answer box 1.2.

### Quality control
#### Motivation

The provided data has been preprocessed by the dataset authors, including downsampling to 
100 Hz and multivariate noise normalization (MVNN) <a href="#gifford2022" title="Gifford et al. (2022) — A large and rich EEG dataset for modeling human visual object recognition">[2]</a>. 
However, the preprocessing pipeline does not include explicit epoch-level artifact rejection, 
meaning individual stimulus-channel pairs with extreme values may still be present. We 
therefore assess recording quality before fitting any models. We expected three types of 
problematic channels: dead channels with near-zero variance, channels with unusually high 
variance caused by movement artifacts or equipment noise, and channels that drift over time.

#### Z-score normalization

For each stimulus $i$ and channel $c$, we computed the variance over the 80 timepoints of 
the 0.8s epoch, then z-scored it using the mean and standard deviation computed across all 
16,540 training stimuli for that channel:

$$z_{i,c} = \frac{\text{var}_{i,c} - \mu_c}{\sigma_c}$$

This normalization was necessary because channels vary considerably in amplitude — some 
consistently show values in the $\pm 30$ range while others stay within $\pm 5$. Using raw 
variance would make any visualization dominated by high-amplitude channels, hiding degradation 
in quieter ones. The z-score instead shows relative changes within each channel compared to 
its own baseline.

#### Visual inspection

With 16,540 training stimuli per subject, inspecting every stimulus-channel pair individually 
is not feasible. We therefore grouped stimuli into blocks of 200 consecutive stimuli, using 
block index as a proxy for time in the experiment. The z-scored variance was averaged within 
each block to produce a matrix of shape $(n_{\text{blocks}}, n_{\text{channels}})$, where a 
uniformly anomalous column indicates a persistently problematic channel and a column that 
changes over time suggests a channel that degraded progressively during the recording.

The blocked heatmap provided a useful first overview but was somewhat hard to interpret on 
its own, as averaging over 200 stimuli smooths out individual artifacts. We therefore built 
an interactive tool to inspect individual blocks at full stimulus resolution using a symmetric 
log scale to handle the wide dynamic range of z-score values. This revealed recurring problems 
at channels 13 and 53, which appeared as mostly flat columns in the heatmap with occasional 
extreme values.

The figures below show representative time courses for subject 1 for channels 13 and 43 over stimuli 
7139–7143. Each panel shows the z-scored variance heatmap on the left and the raw time 
courses on the right. Channel 13 shows extreme artifact values reaching ±600, with the 
artifact appearing to worsen progressively across repetitions — consistent with electrode 
drift or a gradually deteriorating contact. Channel 43 over the same stimuli shows clean, 
stable responses typical of a functioning electrode.

<figure style="text-align: center;">
  <img src="images/stimulus_7139_7143_ch_13_z_score.png" style="width: 80%;"/>
  <figcaption>Channel 13 (anomalous) — stimuli 7139–7143</figcaption>
</figure>
<figure style="text-align: center;">
  <img src="images/stimulus_7139_7143_ch_43_z_score.png" style="width: 80%;"/>
  <figcaption>Channel 43 (normal) — stimuli 7139–7143</figcaption>
</figure>

#### Automated filters

To systematically flag faulty stimulus-channel pairs, we applied two complementary filters. 
The first flags any pair whose z-scored variance exceeds $\tau = 2.5$. Based on visual 
inspection, z-scores above ~3 were clearly artifactual while z-scores below ~1.3 appeared 
normal; we set the threshold conservatively between these values, flagging 2.11% of all 
stimulus-channel pairs.

<figure style="text-align: center;">
  <img src="images/z-score-distribution.png" style="width: 80%;"/>
  <figcaption>Z-score distribution across all stimulus-channel pairs</figcaption>
</figure>

The second filter targets extreme amplitude values that the variance filter can miss. A channel 
that shifts to a very different baseline but remains stable will have low variance and pass the 
z-score filter undetected. We therefore also flag any stimulus-channel pair where the maximum 
absolute amplitude exceeds 50, chosen by inspecting the amplitude distribution. After applying 
the z-score filter first, the amplitude filter flags an additional 0.54% of remaining pairs, 
confirming the two criteria are complementary rather than redundant.

<figure style="text-align: center;">
  <img src="images/max_abs_distribution.png" style="width: 80%;"/>
  <figcaption>Max absolute amplitude distribution</figcaption>
</figure>

The combined mask flags roughly 2.65% of all stimulus-channel pairs as faulty. No persistently 
dead channels were found, suggesting the preprocessing pipeline had already removed the most 
severely degraded channels. The secondary bump in the amplitude distribution around 750–1000 
may reflect a specific recurring artifact type rather than random noise, though we did not 
investigate it further. The thresholds were chosen based on visual inspection rather than a 
formal criterion, as neither distribution showed a clear natural gap that would justify a 
principled cutoff. For encoding model analyses, we prefer to err on the side of exclusion 
since a missing channel causes less harm than a corrupted one influencing the fit.

In [ ]:
# ---- Block-averaged channel quality heatmap ----
BLOCK_SIZE = 200
SUBJECT    = "sub-01"
ROI        = "whole_brain"

with h5py.File(EEG_PATH, "r") as f:
    # shape: (16540, n_channels, 80)
    train_data = f[f"train/neural_data/{SUBJECT}/{ROI}"][:]

n_stimuli, n_channels, n_timepoints = train_data.shape
n_blocks = n_stimuli // BLOCK_SIZE

# For each stimulus, compute variance over the 80 timepoints -> (n_stimuli, n_channels)
epoch_var = train_data.var(axis=2)

# Average within blocks -> (n_blocks, n_channels)
epoch_var_blocked = epoch_var[:n_blocks * BLOCK_SIZE].reshape(n_blocks, BLOCK_SIZE, n_channels).mean(axis=1)
# Z-score each channel across blocks
channel_mean = epoch_var_blocked.mean(axis=0)
channel_std  = epoch_var_blocked.std(axis=0)
epoch_var_blocked_norm = (epoch_var_blocked - channel_mean) / (channel_std + 1e-8)

# Plot
fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(epoch_var_blocked_norm , aspect="auto", cmap="viridis",
               interpolation="nearest")
ax.set_xlabel("Channel index")
ax.set_ylabel(f"Stimulus block (each = {BLOCK_SIZE} stimuli)")
ax.set_title(f"Block-averaged epoch variance — {SUBJECT} / {ROI}")
plt.colorbar(im, ax=ax, label="Normalized variance (relative to channel mean)")
plt.tight_layout()
plt.show()

In [ ]:
# ---- Block-averaged channel quality heatmap ----
BLOCK_SIZE = 200
SUBJECT    = "sub-01"
ROI        = "whole_brain"

with h5py.File(EEG_PATH, "r") as f:
    train_data = f[f"train/neural_data/{SUBJECT}/{ROI}"][:]

n_stimuli, n_channels, n_timepoints = train_data.shape
n_blocks = n_stimuli // BLOCK_SIZE

# For each stimulus, compute variance over the 80 timepoints -> (n_stimuli, n_channels)
epoch_var = train_data.var(axis=2)

# Average within blocks -> (n_blocks, n_channels)
epoch_var_blocked = epoch_var[:n_blocks * BLOCK_SIZE].reshape(n_blocks, BLOCK_SIZE, n_channels).mean(axis=1)

# Z-score each channel across blocks
channel_mean = epoch_var_blocked.mean(axis=0)
channel_std  = epoch_var_blocked.std(axis=0)
epoch_var_blocked_norm = (epoch_var_blocked - channel_mean) / (channel_std + 1e-8)

def plot_threshold(threshold):
    fig, ax = plt.subplots(figsize=(14, 6))
    mask = (epoch_var_blocked_norm < threshold).astype(float)
    im = ax.imshow(mask, aspect="auto", cmap="RdYlGn_r",
                   interpolation="nearest", vmin=0, vmax=1)
    ax.set_xlabel("Channel index")
    ax.set_ylabel(f"Stimulus block (each = {BLOCK_SIZE} stimuli)")
    ax.set_title(f"Dead channel detection (threshold={threshold:.2f}) — {SUBJECT} / {ROI}")
    plt.colorbar(im, ax=ax, label="0 = OK, 1 = dead/degraded")
    plt.tight_layout()
    plt.show()

slider = widgets.FloatSlider(
    value=-1.5, min=-3.0, max=0.0, step=0.1,
    description="Threshold:",
    continuous_update=False,
    style={"description_width": "initial"}
)

widgets.interact(plot_threshold, threshold=slider)

In [ ]:
n_examples = 5

# Compute normalization statistics across ALL stimuli (do this once)
with h5py.File(EEG_PATH, "r") as f:
    all_data = f[f"train/neural_data/{SUBJECT}/{ROI}"][:]

# Variance per stimulus -> (n_stimuli, n_channels)
epoch_var_all = all_data.var(axis=2)

# Statistics across all stimuli per channel
stim_mean = epoch_var_all.mean(axis=0)  # (n_channels,)
stim_std  = epoch_var_all.std(axis=0)   # (n_channels,)

def plot_channel_block_detail(block_idx, channel_idx, stim_offset):
    start = block_idx * BLOCK_SIZE
    end   = start + BLOCK_SIZE
    
    # Load this block
    data_block = all_data[start:end]  # (BLOCK_SIZE, n_channels, 80)
    
    # Variance per stimulus -> (BLOCK_SIZE, n_channels)
    var_block = data_block.var(axis=2)
    
    # Z-score using global statistics
    var_block_norm = (var_block - stim_mean) / (stim_std + 1e-8)

    print(f"Z-scores for shown stimuli (channel {channel_idx}):")
    for i in range(n_examples):
        stim_in_block = stim_offset + i
        z = var_block_norm[stim_in_block, channel_idx]
        print(f"  Stimulus {start + stim_in_block}: z = {z:.2f}")
        
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    # Left: heatmap of all stimuli in this block x all channels
    im = axes[0].imshow(var_block_norm, aspect="auto", cmap="viridis",
                    interpolation="nearest",
                    norm=mcolors.SymLogNorm(linthresh=1))
    axes[0].axvline(x=channel_idx, color="red", linewidth=1.5, 
                    label=f"Channel {channel_idx}")
    # Horizontal bars showing which stimuli are plotted on the right
    axes[0].axhline(y=stim_offset,             color="orange", linewidth=1.5, linestyle="--")
    axes[0].axhline(y=stim_offset + n_examples - 1, color="orange", linewidth=1.5, 
                    linestyle="--", label=f"Stimuli shown on right")
    axes[0].set_xlabel("Channel index")
    axes[0].set_ylabel("Stimulus index within block")
    axes[0].set_title(f"Z-scored variance — block {block_idx} (stimuli {start}–{end})")
    axes[0].legend(fontsize=8)
    plt.colorbar(im, ax=axes[0], label="Z-scored variance")
    
    # Right: time courses of selected channel for n_examples stimuli
    time_axis  = np.linspace(0, 0.8, n_timepoints)
    for i in range(n_examples):
        stim_in_block = stim_offset + i
        axes[1].plot(time_axis, data_block[stim_in_block, channel_idx, :],
                     alpha=0.7, label=f"Stimulus {start + stim_in_block}")
    axes[1].set_xlabel("Time (s)")
    axes[1].set_ylabel("Amplitude")
    axes[1].set_title(f"Time courses — channel {channel_idx}, block {block_idx}")
    axes[1].legend(fontsize=8)
    
    plt.tight_layout()
    plt.show()

n_examples = 5

block_slider = widgets.IntSlider(
    value=0, min=0, max=n_blocks - 1, step=1,
    description="Block index:",
    continuous_update=False,
    style={"description_width": "initial"}
)
channel_slider = widgets.IntSlider(
    value=0, min=0, max=n_channels - 1, step=1,
    description="Channel index:",
    continuous_update=False,
    style={"description_width": "initial"}
)
stim_slider = widgets.IntSlider(
    value=0, min=0, max=BLOCK_SIZE - n_examples, step=1,
    description="Stimulus offset:",
    continuous_update=False,
    style={"description_width": "initial"}
)

widgets.interact(plot_channel_block_detail, 
                 block_idx=block_slider, 
                 channel_idx=channel_slider,
                 stim_offset=stim_slider)

In [ ]:
# Z-score all stimuli using global statistics
epoch_var_norm_all = (epoch_var_all - stim_mean) / (stim_std + 1e-8)  # (n_stimuli, n_channels)

THRESHOLD = 2.5

fig, axes = plt.subplots(1, 2, figsize=(16, 4))

# Linear y-scale
axes[0].hist(epoch_var_norm_all.flatten(), bins=200, color="steelblue", edgecolor="none")
axes[0].axvline(x=THRESHOLD, color="red", linewidth=2, label=f"Threshold τ={THRESHOLD}")
axes[0].set_xlabel("Z-score")
axes[0].set_ylabel("Count")
axes[0].set_title(f"Z-score distribution — {SUBJECT} / {ROI}")
axes[0].legend()

# Log y-scale
axes[1].hist(epoch_var_norm_all.flatten(), bins=200, color="steelblue", edgecolor="none")
axes[1].axvline(x=THRESHOLD, color="red", linewidth=2, label=f"Threshold τ={THRESHOLD}")
axes[1].set_yscale("log")
axes[1].set_xlabel("Z-score")
axes[1].set_ylabel("Count (log scale)")
axes[1].set_title(f"Z-score distribution (log y) — {SUBJECT} / {ROI}")
axes[1].legend()

plt.tight_layout()
plt.show()

# Print fraction flagged
n_flagged = (epoch_var_norm_all > THRESHOLD).sum()
n_total   = epoch_var_norm_all.size
print(f"Flagged: {n_flagged} / {n_total} ({100 * n_flagged / n_total:.2f}%)")

In [ ]:
# Max absolute value per stimulus per channel
max_abs_all = np.abs(all_data).max(axis=2)  # (n_stimuli, n_channels)

THRESHOLD_ABS = 50

fig, axes = plt.subplots(1, 2, figsize=(16, 4))

# Linear y-scale
axes[0].hist(max_abs_all.flatten(), bins=200, color="steelblue", edgecolor="none")
axes[0].axvline(x=THRESHOLD_ABS, color="red", linewidth=2, label=f"Threshold={THRESHOLD_ABS}")
axes[0].set_xlabel("Max absolute amplitude")
axes[0].set_ylabel("Count")
axes[0].set_title(f"Max absolute amplitude distribution — {SUBJECT} / {ROI}")
axes[0].legend()

# Log y-scale
axes[1].hist(max_abs_all.flatten(), bins=200, color="steelblue", edgecolor="none")
axes[1].axvline(x=THRESHOLD_ABS, color="red", linewidth=2, label=f"Threshold={THRESHOLD_ABS}")
axes[1].set_yscale("log")
axes[1].set_xlabel("Max absolute amplitude")
axes[1].set_ylabel("Count (log scale)")
axes[1].set_title(f"Max absolute amplitude distribution (log y) — {SUBJECT} / {ROI}")
axes[1].legend()

plt.tight_layout()
plt.show()

# Print fraction flagged
n_flagged = (max_abs_all > THRESHOLD_ABS).sum()
n_total   = max_abs_all.size
print(f"Flagged: {n_flagged} / {n_total} ({100 * n_flagged / n_total:.2f}%)")

In [ ]:
# Apply z-score filter first
zscore_mask = epoch_var_norm_all > THRESHOLD  # (n_stimuli, n_channels), True = flagged

# Get max abs values for unflagged stimuli only
max_abs_filtered = max_abs_all.copy()
max_abs_filtered[zscore_mask] = np.nan  # mask out z-score flagged pairs

fig, axes = plt.subplots(1, 2, figsize=(16, 4))

# Linear y-scale
axes[0].hist(max_abs_filtered[~zscore_mask].flatten(), bins=100, 
             color="steelblue", edgecolor="none")
axes[0].axvline(x=THRESHOLD_ABS, color="red", linewidth=2, 
                label=f"Threshold={THRESHOLD_ABS}")
axes[0].set_xlabel("Max absolute amplitude")
axes[0].set_ylabel("Count")
axes[0].set_title(f"Max absolute amplitude (after z-score filter) — {SUBJECT} / {ROI}")
axes[0].legend()

# Log y-scale
axes[1].hist(max_abs_filtered[~zscore_mask].flatten(), bins=100, 
             color="steelblue", edgecolor="none")
axes[1].axvline(x=THRESHOLD_ABS, color="red", linewidth=2, 
                label=f"Threshold={THRESHOLD_ABS}")
axes[1].set_yscale("log")
axes[1].set_xlabel("Max absolute amplitude")
axes[1].set_ylabel("Count (log scale)")
axes[1].set_title(f"Max absolute amplitude (after z-score filter, log y) — {SUBJECT} / {ROI}")
axes[1].legend()

plt.tight_layout()
plt.show()

# Print fractions
n_remaining = (~zscore_mask).sum()
n_flagged_abs = (max_abs_filtered[~zscore_mask] > THRESHOLD_ABS).sum()
print(f"After z-score filter: {n_remaining:,} / {max_abs_all.size:,} pairs remaining")
print(f"Additionally flagged by amplitude: {n_flagged_abs} ({100 * n_flagged_abs / n_remaining:.2f}% of remaining)")

In [ ]:
# Combined quality mask
# True = flagged as bad (either high variance or extreme amplitude)
quality_mask = zscore_mask | (max_abs_all > THRESHOLD_ABS)  # (n_stimuli, n_channels)

n_flagged_total = quality_mask.sum()
n_total         = quality_mask.size
n_flagged_z     = zscore_mask.sum()
n_flagged_amp   = (max_abs_all > THRESHOLD_ABS).sum()
n_flagged_both  = (zscore_mask & (max_abs_all > THRESHOLD_ABS)).sum()

print(f"Total stimulus-channel pairs:        {n_total:,}")
print(f"Flagged by z-score (τ={THRESHOLD}):    {n_flagged_z:,} ({100 * n_flagged_z / n_total:.2f}%)")
print(f"Flagged by amplitude (>{THRESHOLD_ABS}): {n_flagged_amp:,} ({100 * n_flagged_amp / n_total:.2f}%)")
print(f"Flagged by both:                     {n_flagged_both:,} ({100 * n_flagged_both / n_total:.2f}%)")
print(f"Flagged by either (total):           {n_flagged_total:,} ({100 * n_flagged_total / n_total:.2f}%)")
print(f"Remaining clean pairs:               {n_total - n_flagged_total:,} ({100 * (1 - n_flagged_total / n_total):.2f}%)")

### Quality control across all subjects

#### Methodology

The quality control analysis above was performed on sub-01. We now repeat it for all 10 
subjects using the same thresholds ($\tau = 2.5$ for z-score, 50 for max absolute amplitude) 
to check whether the thresholds generalize and whether data quality is consistent across 
subjects.

The z-score threshold is self-normalizing by construction since it is computed relative to 
each channel's own mean and standard deviation, the same threshold should produce comparable 
flagging rates across subjects regardless of their absolute signal levels. The amplitude 
threshold of 50 is less principled in this regard, since subjects may differ in their overall 
response amplitude due to inter-subject variability in source reconstruction, cortical 
geometry, or signal-to-noise ratio. One alternative would be to use a subject-specific 
percentile threshold (e.g. flag the top 1% of amplitude values per subject), which adapts 
to each subject's own distribution while flagging a consistent fraction. However, this assumes 
the same fraction of data is artifactual in every subject, which is not necessarily true 
either. We therefore first apply the fixed threshold of 50 to all subjects and inspect the 
resulting flagged percentages. If the percentages are consistent across subjects, the fixed 
threshold is adequate. If a subject shows a dramatically different flagging rate, we will 
consider switching to a percentile-based threshold for that subject.

#### Results

The z-score flagging rates are consistent across subjects (1.98–3.08%), confirming that 
the z-score threshold generalizes well by construction. However, the amplitude flagging 
rates under the fixed threshold of 50 vary dramatically (from 0.00% for sub-06 to 1.55% 
for sub-04) indicating that subjects differ considerably in their overall signal amplitude, 
likely due to inter-subject differences in skull thickness, cortical geometry, or source 
reconstruction. A fixed amplitude threshold therefore flags very different fractions of data 
across subjects, which is undesirable.

**Fixed amplitude threshold (50) across subjects**

| Subject | Z-score flagged % | Amplitude flagged % | Total flagged % | Clean pairs % |
|---|---|---|---|---|
| sub-01 | 2.11 | 0.89 | 2.64 | 97.36 |
| sub-02 | 2.11 | 0.43 | 2.47 | 97.53 |
| sub-03 | 2.71 | 0.52 | 2.89 | 97.11 |
| sub-04 | 3.08 | 1.55 | 3.80 | 96.20 |
| sub-05 | 1.99 | 0.02 | 2.00 | 98.00 |
| sub-06 | 2.50 | 0.00 | 2.50 | 97.50 |
| sub-07 | 2.00 | 0.03 | 2.00 | 98.00 |
| sub-08 | 2.61 | 0.20 | 2.63 | 97.37 |
| sub-09 | 1.98 | 0.09 | 2.01 | 97.99 |
| sub-10 | 2.15 | 0.10 | 2.22 | 97.78 |

We therefore switched to a subject-specific percentile threshold, calibrated to match the 
0.89% amplitude flagging rate of sub-01 under the fixed threshold. This equalizes the 
amplitude filter's aggressiveness across subjects while still flagging a consistent and 
reasonable fraction. The resulting thresholds vary from 17.67 (sub-06) to 58.35 (sub-04), 
confirming the large inter-subject amplitude differences. Total flagging rates under this 
approach are consistent across subjects (2.47–3.51%).

**Subject-specific percentile threshold across subjects**

| Subject | Amp threshold | Z-score flagged % | Amplitude flagged % | Total flagged % | Clean pairs % |
|---|---|---|---|---|---|
| sub-01 | 50.00 | 2.11 | 0.89 | 2.64 | 97.36 |
| sub-02 | 35.73 | 2.11 | 0.89 | 2.87 | 97.13 |
| sub-03 | 44.44 | 2.71 | 0.89 | 3.01 | 96.99 |
| sub-04 | 58.35 | 3.08 | 0.89 | 3.51 | 96.49 |
| sub-05 | 21.83 | 1.99 | 0.89 | 2.61 | 97.39 |
| sub-06 | 17.67 | 2.50 | 0.89 | 3.02 | 96.98 |
| sub-07 | 33.30 | 2.00 | 0.89 | 2.61 | 97.39 |
| sub-08 | 39.70 | 2.61 | 0.89 | 2.88 | 97.12 |
| sub-09 | 28.93 | 1.98 | 0.89 | 2.47 | 97.53 |
| sub-10 | 32.18 | 2.15 | 0.89 | 2.87 | 97.13 |

In [ ]:
subjects = [f"sub-{i:02d}" for i in range(1, 11)]
results = []

for subj in subjects:
    with h5py.File(EEG_PATH, "r") as f:
        data = f[f"train/neural_data/{subj}/{ROI}"][:]
    
    n_stim, n_ch, n_tp = data.shape
    
    # Variance-based z-score
    ep_var = data.var(axis=2)
    s_mean = ep_var.mean(axis=0)
    s_std  = ep_var.std(axis=0)
    ep_var_norm = (ep_var - s_mean) / (s_std + 1e-8)
    
    # Amplitude-based
    max_abs = np.abs(data).max(axis=2)
    
    # Combined mask
    mask = (ep_var_norm > THRESHOLD) | (max_abs > THRESHOLD_ABS)
    
    pct_z   = (ep_var_norm > THRESHOLD).mean() * 100
    pct_amp = (max_abs > THRESHOLD_ABS).mean() * 100
    pct_tot = mask.mean() * 100
    
    results.append({
        "Subject":           subj,
        "Z-score flagged %": round(pct_z, 2),
        "Amplitude flagged %": round(pct_amp, 2),
        "Total flagged %":   round(pct_tot, 2),
        "Clean pairs %":     round(100 - pct_tot, 2),
    })
    
    print(f"{subj}: z={pct_z:.2f}%  amp={pct_amp:.2f}%  total={pct_tot:.2f}%")

results_df = pd.DataFrame(results).set_index("Subject")
results_df

In [ ]:
percentile_50 = (max_abs_all < 50).mean() * 100
print(f"Threshold of 50 corresponds to the {percentile_50:.2f}th percentile for sub-01")

In [ ]:
subjects = [f"sub-{i:02d}" for i in range(1, 11)]
results = []

for subj in subjects:
    with h5py.File(EEG_PATH, "r") as f:
        data = f[f"train/neural_data/{subj}/{ROI}"][:]
    
    n_stim, n_ch, n_tp = data.shape
    
    # Variance-based z-score
    ep_var = data.var(axis=2)
    s_mean = ep_var.mean(axis=0)
    s_std  = ep_var.std(axis=0)
    ep_var_norm = (ep_var - s_mean) / (s_std + 1e-8)
    
    # Amplitude-based — subject-specific percentile threshold
    max_abs = np.abs(data).max(axis=2)
    THRESHOLD_ABS = np.percentile(max_abs, percentile_50)
    
    # Combined mask
    mask = (ep_var_norm > THRESHOLD) | (max_abs > THRESHOLD_ABS)
    
    pct_z   = (ep_var_norm > THRESHOLD).mean() * 100
    pct_amp = (max_abs > THRESHOLD_ABS).mean() * 100
    pct_tot = mask.mean() * 100
    
    results.append({
        "Subject":             subj,
        "Amp threshold":       round(THRESHOLD_ABS, 2),
        "Z-score flagged %":   round(pct_z, 2),
        "Amplitude flagged %": round(pct_amp, 2),
        "Total flagged %":     round(pct_tot, 2),
        "Clean pairs %":       round(100 - pct_tot, 2),
    })
    
    print(f"{subj}: amp_threshold={THRESHOLD_ABS:.1f}  z={pct_z:.2f}%  amp={pct_amp:.2f}%  total={pct_tot:.2f}%")

results_df = pd.DataFrame(results).set_index("Subject")
results_df

### Quality control on test repetitions

The noise ceilings provided in `noise_ceilings/` are computed from the 80 test repetitions.
If individual test repetitions are anomalous (e.g., due to movement artifacts, electrode drift, 
or equipment issues) they will directly corrupt the noise ceiling estimate for the affected 
channel×timepoint combinations. It is therefore important to also assess data quality in the 
test repetitions.

Unlike the training data where repetitions were already averaged before storage, the test 
repetitions file (`things_eeg2-test_reps.h5`) preserves all 80 repetitions individually, 
with shape `(n_stimuli, n_channels, n_repetitions, n_timepoints)`. This actually gives us 
more control: a single bad repetition out of 80 can be cleanly excluded without affecting 
the remaining 79, whereas in the training data a bad repetition is baked into the average 
and cannot be recovered.

We apply the same two quality filters as before: z-score on epoch variance and max absolute 
amplitude, now at the level of individual repetitions. However, the thresholds derived from 
the training data are not directly transferable. The z-score threshold of 2.5 was calibrated 
on averaged responses, where averaging over 4 repetitions reduces variance by roughly 
$1/\sqrt{4}$. Individual repetitions have higher intrinsic variance by construction, so the 
same threshold is more conservative and will flag fewer pairs than it would on averaged data. 
Similarly, the amplitude threshold may behave differently since averaging smooths out 
transient peaks. The 0.91% flagging rate on individual test repetitions is therefore a lower 
bound on what a properly calibrated threshold would yield, and should not be directly compared 
to the 2.6% seen in training.

The provided noise ceilings were computed before this quality control step and may therefore 
include a small fraction of anomalous repetitions. We investigate the effect of applying this 
cleaning to the noise ceiling estimates in Section 1.3, where we find the impact to be minimal 
and confined to channels 13 and 53. We therefore use the uncleaned estimates for the remainder 
of the analysis to stay consistent with the provided ceilings.

In [ ]:
SUBJECT = "sub-01"
ROI     = "whole_brain"

with h5py.File(EEG_REPS_PATH, "r") as f:
    test_data = f[f"test/neural_data/{SUBJECT}/{ROI}"][:]
    # shape: (200, 63, 80, 80) -> (n_stimuli, n_channels, n_reps, n_timepoints)

n_stim, n_ch, n_reps, n_tp = test_data.shape

# Variance over timepoints for each (stimulus, channel, rep)
ep_var_test = test_data.var(axis=3)  # (200, 63, 80)

# Z-score per channel across all stimulus-rep combinations
ep_var_flat = ep_var_test.reshape(n_stim * n_reps, n_ch)  # (16000, 63)
s_mean_test = ep_var_flat.mean(axis=0)  # (63,)
s_std_test  = ep_var_flat.std(axis=0)   # (63,)
ep_var_norm_test = (ep_var_test - s_mean_test[None, :, None]) / (s_std_test[None, :, None] + 1e-8)

# Max absolute amplitude per (stimulus, channel, rep)
max_abs_test = np.abs(test_data).max(axis=3)  # (200, 63, 80)
thresh_abs_test = np.percentile(max_abs_test, percentile_50)

# Quality mask: True = flagged
quality_mask_test = (ep_var_norm_test > THRESHOLD) | (max_abs_test > thresh_abs_test)

n_flagged = quality_mask_test.sum()
n_total   = quality_mask_test.size
print(f"Flagged: {n_flagged} / {n_total} ({100 * n_flagged / n_total:.2f}%)")

In [ ]:
# ---- Heatmap of mean EEG response over channels x time (cleaned) ----

# Identify channels that are anomalous overall
# A channel is considered anomalous if more than 10% of its stimuli are flagged
flagged_per_channel = quality_mask.mean(axis=0)  # (n_channels,)
CHANNEL_THRESHOLD = 0.10
clean_channels = flagged_per_channel < CHANNEL_THRESHOLD
print(f"Clean channels: {clean_channels.sum()} / {n_channels} "
      f"({100 * clean_channels.mean():.1f}%)")

# Average over clean stimuli per channel, respecting the quality mask
clean_mean = np.zeros((n_channels, n_timepoints))
for ch in range(n_channels):
    clean_stimuli = ~quality_mask[:, ch]
    clean_mean[ch] = all_data[clean_stimuli, ch, :].mean(axis=0)

# Only keep clean channels for the heatmap
clean_mean_filtered = clean_mean[clean_channels]
time_axis = np.linspace(0, 0.8, n_timepoints)

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(clean_mean_filtered, aspect="auto", cmap="RdBu_r",
               interpolation="nearest",
               extent=[0, 0.8, clean_mean_filtered.shape[0], 0])
ax.axvline(x=0, color="black", linewidth=1, linestyle="--", label="Stimulus onset")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Channel index (clean channels only)")
ax.set_title(f"Mean EEG response — {SUBJECT} / {ROI} (clean channels only)")
plt.colorbar(im, ax=ax, label="Mean amplitude")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
import importlib
import utils
importlib.reload(utils)
from utils import *

subjects = ["sub-01", "sub-02", "sub-03", "sub-04"]

for roi in ["whole_brain","occipital", "occipital_parietal", "temporal", "frontal"]:
    for subj in subjects:
        plot_mean_eeg_heatmap(EEG_PATH, subj, roi)

In [ ]:
import importlib
import utils
importlib.reload(utils)
from utils import *

subjects = ["sub-01", "sub-02", "sub-03", "sub-04", "sub-05", "sub-06", "sub-07", "sub-08", "sub-09", "sub-10"]

for roi in ["occipital"]:
    for subj in subjects:
        plot_mean_eeg_heatmap(EEG_PATH, subj, roi)

In [ ]:
subjects = ["sub-01", "sub-02", "sub-03"]
for roi in ["whole_brain", "occipital"]:
    for subj in subjects:
        save_path = f"images/eeg_heatmap_{roi}_{subj.replace('-', '')}.png"
        plot_mean_eeg_heatmap(EEG_PATH, subj, roi, save_path=save_path)

### EEG signal visualization

To get a first impression of the EEG data quality and structure, we plotted the mean response 
across all clean training stimuli (after applying the quality control mask described above)
for each channel over the 0.8s post-stimulus window. Each epoch is time-locked to a stimulus 
onset, though due to the RSVP paradigm with a 200ms SOA, the later timepoints of each epoch 
are contaminated by responses to subsequent images. The repeating pattern of evoked responses 
every ~200ms visible in the heatmaps directly reflects this: each peak corresponds to the 
onset of a new image in the rapid sequence.

We show results for the `whole_brain` ROI (all 63 channels) and the `occipital` ROI (3 
channels) for subjects 01, 02, and 03. The `occipital` region is particularly relevant for 
visual processing as it overlies primary visual cortex (V1) and surrounding early visual 
areas. Ideally, these heatmaps would be computed after averaging across all subjects to reduce 
inter-subject noise. We show per-subject results here for transparency, as averaging across 
subjects would obscure the inter-subject variability that is itself informative about data 
quality.

<div style="display: flex; gap: 10px;">
  <figure style="text-align: center;">
    <img src="images/eeg_heatmap_whole_brain_sub01.png" style="width: 100%;"/>
    <figcaption>sub-01 / whole_brain</figcaption>
  </figure>
  <figure style="text-align: center;">
    <img src="images/eeg_heatmap_whole_brain_sub02.png" style="width: 100%;"/>
    <figcaption>sub-02 / whole_brain</figcaption>
  </figure>
  <figure style="text-align: center;">
    <img src="images/eeg_heatmap_whole_brain_sub03.png" style="width: 100%;"/>
    <figcaption>sub-03 / whole_brain</figcaption>
  </figure>
</div>

<div style="display: flex; gap: 10px; margin-top: 15px;">
  <figure style="text-align: center;">
    <img src="images/eeg_heatmap_occipital_sub01.png" style="width: 100%;"/>
    <figcaption>sub-01 / occipital</figcaption>
  </figure>
  <figure style="text-align: center;">
    <img src="images/eeg_heatmap_occipital_sub02.png" style="width: 100%;"/>
    <figcaption>sub-02 / occipital</figcaption>
  </figure>
  <figure style="text-align: center;">
    <img src="images/eeg_heatmap_occipital_sub03.png" style="width: 100%;"/>
    <figcaption>sub-03 / occipital</figcaption>
  </figure>
</div>

**What we observe:**

Looking at the `whole_brain` heatmaps, the absolute amplitudes and overall color patterns 
differ across subjects: sub-01 shows a mix of positive and negative responses while sub-02 
is dominated by negative values. This inter-subject variability in absolute amplitude is 
expected given individual differences in skull thickness, electrode impedance, and cortical 
geometry. Despite these differences, two features are consistent across subjects: channel 30 
shows a persistent positive mean across the entire 0.8s window in all three subjects, 
suggesting this reflects a systematic property of that electrode's position or the online 
referencing to Fz rather than a subject-specific artifact. Additionally, the repeating 
pattern of elevated activity every ~200ms is visible in all subjects, directly reflecting 
the RSVP stimulus onsets.

In the `occipital` ROI, the temporal structure becomes much clearer. Sub-02 shows highly 
consistent responses across all three occipital channels, with clear peaks at approximately 
0.1s, 0.3s, 0.5s, and 0.7s after stimulus onset. Sub-01 shows a similar pattern but with 
more variability across channels: channel 2 consistently shows lower amplitude than channels 
1 and 3. Sub-03 shows an interesting temporal shift: the response peaks appear approximately 
50ms earlier (at ~0.05s, 0.25s, 0.45s, and 0.65s) compared to the other subjects. This 
pattern was also observed in subjects 05, 06, and 10, suggesting it may reflect a subgroup 
difference in neural response latency or a difference in how the data was processed for those 
subjects. The cause of this shift is currently unclear.

Despite the inter-subject variability, the consistent temporal structure across subjects, 
particularly the clear evoked responses in occipital channels, confirms that the data 
contains meaningful stimulus-driven signal rather than pure noise.

In [ ]:
# Load and average noise ceilings across all subjects for whole_brain
roi      = "whole_brain"
subjects = [f"sub-{i:02d}" for i in range(1, 11)]

nc_all = []
with h5py.File(EEG_PATH, "r") as f:
    for subj in subjects:
        nc = f[f"noise_ceilings/{subj}/{roi}"][:]  # (n_channels, 80)
        nc_all.append(nc)

nc_mean = np.mean(nc_all, axis=0) / 100  # (n_channels, 80), convert to [0,1]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Heatmap over channels x time
im = axes[0].imshow(nc_mean, aspect="auto", cmap="viridis",
                    interpolation="nearest",
                    extent=[0, 0.8, nc_mean.shape[0], 0],
                    vmin=0, vmax=1)
axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("Channel index")
axes[0].set_title(f"Mean noise ceiling — {roi} (averaged across subjects)")
plt.colorbar(im, ax=axes[0], label="Noise ceiling [0, 1]")

# Mean over channels per timepoint
time_axis = np.linspace(0, 0.8, 80)
axes[1].plot(time_axis, nc_mean.mean(axis=0), color="steelblue", linewidth=2)
axes[1].fill_between(time_axis, 
                      nc_mean.mean(axis=0) - nc_mean.std(axis=0),
                      nc_mean.mean(axis=0) + nc_mean.std(axis=0),
                      alpha=0.3, color="steelblue", label="±1 std across channels")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("Noise ceiling [0, 1]")
axes[1].set_title(f"Mean noise ceiling over time — {roi}")
axes[1].legend()

plt.tight_layout()
plt.savefig("images/noise_ceiling_heatmap_whole_brain.png", dpi=150, bbox_inches="tight")
plt.show()

### Provided EEG noise ceilings

The noise ceilings provided in `noise_ceilings/` were computed on the 80 test repetitions and reflect how consistently each channel responds to the same 
image across repetitions. A high noise ceiling at a given channel×timepoint means the 
response there is strongly driven by the stimulus: the brain produces a reliable, 
reproducible signal for that image. A low noise ceiling means the response is not well 
explained by the stimulus, either because the channel is noisy or because it is engaged 
in processes unrelated to the visual task (attention, internal state, default mode activity) 
that vary across repetitions.

We averaged the noise ceilings across all 10 subjects and visualized them as a heatmap over 
channels × time, alongside the mean noise ceiling over time averaged across channels.

<div style="display: flex; gap: 10px;">
  <figure style="text-align: center;">
    <img src="images/noise_ceiling_heatmap_whole_brain.png" style="width: 100%;"/>
    <figcaption>Mean noise ceiling across subjects (whole_brain)</figcaption>
  </figure>
</div>

Several patterns are immediately visible. First, the channels with the highest noise ceilings 
(around indices 15-20 and 45-50) are exactly the same channels that showed the strongest 
evoked responses in the mean EEG heatmap. This is a clean validation of the noise ceiling 
concept, the channels most reliably driven by the visual stimulus are the ones with the 
highest theoretical upper bound on model prediction accuracy. Channels with low noise ceilings 
are not simply noisy; they are engaged in processes that our vision models cannot and should 
not be expected to predict.

Second, the noise ceiling is highest in the first 400ms after stimulus onset and drops 
steeply afterwards. This aligns with the neural habituation effect described in the original 
paper <a href="#gifford2022" title="Gifford et al. (2022) — A large and rich EEG dataset for modeling human visual object recognition">[2]</a>: responses to successive images in a rapid sequence diminish over 
time, making later timepoints less reliable. The noise ceiling plateau between roughly 0.1s 
and 0.4s suggests that the brain's response to the first image in the sequence remains 
reliable across multiple stages of visual processing, capturing both early sensory responses 
and later higher-level processing. After 0.4s (corresponding approximately to when the 
third image in the sequence appears) habituation becomes strong enough that responses 
lose their stimulus-specificity and the noise ceiling drops sharply.

Third, the large standard deviation across channels visible in the line plot confirms that 
channels differ substantially in how stimulus-driven they are. This further motivates our 
earlier decision to filter out low-reliability channel×timepoint pairs when computing 
predictive metrics in Section 2 (noise ceiling < 0.1 as specified in the project guidelines).

<div style="background:#fff4c2; border:1px solid #c89b1f; border-left:6px solid #9a6f00; padding:10px 12px; border-radius:6px; margin-top:8px; margin-bottom:4px; color:#241a00; line-height:1.45;"><strong style="color:#5c4300;">Answer box 1.2</strong><br>Which time windows and channel groups appear most informative? Are the responses dominated by noise, or do you observe clear evoked structure?<p>
    
The most informative time window is roughly 0.1s to 0.4s after stimulus onset, where both 
the evoked responses and noise ceilings are highest. Within this window, the first 200ms 
captures the primary visual response to each image before contamination from the next image 
in the RSVP sequence becomes dominant. Channel groups around indices 15-20 and 45-50 are consistently the most informative across subjects, showing the strongest evoked responses and highest noise ceilings. These are likely the visually responsive electrodes overlying occipital and parietal cortex. This is further supported by the provided noise ceilings, which peak at the same channel indices and in the same 0.1–0.4s window, confirming these are the most stimulus-driven regions of the recording. Frontal and central channels show much weaker stimulus-driven responses and lower noise ceilings, suggesting they are engaged in processes unrelated to the visual task.

The responses are not dominated by noise, there is clear evoked structure that is 
consistent across subjects and repeats every ~200ms in line with the RSVP stimulus onsets. 
However, the RSVP paradigm introduces substantial forward contamination: from 200ms onwards, 
each epoch contains overlapping responses to subsequent images, making later timepoints 
harder to interpret as stimulus-specific. The occipital region shows particularly clean 
evoked structure with responses peaking around 100ms post-stimulus, consistent with known 
latencies of early visual cortex responses.
</div>

---

## 1.3 Estimate EEG noise ceilings using two methods

In practice, there are multiple ways to estimate noise ceilings, depending on the available data and the specific research question. When you have repeated measurements of the same stimulus, you can estimate reliability from the consistency of those repetitions. When repeated measurements are not available, reliability can instead be estimated across subjects, which often yields a more conservative ceiling.

In this part, you will implement two different estimators using the `things_eeg2-test_reps.h5` file, which contains the unaveraged test responses with repetitions.

You must implement **two different estimators** using `things_eeg2-test_reps.h5`. You can refer to the cited paper in each method's docstring for details.


### Required estimators

1. **Variance-based estimator**  
2. **Split-half reliability estimator**

<div style="background:#eef5fb; border-left:4px solid #4c78a8; padding:8px 12px; border-radius:6px; font-weight:700; color:#26445e;">What you must do</div>

- Implement both estimators.
- Compute both estimators for EEG2.
- Compare them to the provided EEG noise ceilings stored in `things_eeg2.h5`.

<div style="background:#f3f6fa; border-left:4px solid #7a93ac; padding:8px 12px; border-radius:6px; font-weight:700; color:#32475b;">Required deliverables</div>

You must include all of the following:

1. **Working implementation of the variance-based estimator**
2. **Working implementation of the split-half estimator**
3. **One plot of mean noise ceiling over time**
4. **One plot of mean noise ceiling over channels**
5. **At least one channel × time heatmap for each estimator**
6. **At least one histogram comparing the value distributions**
7. **One direct visual comparison to the stored EEG noise ceilings**

In [ ]:
import importlib
import utils
importlib.reload(utils)
from utils import *

In [ ]:
SUBJECT = "sub-01"
ROI     = "whole_brain"

with h5py.File(EEG_REPS_PATH, "r") as f:
    reps_raw = f["test"]["neural_data"][SUBJECT][ROI][:]
    print("Raw reps shape:", reps_raw.shape)  # (200, ch, 80, 80)

# Axes: (stim, ch, tp, reps) → transpose to (ch, tp, stim, reps)
reps = reps_raw.transpose(1, 2, 0, 3).astype(np.float32)
print("Transposed shape (ch, tp, stim, reps):", reps.shape)

n_ch, n_tp, n_stim, n_reps = reps.shape
times = np.linspace(0.0, 0.8, n_tp)

In [ ]:
print("Computing variance-based ceiling …")
nc_vb = compute_ceiling_variancebased(reps)          # (ch, tp) in percent
print("  shape:", nc_vb.shape, " range: %.1f – %.1f" % (np.nanmin(nc_vb), np.nanmax(nc_vb)))
 
print("Computing split-half ceiling …")
nc_sh = compute_ceiling_splithalf(reps, folds=10, seed=0)  # (ch, tp) in [0,1]
nc_sh_pct = nc_sh * 100.0                                   # convert to percent
print("  shape:", nc_sh_pct.shape, " range: %.1f – %.1f" % (np.nanmin(nc_sh_pct), np.nanmax(nc_sh_pct)))
 
# Load provided ceiling from things_eeg2.h5 for comparison
with h5py.File(EEG_PATH, "r") as f:
    nc_stored = f["noise_ceilings"][SUBJECT][ROI][:]
    print("Stored ceiling shape:", nc_stored.shape,
          " range: %.1f – %.1f" % (nc_stored.min(), nc_stored.max()))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
 
for ax, nc, label, color in zip(
    axes,
    [nc_vb,      nc_sh_pct,  nc_stored],
    ["Variance-based", "Split-half", "Provided"],
    ["steelblue", "darkorange", "forestgreen"],
):
    mean_over_ch = np.nanmean(nc, axis=0)   # (tp,)
    sem          = np.nanstd(nc, axis=0) / np.sqrt(nc.shape[0])
    ax.fill_between(times, mean_over_ch - sem, mean_over_ch + sem, alpha=0.25, color=color)
    ax.plot(times, mean_over_ch, color=color, lw=2, label=label)
    ax.set_title(f"{label}\n(mean ± SEM over channels)", fontsize=11)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Noise ceiling (%)")
    ax.axhline(0, color="k", lw=0.5, ls="--")
    ax.legend(fontsize=9)
 
fig.suptitle(f"Noise ceiling over time — {SUBJECT} / {ROI}", fontsize=13, y=1.01)
fig.tight_layout()
plt.savefig("images/fig_nc_over_time.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

for nc, label, color in zip(
    [nc_vb, nc_sh_pct, nc_stored],
    ["Variance-based", "Split-half", "Provided"],
    ["steelblue", "darkorange", "forestgreen"],
):
    mean_over_ch = np.nanmean(nc, axis=0)  # (tp,)
    ax.plot(times, mean_over_ch, color=color, lw=2, label=label)

ax.set_xlabel("Time (s)")
ax.set_ylabel("Noise ceiling (%)")
ax.set_title(f"Noise ceiling over time — {SUBJECT} / {ROI}\n(mean over channels)")
ax.axhline(0, color="k", lw=0.5, ls="--")
ax.legend()
fig.tight_layout()
plt.savefig("images/fig_nc_over_time_same_plot.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for nc, label, color in zip(
    [nc_vb, nc_sh_pct, nc_stored],
    ["Variance-based", "Split-half", "Provided"],
    ["steelblue", "darkorange", "forestgreen"],
):
    mean_over_tp = np.nanmean(nc, axis=1)   # (ch,)
    ax.plot(range(n_ch), mean_over_tp, color=color, lw=1.8, label=label, alpha=0.85)
 
ax.set_xlabel("Channel index")
ax.set_ylabel("Noise ceiling (%)")
ax.set_title(f"Noise ceiling over channels (mean over time) — {SUBJECT} / {ROI}")
ax.legend()
fig.tight_layout()
plt.savefig("images/fig_nc_over_channels.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
labels = ["Variance-based (%)", "Split-half (%)", "Provided (%)"]
ncs    = [nc_vb, nc_sh_pct, nc_stored]
 
for ax, nc, label in zip(axes, ncs, labels):
    vmax = np.nanpercentile(nc, 95)
    im = ax.imshow(nc, aspect="auto", origin="lower",
                   extent=[times[0], times[-1], 0, n_ch],
                   vmin=0, vmax=vmax, cmap="viridis")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Channel index")
    ax.set_title(label)
    fig.colorbar(im, ax=ax, shrink=0.85)
 
fig.suptitle(f"Channel × time noise ceiling heatmaps — {SUBJECT} / {ROI}", fontsize=13)
fig.tight_layout()
plt.savefig("images/fig_nc_heatmaps.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, nc_est, label, color in zip(
    axes,
    [nc_vb, nc_sh_pct],
    ["Variance-based − Provided", "Split-half − Provided"],
    ["RdBu_r", "RdBu_r"],
):
    diff = nc_est - nc_stored
    vmax = np.nanpercentile(np.abs(diff), 95)
    im = ax.imshow(diff, aspect="auto", origin="lower",
                   extent=[times[0], times[-1], 0, n_ch],
                   vmin=-vmax, vmax=vmax, cmap="RdBu_r")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Channel index")
    ax.set_title(f"{label}\n(mean diff = {np.nanmean(diff):.2f}%)")
    fig.colorbar(im, ax=ax, shrink=0.85)

fig.suptitle(f"Estimator difference from provided ceiling — {SUBJECT} / {ROI}", fontsize=13)
fig.tight_layout()
plt.savefig("images/fig_nc_diff_heatmaps.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
bins = np.linspace(0, 100, 60)
for nc, label, color in zip(
    [nc_vb.ravel(), nc_sh_pct.ravel(), nc_stored.ravel()],
    ["Variance-based", "Split-half", "Provided"],
    ["steelblue", "darkorange", "forestgreen"],
):
    ax.hist(nc[np.isfinite(nc)], bins=bins, histtype="step", lw=2, color=color,
        label=f"{label}  (mean={np.nanmean(nc):.1f}%)")
 
ax.set_xlabel("Noise ceiling (%)")
ax.set_ylabel("Density")
ax.set_title(f"Distribution of noise ceiling values — {SUBJECT} / {ROI}")
ax.legend()
fig.tight_layout()
plt.savefig("images/fig_nc_histogram.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
bins = np.linspace(0, 100, 60)

counts_sh,  _ = np.histogram(nc_sh_pct.ravel()[np.isfinite(nc_sh_pct.ravel())], bins=bins, density=True)
counts_ref, _ = np.histogram(nc_stored.ravel()[np.isfinite(nc_stored.ravel())], bins=bins, density=True)

bin_centers = 0.5 * (bins[:-1] + bins[1:])
diff_counts = counts_sh - counts_ref

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(bin_centers, diff_counts, width=bins[1]-bins[0], color="purple", alpha=0.7)
ax.axhline(0, color="k", lw=1)
ax.set_xlabel("Noise ceiling (%)")
ax.set_ylabel("Density difference (Split-half − Provided)")
ax.set_title(f"Where split-half differs from provided — {SUBJECT} / {ROI}")
fig.tight_layout()
plt.savefig("images/fig_nc_diff_counts.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
 
for ax, nc_est, label, color in zip(
    axes,
    [nc_vb, nc_sh_pct],
    ["Variance-based vs Provided", "Split-half vs Provided"],
    ["steelblue", "darkorange"],
):
    flat_stored = nc_stored.ravel()
    flat_est    = nc_est.ravel()
    mask        = np.isfinite(flat_stored) & np.isfinite(flat_est)
    r, p        = stats.pearsonr(flat_stored[mask], flat_est[mask])
    ax.scatter(flat_stored[mask], flat_est[mask], s=2, alpha=0.3, color=color)
    lim = [0, 100]
    ax.plot(lim, lim, "k--", lw=1)
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel("Provided ceiling (%)")
    ax.set_ylabel(f"{label.split(' vs')[0]} (%)")
    ax.set_title(f"{label}\nr = {r:.3f}, p = {p:.2e}")
 
fig.suptitle(f"Estimator vs provided ceiling — {SUBJECT} / {ROI}", fontsize=13)
fig.tight_layout()
plt.savefig("images/fig_nc_scatter_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
import importlib
import utils
importlib.reload(utils)
from utils import compute_ceiling_variancebased_clean, compute_ceiling_splithalf_clean

# Apply QC mask to reps — set flagged (stim, ch, rep) triples to NaN
# reps shape: (ch, tp, stim, reps) — need to broadcast mask accordingly
# quality_mask_test shape: (stim, ch, reps) → transpose to (ch, stim, reps)
mask_sch = quality_mask_test.transpose(1, 0, 2)          # (ch, stim, reps)
mask_bcast = mask_sch[:, np.newaxis, :, :]               # (ch, 1, stim, reps)
mask_bcast = np.broadcast_to(mask_bcast, reps.shape)     # (ch, tp, stim, reps)

reps_clean = reps.copy()
reps_clean[mask_bcast] = np.nan
print(f"Flagged {quality_mask_test.mean()*100:.2f}% of (stim, ch, rep) triples set to NaN")
print(f"Channel 13: {quality_mask_test[:, 13, :].mean()*100:.2f}% flagged")
print(f"Channel 53: {quality_mask_test[:, 53, :].mean()*100:.2f}% flagged")

# Recompute estimators on cleaned data using nan-safe versions
print("\nComputing variance-based ceiling on cleaned data...")
nc_vb_clean = compute_ceiling_variancebased_clean(reps_clean)
print("Computing split-half ceiling on cleaned data...")
nc_sh_clean = compute_ceiling_splithalf_clean(reps_clean, folds=10, seed=0) * 100

print(f"\nVB  raw: {np.nanmean(nc_vb):.2f}%  |  clean: {np.nanmean(nc_vb_clean):.2f}%")
print(f"SH  raw: {np.nanmean(nc_sh_pct):.2f}%  |  clean: {np.nanmean(nc_sh_clean):.2f}%")
print(f"\nChannel 13 — VB raw: {nc_vb[13].mean():.2f}%  |  clean: {np.nanmean(nc_vb_clean[13]):.2f}%")
print(f"Channel 53 — VB raw: {nc_vb[53].mean():.2f}%  |  clean: {np.nanmean(nc_vb_clean[53]):.2f}%")
print(f"Channel 13 — SH raw: {nc_sh_pct[13].mean():.2f}%  |  clean: {np.nanmean(nc_sh_clean[13]):.2f}%")
print(f"Channel 53 — SH raw: {nc_sh_pct[53].mean():.2f}%  |  clean: {np.nanmean(nc_sh_clean[53]):.2f}%")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 8))

# Plot 1: noise ceiling over channels
ax = axes[0]
for nc, label, color, ls in zip(
    [nc_vb, nc_sh_pct, nc_stored, nc_vb_clean, nc_sh_clean],
    ["VB raw", "SH raw", "Provided", "VB clean", "SH clean"],
    ["steelblue", "darkorange", "forestgreen", "steelblue", "darkorange"],
    ["-", "-", "-", "--", "--"],
):
    mean_over_tp = np.nanmean(nc, axis=1)
    ax.plot(range(n_ch), mean_over_tp, color=color, lw=1.8,
            label=label, alpha=0.85, ls=ls)

ax.set_xlabel("Channel index")
ax.set_ylabel("Noise ceiling (%)")
ax.set_title(f"Noise ceiling over channels — raw vs cleaned — {SUBJECT} / {ROI}")
ax.legend(ncol=2)

# Plot 2: difference between raw and clean for each estimator
ax = axes[1]
for nc_raw, nc_cl, label, color in zip(
    [nc_vb, nc_sh_pct],
    [nc_vb_clean, nc_sh_clean],
    ["VB clean − raw", "SH clean − raw"],
    ["steelblue", "darkorange"],
):
    diff = np.nanmean(nc_cl - nc_raw, axis=1)  # mean over time
    ax.plot(range(n_ch), diff, color=color, lw=1.8, label=label)

ax.axhline(0, color="k", lw=1, ls="--")
ax.set_xlabel("Channel index")
ax.set_ylabel("Difference (%)")
ax.set_title(f"Effect of QC cleaning on noise ceiling — {SUBJECT} / {ROI}")
ax.legend()

fig.tight_layout()
plt.savefig("images/fig_nc_cleaning_effect_channels.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# quality_mask_test shape: (stim, ch, reps)
flagged_per_channel = quality_mask_test.mean(axis=(0, 2)) * 100  # mean over stim and reps
for ch in [13, 33, 53]:
    print(f"Channel {ch}: {flagged_per_channel[ch]:.2f}% of (stim, rep) pairs flagged")

print(f"\nAll channels mean: {flagged_per_channel.mean():.2f}%")
print(f"Max flagged channel: {flagged_per_channel.argmax()} ({flagged_per_channel.max():.2f}%)")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 8))

ncs    = [nc_vb, nc_sh_pct, nc_stored, nc_vb_clean, nc_sh_clean, nc_sh_clean - nc_sh_pct]
labels = ["VB raw", "SH raw", "Provided", "VB clean", "SH clean", "SH clean − SH raw"]
cmaps  = ["viridis", "viridis", "viridis", "viridis", "viridis", "RdBu_r"]

for ax, nc, label, cmap in zip(axes.ravel(), ncs, labels, cmaps):
    vmax = np.nanpercentile(np.abs(nc), 95)
    vmin = -vmax if cmap == "RdBu_r" else 0
    im = ax.imshow(nc, aspect="auto", origin="lower",
                   extent=[times[0], times[-1], 0, n_ch],
                   vmin=vmin, vmax=vmax, cmap=cmap)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Channel index")
    ax.set_title(label)
    fig.colorbar(im, ax=ax, shrink=0.85)

fig.suptitle(f"Raw vs cleaned noise ceilings — {SUBJECT} / {ROI}", fontsize=13)
fig.tight_layout()
plt.savefig("images/fig_nc_cleaning_heatmaps.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import matplotlib.colors as mcolors

# Load test repetitions for inspection
# reps_raw shape: (200, n_ch, 80, 80) — (stim, ch, reps, tp)
with h5py.File(EEG_REPS_PATH, "r") as f:
    test_reps_raw = f["test"]["neural_data"][SUBJECT][ROI][:]

n_stim_test, n_ch_test, n_reps_test, n_tp_test = test_reps_raw.shape
time_axis_test = np.linspace(0, 0.8, n_tp_test)

# Compute variance per (stim, ch, rep) for z-scoring
ep_var_test_raw = test_reps_raw.var(axis=3)  # (200, ch, 80)
ep_var_flat = ep_var_test_raw.reshape(n_stim_test * n_reps_test, n_ch_test)
stim_mean_test = ep_var_flat.mean(axis=0)
stim_std_test  = ep_var_flat.std(axis=0)

n_examples = 5

def plot_test_channel_detail(stim_idx, channel_idx, rep_offset):
    # Z-score variance for all (stim, rep) for this channel
    var_norm = (ep_var_test_raw[:, channel_idx, :] - stim_mean_test[channel_idx]) / \
               (stim_std_test[channel_idx] + 1e-8)  # (200, 80)

    print(f"Z-scores for shown repetitions (stim {stim_idx}, channel {channel_idx}):")
    for i in range(n_examples):
        rep = rep_offset + i
        z = var_norm[stim_idx, rep]
        flagged = quality_mask_test[stim_idx, channel_idx, rep]
        print(f"  Rep {rep}: z = {z:.2f}  {'← FLAGGED' if flagged else ''}")

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Left: heatmap of z-scored variance across all reps x all stimuli for this channel
    im = axes[0].imshow(var_norm.T, aspect="auto", cmap="viridis",
                        interpolation="nearest",
                        norm=mcolors.SymLogNorm(linthresh=1))
    axes[0].axhline(y=rep_offset, color="orange", linewidth=1.5, linestyle="--")
    axes[0].axhline(y=rep_offset + n_examples - 1, color="orange", linewidth=1.5,
                    linestyle="--", label=f"Reps shown on right")
    axes[0].axvline(x=stim_idx, color="red", linewidth=1.5,
                    label=f"Stimulus {stim_idx}")
    axes[0].set_xlabel("Stimulus index")
    axes[0].set_ylabel("Repetition index")
    axes[0].set_title(f"Z-scored variance — channel {channel_idx}")
    axes[0].legend(fontsize=8)
    plt.colorbar(im, ax=axes[0], label="Z-scored variance")

    # Right: time courses of selected repetitions for this stim and channel
    for i in range(n_examples):
        rep = rep_offset + i
        flagged = quality_mask_test[stim_idx, channel_idx, rep]
        label = f"Rep {rep} {'[FLAGGED]' if flagged else ''}"
        axes[1].plot(time_axis_test, test_reps_raw[stim_idx, channel_idx, rep, :],
                     alpha=0.7, label=label,
                     linestyle="--" if flagged else "-")

    axes[1].set_xlabel("Time (s)")
    axes[1].set_ylabel("Amplitude")
    axes[1].set_title(f"Time courses — channel {channel_idx}, stimulus {stim_idx}")
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

stim_slider = widgets.IntSlider(
    value=92, min=0, max=n_stim_test - 1, step=1,
    description="Stimulus index:",
    continuous_update=False,
    style={"description_width": "initial"}
)
channel_slider = widgets.IntSlider(
    value=13, min=0, max=n_ch_test - 1, step=1,
    description="Channel index:",
    continuous_update=False,
    style={"description_width": "initial"}
)
rep_slider = widgets.IntSlider(
    value=9, min=0, max=n_reps_test - n_examples, step=1,
    description="Rep offset:",
    continuous_update=False,
    style={"description_width": "initial"}
)

widgets.interact(plot_test_channel_detail,
                 stim_idx=stim_slider,
                 channel_idx=channel_slider,
                 rep_offset=rep_slider)

<h3>Channel quality investigation in the test repetitions</h3>

Out of curiosity, we investigated whether the QC mask derived from the training data in 
Section 1.2 also captures problematic behavior in the test repetitions. We applied the same 
two filters: z-score on epoch variance and maximum absolute amplitude — to the test 
repetitions file, which contains all 80 repetitions for the 200 test stimuli. The overall 
flagging rate was 0.91%, but this varied dramatically across channels: channel 13 had 50.4% 
of its stimulus-repetition pairs flagged and channel 53 had 7.17% flagged, while most other 
channels had near-zero flagging rates.

We then recomputed both noise ceiling estimators using nan-safe versions that ignore flagged 
entries rather than dropping entire channels. The effect of cleaning on the noise ceiling 
estimates is shown below.

<div style="display: flex; gap: 10px; align-items: flex-start; margin-top: 15px;">
  <figure style="text-align: center; flex: 1;">
    <img src="images/fig_nc_cleaning_effect_channels.png" style="width: 60%;"/>
    <figcaption>Top: mean noise ceiling over channels for raw and cleaned estimators. 
    Bottom: pointwise difference (clean minus raw) averaged over time. The variance-based 
    estimator drops substantially at channels 13 and 53 after cleaning, while the split-half 
    estimator is largely unaffected.</figcaption>
  </figure>
</div>

<div style="display: flex; gap: 10px; align-items: flex-start; margin-top: 15px;">
  <figure style="text-align: center; flex: 1;">
    <img src="images/fig_nc_cleaning_heatmaps.png" style="width: 100%;"/>
    <figcaption>Channel × time heatmaps for raw and cleaned estimators. The effect of 
    cleaning is concentrated at channels 13 and 53 and is essentially invisible elsewhere.
    </figcaption>
  </figure>
</div>

For the vast majority of channels the cleaning has no visible effect on either estimator, 
which is reassuring given the low overall flagging rate. The only notable changes occur at 
channels 13 and 53. The variance-based estimator drops by around 24 percentage points at 
channel 13 and around 12 percentage points at channel 53 after cleaning. The split-half 
estimator is largely unaffected. It had already assigned near-zero reliability to those 
channels before cleaning, suggesting it was already sensitive to their poor quality without 
requiring the explicit QC step. We do not over-interpret this difference between the two 
estimators, as the precise reason depends on how each method aggregates information across 
repetitions and would require further investigation.

To understand what is happening at channel 13 specifically, we used the interactive widget 
to inspect individual test repetition time courses. As noted in Section 1.2, the amplitude threshold for the test repetitions was computed using the same percentile as derived for sub-01 in training (99.11th percentile), which yielded a threshold of 1073 — far above the intended range of around 50. This inflation occurred because channel 13's extreme artifact values dominated the percentile computation, effectively disabling the amplitude filter for the test set. With a corrected threshold computed excluding channels 13 and 53, the threshold would be 
53.03, flagging 99.13% of channel 13's repetitions and 53.17% of channel 53's repetitions.

We show three representative examples from channel 13 below.

<div style="display: flex; gap: 10px; align-items: flex-start; margin-top: 15px;">
  <figure style="text-align: center; flex: 1;">
    <img src="images/ch13_artifact_stim73.png" style="width: 100%;"/>
    <figcaption>Channel 13, stimulus 73 — all five shown repetitions are flagged by the 
    z-score filter and show highly reproducible sharp spikes reaching amplitudes of ±2500, 
    an order of magnitude larger than any other channel. The consistency of the artifact 
    across repetitions is notable, it looks more like a stimulus-locked hardware artifact 
    than random noise.</figcaption>
  </figure>
</div>

<div style="display: flex; gap: 10px; align-items: flex-start; margin-top: 15px;">
  <figure style="text-align: center; flex: 1;">
    <img src="images/ch13_normal_stim92.png" style="width: 100%;"/>
    <figcaption>Channel 13, stimulus 92 — these repetitions were not flagged under the 
    original threshold of 1073 and show much lower amplitudes (roughly ±80). Whether 
    these represent clean signal or a milder form of the same artifact is unclear — a 
    threshold of 50 would flag them, but determining the appropriate threshold requires 
    more knowledge of what constitutes signal versus artifact for this channel and electrode 
    location. The variance-based estimator does not identify these as artifacts since their 
    variance is within the normal range.</figcaption>
  </figure>
</div>

<div style="display: flex; gap: 10px; align-items: flex-start; margin-top: 15px;">
  <figure style="text-align: center; flex: 1;">
    <img src="images/ch13_mixed_stim128.png" style="width: 100%;"/>
    <figcaption>Channel 13, stimulus 128 — a mixed case where some repetitions are flagged 
    (reps 10, 12, 13) and one unflagged repetition (rep 11, green) shows a spike reaching 
    around -500, which would have been caught by an amplitude threshold of 50 but slipped 
    through the inflated threshold of 1073. This illustrates a limitation of the 
    percentile-based threshold approach: when one channel has pathologically extreme values, 
    it inflates the threshold and reduces the sensitivity of the amplitude filter for that 
    same channel.</figcaption>
  </figure>
</div>

Additionally, the z-scored variance heatmap for channel 13 (visible in all three figures 
above as the left panel) shows an interesting pattern: early repetitions (top rows) tend 
to have lower z-scored variance than later repetitions, suggesting the artifact may have 
worsened progressively over the course of the recording session. This is consistent with 
electrode drift or a gradually deteriorating contact, though we cannot confirm this without 
access to the raw recording metadata.

In summary, visual inspection confirms that channel 13 is severely corrupted in the test 
repetitions and should be excluded from any downstream analysis. Channel 53 is more 
ambiguous with 7.17% flagged under the original threshold and 53.17% under the corrected 
threshold, its status depends strongly on the threshold choice. Resolving this would require 
knowledge of the electrode layout and ideally a systematic inspection of its time courses 
across stimuli, which is beyond the scope of this section. We flag both channels as 
potentially unreliable and use the uncleaned noise ceiling estimates for the remainder of 
the analysis to stay consistent with the provided ceilings.

### Noise ceiling over time

We estimate noise ceilings using two methods: the variance-based estimator following 
<a href="#allen2022" title="Allen et al. (2022) — A massive 7T fMRI dataset to bridge cognitive neuroscience and artificial intelligence">[3]</a> and the split-half reliability estimator following 
<a href="#vanbree2025" title="van Bree et al. (2025) — How Much Variance Does Your Model Explain? A Clarifying Note on the Use of Split-Half Reliability for Computing Noise Ceilings">[9]</a>. We compare 
both to the provided ceilings stored in <code>things_eeg2.h5</code>. We use sub-01 and the 
<code>whole_brain</code> ROI throughout. Computing both estimators and comparing them to the 
stored values serves two purposes: it validates that our implementations are correct, and it tells 
us which method was used to generate the provided ceilings, which matters for interpreting how 
conservative or liberal those ceilings are.

The variance-based estimator works by decomposing total response variance into signal and noise 
components. For each channel × time unit, responses are first z-scored across stimuli for each 
repetition separately — this removes slow amplitude drifts across repetitions and ensures total 
variance is approximately 1. Noise variance is then estimated as the variance across repetitions 
averaged over stimuli, and signal variance is whatever remains. The noise ceiling is derived from 
the signal-to-noise ratio, corrected for the finite number of repetitions. The split-half estimator 
takes a different approach: it randomly splits the 80 repetitions into two halves of 40, averages 
each half, and computes the Pearson correlation of the two resulting response vectors across 
stimuli. This is repeated over 10 random splits and the Spearman-Brown correction is applied to 
each fold to project the half-repetition reliability to full-repetition reliability. The average 
across folds gives the final estimate.

We first plot the mean noise ceiling over time averaged across channels, shown separately for each 
estimator and then overlaid in a single plot to allow direct comparison. Plotting them separately 
first makes the temporal structure of each estimator visible including the standard deviation 
across channels, while the combined plot makes the differences between estimators easy to spot.

<div style="display: flex; gap: 10px; align-items: flex-start;">
  <figure style="text-align: center; flex: 1;">
    <img src="images/fig_nc_over_time.png" style="width: 100%;"/>
    <figcaption>Mean noise ceiling over time with ± SEM across channels — separate subplots for each estimator</figcaption>
  </figure>
</div>

<div style="display: flex; gap: 10px; align-items: flex-start; margin-top: 15px;">
  <figure style="text-align: center; flex: 1;">
    <img src="images/fig_nc_over_time_same_plot.png" style="width: 60%;"/>
    <figcaption>Mean noise ceiling over time — all three estimators overlaid</figcaption>
  </figure>
</div>

All three estimators show the same temporal structure. The noise ceiling rises sharply from near 
zero at stimulus onset to around 30% by 0.1s, reflecting the onset of early visual responses. It 
remains elevated and peaks twice (first around 0.1–0.2s and again around 0.25–0.3s where it 
reaches its highest value of almost 35%) likely reflecting early and mid-latency visual 
processing stages. There is a local minimum around 0.35s before a second peak near 0.4s of 
similar height. After 0.4s the ceiling drops steeply and stays low for the remainder of the 0.8s 
window. This drop corresponds to when the third image in the RSVP sequence appears, at which point 
overlapping responses dominate and the signal is no longer stimulus-specific. The standard 
deviation across channels is largest around the peaks, confirming that channels differ 
substantially in how strongly they are driven by the visual stimulus. The low mean values after 
0.45s mean that responses there are not reliably stimulus-driven, and no model should be expected 
to predict them well. In the combined plot, the variance-based trace is invisible because it 
overlaps exactly with the provided ceiling. The split-half trace follows the same shape but sits 
slightly below the other two, particularly at the peaks, indicating it is consistently more 
conservative. This already suggests that the stored ceilings were generated using the 
variance-based method.

We next plot the mean noise ceiling over channels, averaged across all 80 time points, to identify 
which channels are reliably driven by the visual stimulus.

<div style="display: flex; gap: 10px; align-items: flex-start; margin-top: 15px;">
  <figure style="text-align: center; flex: 1;">
    <img src="images/fig_nc_over_channels.png" style="width: 70%;"/>
    <figcaption>Mean noise ceiling over channels (averaged across time) — all three estimators overlaid</figcaption>
  </figure>
</div>

Channels around indices 10–20 and 40–50 show the highest mean noise ceilings, suggesting these 
electrodes overlie visually responsive cortex. Most other channels show low average reliability. 
Again the variance-based and provided ceilings overlap completely. The split-half estimator follows 
the same overall pattern but shows more pronounced negative peaks at channels 13 and 53, and to a 
lesser extent channel 33. These are exactly the channels flagged as problematic during QC in 
Section 1.2, where they showed extreme amplitude jumps and were identified as likely faulty. The 
split-half estimator is more sensitive to these channels because the Spearman-Brown correction 
becomes unstable when the raw half-rep correlation is near zero or negative, which happens for 
genuinely noisy channels. The variance-based estimator is more robust in this regime since it uses 
all 80 repetitions simultaneously rather than pairs of 40-rep halves.

To see both the spatial and temporal structure of reliability at once, we plot channel × time 
heatmaps for all three estimators side by side. This is more informative than either of the 
previous plots alone since it shows whether the low-reliability channels are consistently low 
across all time points or only at specific windows.

<div style="display: flex; gap: 10px; align-items: flex-start; margin-top: 15px;">
  <figure style="text-align: center; flex: 1;">
    <img src="images/fig_nc_heatmaps.png" style="width: 100%;"/>
    <figcaption>Channel × time noise ceiling heatmaps — variance-based, split-half, and provided side by side</figcaption>
  </figure>
</div>

High-reliability regions appear in the first 0.4s for channels around indices 10–20 and 40–50. 
After 0.4s even those channels drop to near zero. Channels 13, 53 and to a lesser extent 33 show 
visibly lower values in the split-half heatmap compared to the other two, while the variance-based 
and provided heatmaps are indistinguishable by eye. These channels are consistently low across the 
entire time window, confirming they are genuinely unreliable rather than just noisy at specific 
time points.

To quantify where the estimators differ from the provided ceiling, we subtract the provided ceiling 
from each estimator pointwise and plot the resulting difference heatmaps. This makes any systematic 
spatial or temporal bias immediately visible.

<div style="display: flex; gap: 10px; align-items: flex-start; margin-top: 15px;">
  <figure style="text-align: center; flex: 1;">
    <img src="images/fig_nc_diff_heatmaps.png" style="width: 70%;"/>
    <figcaption>Pointwise difference from provided ceiling — variance-based (left) and split-half (right). Note that the colorbar range of the left panel is on the order of 1e-7, meaning the variance-based estimator is numerically identical to the provided ceiling up to floating point precision.</figcaption>
  </figure>
</div>

The left panel confirms that the variance-based and provided ceilings are numerically identical, 
the colorbar range of 1e-7 reflects only floating point rounding errors. The right panel shows the 
split-half minus provided difference. No systematic spatial or temporal trend is visible beyond the 
channel-specific effects at channels 13 and 53, which appear as strongly negative patches 
confirming that split-half assigns much lower reliability to those channels than the provided 
ceiling does.

To compare the overall value distributions, we plot step histograms of all three estimators and 
then the bin-wise density difference between split-half and provided. The histogram shows the 
shape of each distribution while the difference plot makes the shift between split-half and 
provided explicit at each ceiling value.

<div style="display: flex; gap: 10px; align-items: flex-start; margin-top: 15px;">
  <figure style="text-align: center; flex: 1;">
    <img src="images/fig_nc_histogram.png" style="width: 70%;"/>
    <figcaption>Distribution of noise ceiling values across all channel × time units — step histograms for all three estimators</figcaption>
  </figure>
</div>

The variance-based and provided distributions are identical with a mean of 14.7%. The split-half 
distribution has a mean of 13.0%, 1.7 percentage points lower. Both distributions are strongly 
right-skewed with the majority of channel × time units having low noise ceilings, reflecting that 
most electrodes and time points are not strongly driven by the visual stimulus.

<div style="display: flex; gap: 10px; align-items: flex-start; margin-top: 15px;">
  <figure style="text-align: center; flex: 1;">
    <img src="images/fig_nc_diff_counts.png" style="width: 70%;"/>
    <figcaption>Bin-wise density difference between split-half and provided ceiling. Positive bars mean split-half has more units at that ceiling value, negative bars mean fewer.</figcaption>
  </figure>
</div>

The density difference plot makes the distributional shift explicit. Split-half has substantially 
more mass in the first bin (0–2%) and slightly more in the second and third bins. From the fourth 
bin onwards the difference flips and the provided ceiling consistently has more mass, meaning the 
variance-based method assigns more units to moderate and high reliability values. This pattern is 
consistent with the split-half estimator being more conservative overall, driven primarily by the 
problematic channels pulling mass towards zero.

Finally we plot the direct scatter comparison between each estimator and the provided ceiling, 
which is the most direct test of how well each estimator reproduces the stored values point by 
point across all channel × time units.

<div style="display: flex; gap: 10px; align-items: flex-start; margin-top: 15px;">
  <figure style="text-align: center; flex: 1;">
    <img src="images/fig_nc_scatter_comparison.png" style="width: 60%;"/>
    <figcaption>Scatter comparison of each estimator against the provided ceiling across all channel × time units</figcaption>
  </figure>
</div>

The variance-based estimator achieves r = 1.000 with the provided ceiling which is a perfect match 
confirming these are numerically identical. The split-half estimator achieves r = 0.957, which is 
high but not perfect. The scatter shows that most points lie close to the diagonal, but there is a 
clear population of outliers with high provided ceiling values of up to 50% that correspond to 
near-zero split-half values. These outliers are channels 13 and 53, and they drive both the lower 
correlation and the excess low-value mass seen in the histogram. For well-behaved channels the two 
estimators agree closely, suggesting the difference is not a fundamental methodological 
disagreement but rather a sensitivity difference for genuinely noisy channels.

In summary, the variance-based estimator reproduces the stored ceilings exactly, confirming it was 
used to generate them. The split-half estimator is slightly more conservative on average and 
substantially more sensitive to problematic channels, which can be an advantage when the goal is 
to identify unreliable units but a disadvantage when the goal is to match a provided reference.

</div>

<div style="background:#fff4c2; border:1px solid #c89b1f; border-left:6px solid #9a6f00; padding:10px 12px; border-radius:6px; margin-top:8px; margin-bottom:4px; color:#241a00; line-height:1.45;"><strong style="color:#5c4300;">Answer box 1.3</strong><br>Compare the two estimators. Do they produce similar patterns across channels and time? Where do they differ most?<p>

Both estimators show the same overall temporal structure: noise ceilings rise sharply after 
stimulus onset, peak twice around 0.1–0.3s reflecting early and mid-latency visual responses, 
and drop steeply after 0.4s when overlapping RSVP responses dominate. Spatially, both 
identify the same channel groups (indices 10–20 and 40–50) as most reliable. The variance-based 
estimator is numerically identical to the provided ceilings (r = 1.000), confirming it was used 
to generate them. The split-half estimator is slightly more conservative overall (mean 13.0% vs 
14.7%) and differs most at channels 13 and 53, where it assigns near-zero reliability while the 
variance-based estimator does not. This difference reflects a sensitivity advantage of split-half 
for genuinely noisy channels, where the Spearman-Brown correction amplifies the signal of poor 
inter-repetition consistency, while the variance-based estimator is more robust in this regime 
by using all repetitions simultaneously.
</p></div>



---

## 1.4 Compare the noise ceiling estimators statistically on EEG2

In `things_eeg2.h5`, we provided noise ceilings computed using one of the two methods you implemented. Can you determine which one it is by comparing the stored ceilings to your computed ones?

Perform a hypothesis test to compare the stored ceilings to each of your computed estimators. For example, you could compute the mean squared error between the stored ceilings and each estimator per subject/time/channel, and then use a paired t-test to see if one estimator is significantly closer to the stored values than the other.

<div style="background:#eef5fb; border-left:4px solid #4c78a8; padding:8px 12px; border-radius:6px; font-weight:700; color:#26445e;">What you must do</div>

- State clearly what each estimator assumes.
- Define a quantitative comparison to the stored EEG noise ceilings.
- Run at least one simple statistical test or formal comparison.

Examples:

- mean absolute deviation from the stored values,
- paired comparison across channel × time units,
- correlation with the stored values.

<div style="background:#f3f6fa; border-left:4px solid #7a93ac; padding:8px 12px; border-radius:6px; font-weight:700; color:#32475b;">Required deliverables</div>

You must include all of the following:

1. **One quantitative comparison table** comparing both estimators to the stored EEG noise ceilings.
2. **One statistical test or one formal quantitative comparison** such as a paired test, correlation analysis, or mean absolute deviation analysis.
3. **One concise written conclusion** stating which estimator better matches the stored values and why.

In [ ]:
# Flatten all arrays to 1D for comparison
flat_stored = nc_stored.ravel()
flat_vb     = nc_vb.ravel()
flat_sh     = nc_sh_pct.ravel()
mask        = np.isfinite(flat_stored) & np.isfinite(flat_vb) & np.isfinite(flat_sh)

# --- Quantitative comparison table ---
def comparison_metrics(nc_ref, nc_est, name):
    r, p   = stats.pearsonr(nc_ref[mask], nc_est[mask])
    mad    = np.abs(nc_ref[mask] - nc_est[mask]).mean()
    mse    = ((nc_ref[mask] - nc_est[mask]) ** 2).mean()
    max_d  = np.abs(nc_ref[mask] - nc_est[mask]).max()
    return dict(Estimator=name,
                Pearson_r=round(r, 6),
                MAD=round(mad, 6),
                MSE=round(mse, 6),
                Max_abs_diff=round(max_d, 6))

rows = [
    comparison_metrics(flat_stored, flat_vb, "Variance-based"),
    comparison_metrics(flat_stored, flat_sh, "Split-half"),
]
df_cmp = pd.DataFrame(rows).set_index("Estimator")
print("Quantitative comparison to stored noise ceilings\n")
print(df_cmp.to_string())

# --- Paired t-test on absolute errors ---
err_vb = np.abs(flat_stored[mask] - flat_vb[mask])
err_sh = np.abs(flat_stored[mask] - flat_sh[mask])
t_stat, p_val = stats.ttest_rel(err_vb, err_sh)

print(f"\nPaired t-test: |VB − stored| vs |SH − stored|")
print(f"  t = {t_stat:.4f},  p = {p_val:.4e}")
print(f"  Mean absolute error — VB: {err_vb.mean():.6f}%,  SH: {err_sh.mean():.3f}%")

better = "Variance-based" if err_vb.mean() < err_sh.mean() else "Split-half"
if p_val < 0.05:
    print(f"  → {better} is significantly closer to the stored values (p < 0.05).")
else:
    print(f"  → No significant difference between the two estimators (p = {p_val:.3f}).")

<div style="background:#fff4c2; border:1px solid #c89b1f; border-left:6px solid #9a6f00; padding:10px 12px; border-radius:6px; margin-top:8px; margin-bottom:4px; color:#241a00; line-height:1.45;"><strong style="color:#5c4300;">Answer box 1.4</strong><br>Which estimator is more likely to have been used to generate the stored EEG noise ceilings? Justify your answer with both visual and quantitative evidence
<p></p>
To formally determine which estimator was used to generate the stored noise ceilings we 
computed four comparison metrics across all channel × time units for sub-01 / whole_brain: 
Pearson correlation, mean absolute deviation, mean squared error, and the maximum absolute 
difference anywhere.

<table style="border-collapse: collapse; width: 80%; margin: 10px auto;">
  <thead>
    <tr style="border-bottom: 2px solid #333;">
      <th style="text-align: left; padding: 6px 12px;">Estimator</th>
      <th style="text-align: center; padding: 6px 12px;">Pearson r</th>
      <th style="text-align: center; padding: 6px 12px;">MAD (%)</th>
      <th style="text-align: center; padding: 6px 12px;">MSE (%²)</th>
      <th style="text-align: center; padding: 6px 12px;">Max |diff| (%)</th>
    </tr>
  </thead>
  <tbody>
    <tr style="border-bottom: 1px solid #ccc;">
      <td style="text-align: left; padding: 6px 12px;">Variance-based</td>
      <td style="text-align: center; padding: 6px 12px;">1.000000</td>
      <td style="text-align: center; padding: 6px 12px;">0.000000</td>
      <td style="text-align: center; padding: 6px 12px;">0.000000</td>
      <td style="text-align: center; padding: 6px 12px;">0.000001</td>
    </tr>
    <tr>
      <td style="text-align: left; padding: 6px 12px;">Split-half</td>
      <td style="text-align: center; padding: 6px 12px;">0.957</td>
      <td style="text-align: center; padding: 6px 12px;">2.298</td>
      <td style="text-align: center; padding: 6px 12px;">29.250</td>
      <td style="text-align: center; padding: 6px 12px;">66.383</td>
    </tr>
  </tbody>
</table>

The variance-based numbers are essentially zero everywhere — the maximum absolute difference 
across all 5040 channel × time units is 0.000001 percentage points, which is floating point 
rounding and nothing else. The split-half estimator correlates well (r = 0.957) but deviates 
on average by 2.3 percentage points per unit, with a worst case of 66.4 percentage points 
at the problematic channels we identified earlier.

We ran a paired t-test on the absolute errors of the two estimators across all channel × 
time units to confirm this statistically. The result is t = −33.32, p = 3.71 × 10⁻²²⁰, 
meaning the variance-based errors are significantly smaller at essentially every unit. The 
p-value is so extreme that it really just confirms what the table already shows — the 
variance-based estimator and the stored ceiling are the same thing up to numerical precision.

The stored noise ceilings were generated using the variance-based estimator. The two agree 
to floating point precision across all channel × time units, with r = 1.000 and a mean 
absolute deviation of zero. A paired t-test confirms this is not a coincidence — the 
variance-based errors are significantly smaller than the split-half errors at essentially 
every unit (t = −33.32, p = 3.71 × 10⁻²²⁰). The split-half estimator tracks the same 
overall pattern well but is consistently more conservative, particularly at channels 13 and 
53 where it assigns near-zero reliability while the stored ceiling does not. As discussed 
in Section 1.3, this reflects a sensitivity difference between the two methods rather than 
an error in either one.

.</div>

---

## 1.5 Convert NSD ncsnr to noise ceiling and visualize it on cortex

Some datasets, such as NSD, provide reliability estimates with the data release. In this section, you will visualize the provided NSD reliability estimates on the cortical surface and convert them into noise ceilings for later use in predictive analyses.

The provided NSD reliability estimates are stored as **ncsnr** values on the fsaverage surface. To use them as noise ceilings for voxel-wise analyses, you need to convert ncsnr to noise ceiling using the formula provided in the NSD methods paper.

Parcellations and atlases provide group-level anatomical labels for brain regions. They are often defined on a standard surface or volume space (e.g., fsaverage, MNI) and can be used to summarize or interpret neural data. For this exercise, use the Destrieux atlas to anatomically label the regions with the highest and lowest noise ceilings. It is available in fsaverage space and can be accessed through `nilearn`. Compute the average noise ceiling within each atlas region and identify which regions have the highest and lowest reliability.

If the available atlas is in a different surface resolution (e.g. `fsaverage5`), you can interpolate either the atlas or the noise ceiling map to the same space before visualization. Prefer downsampling rather than upsampling to avoid introducing artificial precision.

You can use `nibabel` to load the `.mgh` files and `nilearn` to visualize the resulting noise ceiling on the fsaverage surface.


<div style="background:#eef5fb; border-left:4px solid #4c78a8; padding:8px 12px; border-radius:6px; font-weight:700; color:#26445e;">What you must do</div>

- Load the provided `.mgh` files for subject 01.
- Convert **ncsnr** to a **noise ceiling estimate** using the formula described in the NSD paper.
- Visualize the resulting noise ceiling on the fsaverage surface.
- Overlay a cortical parcellation.
- Compute parcel-wise average values.

<div style="background:#f3f6fa; border-left:4px solid #7a93ac; padding:8px 12px; border-radius:6px; font-weight:700; color:#32475b;">Required deliverables</div>

You must include all of the following:

1. **One histogram of ncsnr values**
2. **One cortical surface plot** of ncsnr or the derived noise ceiling
3. **One cortical surface plot with parcel overlay**
4. **One parcel-wise summary figure or one parcel-wise summary table**

In [ ]:
# Load ncsnr maps for subject 01
lh_img = nib.load(NSD_NCSNR_LH)
rh_img = nib.load(NSD_NCSNR_RH)

lh_ncsnr = lh_img.get_fdata().squeeze()
rh_ncsnr = rh_img.get_fdata().squeeze()

print(f"LH ncsnr shape: {lh_ncsnr.shape}, range: {lh_ncsnr.min():.3f} – {lh_ncsnr.max():.3f}")
print(f"RH ncsnr shape: {rh_ncsnr.shape}, range: {rh_ncsnr.min():.3f} – {rh_ncsnr.max():.3f}")

# Convert ncsnr to noise ceiling using the NSD formula (Allen et al. 2022)
# ncsnr = signal / noise
# noise ceiling (explained variance) = ncsnr^2 / (ncsnr^2 + 1/n_avg)
# For NSD, n_avg = 3 (average number of repetitions per image for the test set)
n_avg = 3
lh_nc = lh_ncsnr**2 / (lh_ncsnr**2 + 1.0 / n_avg)
rh_nc = rh_ncsnr**2 / (rh_ncsnr**2 + 1.0 / n_avg)

print(f"\nLH noise ceiling range: {lh_nc.min():.3f} – {lh_nc.max():.3f}")
print(f"RH noise ceiling range: {rh_nc.min():.3f} – {rh_nc.max():.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, ncsnr, nc, hemi in zip(
    axes,
    [lh_ncsnr, rh_ncsnr],
    [lh_nc, rh_nc],
    ["Left hemisphere", "Right hemisphere"],
):
    # only plot non-zero vertices (zero = outside cortex / medial wall)
    mask = ncsnr > 0
    ax2 = ax.twinx()
    ax.hist(ncsnr[mask], bins=80, color="steelblue", histtype="step", lw=2,
            label="ncsnr")
    ax2.hist(nc[mask], bins=80, color="darkorange", histtype="step", lw=2,
             label="noise ceiling")
    ax.set_xlabel("Value")
    ax.set_ylabel("Count (ncsnr)", color="steelblue")
    ax2.set_ylabel("Count (noise ceiling)", color="darkorange")
    ax.set_title(f"{hemi}")
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc="upper right")

fig.suptitle("Distribution of ncsnr and derived noise ceiling — subj01", fontsize=13)
fig.tight_layout()
plt.savefig("images/fig_nsd_ncsnr_histogram.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
views = ["lateral", "medial"]
hemis = [("left", lh_nc), ("right", rh_nc)]

sm = cm.ScalarMappable(cmap="hot", norm=mcolors.Normalize(vmin=0, vmax=0.9))
sm.set_array([])

for hemi, nc in hemis:
    for view in views:
        fig = plt.figure(figsize=(7, 5))
        ax = fig.add_subplot(111, projection="3d")
        nlplt.plot_surf_stat_map(
            fsaverage[f"infl_{hemi}"],
            stat_map=nc,
            hemi=hemi,
            view=view,
            colorbar=False,
            cmap="hot",
            vmax=0.9,
            bg_map=fsaverage[f"sulc_{hemi}"],
            axes=ax,
            title=f"{hemi} — {view}",
        )
        fig.colorbar(sm, ax=ax, shrink=0.5, pad=0.05, label="Noise ceiling")
        plt.savefig(f"images/fig_nsd_nc_{hemi}_{view}.png",
                    dpi=150, bbox_inches="tight")
        plt.close()
        print(f"Saved: {hemi} — {view}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5),
                         subplot_kw={"projection": "3d"})
nlplt.plot_surf_stat_map(
    fsaverage["infl_left"],
    stat_map=lh_nc,
    hemi="left",
    view="lateral",
    colorbar=False,
    cmap="hot",
    vmax=0.9,
    bg_map=fsaverage["sulc_left"],
    axes=axes[0],
    title="Left — lateral",
)
nlplt.plot_surf_stat_map(
    fsaverage["infl_right"],
    stat_map=rh_nc,
    hemi="right",
    view="lateral",
    colorbar=False,
    cmap="hot",
    vmax=0.9,
    bg_map=fsaverage["sulc_right"],
    axes=axes[1],
    title="Right — lateral",
)
sm = cm.ScalarMappable(cmap="hot", norm=mcolors.Normalize(vmin=0, vmax=0.9))
sm.set_array([])
fig.colorbar(sm, ax=axes, shrink=0.5, label="Noise ceiling")
fig.suptitle("NSD noise ceiling — subj01", fontsize=13)
plt.savefig("images/fig_nsd_nc_surface.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Load Destrieux atlas — labels on fsaverage5 (10242 vertices per hemi)
destrieux = datasets.fetch_atlas_surf_destrieux()
lh_labels = destrieux["map_left"]   # (10242,) integer labels
rh_labels = destrieux["map_right"]  # (10242,) integer labels
label_names = [name.decode() if isinstance(name, bytes) else name 
               for name in destrieux["labels"]]

print(f"LH labels shape: {lh_labels.shape}")
print(f"Number of regions: {len(label_names)}")
print(f"NC map LH shape: {lh_nc.shape}")  # 163842

# Downsample fsaverage NC to fsaverage5 by loading fsaverage5 coordinates
# and finding nearest fsaverage vertex for each fsaverage5 vertex
fsaverage5 = datasets.fetch_surf_fsaverage(mesh='fsaverage5')
fsaverage_full = datasets.fetch_surf_fsaverage(mesh='fsaverage')

# Load coordinates of both meshes
coords_fs5_lh, _ = surface.load_surf_mesh(fsaverage5["pial_left"])
coords_fs_lh, _  = surface.load_surf_mesh(fsaverage_full["pial_left"])
coords_fs5_rh, _ = surface.load_surf_mesh(fsaverage5["pial_right"])
coords_fs_rh, _  = surface.load_surf_mesh(fsaverage_full["pial_right"])

# For each fsaverage5 vertex find nearest fsaverage vertex
from scipy.spatial import cKDTree

tree_lh = cKDTree(coords_fs_lh)
_, idx_lh = tree_lh.query(coords_fs5_lh)
lh_nc_fs5 = lh_nc[idx_lh]

tree_rh = cKDTree(coords_fs_rh)
_, idx_rh = tree_rh.query(coords_fs5_rh)
rh_nc_fs5 = rh_nc[idx_rh]

print(f"Downsampled LH NC shape: {lh_nc_fs5.shape}")
print(f"Downsampled RH NC shape: {rh_nc_fs5.shape}")

# Compute parcel-wise averages
rows = []
for i, name in enumerate(label_names):
    lh_mask = lh_labels == i
    rh_mask = rh_labels == i
    lh_mean = lh_nc_fs5[lh_mask].mean() if lh_mask.sum() > 0 else np.nan
    rh_mean = rh_nc_fs5[rh_mask].mean() if rh_mask.sum() > 0 else np.nan
    avg = np.nanmean([lh_mean, rh_mean])
    rows.append(dict(Region=name, LH=round(lh_mean, 3), 
                     RH=round(rh_mean, 3), Average=round(avg, 3)))

df_parcels = pd.DataFrame(rows).set_index("Region")
df_parcels = df_parcels.sort_values("Average", ascending=False)

print("\nTop 10 most reliable regions:")
print(df_parcels.head(10).to_string())
print("\nBottom 10 least reliable regions:")
print(df_parcels.tail(10).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, df_sub, title in zip(
    axes,
    [df_parcels.dropna().head(10), df_parcels.dropna().iloc[-10:]],
    ["Top 10 most reliable regions", "Bottom 10 least reliable regions"],
):
    regions = df_sub.index[::-1]
    y = np.arange(len(regions))
    width = 0.3

    ax.barh(y + width/2, df_sub["LH"][::-1], width, 
            color="steelblue", alpha=0.8, label="LH")
    ax.barh(y - width/2, df_sub["RH"][::-1], width,
            color="darkorange", alpha=0.8, label="RH")
    ax.plot(df_sub["Average"][::-1], y, "k|", 
            markersize=10, markeredgewidth=2, label="Average")

    ax.set_yticks(y)
    ax.set_yticklabels(regions, fontsize=9)
    ax.set_xlabel("Noise ceiling")
    ax.set_title(title)
    ax.legend()
    ax.axvline(0, color="k", lw=0.5)

fig.suptitle("Parcel-wise noise ceiling — subj01 (Destrieux atlas)", fontsize=13)
fig.tight_layout()
plt.savefig("images/fig_nsd_nc_parcels.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Get top 10 region names (excluding Unknown/NaN)
top10_names = list(df_parcels.dropna().head(10).index)

# Create label maps that only show top 10 regions, rest = 0
def make_top10_map(labels, label_names, top10_names):
    out = np.zeros_like(labels, dtype=float)
    for i, name in enumerate(label_names):
        if name in top10_names:
            rank = top10_names.index(name) + 1  # 1 = most reliable
            out[labels == i] = rank
    return out

lh_top10_map = make_top10_map(lh_labels, label_names, top10_names)
rh_top10_map = make_top10_map(rh_labels, label_names, top10_names)

# Plot all 4 views
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches

configs = [
    ("left",  lh_top10_map, "lateral",  fsaverage5["infl_left"],  fsaverage5["sulc_left"]),
    ("left",  lh_top10_map, "medial",   fsaverage5["infl_left"],  fsaverage5["sulc_left"]),
    ("right", rh_top10_map, "lateral",  fsaverage5["infl_right"], fsaverage5["sulc_right"]),
    ("right", rh_top10_map, "medial",   fsaverage5["infl_right"], fsaverage5["sulc_right"]),
]

for hemi, nc_map, view, surf, sulc in configs:
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection="3d")
    nlplt.plot_surf_stat_map(
        surf,
        stat_map=nc_map,
        hemi=hemi,
        view=view,
        colorbar=False,
        cmap="tab10",
        vmin=0,
        vmax=10,
        bg_map=sulc,
        axes=ax,
        title=f"{hemi} — {view}",
    )
    # add legend for top 10 regions
    cmap = cm.get_cmap("tab10", 10)
    patches = [mpatches.Patch(color=cmap(i), 
                              label=f"{i+1}. {top10_names[i]}") 
               for i in range(10)]
    ax.legend(handles=patches, loc="lower left", fontsize=7,
              bbox_to_anchor=(0, 0))
    plt.savefig(f"images/fig_nsd_nc_top10_{hemi}_{view}.png",
                dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved: {hemi} — {view}")

### NDS reliability analysis

The NSD dataset provides reliability estimates as ncsnr values on the fsaverage surface 
rather than directly as noise ceilings. We load the provided <code>.mgh</code> files for 
subject 01 and convert them to noise ceilings using the formula from Allen et al. (2022):

$$NC = \frac{\text{ncsnr}^2}{\text{ncsnr}^2 + \frac{1}{n}}$$

where $n = 3$ reflects the fact that trial-wise betas are averaged across three presentations 
of each image before fitting the encoding model. The resulting noise ceiling is in the range 
[0, 1] and directly interpretable as the maximum fraction of response variance that any model 
could theoretically explain for a given voxel.

We first inspect the distribution of ncsnr values and the derived noise ceilings across all 
fsaverage vertices.

<div style="display: flex; gap: 10px; align-items: flex-start; margin-top: 15px;">
  <figure style="text-align: center; flex: 1;">
    <img src="images/fig_nsd_ncsnr_histogram.png" style="width: 80%;"/>
    <figcaption>Distribution of ncsnr values (blue) and derived noise ceilings (orange) 
    across all fsaverage vertices for subject 01. Only non-zero vertices are shown. Both 
    distributions are strongly right-skewed, reflecting that most cortical vertices are not 
    reliably driven by the visual stimuli. The compression from ncsnr to noise ceiling is 
    a nonlinear effect of the conversion formula, even the most reliable voxels only reach 
    a ceiling of around 0.87, reflecting the fundamental limit imposed by averaging only 
    three repetitions.</figcaption>
  </figure>
</div>

Both hemispheres show nearly identical distributions, confirming the data quality is 
consistent across hemispheres. The large spike near zero reflects the majority of cortical 
vertices that lie outside visually responsive areas. The long right tail of the ncsnr 
distribution represents a small number of highly reliable voxels, most likely in early 
visual cortex.

We then visualize the noise ceiling on the inflated fsaverage surface alongside the top 10 
most reliable Destrieux parcels to allow direct comparison between the continuous reliability 
map and the discrete anatomical boundaries.

<div style="display: flex; gap: 10px; align-items: flex-start; margin-top: 15px;">
  <figure style="text-align: center; flex: 1;">
    <img src="images/fig_nsd_nc_left_lateral.png" style="width: 93%;"/>
    <figcaption>Left lateral — noise ceiling</figcaption>
  </figure>
  <figure style="text-align: center; flex: 1;">
    <img src="images/fig_nsd_nc_top10_left_lateral.png" style="width: 80%;"/>
    <figcaption>Left lateral — top 10 parcels</figcaption>
  </figure>
</div>

<div style="display: flex; gap: 10px; align-items: flex-start; margin-top: 15px;">
  <figure style="text-align: center; flex: 1;">
    <img src="images/fig_nsd_nc_left_medial.png" style="width: 93%;"/>
    <figcaption>Left medial — noise ceiling</figcaption>
  </figure>
  <figure style="text-align: center; flex: 1;">
    <img src="images/fig_nsd_nc_top10_left_medial.png" style="width: 80%;"/>
    <figcaption>Left medial — top 10 parcels</figcaption>
  </figure>
</div>

<div style="display: flex; gap: 10px; align-items: flex-start; margin-top: 15px;">
  <figure style="text-align: center; flex: 1;">
    <img src="images/fig_nsd_nc_right_lateral.png" style="width: 93%;"/>
    <figcaption>Right lateral — noise ceiling</figcaption>
  </figure>
  <figure style="text-align: center; flex: 1;">
    <img src="images/fig_nsd_nc_top10_right_lateral.png" style="width: 80%;"/>
    <figcaption>Right lateral — top 10 parcels</figcaption>
  </figure>
</div>

<div style="display: flex; gap: 10px; align-items: flex-start; margin-top: 15px;">
  <figure style="text-align: center; flex: 1;">
    <img src="images/fig_nsd_nc_right_medial.png" style="width: 93%;"/>
    <figcaption>Right medial — noise ceiling</figcaption>
  </figure>
  <figure style="text-align: center; flex: 1;">
    <img src="images/fig_nsd_nc_top10_right_medial.png" style="width: 80%;"/>
    <figcaption>Right medial — top 10 parcels</figcaption>
  </figure>
</div>

The pattern is clear and anatomically consistent across both hemispheres. The highest noise 
ceilings are concentrated in the posterior occipital cortex, particularly in the lateral 
views where the occipital pole is visible at the back of the brain, and in the medial views 
where a bright strip runs along the bottom of the surface corresponding to the calcarine 
sulcus, which is the location of primary visual cortex. Reliability drops off steeply as you move 
anteriorly towards frontal and temporal cortex. The two hemispheres look nearly symmetric.

To identify which specific brain regions are most and least reliable, we use the Destrieux 
atlas to compute parcel-wise average noise ceilings. The Destrieux atlas divides the cortical 
surface into 74 named regions per hemisphere based on sulcal and gyral landmarks. Since the 
atlas is defined on fsaverage5 (10242 vertices per hemisphere) rather than the full fsaverage 
resolution, we first downsample our noise ceiling map using nearest-neighbor interpolation 
before averaging within each parcel.

<div style="display: flex; gap: 10px; align-items: flex-start; margin-top: 15px;">
  <figure style="text-align: center; flex: 1;">
    <img src="images/fig_nsd_nc_parcels.png" style="width: 90%;"/>
    <figcaption>Top 10 most reliable (left) and bottom 10 least reliable (right) Destrieux 
    regions by average noise ceiling across hemispheres. Blue bars show left hemisphere 
    values, orange bars show right hemisphere values, and the black marker shows the 
    bilateral average.</figcaption>
  </figure>
</div>

The top 10 regions are entirely occipital: the occipital pole, the posterior collateral 
sulcus, the middle occipital gyrus, the anterior occipital sulcus and the lateral 
occipito-temporal gyrus which includes the fusiform face area. The most reliable region 
overall is the posterior transverse collateral sulcus with an average noise ceiling of 0.54, 
followed by the anterior occipital sulcus at 0.52 and the occipital pole at 0.51. These 
values mean that even the most reliable regions have around half their response variance 
explained by the stimulus, with the other half being noise — reflecting the fundamental 
challenge of single-subject fMRI with limited repetitions.

The bottom 10 regions are all non-visual — motor cortex, frontal cortex, temporal pole and 
auditory cortex, with average noise ceilings below 0.03, essentially zero. This means they 
produce no reliable stimulus-driven responses to the NSD images as expected.

The most reliable cortical regions are concentrated in occipital cortex, particularly 
around the occipital pole, the posterior collateral sulcus, and the calcarine sulcus where 
primary visual cortex sits. Average noise ceilings in these regions reach around 0.50 to 
0.54, meaning roughly half of the response variance is reliably stimulus-driven. The least 
reliable regions are motor cortex, frontal cortex, temporal pole and auditory cortex, all 
with noise ceilings below 0.03. This pattern is exactly what you would expect from a 
dataset of visual images — areas that process visual information are reliably driven by 
the stimuli, while areas involved in motor control, language and audition are not.

The Destrieux parcellation is useful here because it translates the continuous surface map 
into specific anatomical names, making it possible to say concretely which regions are 
reliable rather than just pointing at blobs. It also reveals subtle differences between 
hemispheres, for example the posterior collateral sulcus has a noise ceiling of 0.516 in 
the left hemisphere but 0.566 in the right, suggesting slightly stronger visual responses 
on the right side for this subject. Whether this reflects a genuine hemispheric asymmetry 
or subject-specific variability would require comparison across subjects.
</div>

<div style="background:#fff4c2; border:1px solid #c89b1f; border-left:6px solid #9a6f00; padding:10px 12px; border-radius:6px; margin-top:8px; margin-bottom:4px; color:#241a00; line-height:1.45;"><strong style="color:#5c4300;">Answer box 1.5</strong><br>Which cortical regions appear most reliable, and which appear least reliable? Explain how the parcellation helps interpret the surface maps.<p>

The most reliable regions are concentrated in occipital cortex, particularly the occipital 
pole, the posterior collateral sulcus, and the calcarine sulcus where primary visual cortex 
sits, with average noise ceilings of 0.50–0.54. The least reliable regions are motor, 
frontal, temporal pole and auditory cortex, all below 0.03. This pattern is exactly what 
you would expect from a visual image dataset: areas that process vision are reliably driven 
by the stimuli, while areas involved in motor control, language and audition are not.

The Destrieux parcellation translates the continuous surface map into specific anatomical 
names, making it possible to identify reliably which regions are most stimulus-driven rather 
than just pointing at blobs on the surface. It also reveals subtle hemispheric differences, 
for example the posterior collateral sulcus has a noise ceiling of 0.516 in the left 
hemisphere but 0.566 in the right, though whether this reflects a genuine asymmetry or 
subject-specific variability would require comparison across subjects.
</p></div>

---

# 2. Brain–Model Alignment

In this section, you will compare neural responses and model features using **both representational metrics and predictive linear models**. You must complete both parts of this section. The goal is not only to report scores, but also to compare what different metrics reveal about model–brain alignment.

## 2.1 Representational alignment: RSA

RSA stands for representational similarity analysis. It is one of the most widely used analyses in fMRI and model–brain alignment research. It compares the geometry of two representational spaces through their representational dissimilarity matrices (RDMs). Given two response matrices, `X` and `Y`, with rows corresponding to the same stimuli, we first compute an RDM for each matrix using correlation distance:

$$
D^X_{ij} = 1 - \mathrm{corr}(X[i,:], X[j,:]),
\qquad
D^Y_{ij} = 1 - \mathrm{corr}(Y[i,:], Y[j,:]),
$$

for stimulus pairs $i \neq j$.

We then vectorize the upper triangle of each RDM and compute RSA as the Spearman correlation between these two vectors:

$$
\mathrm{RSA}(X, Y)
=
\rho_{\mathrm{Spearman}}
\left(
\mathrm{vec}(D^X),\,
\mathrm{vec}(D^Y)
\right).
$$

In this project, `X` will usually denote model features from one candidate layer, and `Y` will denote neural responses from one dataset, ROI, subject, or time slice, depending on the analysis.

- Implement RSA between two representation matrices.
- Support at least one dissimilarity measure and one similarity measure.
- Use your implementation to compare model layers to neural responses.

In [ ]:
import importlib
import utils
import numpy as np
importlib.reload(utils)
from utils import RepresentationalSimilarityAnalysis

# quick sanity check
rsa = RepresentationalSimilarityAnalysis(dissimilarity="correlation", 
                                          similarity_metric="spearman")
X = np.random.randn(20, 100)
Y = np.random.randn(20, 50)
print(f"RSA between random matrices: {rsa(X, Y):.4f}  (should be near 0)")
print(f"RSA of matrix with itself:   {rsa(X, X):.4f}  (should be 1.0)")

#### Sanity check results

We sanity-check our RSA implementation on two synthetic cases. Two independent random matrices over the same 20 stimuli should be representationally unrelated, so their RSA score should be close to zero; a matrix compared with itself has identical RDMs, so RSA should equal exactly 1. Both checks pass:

    RSA between random matrices: 0.0324  (should be near 0)
    RSA of matrix with itself:   1.0000  (should be 1.0)

The small non-zero value for unrelated matrices reflects the finite sample size: with only 20 stimuli, the 190 pairwise distances in each RDM produce a noisy estimate of zero correlation rather than exactly 0.

## 2.2 Representational alignment: unbiased linear CKA

CKA stands for centered kernel alignment. It is commonly used in interpretability and representation analysis to test how strongly the internal computations of two systems align. As a second mapping-free alignment metric, we want to compute unbiased linear centered kernel alignment (CKA) between model features and neural responses. Let

$$
X \in \mathbb{R}^{n \times d}, \qquad
Y \in \mathbb{R}^{n \times p},
$$

where both matrices are measured on the same $n$ stimuli. We form linear Gram matrices

$$
K = XX^\top,
\qquad
L = YY^\top.
$$

We then estimate dependence using the unbiased (U-statistic) HSIC estimator, $\mathrm{HSIC}_u(K, L)$, and define CKA as

$$
\mathrm{CKA}(X, Y)
=
\frac{\mathrm{HSIC}_u(K, L)}
{\sqrt{\mathrm{HSIC}_u(K, K)\,\mathrm{HSIC}_u(L, L)}}.
$$

Like RSA, CKA compares representational structure directly without fitting a predictive mapping. In this notebook, `X` and `Y` again refer to aligned model and neural response matrices evaluated on the same set of stimuli.

- Implement **unbiased linear CKA** only.
- Use your implementation to compare model layers to neural responses.

In [ ]:
import importlib
import utils
importlib.reload(utils)
from utils import CenteredKernelAlignment

cka = CenteredKernelAlignment()

X = np.random.randn(100, 50)
Y = np.random.randn(100, 30)

print(f"CKA between random matrices:  {cka(X, Y):.4f}  (should be near 0)")
print(f"CKA of matrix with itself:    {cka(X, X):.4f}  (should be near 1)")
print(f"CKA with scaled version:      {cka(X, X * 2):.4f}  (should be near 1 — scale invariant)")
print(f"CKA with orthogonal rotation: {cka(X, X @ np.linalg.qr(np.random.randn(50,50))[0]):.4f}  (should be near 1)")

#### Sanity check results

We sanity-check our unbiased linear CKA on four synthetic cases. Two independent random matrices should be unrelated (CKA near 0), a matrix compared with itself should yield CKA = 1, and CKA should be invariant to isotropic scaling and to orthogonal rotations of the feature dimensions. All four properties hold:

    CKA between random matrices:  -0.0052  (should be near 0)
    CKA of matrix with itself:    1.0000  (should be near 1)
    CKA with scaled version:      1.0000  (should be near 1 — scale invariant)
    CKA with orthogonal rotation: 1.0000  (should be near 1)

The slightly negative value for unrelated matrices is expected: the unbiased HSIC estimator can produce small negative values by construction, unlike the biased version which is non-negative. Scale and rotation invariance hold exactly because linear CKA depends only on the Gram matrix $XX^\top$, which is unchanged when $X$ is multiplied by a scalar or by an orthogonal matrix on the right.

## 2.3 Apply RSA and CKA

<div style="background:#eef5fb; border-left:4px solid #4c78a8; padding:8px 12px; border-radius:6px; font-weight:700; color:#26445e;">What you must do</div>

- Compare layers within each model.
- Compare the two models.
- For EEG, show how representational similarity changes over time.
- For TVSD and NSD, compare across ROIs.

<div style="background:#f3f6fa; border-left:4px solid #7a93ac; padding:8px 12px; border-radius:6px; font-weight:700; color:#32475b;">Required deliverables</div>

You must include all of the following:

1. **Layer-wise RSA results** for both models.
2. **Layer-wise CKA results** for both models.
3. **One direct comparison between the two models** using representational metrics.
4. **One EEG time-resolved analysis** or **one ROI-wise analysis** for TVSD/NSD.
5. **One short written interpretation** in Answer box 2.1.

In [ ]:
with h5py.File(FEAT_A_THINGS, "r") as f:
    f.visititems(lambda name, obj: print(name, "→", obj.shape if hasattr(obj, "shape") else "group"))

In [ ]:
with h5py.File(FEAT_B_THINGS, "r") as f:
    f.visititems(lambda name, obj: print(name, "→", obj.shape if hasattr(obj, "shape") else "group"))

In [ ]:
with h5py.File(TVSD_PATH, "r") as f:
    f.visititems(lambda name, obj: print(name, "→", obj.shape if hasattr(obj, "shape") else "group"))

In [ ]:
import importlib
import utils
importlib.reload(utils)
from utils import RepresentationalSimilarityAnalysis, CenteredKernelAlignment, get_feat_rows

# Define layers for both models
with h5py.File(FEAT_A_THINGS, "r") as f:
    layers_A = sorted(f["features"].keys())

with h5py.File(FEAT_B_THINGS, "r") as f:
    layers_B = sorted(f["features"].keys())

print("Model A layers:", layers_A)
print("Model B layers:", layers_B)

# Instantiate metrics
rsa = RepresentationalSimilarityAnalysis(dissimilarity="correlation",
                                          similarity_metric="spearman")
cka = CenteredKernelAlignment()

In [ ]:
# Load TVSD test data — all ROIs and both monkeys
tvsd_rois   = ["V1", "V4", "IT"]
tvsd_monkeys = ["monkeyF", "monkeyN"]
tvsd_test    = {}
tvsd_nc      = {}

with h5py.File(TVSD_PATH, "r") as f:
    # stimulus IDs are shared
    tvsd_test_ids = f["test/stimulus_ids"][:]
    
    for monkey in tvsd_monkeys:
        for roi in tvsd_rois:
            key = f"{monkey}_{roi}"
            tvsd_test[key]  = f[f"test/neural_data/{monkey}/{roi}"][:]
            tvsd_nc[key]    = f[f"noise_ceilings/{monkey}/{roi}"][:]
            print(f"{key}: responses {tvsd_test[key].shape}, "
                  f"nc {tvsd_nc[key].shape}")

print(f"\nTest stimulus IDs shape: {tvsd_test_ids.shape}")
print(f"Example IDs: {tvsd_test_ids[:3]}")

In [ ]:
# Match TVSD stimulus IDs to feature rows
with h5py.File(FEAT_A_THINGS, "r") as f:
    feat_ids_A = f["ids"][:]

with h5py.File(FEAT_B_THINGS, "r") as f:
    feat_ids_B = f["ids"][:]

# Build index maps
id_to_idx_A = {id_: i for i, id_ in enumerate(feat_ids_A)}
id_to_idx_B = {id_: i for i, id_ in enumerate(feat_ids_B)}

# Get feature row indices for TVSD test stimuli
feat_idx_A = np.array([id_to_idx_A[x] for x in tvsd_test_ids])
feat_idx_B = np.array([id_to_idx_B[x] for x in tvsd_test_ids])

print(f"Feature indices shape: {feat_idx_A.shape}")
print(f"First 3 indices A: {feat_idx_A[:3]}")

# Compute RSA and CKA across layers for each monkey/ROI combination
results = []

for layer_key, feat_path, feat_idx, model_name in [
    *[(f"features/{l}", FEAT_A_THINGS, feat_idx_A, "ResNet152") for l in layers_A],
    *[(f"features/{l}", FEAT_B_THINGS, feat_idx_B, "Qwen3-VL") for l in layers_B],
]:
    # Load features for matched stimuli
    sort_idx = np.argsort(feat_idx)
    unsort_idx = np.argsort(sort_idx)
    with h5py.File(feat_path, "r") as f:
        feats = f[layer_key][feat_idx[sort_idx]][unsort_idx]  # (100, 30000)

    layer_name = layer_key.replace("features/", "")

    for key, neural in tvsd_test.items():
        rsa_score = rsa(feats, neural)
        cka_score = cka(feats, neural)
        monkey, roi = key.split("_")
        results.append(dict(
            model=model_name,
            layer=layer_name,
            monkey=monkey,
            roi=roi,
            rsa=rsa_score,
            cka=cka_score,
        ))
        print(f"{model_name} | {layer_name} | {key}: RSA={rsa_score:.3f}, CKA={cka_score:.3f}")

import pandas as pd
df_tvsd = pd.DataFrame(results)
print("\nDone. Shape:", df_tvsd.shape)

In [ ]:
# Define layer order for each model (by depth)
layer_order_A = ['layer1-0', 'layer2-0', 'layer3-0', 'layer3-5', 'layer3-10',
                 'layer3-15', 'layer3-20', 'layer3-25', 'layer3-30', 'layer4-1']
layer_order_B = ['visual-blocks-2', 'visual-blocks-6', 'visual-blocks-10',
                 'visual-blocks-14', 'visual-blocks-18', 'visual-blocks-22',
                 'language_model-layers-3', 'language_model-layers-8',
                 'language_model-layers-11', 'language_model-layers-16']

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for col, (model_name, layer_order) in enumerate([
    ("ResNet152", layer_order_A),
    ("Qwen3-VL", layer_order_B),
]):
    df_model = df_tvsd[df_tvsd["model"] == model_name].copy()
    df_model["layer"] = pd.Categorical(df_model["layer"], 
                                        categories=layer_order, ordered=True)
    df_model = df_model.sort_values("layer")

    for row, metric in enumerate(["rsa", "cka"]):
        ax = axes[row, col]
        for roi in ["V1", "V4", "IT"]:
            # average over monkeys
            df_roi = df_model[df_model["roi"] == roi].groupby("layer")[metric].mean()
            ax.plot(range(len(layer_order)), df_roi.values, 
                    marker="o", lw=2, label=roi)
        
        ax.set_xticks(range(len(layer_order)))
        ax.set_xticklabels(layer_order, rotation=45, ha="right", fontsize=8)
        ax.set_xlabel("Layer")
        ax.set_ylabel(metric.upper())
        ax.set_title(f"{model_name} — {metric.upper()} — TVSD")
        ax.legend()
        ax.axhline(0, color="k", lw=0.5, ls="--")

fig.suptitle("Layer-wise RSA and CKA — TVSD (averaged over monkeys)", fontsize=13)
fig.tight_layout()
plt.savefig("images/fig_tvsd_rsa_cka_layers.png", dpi=150, bbox_inches="tight")
plt.show()

![Alt text](images/fig_tvsd_rsa_cka_layers.png)

In [ ]:
# For each model, find the best layer per ROI (highest RSA and CKA)
# then plot a bar chart comparing ROIs side by side

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

metrics = ["rsa", "cka"]
colors = {"V1": "steelblue", "V4": "darkorange", "IT": "forestgreen"}
x = np.arange(2)  # rsa and cka
width = 0.25

for col, model_name in enumerate(["ResNet152", "Qwen3-VL"]):
    ax = axes[col]
    df_model = df_tvsd[df_tvsd["model"] == model_name]
    
    for i, roi in enumerate(["V1", "V4", "IT"]):
        df_roi = df_model[df_model["roi"] == roi]
        # best layer = layer with highest average score across monkeys
        best_rsa = df_roi.groupby("layer")["rsa"].mean().max()
        best_cka = df_roi.groupby("layer")["cka"].mean().max()
        ax.bar(x + i * width, [best_rsa, best_cka], width,
               label=roi, color=colors[roi], alpha=0.8)
    
    ax.set_xticks(x + width)
    ax.set_xticklabels(["RSA", "CKA"])
    ax.set_ylabel("Best-layer score")
    ax.set_title(f"{model_name} — best layer per ROI")
    ax.legend()
    ax.set_ylim(0, None)

fig.suptitle("ROI comparison — best layer score — TVSD", fontsize=13)
fig.tight_layout()
plt.savefig("images/fig_tvsd_roi_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

![Alt text](images/fig_tvsd_roi_comparison.png)

In [ ]:
# new (from Anastasis)
# TVSD summary table — best layer per (ROI, model), averaged across monkeys
# Reports: best RSA / CKA values and the relative depth (0..1) of the argmax layer.
# Relative depth makes layer hierarchy comparable across the two architectures.

tvsd_summary_rows = []

for model_name, layer_order in [("ResNet152", layer_order_A),
                                 ("Qwen3-VL",  layer_order_B)]:
    L = len(layer_order) - 1   # max layer index for normalization
    df_m = df_tvsd[df_tvsd["model"] == model_name]
    for roi in ["V1", "V4", "IT"]:
        # average across the two monkeys, ordered by depth
        rsa_vec = (df_m[df_m["roi"] == roi]
                   .groupby("layer")["rsa"].mean()
                   .reindex(layer_order).values)
        cka_vec = (df_m[df_m["roi"] == roi]
                   .groupby("layer")["cka"].mean()
                   .reindex(layer_order).values)
        i_rsa = int(np.nanargmax(rsa_vec))
        i_cka = int(np.nanargmax(cka_vec))
        tvsd_summary_rows.append({
            "ROI":        roi,
            "model":      model_name,
            "best_RSA":   float(rsa_vec[i_rsa]),
            "RSA_layer":  layer_order[i_rsa],
            "RSA_depth":  i_rsa / L,
            "best_CKA":   float(cka_vec[i_cka]),
            "CKA_layer":  layer_order[i_cka],
            "CKA_depth":  i_cka / L,
        })

df_tvsd_summary = (pd.DataFrame(tvsd_summary_rows)
                     .round(3)
                     .sort_values(["ROI", "model"])
                     .reset_index(drop=True))

# Save for the report and for inclusion in markdown
import os
os.makedirs("results", exist_ok=True)
df_tvsd_summary.to_csv("results/tvsd_summary_best_layer.csv", index=False)

print("TVSD — best layer per (ROI, model), averaged across monkeys")
print(df_tvsd_summary.to_string(index=False))

#### Motivation & Methodology
Task-optimised hierarchical neural networks, though never fit to brain data, develop 
internal representations that mirror the ventral stream hierarchy: early layers predict 
V4 responses while deeper layers predict IT, establishing a direct correspondence between 
model depth and cortical area <a href="#yamins2014" title="Yamins et al. (2014) — 
Performance-optimized hierarchical models predict neural responses in higher visual 
cortex">[10]</a>. The table below tests whether this holds for both our models on the 
TVSD macaque electrophysiology dataset, reporting the best-scoring layer per ROI averaged 
across both monkeys. The `_depth` columns normalize the argmax layer position to [0, 1] 
so ResNet152 and Qwen3-VL can be compared on the same scale. For Qwen3-VL, layers are 
ordered architecturally (visual blocks first, language-model layers last) so that 
"deeper" reflects later in the actual computation.

#### Results

| ROI | Model     | best_RSA | RSA_layer               | RSA_depth | best_CKA | CKA_layer                | CKA_depth |
|-----|-----------|----------|-------------------------|-----------|----------|--------------------------|-----------|
| V1  | Qwen3-VL  | 0.105    | visual-blocks-6         | 0.111     | 0.159    | visual-blocks-2          | 0.000     |
| V1  | ResNet152 | 0.198    | layer3-0                | 0.222     | 0.362    | layer2-0                 | 0.111     |
| V4  | Qwen3-VL  | 0.068    | visual-blocks-22        | 0.556     | 0.106    | language_model-layers-3  | 0.667     |
| V4  | ResNet152 | 0.232    | layer3-0                | 0.222     | 0.281    | layer3-0                 | 0.222     |
| IT  | Qwen3-VL  | 0.086    | visual-blocks-22        | 0.556     | 0.129    | language_model-layers-11 | 0.889     |
| IT  | ResNet152 | 0.223    | layer4-1                | 1.000     | 0.252    | layer3-25                | 0.778     |

For ResNet152 the correspondence is clean: V1 peaks at depth 0.11–0.22 (early layers 
processing edges and low-level structure), V4 at 0.22, and IT at 0.78–1.00 (deep layers 
encoding complex object features). Qwen3-VL shows the same ordering, with V1 peaking 
earliest (depth 0.00–0.11), V4 in the late visual blocks (0.56–0.67), and IT in the 
language-model layers (0.56–0.89), but with scores roughly half of ResNet152's across 
all six ROI x metric cells. CKA is systematically higher than RSA for both models 
(e.g. ResNet V1: 0.362 vs 0.198, Qwen V1: 0.159 vs 0.105), consistent with CKA being 
more sensitive to large geometric differences than rank-based RSA. Overall, ResNet152 
wins on every cell, but Qwen3-VL's argmax ordering still tracks the brain hierarchy 
approximately, suggesting both models capture similar macro-structure with different 
fidelity.

**Key takeaways**
- ResNet152 cleanly recovers the V1 to V4 to IT depth correspondence; Qwen3-VL tracks 
  the same ordering but with ~50% lower scores
- CKA is consistently higher than RSA for both models, reflecting its greater sensitivity 
  to geometric structure
- Qwen's middle visual blocks show weak alignment across all ROIs, suggesting those 
  layers serve language grounding rather than ventral stream processing

#### Motivation
The best-layer table summarizes peak alignment, but looking at the full layer-wise 
curves reveals how alignment evolves across depth and whether the hierarchy is gradual 
or abrupt. This is important for understanding whether the correspondence between model 
depth and cortical area is a general property of the representation or just an artifact 
of a single well-aligned layer.

#### Results

For ResNet152, both RSA and CKA reveal a clear correspondence between model depth and 
the neural hierarchy, averaged across both monkeys. V1 peaks early at layer3-0 and 
declines steadily, while IT remains consistently higher from layer3-5 onwards and 
increases again at layer4-1. V4 follows an intermediate trajectory, peaking at layer3-0 
alongside V1 and then settling between V1 and IT for deeper layers. In the CKA plot 
this picture is sharper: V1 starts much higher than the other ROIs at layer1-0 (around 
0.35 vs 0.18 for V4 and 0.10 for IT), then drops as depth increases while IT gradually 
rises. The crossover point is cleaner in CKA than in RSA, suggesting CKA is more 
sensitive to the geometric reorganization happening across layers.

Qwen3-VL shows a weaker and noisier pattern. RSA values are roughly half those of 
ResNet152 across all layers and ROIs. The ROI ordering is broadly preserved but the 
curves are less structured, and the middle visual blocks (visual-blocks-10 through 
visual-blocks-18) show a drop in alignment for all ROIs before a partial recovery. The 
most striking finding is a large CKA jump for V4 and IT at the first language-model 
layer, which is much less visible in RSA. This suggests the language-model layers 
reorganize the inner-product geometry of the representation in a way that makes it more 
similar to higher visual areas, without necessarily changing the rank ordering of 
pairwise stimulus distances that RSA captures.

Taken together, ResNet152 provides substantially better alignment with macaque ventral 
stream responses than Qwen3-VL across all layers, ROIs, and metrics. Whether this 
advantage generalizes to other modalities is what we examine next with EEG and NSD.

<div style="display: flex; gap: 10px; align-items: flex-start; margin-top: 15px;">
  <figure style="text-align: center; flex: 1;">
    <img src="images/fig_tvsd_rsa_cka_layers.png" style="width: 100%;"/>
    <figcaption>Layer-wise RSA (top) and CKA (bottom) for ResNet152 (left) and Qwen3-VL 
    (right) on the TVSD test set, averaged across the two monkeys. Each line corresponds 
    to one ROI (V1, V4, IT).</figcaption>
  </figure>
</div>

**Key takeaways**
- ResNet152 shows a gradual and consistent depth-to-hierarchy correspondence across 
  all three ROIs, cleaner in CKA than RSA
- Qwen3-VL is noisier overall, with a notable drop in the middle visual blocks and a 
  sharp CKA recovery at the language-model boundary
- The CKA jump at Qwen's language-model layers is not visible in RSA, pointing to a 
  geometric reorganization that rank-based RSA cannot detect

In [ ]:
with h5py.File(EEG_PATH, "r") as f:
    f.visititems(lambda name, obj: print(name, "→", obj.shape if hasattr(obj, "shape") else "group"))

In [ ]:
SUBJECT_EEG = "sub-01"
ROI_EEG     = "occipital_parietal"

with h5py.File(EEG_PATH, "r") as f:
    eeg_test      = f[f"test/neural_data/{SUBJECT_EEG}/{ROI_EEG}"][:]  # (200, 17, 80)
    eeg_test_ids  = f["test/stimulus_ids"][:]                           # (200,)
    eeg_nc        = f[f"noise_ceilings/{SUBJECT_EEG}/{ROI_EEG}"][:]    # (17, 80)

print(f"EEG test shape: {eeg_test.shape}")
print(f"EEG test IDs shape: {eeg_test_ids.shape}")
print(f"EEG noise ceiling shape: {eeg_nc.shape}")
print(f"Example IDs: {eeg_test_ids[:3]}")
print(f"Times: 0.0s to 0.8s, {eeg_test.shape[2]} time points")

In [ ]:
# Match EEG stimulus IDs to feature rows
with h5py.File(FEAT_A_THINGS, "r") as f:
    feat_ids_A = f["ids"][:]
with h5py.File(FEAT_B_THINGS, "r") as f:
    feat_ids_B = f["ids"][:]

id_to_idx_A = {id_: i for i, id_ in enumerate(feat_ids_A)}
id_to_idx_B = {id_: i for i, id_ in enumerate(feat_ids_B)}

feat_idx_eeg_A = np.array([id_to_idx_A[x] for x in eeg_test_ids])
feat_idx_eeg_B = np.array([id_to_idx_B[x] for x in eeg_test_ids])

print(f"Matched {len(feat_idx_eeg_A)} stimuli for Model A")
print(f"Matched {len(feat_idx_eeg_B)} stimuli for Model B")

# Load features for matched stimuli
sort_A = np.argsort(feat_idx_eeg_A)
unsort_A = np.argsort(sort_A)
sort_B = np.argsort(feat_idx_eeg_B)
unsort_B = np.argsort(sort_B)

eeg_feats_A = {}
eeg_feats_B = {}

with h5py.File(FEAT_A_THINGS, "r") as f:
    for layer in layers_A:
        eeg_feats_A[layer] = f[f"features/{layer}"][feat_idx_eeg_A[sort_A]][unsort_A]

with h5py.File(FEAT_B_THINGS, "r") as f:
    for layer in layers_B:
        eeg_feats_B[layer] = f[f"features/{layer}"][feat_idx_eeg_B[sort_B]][unsort_B]

print("Features loaded.")

In [ ]:
import importlib
import utils
importlib.reload(utils)
from utils import safe_normalize
from scipy.stats import spearmanr
from scipy.spatial.distance import pdist

print("Precomputing model RDMs and normalized Gram matrices...")
model_rdms_A = {}
model_K_A = {}
for layer in layers_A:
    feats = safe_normalize(eeg_feats_A[layer])
    model_rdms_A[layer] = pdist(feats, metric="correlation")
    K = feats @ feats.T
    np.fill_diagonal(K, 0)
    model_K_A[layer] = K
    print(f"  ResNet152 {layer} done")

model_rdms_B = {}
model_K_B = {}
for layer in layers_B:
    feats = safe_normalize(eeg_feats_B[layer])
    model_rdms_B[layer] = pdist(feats, metric="correlation")
    K = feats @ feats.T
    np.fill_diagonal(K, 0)
    model_K_B[layer] = K
    print(f"  Qwen3-VL {layer} done")

print("All model RDMs and Gram matrices precomputed.")

times = np.linspace(0.0, 0.8, 80)
results_eeg = []

print("Computing time-resolved RSA and CKA...")
for model_name, layers, model_rdms, model_K in [
    ("ResNet152", layers_A, model_rdms_A, model_K_A),
    ("Qwen3-VL",  layers_B, model_rdms_B, model_K_B),
]:
    for layer in layers:
        rdm_model = model_rdms[layer]
        K = model_K[layer]

        for t_idx in range(80):
            neural_t = eeg_test[:, :, t_idx].astype(np.float64)

            # RSA
            rdm_neural = pdist(neural_t, metric="correlation")
            rsa_score, _ = spearmanr(rdm_model, rdm_neural)

            # CKA
            neural_norm = safe_normalize(neural_t)
            L = neural_norm @ neural_norm.T
            np.fill_diagonal(L, 0)
            n = 200
            KL = K @ L
            hsic_kl = (np.trace(KL) + K.sum() * L.sum() / ((n-1)*(n-2))
                       - 2 * KL.sum() / (n-2)) / (n * (n-3))
            hsic_kk = (np.trace(K @ K) + K.sum()**2 / ((n-1)*(n-2))
                       - 2 * (K @ K).sum() / (n-2)) / (n * (n-3))
            hsic_ll = (np.trace(L @ L) + L.sum()**2 / ((n-1)*(n-2))
                       - 2 * (L @ L).sum() / (n-2)) / (n * (n-3))
            denom = np.sqrt(max(hsic_kk, 0) * max(hsic_ll, 0)) + 1e-8
            cka_score = hsic_kl / denom

            results_eeg.append(dict(
                model=model_name,
                layer=layer,
                time=times[t_idx],
                rsa=float(rsa_score),
                cka=float(cka_score),
            ))

        print(f"  {model_name} {layer} done")

df_eeg = pd.DataFrame(results_eeg)
print(f"\nDone. Shape: {df_eeg.shape}")

df_eeg.to_pickle("results/df_eeg_rsa_cka.pkl")
print("Saved.")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for col, model_name in enumerate(["ResNet152", "Qwen3-VL"]):
    df_model = df_eeg[df_eeg["model"] == model_name]
    
    for row, metric in enumerate(["rsa", "cka"]):
        ax = axes[row, col]
        
        # Build heatmap matrix: layers x time
        if model_name == "ResNet152":
            layer_order = layer_order_A
        else:
            layer_order = layer_order_B
        
        heatmap = np.zeros((len(layer_order), 80))
        for i, layer in enumerate(layer_order):
            df_layer = df_model[df_model["layer"] == layer].sort_values("time")
            heatmap[i] = df_layer[metric].values
        
        im = ax.imshow(heatmap, aspect="auto", origin="lower",
                       extent=[0, 0.8, -0.5, len(layer_order) - 0.5],
                       cmap="RdBu_r" if metric == "rsa" else "viridis",
                       vmin=-heatmap.max() if metric == "rsa" else 0,
                       vmax=heatmap.max())
        ax.set_yticks(range(len(layer_order)))
        ax.set_yticklabels(layer_order, fontsize=8)
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Layer")
        ax.set_title(f"{model_name} — {metric.upper()} — EEG occipital_parietal")
        ax.axvline(0.0, color="k", lw=1, ls="--")
        fig.colorbar(im, ax=ax, shrink=0.85)

fig.suptitle(f"Time-resolved RSA and CKA — {SUBJECT_EEG} / {ROI_EEG}", fontsize=13)
fig.tight_layout()
plt.savefig("images/fig_eeg_rsa_cka_time.png", dpi=150, bbox_inches="tight")
plt.show()

![Alt text](images/fig_eeg_rsa_cka_time.png)

In [ ]:
subjects_eeg = [f"sub-{i:02d}" for i in range(1, 11)]
ROI_EEG = "occipital_parietal"
flagged_per_channel = {}

for subj in subjects_eeg:
    with h5py.File(EEG_REPS_PATH, "r") as f:
        reps = f["test"]["neural_data"][subj][ROI_EEG][:]  # (200, ch, reps, tp)
    
    n_stim, n_ch, n_reps, n_tp = reps.shape
    
    # variance z-score
    ep_var = reps.var(axis=3)  # (200, ch, reps)
    ep_var_flat = ep_var.reshape(n_stim * n_reps, n_ch)
    mu = ep_var_flat.mean(axis=0)
    sd = ep_var_flat.std(axis=0)
    ep_var_norm = (ep_var - mu[None, :, None]) / (sd[None, :, None] + 1e-8)
    
    # amplitude threshold — percentile excluding channels 13 and 53
    good_ch = [c for c in range(n_ch) if c not in [13, 53]]
    thresh_abs = np.percentile(np.abs(reps[:, good_ch, :, :]), percentile_50)
    max_abs = np.abs(reps).max(axis=3)  # (200, ch, reps)
    
    # combined mask
    mask = (ep_var_norm > THRESHOLD) | (max_abs > thresh_abs)
    
    # flagged fraction per channel
    flagged_frac = mask.mean(axis=(0, 2))  # (n_ch,)
    flagged_per_channel[subj] = flagged_frac
    print(f"{subj}: thresh_abs={thresh_abs:.1f}, "
          f"max flagged channel={flagged_frac.argmax()} "
          f"({flagged_frac.max()*100:.1f}%), "
          f"mean={flagged_frac.mean()*100:.2f}%")

In [ ]:
SUBJECT_EEG = "sub-01"
ROI_WB = "whole_brain"

with h5py.File(EEG_REPS_PATH, "r") as f:
    reps_wb = f["test"]["neural_data"][SUBJECT_EEG][ROI_WB][:]  # (200, 63, reps, tp)

n_stim, n_ch_wb, n_reps, n_tp = reps_wb.shape

# variance per (stim, ch, rep)
ep_var = reps_wb.var(axis=3)  # (200, 63, 80)

# z-score WITHIN each channel across (stim, rep) pairs
ep_var_flat = ep_var.reshape(n_stim * n_reps, n_ch_wb)  # (16000, 63)
mu = ep_var_flat.mean(axis=0)   # (63,) — per channel mean
sd = ep_var_flat.std(axis=0)    # (63,) — per channel std
ep_var_norm = (ep_var - mu[None, :, None]) / (sd[None, :, None] + 1e-8)

# amplitude threshold — per channel percentile
max_abs = np.abs(reps_wb).max(axis=3)  # (200, 63, 80)
# compute threshold per channel from its own distribution
thresh_per_channel = np.percentile(max_abs.reshape(n_stim * n_reps, n_ch_wb), 
                                    percentile_50, axis=0)  # (63,)

# combined mask — each channel uses its own threshold
mask_wb = (ep_var_norm > THRESHOLD) | (max_abs > thresh_per_channel[None, :, None])

# flagged fraction per channel
flagged_frac_wb = mask_wb.mean(axis=(0, 2))

print(f"Flagged fraction per channel (whole_brain, per-channel threshold):")
for ch in range(n_ch_wb):
    flag = " ← HIGH" if flagged_frac_wb[ch] > 0.1 else ""
    print(f"  Channel {ch:2d}: {flagged_frac_wb[ch]*100:.2f}%  "
          f"(thresh={thresh_per_channel[ch]:.1f}){flag}")

print(f"\nOverall flagged: {mask_wb.mean()*100:.2f}%")

In [ ]:
SUBJECT_EEG = "sub-10"
ROI_EEG = "occipital_parietal"

with h5py.File(EEG_REPS_PATH, "r") as f:
    reps_op = f["test"]["neural_data"][SUBJECT_EEG][ROI_EEG][:]  # (200, 17, reps, tp)

n_stim, n_ch, n_reps, n_tp = reps_op.shape

# variance per (stim, ch, rep)
ep_var = reps_op.var(axis=3)  # (200, 17, 80)

# z-score WITHIN each channel across (stim, rep) pairs
ep_var_flat = ep_var.reshape(n_stim * n_reps, n_ch)  # (16000, 17)
mu = ep_var_flat.mean(axis=0)   # (17,) — per channel mean
sd = ep_var_flat.std(axis=0)    # (17,) — per channel std
ep_var_norm = (ep_var - mu[None, :, None]) / (sd[None, :, None] + 1e-8)

# amplitude threshold — per channel percentile
max_abs = np.abs(reps_op).max(axis=3)  # (200, 17, 80)
thresh_per_channel = np.percentile(max_abs.reshape(n_stim * n_reps, n_ch),
                                    percentile_50, axis=0)  # (17,)

# combined mask
mask_op = (ep_var_norm > THRESHOLD) | (max_abs > thresh_per_channel[None, :, None])

# flagged fraction per channel
flagged_frac_op = mask_op.mean(axis=(0, 2))

print(f"Flagged fraction per channel (occipital_parietal, per-channel threshold):")
for ch in range(n_ch):
    flag = " ← HIGH" if flagged_frac_op[ch] > 0.1 else ""
    print(f"  Channel {ch:2d}: {flagged_frac_op[ch]*100:.2f}%  "
          f"(thresh={thresh_per_channel[ch]:.1f}){flag}")

print(f"\nOverall flagged: {mask_op.mean()*100:.2f}%")

In [ ]:
SUBJECT_EEG = "sub-01"
ROI_WB = "whole_brain"

with h5py.File(EEG_PATH, "r") as f:
    train_wb = f[f"train/neural_data/{SUBJECT_EEG}/{ROI_WB}"][:]  # (16540, 63, 80)

n_stim_tr, n_ch_wb, n_tp = train_wb.shape

# variance per (stim, ch)
ep_var = train_wb.var(axis=2)  # (16540, 63)

# z-score WITHIN each channel across stimuli
mu = ep_var.mean(axis=0)   # (63,)
sd = ep_var.std(axis=0)    # (63,)
ep_var_norm = (ep_var - mu[None, :]) / (sd[None, :] + 1e-8)

# amplitude threshold — per channel percentile
max_abs = np.abs(train_wb).max(axis=2)  # (16540, 63)
thresh_per_channel = np.percentile(max_abs, percentile_50, axis=0)  # (63,)

# combined mask
mask_tr = (ep_var_norm > THRESHOLD) | (max_abs > thresh_per_channel[None, :])

# flagged fraction per channel
flagged_frac_tr = mask_tr.mean(axis=0)

print(f"Flagged fraction per channel (whole_brain training, per-channel threshold):")
for ch in range(n_ch_wb):
    flag = " ← HIGH" if flagged_frac_tr[ch] > 0.1 else ""
    print(f"  Channel {ch:2d}: {flagged_frac_tr[ch]*100:.2f}%  "
          f"(thresh={thresh_per_channel[ch]:.1f}){flag}")

print(f"\nOverall flagged: {mask_tr.mean()*100:.2f}%")

In [ ]:
SUBJECT_EEG = "sub-01"
ROI_EEG     = "occipital_parietal"

with h5py.File(EEG_PATH, "r") as f:
    eeg_test_raw  = f[f"test/neural_data/{SUBJECT_EEG}/{ROI_EEG}"][:]
    eeg_test_ids  = f["test/stimulus_ids"][:]
    eeg_nc        = f[f"noise_ceilings/{SUBJECT_EEG}/{ROI_EEG}"][:]

bad_channels_op = [2]
good_channels_op = [c for c in range(eeg_test_raw.shape[1]) if c not in bad_channels_op]
eeg_test_clean = eeg_test_raw[:, good_channels_op, :]
eeg_nc_clean   = eeg_nc[good_channels_op, :]

print(f"EEG test shape after excluding channel 2: {eeg_test_clean.shape}")
print(f"EEG noise ceiling shape: {eeg_nc_clean.shape}")
print(f"EEG test IDs shape: {eeg_test_ids.shape}")

In [ ]:
feat_idx_eeg_A = np.array([id_to_idx_A[x] for x in eeg_test_ids])
feat_idx_eeg_B = np.array([id_to_idx_B[x] for x in eeg_test_ids])

print(f"Matched {len(feat_idx_eeg_A)} stimuli for Model A")
print(f"Matched {len(feat_idx_eeg_B)} stimuli for Model B")

sort_A = np.argsort(feat_idx_eeg_A)
unsort_A = np.argsort(sort_A)
sort_B = np.argsort(feat_idx_eeg_B)
unsort_B = np.argsort(sort_B)

eeg_feats_A = {}
eeg_feats_B = {}

with h5py.File(FEAT_A_THINGS, "r") as f:
    for layer in layers_A:
        eeg_feats_A[layer] = f[f"features/{layer}"][feat_idx_eeg_A[sort_A]][unsort_A]

with h5py.File(FEAT_B_THINGS, "r") as f:
    for layer in layers_B:
        eeg_feats_B[layer] = f[f"features/{layer}"][feat_idx_eeg_B[sort_B]][unsort_B]

print("Features loaded.")

In [ ]:
import importlib
import utils
importlib.reload(utils)
from utils import safe_normalize
from scipy.spatial.distance import pdist
from scipy.stats import spearmanr

print("Precomputing model RDMs and normalized Gram matrices...")
model_rdms_A = {}
model_K_A = {}
for layer in layers_A:
    feats = safe_normalize(eeg_feats_A[layer])
    model_rdms_A[layer] = pdist(feats, metric="correlation")
    K = feats @ feats.T
    np.fill_diagonal(K, 0)
    model_K_A[layer] = K
    print(f"  ResNet152 {layer} done")

model_rdms_B = {}
model_K_B = {}
for layer in layers_B:
    feats = safe_normalize(eeg_feats_B[layer])
    model_rdms_B[layer] = pdist(feats, metric="correlation")
    K = feats @ feats.T
    np.fill_diagonal(K, 0)
    model_K_B[layer] = K
    print(f"  Qwen3-VL {layer} done")

print("All model RDMs and Gram matrices precomputed.")

In [ ]:
times = np.linspace(0.0, 0.8, 80)
results_eeg_clean = []

print("Computing time-resolved RSA and CKA (cleaned)...")
for model_name, layers, model_rdms, model_K in [
    ("ResNet152", layers_A, model_rdms_A, model_K_A),
    ("Qwen3-VL",  layers_B, model_rdms_B, model_K_B),
]:
    for layer in layers:
        rdm_model = model_rdms[layer]
        K = model_K[layer]

        for t_idx in range(80):
            neural_t = eeg_test_clean[:, :, t_idx].astype(np.float64)

            # RSA
            rdm_neural = pdist(neural_t, metric="correlation")
            rsa_score, _ = spearmanr(rdm_model, rdm_neural)

            # CKA
            neural_norm = safe_normalize(neural_t)
            L = neural_norm @ neural_norm.T
            np.fill_diagonal(L, 0)
            n = 200
            KL = K @ L
            hsic_kl = (np.trace(KL) + K.sum() * L.sum() / ((n-1)*(n-2))
                       - 2 * KL.sum() / (n-2)) / (n * (n-3))
            hsic_kk = (np.trace(K @ K) + K.sum()**2 / ((n-1)*(n-2))
                       - 2 * (K @ K).sum() / (n-2)) / (n * (n-3))
            hsic_ll = (np.trace(L @ L) + L.sum()**2 / ((n-1)*(n-2))
                       - 2 * (L @ L).sum() / (n-2)) / (n * (n-3))
            denom = np.sqrt(max(hsic_kk, 0) * max(hsic_ll, 0)) + 1e-8
            cka_score = hsic_kl / denom

            results_eeg_clean.append(dict(
                model=model_name,
                layer=layer,
                time=times[t_idx],
                rsa=float(rsa_score),
                cka=float(cka_score),
            ))

        print(f"  {model_name} {layer} done")

df_eeg_clean = pd.DataFrame(results_eeg_clean)
print(f"\nDone. Shape: {df_eeg_clean.shape}")

df_eeg_clean.to_pickle("results/df_eeg_rsa_cka_clean.pkl")
print("Saved to results/df_eeg_rsa_cka_clean.pkl")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for col, model_name in enumerate(["ResNet152", "Qwen3-VL"]):
    df_model = df_eeg_clean[df_eeg_clean["model"] == model_name]
    
    for row, metric in enumerate(["rsa", "cka"]):
        ax = axes[row, col]
        
        # Build heatmap matrix: layers x time
        if model_name == "ResNet152":
            layer_order = layer_order_A
        else:
            layer_order = layer_order_B
        
        heatmap = np.zeros((len(layer_order), 80))
        for i, layer in enumerate(layer_order):
            df_layer = df_model[df_model["layer"] == layer].sort_values("time")
            heatmap[i] = df_layer[metric].values
        
        im = ax.imshow(heatmap, aspect="auto", origin="lower",
                       extent=[0, 0.8, -0.5, len(layer_order) - 0.5],
                       cmap="RdBu_r" if metric == "rsa" else "viridis",
                       vmin=-heatmap.max() if metric == "rsa" else 0,
                       vmax=heatmap.max())
        ax.set_yticks(range(len(layer_order)))
        ax.set_yticklabels(layer_order, fontsize=8)
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Layer")
        ax.set_title(f"{model_name} — {metric.upper()} — EEG occipital_parietal")
        ax.axvline(0.0, color="k", lw=1, ls="--")
        fig.colorbar(im, ax=ax, shrink=0.85)

fig.suptitle(f"Time-resolved RSA and CKA — {SUBJECT_EEG} / {ROI_EEG} (channel 2 excluded)", fontsize=13)
fig.tight_layout()
plt.savefig("images/fig_eeg_rsa_cka_time_clean.png", dpi=150, bbox_inches="tight")
plt.show()

![Alt text](images/fig_eeg_rsa_cka_time_clean.png)

In [ ]:
import matplotlib.colors as mcolors

np.random.seed(42)
n_show = 5

with h5py.File(EEG_REPS_PATH, "r") as f:
    reps_op = f["test"]["neural_data"][SUBJECT_EEG][ROI_EEG][:]  # (200, 17, reps, tp)

n_stim, n_ch, n_reps, n_tp = reps_op.shape
times_rep = np.linspace(0, 0.8, n_tp)

# compute mask for channel 2
ep_var = reps_op.var(axis=3)  # (200, 17, 80)
ep_var_flat = ep_var.reshape(n_stim * n_reps, n_ch)
mu = ep_var_flat.mean(axis=0)
sd = ep_var_flat.std(axis=0)
ep_var_norm = (ep_var - mu[None, :, None]) / (sd[None, :, None] + 1e-8)
max_abs = np.abs(reps_op).max(axis=3)
thresh_per_channel = np.percentile(max_abs.reshape(n_stim * n_reps, n_ch),
                                    percentile_50, axis=0)
mask_op = (ep_var_norm > THRESHOLD) | (max_abs > thresh_per_channel[None, :, None])

ch = 2
stim_idx = np.random.choice(n_stim, n_show, replace=False)
rep_idx  = np.random.choice(n_reps, n_show, replace=False)

fig, axes = plt.subplots(n_show, 1, figsize=(12, 10), sharex=True)

for i, (s, r) in enumerate(zip(stim_idx, rep_idx)):
    ax = axes[i]
    flagged = mask_op[s, ch, r]
    color = "red" if flagged else "steelblue"
    label = f"stim {s}, rep {r} — {'FLAGGED' if flagged else 'clean'}"
    ax.plot(times_rep, reps_op[s, ch, r, :], color=color, lw=1.5, label=label)
    ax.legend(fontsize=8, loc="upper right")
    ax.set_ylabel("Amplitude")
    if flagged:
        ax.set_facecolor("#fff0f0")

axes[-1].set_xlabel("Time (s)")
fig.suptitle(f"Channel 2 time courses — {SUBJECT_EEG} / {ROI_EEG}\n"
             f"red = flagged, blue = clean", fontsize=12)
fig.tight_layout()
plt.savefig("images/fig_ch2_timecourses.png", dpi=150, bbox_inches="tight")
plt.show()

![Alt text](images/fig_ch2_timecourses.png)

#### Motivation
Unlike TVSD and NSD, EEG gives us temporal resolution: we can ask not just which layer 
aligns best with the brain, but *when* that alignment peaks. This is meaningful because 
the ventral stream processes information sequentially, with known latencies for each 
area: V1 responds at 40–60ms, V4 at 60–80ms, and IT at 80–100ms after stimulus onset 
<a href="#kravitz2013" title="Kravitz et al. (2013) — The ventral visual pathway: an 
expanded neural framework for the processing of object quality">[11]</a>. If model 
layers capture brain-like representations, their alignment with EEG should peak at 
latencies consistent with those areas.

<div style="display: flex; gap: 10px; align-items: flex-start; margin-top: 15px;">
  <figure style="text-align: center; flex: 1;">
    <img src="images/kravitz_latencies.jpg" style="width: 60%;"/>
    <figcaption>Known response latencies along the macaque ventral stream 
    <a href="#kravitz2013" title="Kravitz et al. (2013) — The ventral visual pathway: 
    an expanded neural framework for the processing of object quality">[11]</a>. 
    These latencies motivate our interpretation of the temporal RSA and CKA peaks 
    below.</figcaption>
  </figure>
</div>

We start with sub-01 on the occipital_parietal ROI without any artifact correction, to 
establish a baseline before applying the full cleaning pipeline. We flag upfront that 
channel 2 in sub-01 is corrupted (identified in Section 1.2), and several features 
visible here weaken or disappear once it is removed and once we average across all 10 
subjects. The interpretations below are therefore observations on this single uncleaned 
subject, not general conclusions. Those are drawn from the group-level cleaned results 
further below.

#### Results

**ResNet152 RSA**

Alignment becomes positive starting around 50ms and peaks around 60–100ms for the early 
layers (layer1-0 and layer2-0), consistent with the V1 latency range from Kravitz et 
al. <a href="#kravitz2013" title="Kravitz et al. (2013) — The ventral visual pathway: 
an expanded neural framework for the processing of object quality">[11]</a>. However, 
we note that this early peak is partially driven by the corrupted channel 2. It 
substantially weakens once that channel is excluded, as shown in the next section. The 
layer1-0 negative patch around 100ms is also artifact-driven and disappears after 
cleaning (forward reference: see group-level results below).

Deeper layers (layer3-0 and layer4-1) peak slightly later around 100ms, consistent with 
V4 and IT latencies. Around 190ms a second positive peak is visible across several 
layers. Given that the second RSVP image appears at 200ms and feedforward latencies are 
around 60ms, this is slightly early for a feedforward response and may reflect 
anticipatory activity. This feature varies across subjects in the cleaned group analysis, 
so we treat it as tentative. A third weaker period around 300–330ms may reflect 
recurrent processing, though we make no strong claims here. After 400ms alignment drops, 
consistent with the third image appearing and overlapping responses making it hard to 
attribute signal to any single stimulus.

**Qwen3-VL RSA**

The early visual blocks (visual-blocks-2 and visual-blocks-6) show a similar temporal 
structure to the early ResNet layers, with alignment starting around 50ms and a weak 
anticipatory peak around 190ms. The middle visual blocks show low alignment throughout, 
consistent with the TVSD finding. The language model layers show the strongest RSA in 
this model, with two distinct peaks around 90ms and 140ms, suggesting they capture 
higher-order visual structure present in the EEG signal. The anticipatory and recurrent 
features visible in ResNet152 are much weaker in Qwen3-VL.

**ResNet152 CKA**

CKA tells a strikingly different story. Alignment is highly localized at layer3-0 and 
layer3-5, concentrated in a narrow 100–150ms window. Outside this region CKA is close 
to zero throughout. There is no temporal layer hierarchy (early layers do not peak 
earlier than deep layers) which directly contradicts the RSA picture. From layer3-0 
onwards CKA decreases monotonically with depth.

**Qwen3-VL CKA**

Two regions of elevated CKA are visible: the language model layers peak around 90–100ms, 
and the early visual blocks peak slightly later at 100–120ms, both at roughly half the 
values ResNet152 achieves. The middle visual blocks remain near zero, again consistent 
with TVSD.

**RSA vs CKA**

The most important observation is that RSA suggests a temporal layer hierarchy (early 
layers peak early, deeper layers peak later) while CKA does not. RSA only requires the 
rank ordering of pairwise stimulus distances to be preserved, while CKA requires the 
actual inner-product geometry to match. The temporal hierarchy visible in RSA may 
reflect genuine feedforward dynamics, or it may reflect that RSA is more sensitive to 
weak monotonic relationships that do not represent genuine geometric similarity. The 
group-level cleaned results below are the basis for drawing firmer conclusions.

<div style="display: flex; gap: 10px; align-items: flex-start; margin-top: 15px;">
  <figure style="text-align: center; flex: 1;">
    <img src="images/fig_eeg_rsa_cka_time.png" style="width: 100%;"/>
    <figcaption>Time-resolved RSA and CKA — sub-01 / occipital_parietal, all 17 
    channels, no artifact correction. Rows: RSA (top) and CKA (bottom). Columns: 
    ResNet152 (left) and Qwen3-VL (right).</figcaption>
  </figure>
</div>

**Key takeaways**
- Early RSA peaks around 50–100ms are consistent with known V1/V4 latencies but are 
  partially artifact-driven in this single-subject uncleaned analysis
- CKA shows no temporal layer hierarchy, concentrating alignment at layer3-0/layer3-5 
  around 100–150ms regardless of depth
- Qwen3-VL's language model layers show the strongest RSA in that model, consistent 
  with the TVSD finding
- All findings here should be treated as observations on a single uncleaned subject; 
  the group-level cleaned results are the basis for conclusions

Having flagged channel 2 as corrupted (40.81% of epochs flagged under its own amplitude 
threshold of 2285), we now show the effect of removing it directly.

#### Results

Excluding channel 2 removes the spurious early RSA patch for layer1-0 and layer2-0 in 
the 0–70ms window, confirming it was artifact-driven rather than reflecting genuine 
early visual processing. The layer1-0 negative patch around 100ms also substantially 
weakens. Outside the 0–70ms window the two heatmaps are broadly consistent, confirming 
that the main temporal features survive artifact removal. CKA is unchanged by the 
exclusion, reinforcing that it is more robust to structured noise than RSA.

<div style="display: flex; gap: 10px; align-items: flex-start; margin-top: 15px;">
  <figure style="text-align: center; flex: 1;">
    <img src="images/fig_eeg_rsa_cka_time_clean.png" style="width: 100%;"/>
    <figcaption>Time-resolved RSA and CKA — sub-01 / occipital_parietal, 16 channels 
    (channel 2 excluded).</figcaption>
  </figure>
</div>

**Key takeaways**
- Removing channel 2 eliminates the spurious 0–70ms RSA signal, confirming it was 
  artifact-driven
- CKA is unaffected, consistent with its greater robustness to structured noise
- The main temporal features (peak around 100ms, language model layer advantage for 
  Qwen3-VL) survive the exclusion

In [ ]:
SUBJECT_EEG_2 = "sub-02"
ROI_EEG = "occipital_parietal"

with h5py.File(EEG_PATH, "r") as f:
    eeg_test_2      = f[f"test/neural_data/{SUBJECT_EEG_2}/{ROI_EEG}"][:]
    eeg_test_ids_2  = f["test/stimulus_ids"][:]
    eeg_nc_2        = f[f"noise_ceilings/{SUBJECT_EEG_2}/{ROI_EEG}"][:]

print(f"EEG test shape: {eeg_test_2.shape}")
print(f"EEG noise ceiling shape: {eeg_nc_2.shape}")
print(f"EEG test IDs shape: {eeg_test_ids_2.shape}")

In [ ]:
feat_idx_eeg_A_2 = np.array([id_to_idx_A[x] for x in eeg_test_ids_2])
feat_idx_eeg_B_2 = np.array([id_to_idx_B[x] for x in eeg_test_ids_2])

print(f"Matched {len(feat_idx_eeg_A_2)} stimuli for Model A")
print(f"Matched {len(feat_idx_eeg_B_2)} stimuli for Model B")

sort_A_2 = np.argsort(feat_idx_eeg_A_2)
unsort_A_2 = np.argsort(sort_A_2)
sort_B_2 = np.argsort(feat_idx_eeg_B_2)
unsort_B_2 = np.argsort(sort_B_2)

eeg_feats_A_2 = {}
eeg_feats_B_2 = {}

with h5py.File(FEAT_A_THINGS, "r") as f:
    for layer in layers_A:
        eeg_feats_A_2[layer] = f[f"features/{layer}"][feat_idx_eeg_A_2[sort_A_2]][unsort_A_2]

with h5py.File(FEAT_B_THINGS, "r") as f:
    for layer in layers_B:
        eeg_feats_B_2[layer] = f[f"features/{layer}"][feat_idx_eeg_B_2[sort_B_2]][unsort_B_2]

print("Features loaded.")

In [ ]:
print("Precomputing model RDMs and normalized Gram matrices for sub-02...")
model_rdms_A_2 = {}
model_K_A_2 = {}
for layer in layers_A:
    feats = safe_normalize(eeg_feats_A_2[layer])
    model_rdms_A_2[layer] = pdist(feats, metric="correlation")
    K = feats @ feats.T
    np.fill_diagonal(K, 0)
    model_K_A_2[layer] = K
    print(f"  ResNet152 {layer} done")

model_rdms_B_2 = {}
model_K_B_2 = {}
for layer in layers_B:
    feats = safe_normalize(eeg_feats_B_2[layer])
    model_rdms_B_2[layer] = pdist(feats, metric="correlation")
    K = feats @ feats.T
    np.fill_diagonal(K, 0)
    model_K_B_2[layer] = K
    print(f"  Qwen3-VL {layer} done")

print("All model RDMs and Gram matrices precomputed.")

In [ ]:
times = np.linspace(0.0, 0.8, 80)
results_eeg_2 = []

print("Computing time-resolved RSA and CKA for sub-02...")
for model_name, layers, model_rdms, model_K in [
    ("ResNet152", layers_A, model_rdms_A_2, model_K_A_2),
    ("Qwen3-VL",  layers_B, model_rdms_B_2, model_K_B_2),
]:
    for layer in layers:
        rdm_model = model_rdms[layer]
        K = model_K[layer]

        for t_idx in range(80):
            neural_t = eeg_test_2[:, :, t_idx].astype(np.float64)

            # RSA
            rdm_neural = pdist(neural_t, metric="correlation")
            rsa_score, _ = spearmanr(rdm_model, rdm_neural)

            # CKA
            neural_norm = safe_normalize(neural_t)
            L = neural_norm @ neural_norm.T
            np.fill_diagonal(L, 0)
            n = 200
            KL = K @ L
            hsic_kl = (np.trace(KL) + K.sum() * L.sum() / ((n-1)*(n-2))
                       - 2 * KL.sum() / (n-2)) / (n * (n-3))
            hsic_kk = (np.trace(K @ K) + K.sum()**2 / ((n-1)*(n-2))
                       - 2 * (K @ K).sum() / (n-2)) / (n * (n-3))
            hsic_ll = (np.trace(L @ L) + L.sum()**2 / ((n-1)*(n-2))
                       - 2 * (L @ L).sum() / (n-2)) / (n * (n-3))
            denom = np.sqrt(max(hsic_kk, 0) * max(hsic_ll, 0)) + 1e-8
            cka_score = hsic_kl / denom

            results_eeg_2.append(dict(
                model=model_name,
                layer=layer,
                time=times[t_idx],
                rsa=float(rsa_score),
                cka=float(cka_score),
            ))

        print(f"  {model_name} {layer} done")

df_eeg_2 = pd.DataFrame(results_eeg_2)
print(f"\nDone. Shape: {df_eeg_2.shape}")

df_eeg_2.to_pickle("results/df_eeg_rsa_cka_sub02.pkl")
print("Saved to results/df_eeg_rsa_cka_sub02.pkl")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for col, model_name in enumerate(["ResNet152", "Qwen3-VL"]):
    df_model = df_eeg_2[df_eeg_2["model"] == model_name]
    
    for row, metric in enumerate(["rsa", "cka"]):
        ax = axes[row, col]
        
        if model_name == "ResNet152":
            layer_order = layer_order_A
        else:
            layer_order = layer_order_B
        
        heatmap = np.zeros((len(layer_order), 80))
        for i, layer in enumerate(layer_order):
            df_layer = df_model[df_model["layer"] == layer].sort_values("time")
            heatmap[i] = df_layer[metric].values
        
        im = ax.imshow(heatmap, aspect="auto", origin="lower",
                       extent=[0, 0.8, -0.5, len(layer_order) - 0.5],
                       cmap="RdBu_r" if metric == "rsa" else "viridis",
                       vmin=-heatmap.max() if metric == "rsa" else 0,
                       vmax=heatmap.max())
        ax.set_yticks(range(len(layer_order)))
        ax.set_yticklabels(layer_order, fontsize=8)
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Layer")
        ax.set_title(f"{model_name} — {metric.upper()} — EEG {ROI_EEG} (sub-02)")
        ax.axvline(0.0, color="k", lw=1, ls="--")
        fig.colorbar(im, ax=ax, shrink=0.85)

fig.suptitle(f"Time-resolved RSA and CKA — {SUBJECT_EEG_2} / {ROI_EEG}",
             fontsize=13)
fig.tight_layout()
plt.savefig("images/fig_eeg_rsa_cka_time_sub02.png", dpi=150, bbox_inches="tight")
plt.show()

#### Motivation
Sub-01 had a clearly corrupted channel that complicated interpretation. Sub-02 has no 
such extreme anomaly, making it a useful sanity check: if the main temporal features 
from sub-01 replicate in a cleaner subject, they are more likely to reflect genuine 
signal rather than artifact.

#### Results

The overall pattern is broadly consistent with sub-01. RSA peaks around 100ms for most 
layers, the CKA peak is again concentrated at layer3-0 and layer3-5, and the layer1-0 
negative RSA patch is present at a similar latency. Sub-02 shows an early RSA peak for 
the first two layers around 70–80ms. Given that the equivalent peak in sub-01 turned 
out to be largely artifact-driven, we are cautious about interpreting this as genuine 
early visual processing as it may reflect a different artifact or simply noise in this 
subject. RSA values are higher in magnitude for sub-02, reaching above 0.2 compared to 
around 0.1 for sub-01, suggesting sub-02 has stronger or more reliable stimulus-driven 
responses in this ROI. For Qwen3-VL, the language model layers show more sustained 
alignment for sub-02, with RSA extending well beyond 200ms and CKA remaining elevated 
up to around 400ms which is less prominent in sub-01. Whether this reflects a genuine subject 
difference or recording quality is unclear from two subjects alone, which is exactly 
what motivates the full multi-subject analysis with artifact correction that follows.

<div style="display: flex; gap: 10px; align-items: flex-start; margin-top: 15px;">
  <figure style="text-align: center; flex: 1;">
    <img src="images/fig_eeg_rsa_cka_time_sub02.png" style="width: 100%;"/>
    <figcaption>Time-resolved RSA and CKA — sub-02 / occipital_parietal, no artifact 
    correction applied.</figcaption>
  </figure>
</div>

**Key takeaways**
- Main temporal features from sub-01 replicate in sub-02, supporting their genuineness
- The early RSA peak in sub-02 should not be interpreted as genuine early visual 
  processing given that the equivalent feature in sub-01 was artifact-driven
- Qwen3-VL's language model layers show more sustained alignment in sub-02, but 
  whether this is subject-specific or general requires the group analysis

In [ ]:
with h5py.File(NSD_PATH, "r") as f:
    f.visititems(lambda name, obj: print(name, "→", obj.shape if hasattr(obj, "shape") else "group"))

In [ ]:
with h5py.File(NSD_PATH, "r") as f:
    f.visititems(lambda name, obj: 
        print(name, "→", obj.shape if hasattr(obj, "shape") else "group")
        if "test" in name.lower() else None)

In [ ]:
with h5py.File(NSD_PATH, "r") as f:
    f.visititems(lambda name, obj: 
        print(name, "→", obj.shape if hasattr(obj, "shape") else "group")
        if "stimulus" in name.lower() or "ids" in name.lower() else None)


In [ ]:
SUBJECT_NSD = "subj01"
NSD_ROIS = ["V1v", "V2v", "V3v", "hV4", "FFA-1", "VWFA-1", "PPA", "OPA", "EBA"]

nsd_test = {}
nsd_nc   = {}

with h5py.File(NSD_PATH, "r") as f:
    nsd_test_ids = f[f"test/stimulus_ids/{SUBJECT_NSD}"][:]
    for roi in NSD_ROIS:
        nsd_test[roi] = f[f"test/neural_data/{SUBJECT_NSD}/{roi}"][:]
        nsd_nc[roi]   = f[f"noise_ceilings/{SUBJECT_NSD}/{roi}"][:]
        print(f"{roi}: responses {nsd_test[roi].shape}, nc {nsd_nc[roi].shape}")

print(f"\nTest stimulus IDs shape: {nsd_test_ids.shape}")
print(f"Example IDs: {nsd_test_ids[:3]}")

In [ ]:
# Load NSD feature IDs
with h5py.File(FEAT_A_NSD, "r") as f:
    feat_ids_A_nsd = f["ids"][:]
    print(f"Model A NSD feature IDs shape: {feat_ids_A_nsd.shape}")
    print(f"Example IDs: {feat_ids_A_nsd[:3]}")

with h5py.File(FEAT_B_NSD, "r") as f:
    feat_ids_B_nsd = f["ids"][:]
    print(f"Model B NSD feature IDs shape: {feat_ids_B_nsd.shape}")

In [ ]:
# Build index maps for NSD features
id_to_idx_A_nsd = {id_: i for i, id_ in enumerate(feat_ids_A_nsd)}
id_to_idx_B_nsd = {id_: i for i, id_ in enumerate(feat_ids_B_nsd)}

# Get feature row indices for NSD test stimuli
feat_idx_nsd_A = np.array([id_to_idx_A_nsd[x] for x in nsd_test_ids])
feat_idx_nsd_B = np.array([id_to_idx_B_nsd[x] for x in nsd_test_ids])

print(f"Feature indices shape: {feat_idx_nsd_A.shape}")
print(f"First 3 indices: {feat_idx_nsd_A[:3]}")

# Load features for matched stimuli
sort_A_nsd = np.argsort(feat_idx_nsd_A)
unsort_A_nsd = np.argsort(sort_A_nsd)
sort_B_nsd = np.argsort(feat_idx_nsd_B)
unsort_B_nsd = np.argsort(sort_B_nsd)

nsd_feats_A = {}
nsd_feats_B = {}

with h5py.File(FEAT_A_NSD, "r") as f:
    for layer in layers_A:
        nsd_feats_A[layer] = f[f"features/{layer}"][feat_idx_nsd_A[sort_A_nsd]][unsort_A_nsd]
        print(f"  Loaded ResNet152 {layer}: {nsd_feats_A[layer].shape}")

with h5py.File(FEAT_B_NSD, "r") as f:
    for layer in layers_B:
        nsd_feats_B[layer] = f[f"features/{layer}"][feat_idx_nsd_B[sort_B_nsd]][unsort_B_nsd]
        print(f"  Loaded Qwen3-VL {layer}: {nsd_feats_B[layer].shape}")

print("Features loaded.")

In [ ]:
# Precompute model RDMs for NSD — expensive step done once per layer
from scipy.spatial.distance import pdist
from scipy.stats import spearmanr

print("Precomputing model RDMs for NSD...")
nsd_model_rdms_A = {}
nsd_model_K_A = {}
for layer in layers_A:
    feats = safe_normalize(nsd_feats_A[layer])
    nsd_model_rdms_A[layer] = pdist(feats, metric="correlation")
    K = feats @ feats.T
    np.fill_diagonal(K, 0)
    nsd_model_K_A[layer] = K
    print(f"  ResNet152 {layer} done")

nsd_model_rdms_B = {}
nsd_model_K_B = {}
for layer in layers_B:
    feats = safe_normalize(nsd_feats_B[layer])
    nsd_model_rdms_B[layer] = pdist(feats, metric="correlation")
    K = feats @ feats.T
    np.fill_diagonal(K, 0)
    nsd_model_K_B[layer] = K
    print(f"  Qwen3-VL {layer} done")

print("All NSD model RDMs precomputed.")

# Compute RSA and CKA for each layer x ROI combination
results_nsd = []
n = 1000

print("Computing RSA and CKA...")
for model_name, layers, model_rdms, model_K in [
    ("ResNet152", layers_A, nsd_model_rdms_A, nsd_model_K_A),
    ("Qwen3-VL",  layers_B, nsd_model_rdms_B, nsd_model_K_B),
]:
    for layer in layers:
        rdm_model = model_rdms[layer]
        K = model_K[layer]

        for roi in NSD_ROIS:
            neural = nsd_test[roi].astype(np.float64)  # (1000, n_voxels)

            # RSA
            rdm_neural = pdist(neural, metric="correlation")
            rsa_score, _ = spearmanr(rdm_model, rdm_neural)

            # CKA
            neural_norm = safe_normalize(neural)
            L = neural_norm @ neural_norm.T
            np.fill_diagonal(L, 0)
            KL = K @ L
            hsic_kl = (np.trace(KL) + K.sum() * L.sum() / ((n-1)*(n-2))
                       - 2 * KL.sum() / (n-2)) / (n * (n-3))
            hsic_kk = (np.trace(K @ K) + K.sum()**2 / ((n-1)*(n-2))
                       - 2 * (K @ K).sum() / (n-2)) / (n * (n-3))
            hsic_ll = (np.trace(L @ L) + L.sum()**2 / ((n-1)*(n-2))
                       - 2 * (L @ L).sum() / (n-2)) / (n * (n-3))
            denom = np.sqrt(max(hsic_kk, 0) * max(hsic_ll, 0)) + 1e-8
            cka_score = hsic_kl / denom

            results_nsd.append(dict(
                model=model_name,
                layer=layer,
                roi=roi,
                rsa=float(rsa_score),
                cka=float(cka_score),
            ))

        print(f"  {model_name} {layer} done")

df_nsd = pd.DataFrame(results_nsd)
print(f"\nDone. Shape: {df_nsd.shape}")

df_nsd.to_pickle("results/df_nsd_rsa_cka.pkl")
print("Saved to results/df_nsd_rsa_cka.pkl")

In [ ]:
print(df_nsd.shape)
print(df_nsd.head(10).to_string())
print(f"\nRSA range: {df_nsd['rsa'].min():.4f} – {df_nsd['rsa'].max():.4f}")
print(f"CKA range: {df_nsd['cka'].min():.4f} – {df_nsd['cka'].max():.4f}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

roi_colors = {
    "V1v": "steelblue", "V2v": "dodgerblue", "V3v": "deepskyblue",
    "hV4": "forestgreen", "FFA-1": "darkorange", "VWFA-1": "gold",
    "PPA": "red", "OPA": "purple", "EBA": "brown"
}

for col, model_name in enumerate(["ResNet152", "Qwen3-VL"]):
    df_model = df_nsd[df_nsd["model"] == model_name]

    if model_name == "ResNet152":
        layer_order = layer_order_A
    else:
        layer_order = layer_order_B

    for row, metric in enumerate(["rsa", "cka"]):
        ax = axes[row, col]

        for roi in NSD_ROIS:
            df_roi = df_model[df_model["roi"] == roi].copy()
            df_roi["layer"] = pd.Categorical(df_roi["layer"],
                                              categories=layer_order, ordered=True)
            df_roi = df_roi.sort_values("layer")
            ax.plot(range(len(layer_order)), df_roi[metric].values,
                    marker="o", lw=2, label=roi, color=roi_colors[roi])

        ax.set_xticks(range(len(layer_order)))
        ax.set_xticklabels(layer_order, rotation=45, ha="right", fontsize=8)
        ax.set_xlabel("Layer")
        ax.set_ylabel(metric.upper())
        ax.set_title(f"{model_name} — {metric.upper()} — NSD")
        ax.legend(fontsize=7, ncol=2)
        ax.axhline(0, color="k", lw=0.5, ls="--")

fig.suptitle(f"Layer-wise RSA and CKA — NSD ({SUBJECT_NSD})", fontsize=13)
fig.tight_layout()
plt.savefig("images/fig_nsd_rsa_cka_layers.png", dpi=150, bbox_inches="tight")
plt.show()

#### Motivation
NSD gives us the spatially cleanest test of the layer-to-hierarchy correspondence. 
Unlike TVSD (two monkeys, three broad areas) and EEG (aggregated scalp channels), NSD 
provides fMRI responses from functionally defined human visual ROIs. We restrict the 
analysis to nine required ROIs spanning the full ventral hierarchy: early retinotopic 
areas (V1v, V2v, V3v, hV4) and category-selective regions (FFA-1, VWFA-1, PPA, OPA, 
EBA). This selection is deliberate as the retinotopic areas test whether early model 
layers capture low-level visual structure, while the category-selective regions test 
whether deeper layers capture higher-order object and scene properties, making this the 
most diagnostic comparison between the two models. We start with a single subject 
(subj01) to allow direct inspection of the layer-wise curves; the group average across 
all 8 subjects follows to assess replicability.

#### Results

**ResNet152 RSA** shows a remarkably clean hierarchy. V1v peaks around layer3-0 at 
roughly 0.28 and then declines steadily, while EBA, PPA and OPA start low and increase 
monotonically with depth, with EBA reaching around 0.29 at layer4-1 and overtaking V1v 
around layer3-5. V2v and V3v track V1v closely but slightly lower, while FFA-1, VWFA-1 
and hV4 sit in the middle throughout. This ordering maps cleanly onto the known 
functional hierarchy of human visual cortex.

**ResNet152 CKA** tells a complementary story. V1v starts dramatically higher than all 
other ROIs (around 0.31 at layer1-0, peaking at 0.36 at layer3-0), then drops sharply 
as depth increases. By layer3-10 all ROIs converge to similar values around 0.20–0.25. 
CKA captures the early V1v dominance very strongly but is less sensitive to the finer 
hierarchy among higher-level areas, where most ROIs bunch together in the deeper layers.

**Qwen3-VL** shows the same pattern as on TVSD. V1v starts highest and drops after 
visual-blocks-6, the middle visual blocks show a dip for all ROIs, and the first 
language model layer triggers a strong jump in both RSA and CKA for EBA, PPA and OPA, 
while V1v barely recovers. The CKA jump is particularly striking: EBA, PPA and OPA 
reach 0.24–0.25 while V1v stays below 0.13. This contrast is more pronounced here than 
in TVSD, likely because the NSD ROIs are spatially specific and free from the signal 
mixing that affects scalp EEG. It is consistent with the interpretation that Qwen's 
language model layers reorganize visual representations to capture higher-order object 
and scene properties, without affecting low-level structure.

Compared to ResNet152, Qwen3-VL achieves lower peak alignment for early visual areas 
but comparable or higher alignment for higher-level areas in the language model layers, 
suggesting the two models capture complementary aspects of the visual hierarchy rather 
than one being uniformly better.

<div style="display: flex; gap: 10px; align-items: flex-start; margin-top: 15px;">
  <figure style="text-align: center; flex: 1;">
    <img src="images/fig_nsd_rsa_cka_layers.png" style="width: 100%;"/>
    <figcaption>Layer-wise RSA and CKA for both models on NSD (subj01), restricted to 
    nine ROIs spanning the ventral hierarchy. Each line corresponds to one ROI.</figcaption>
  </figure>
</div>

**Key takeaways**
- ResNet152 cleanly recovers the full retinotopic-to-categorical hierarchy on NSD, 
  with V1v dominating early layers and EBA/PPA/OPA dominating deep layers
- CKA captures the sharp early V1v dominance but loses sensitivity to the finer 
  hierarchy among higher-level areas, where RSA is more informative
- Qwen3-VL's language model layers produce a striking jump in alignment for 
  higher-level areas, more pronounced here than on TVSD due to NSD's spatial specificity
- These are single-subject results; the group average across 8 subjects follows

In [ ]:
NSD_SUBJECTS = ["subj01", "subj02", "subj03", "subj04", 
                "subj05", "subj06", "subj07", "subj08"]
NSD_ROIS = ["V1v", "V2v", "V3v", "hV4", "FFA-1", "VWFA-1", "PPA", "OPA", "EBA"]

nsd_test_all = {}   # {subj: {roi: array}}
nsd_ids_all  = {}   # {subj: array}
nsd_nc_all   = {}   # {subj: {roi: array}}

with h5py.File(NSD_PATH, "r") as f:
    for subj in NSD_SUBJECTS:
        nsd_ids_all[subj] = f[f"test/stimulus_ids/{subj}"][:]
        nsd_test_all[subj] = {}
        nsd_nc_all[subj]   = {}
        for roi in NSD_ROIS:
            nsd_test_all[subj][roi] = f[f"test/neural_data/{subj}/{roi}"][:]
            nsd_nc_all[subj][roi]   = f[f"noise_ceilings/{subj}/{roi}"][:]
        print(f"{subj}: {len(nsd_ids_all[subj])} stimuli")

In [ ]:
results_nsd_all = []

for subj in NSD_SUBJECTS:
    print(f"\nProcessing {subj}...")
    
    # Match stimulus IDs to feature rows
    feat_idx_A = np.array([id_to_idx_A_nsd[x] for x in nsd_ids_all[subj]])
    feat_idx_B = np.array([id_to_idx_B_nsd[x] for x in nsd_ids_all[subj]])
    
    sort_A = np.argsort(feat_idx_A)
    unsort_A = np.argsort(sort_A)
    sort_B = np.argsort(feat_idx_B)
    unsort_B = np.argsort(sort_B)
    
    n = len(nsd_ids_all[subj])
    
    # Precompute model RDMs and Gram matrices for this subject
    subj_rdms_A = {}
    subj_K_A    = {}
    subj_rdms_B = {}
    subj_K_B    = {}
    
    with h5py.File(FEAT_A_NSD, "r") as f:
        for layer in layers_A:
            feats = safe_normalize(
                f[f"features/{layer}"][feat_idx_A[sort_A]][unsort_A].astype(np.float64))
            subj_rdms_A[layer] = pdist(feats, metric="correlation")
            K = feats @ feats.T
            np.fill_diagonal(K, 0)
            subj_K_A[layer] = K

    with h5py.File(FEAT_B_NSD, "r") as f:
        for layer in layers_B:
            feats = safe_normalize(
                f[f"features/{layer}"][feat_idx_B[sort_B]][unsort_B].astype(np.float64))
            subj_rdms_B[layer] = pdist(feats, metric="correlation")
            K = feats @ feats.T
            np.fill_diagonal(K, 0)
            subj_K_B[layer] = K

    print(f"  Model RDMs precomputed.")

    # Compute RSA and CKA
    for model_name, layers, model_rdms, model_K in [
        ("ResNet152", layers_A, subj_rdms_A, subj_K_A),
        ("Qwen3-VL",  layers_B, subj_rdms_B, subj_K_B),
    ]:
        for layer in layers:
            rdm_model = model_rdms[layer]
            K = model_K[layer]

            for roi in NSD_ROIS:
                neural = nsd_test_all[subj][roi].astype(np.float64)

                # RSA
                rdm_neural = pdist(neural, metric="correlation")
                rsa_score, _ = spearmanr(rdm_model, rdm_neural)

                # CKA
                neural_norm = safe_normalize(neural)
                L = neural_norm @ neural_norm.T
                np.fill_diagonal(L, 0)
                KL = K @ L
                hsic_kl = (np.trace(KL) + K.sum() * L.sum() / ((n-1)*(n-2))
                           - 2 * KL.sum() / (n-2)) / (n * (n-3))
                hsic_kk = (np.trace(K @ K) + K.sum()**2 / ((n-1)*(n-2))
                           - 2 * (K @ K).sum() / (n-2)) / (n * (n-3))
                hsic_ll = (np.trace(L @ L) + L.sum()**2 / ((n-1)*(n-2))
                           - 2 * (L @ L).sum() / (n-2)) / (n * (n-3))
                denom = np.sqrt(max(hsic_kk, 0) * max(hsic_ll, 0)) + 1e-8
                cka_score = hsic_kl / denom

                results_nsd_all.append(dict(
                    subject=subj,
                    model=model_name,
                    layer=layer,
                    roi=roi,
                    rsa=float(rsa_score),
                    cka=float(cka_score),
                ))

        print(f"  {model_name} done.")

df_nsd_all = pd.DataFrame(results_nsd_all)
print(f"\nDone. Shape: {df_nsd_all.shape}")

df_nsd_all.to_pickle("results/df_nsd_rsa_cka_all_subjects.pkl")
print("Saved to results/df_nsd_rsa_cka_all_subjects.pkl")

In [ ]:
df_nsd_all = pd.read_pickle("results/df_nsd_rsa_cka_all_subjects.pkl")
print(f"Loaded. Shape: {df_nsd_all.shape}")
print(f"Subjects: {df_nsd_all['subject'].unique()}")
print(f"RSA range: {df_nsd_all['rsa'].min():.4f} – {df_nsd_all['rsa'].max():.4f}")
print(f"CKA range: {df_nsd_all['cka'].min():.4f} – {df_nsd_all['cka'].max():.4f}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

roi_colors = {
    "V1v": "steelblue", "V2v": "dodgerblue", "V3v": "deepskyblue",
    "hV4": "forestgreen", "FFA-1": "darkorange", "VWFA-1": "gold",
    "PPA": "red", "OPA": "purple", "EBA": "brown"
}

for col, model_name in enumerate(["ResNet152", "Qwen3-VL"]):
    df_model = df_nsd_all[df_nsd_all["model"] == model_name]

    if model_name == "ResNet152":
        layer_order = layer_order_A
    else:
        layer_order = layer_order_B

    for row, metric in enumerate(["rsa", "cka"]):
        ax = axes[row, col]

        for roi in NSD_ROIS:
            df_roi = df_model[df_model["roi"] == roi].copy()
            df_roi["layer"] = pd.Categorical(df_roi["layer"],
                                              categories=layer_order, ordered=True)
            df_roi = df_roi.sort_values("layer")

            # average and SEM across subjects
            mean_vals = df_roi.groupby("layer")[metric].mean().reindex(layer_order)
            sem_vals  = df_roi.groupby("layer")[metric].sem().reindex(layer_order)

            x = range(len(layer_order))
            ax.plot(x, mean_vals.values, marker="o", lw=2,
                    label=roi, color=roi_colors[roi])
            ax.fill_between(x,
                            mean_vals.values - sem_vals.values,
                            mean_vals.values + sem_vals.values,
                            alpha=0.15, color=roi_colors[roi])

        ax.set_xticks(range(len(layer_order)))
        ax.set_xticklabels(layer_order, rotation=45, ha="right", fontsize=8)
        ax.set_xlabel("Layer")
        ax.set_ylabel(metric.upper())
        ax.set_title(f"{model_name} — {metric.upper()} — NSD (mean ± SEM, 8 subjects)")
        ax.legend(fontsize=7, ncol=2)
        ax.axhline(0, color="k", lw=0.5, ls="--")

fig.suptitle("Layer-wise RSA and CKA — NSD (averaged across 8 subjects)", fontsize=13)
fig.tight_layout()
plt.savefig("images/fig_nsd_rsa_cka_layers_all_subjects.png", dpi=150, bbox_inches="tight")
plt.show()

#### Motivation
Having established the layer-hierarchy correspondence on a single subject, we now ask 
whether it replicates across all 8 NSD subjects. Cross-subject averaging is more 
meaningful here than for EEG: NSD uses 7T fMRI measured directly from brain tissue, 
all subjects are registered to a common anatomical space, and ROIs are defined using 
functional localizers — meaning V1v in one subject is genuinely comparable to V1v in 
another. The same electrode in EEG captures different mixtures of underlying sources 
across subjects depending on individual anatomy, making cross-subject averaging noisier 
and harder to interpret.

#### Results

The group average confirms that all main findings from subj01 replicate. The SEM bands 
are narrow relative to the differences between ROIs, indicating low inter-subject 
variability. The ResNet152 hierarchy — V1v peaking early and EBA/PPA/OPA dominating in 
deeper layers — is if anything cleaner in the group average than in the single subject. 
The Qwen3-VL language model jump is equally consistent across subjects, with EBA and 
PPA showing the strongest recovery at the language model boundary. SubjO1 was 
marginally below average in absolute RSA values for some ROIs, but the qualitative 
pattern is identical to the group. This replicability, made possible by the spatial 
specificity of fMRI and the anatomically grounded ROI definitions, gives us 
considerably more confidence in the NSD findings than either TVSD or EEG alone. Across 
all 8 subjects, Qwen3-VL achieves lower peak alignment for early visual areas but 
comparable or higher alignment for higher-level areas in the language model layers, 
confirming that the two models capture complementary aspects of the visual hierarchy.

<div style="display: flex; gap: 10px; align-items: flex-start; margin-top: 15px;">
  <figure style="text-align: center; flex: 1;">
    <img src="images/fig_nsd_rsa_cka_layers_all_subjects.png" style="width: 100%;"/>
    <figcaption>Layer-wise RSA and CKA for both models on NSD — mean across 8 subjects 
    with ± SEM shading. Each line corresponds to one ROI.</figcaption>
  </figure>
</div>

**Key takeaways**
- All single-subject findings replicate in the group average with narrow SEM bands, 
  confirming high replicability across subjects
- Cross-subject averaging is more meaningful for fMRI than EEG due to anatomical 
  registration and functional ROI definitions
- The complementary pattern between ResNet152 (strong early areas) and Qwen3-VL 
  (strong higher-level areas at the language model boundary) holds consistently 
  across all 8 subjects

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import matplotlib.colors as mcolors

ROI_EEG = "occipital_parietal"
SUBJECTS_EEG = [f"sub-{i:02d}" for i in range(1, 11)]

# Load all training data upfront for the selected ROI
train_data_all = {}
with h5py.File(EEG_PATH, "r") as f:
    for subj in SUBJECTS_EEG:
        train_data_all[subj] = f[f"train/neural_data/{subj}/{ROI_EEG}"][:]
        
print("Training data loaded.")
for subj in SUBJECTS_EEG:
    print(f"  {subj}: {train_data_all[subj].shape}")  # (16540, n_ch, 80)

n_ch_train = train_data_all["sub-01"].shape[1]
n_tp_train = train_data_all["sub-01"].shape[2]
times_train = np.linspace(0, 0.8, n_tp_train)
n_stim_train = train_data_all["sub-01"].shape[0]
BLOCK_SIZE = 200
n_blocks = n_stim_train // BLOCK_SIZE

def plot_train_detail(subject, block_idx, channel_idx, stim_offset):
    data = train_data_all[subject]  # (16540, n_ch, 80)
    n_stim = data.shape[0]
    
    start = block_idx * BLOCK_SIZE
    end   = start + BLOCK_SIZE
    data_block = data[start:end]  # (BLOCK_SIZE, n_ch, 80)
    
    # variance per stimulus
    ep_var = data_block.var(axis=2)  # (BLOCK_SIZE, n_ch)
    stim_mean = data.var(axis=2).mean(axis=0)  # (n_ch,)
    stim_std  = data.var(axis=2).std(axis=0)
    ep_var_norm = (ep_var - stim_mean) / (stim_std + 1e-8)
    
    n_examples = 5
    print(f"Z-scores for shown stimuli (channel {channel_idx}):")
    for i in range(n_examples):
        si = stim_offset + i
        z = ep_var_norm[si, channel_idx]
        print(f"  Stimulus {start + si}: z = {z:.2f}")
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    im = axes[0].imshow(ep_var_norm, aspect="auto", cmap="viridis",
                        interpolation="nearest",
                        norm=mcolors.SymLogNorm(linthresh=1))
    axes[0].axvline(x=channel_idx, color="red", lw=1.5, label=f"Channel {channel_idx}")
    axes[0].axhline(y=stim_offset, color="orange", lw=1.5, ls="--")
    axes[0].axhline(y=stim_offset + n_examples - 1, color="orange", lw=1.5,
                    ls="--", label="Stimuli shown right")
    axes[0].set_xlabel("Channel index")
    axes[0].set_ylabel("Stimulus index within block")
    axes[0].set_title(f"Z-scored variance — {subject} / {ROI_EEG} — block {block_idx}")
    axes[0].legend(fontsize=8)
    plt.colorbar(im, ax=axes[0], label="Z-scored variance")
    
    for i in range(n_examples):
        si = stim_offset + i
        axes[1].plot(times_train, data_block[si, channel_idx, :],
                     alpha=0.7, label=f"Stimulus {start + si}")
    axes[1].set_xlabel("Time (s)")
    axes[1].set_ylabel("Amplitude")
    axes[1].set_title(f"Time courses — {subject} / channel {channel_idx} / block {block_idx}")
    axes[1].legend(fontsize=8)
    
    plt.tight_layout()
    plt.show()

subj_slider_tr = widgets.Dropdown(options=SUBJECTS_EEG, value="sub-01",
    description="Subject:", style={"description_width": "initial"})
block_slider_tr = widgets.IntSlider(value=0, min=0, max=n_blocks-1, step=1,
    description="Block:", continuous_update=False,
    style={"description_width": "initial"})
channel_slider_tr = widgets.IntSlider(value=0, min=0, max=n_ch_train-1, step=1,
    description="Channel:", continuous_update=False,
    style={"description_width": "initial"})
stim_slider_tr = widgets.IntSlider(value=0, min=0, max=BLOCK_SIZE-5, step=1,
    description="Stim offset:", continuous_update=False,
    style={"description_width": "initial"})

widgets.interact(plot_train_detail,
                 subject=subj_slider_tr,
                 block_idx=block_slider_tr,
                 channel_idx=channel_slider_tr,
                 stim_offset=stim_slider_tr)

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import matplotlib.colors as mcolors

ROI_EEG = "occipital_parietal"
SUBJECTS_EEG = [f"sub-{i:02d}" for i in range(1, 11)]

# Load all training data upfront for the selected ROI
train_data_all = {}
with h5py.File(EEG_PATH, "r") as f:
    for subj in SUBJECTS_EEG:
        train_data_all[subj] = f[f"train/neural_data/{subj}/{ROI_EEG}"][:]
        
print("Training data loaded.")
for subj in SUBJECTS_EEG:
    print(f"  {subj}: {train_data_all[subj].shape}")  # (16540, n_ch, 80)

n_ch_train = train_data_all["sub-01"].shape[1]
n_tp_train = train_data_all["sub-01"].shape[2]
times_train = np.linspace(0, 0.8, n_tp_train)
n_stim_train = train_data_all["sub-01"].shape[0]
BLOCK_SIZE = 200
n_blocks = n_stim_train // BLOCK_SIZE

def plot_train_detail(subject, block_idx, channel_idx, stim_offset):
    data = train_data_all[subject]  # (16540, n_ch, 80)
    n_stim = data.shape[0]
    
    start = block_idx * BLOCK_SIZE
    end   = start + BLOCK_SIZE
    data_block = data[start:end]  # (BLOCK_SIZE, n_ch, 80)
    
    # variance per stimulus
    ep_var = data_block.var(axis=2)  # (BLOCK_SIZE, n_ch)
    stim_mean = data.var(axis=2).mean(axis=0)  # (n_ch,)
    stim_std  = data.var(axis=2).std(axis=0)
    ep_var_norm = (ep_var - stim_mean) / (stim_std + 1e-8)
    
    n_examples = 5
    print(f"Z-scores for shown stimuli (channel {channel_idx}):")
    for i in range(n_examples):
        si = stim_offset + i
        z = ep_var_norm[si, channel_idx]
        print(f"  Stimulus {start + si}: z = {z:.2f}")
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    im = axes[0].imshow(ep_var_norm, aspect="auto", cmap="viridis",
                        interpolation="nearest",
                        norm=mcolors.SymLogNorm(linthresh=1))
    axes[0].axvline(x=channel_idx, color="red", lw=1.5, label=f"Channel {channel_idx}")
    axes[0].axhline(y=stim_offset, color="orange", lw=1.5, ls="--")
    axes[0].axhline(y=stim_offset + n_examples - 1, color="orange", lw=1.5,
                    ls="--", label="Stimuli shown right")
    axes[0].set_xlabel("Channel index")
    axes[0].set_ylabel("Stimulus index within block")
    axes[0].set_title(f"Z-scored variance — {subject} / {ROI_EEG} — block {block_idx}")
    axes[0].legend(fontsize=8)
    plt.colorbar(im, ax=axes[0], label="Z-scored variance")
    
    for i in range(n_examples):
        si = stim_offset + i
        axes[1].plot(times_train, data_block[si, channel_idx, :],
                     alpha=0.7, label=f"Stimulus {start + si}")
    axes[1].set_xlabel("Time (s)")
    axes[1].set_ylabel("Amplitude")
    axes[1].set_title(f"Time courses — {subject} / channel {channel_idx} / block {block_idx}")
    axes[1].legend(fontsize=8)
    
    plt.tight_layout()
    plt.show()

subj_slider_tr = widgets.Dropdown(options=SUBJECTS_EEG, value="sub-01",
    description="Subject:", style={"description_width": "initial"})
block_slider_tr = widgets.IntSlider(value=0, min=0, max=n_blocks-1, step=1,
    description="Block:", continuous_update=False,
    style={"description_width": "initial"})
channel_slider_tr = widgets.IntSlider(value=0, min=0, max=n_ch_train-1, step=1,
    description="Channel:", continuous_update=False,
    style={"description_width": "initial"})
stim_slider_tr = widgets.IntSlider(value=0, min=0, max=BLOCK_SIZE-5, step=1,
    description="Stim offset:", continuous_update=False,
    style={"description_width": "initial"})

widgets.interact(plot_train_detail,
                 subject=subj_slider_tr,
                 block_idx=block_slider_tr,
                 channel_idx=channel_slider_tr,
                 stim_offset=stim_slider_tr)

In [ ]:
# Load test repetitions for all subjects
test_reps_all = {}
with h5py.File(EEG_REPS_PATH, "r") as f:
    for subj in SUBJECTS_EEG:
        test_reps_all[subj] = f["test"]["neural_data"][subj][ROI_EEG][:]
        
print("Test repetitions loaded.")
for subj in SUBJECTS_EEG:
    print(f"  {subj}: {test_reps_all[subj].shape}")  # (200, n_ch, 80, 80)

# shape: (stim, ch, reps, tp)
n_stim_reps = test_reps_all["sub-01"].shape[0]
n_ch_reps   = test_reps_all["sub-01"].shape[1]
n_reps      = test_reps_all["sub-01"].shape[2]
times_reps  = np.linspace(0, 0.8, 80)

def plot_reps_detail(subject, stimulus_idx, channel_idx, rep_offset):
    data = test_reps_all[subject]  # (200, n_ch, 80, 80)
    n_examples = 5
    
    # variance per (stim, ch, rep)
    ep_var = data.var(axis=3)  # (200, n_ch, 80)
    ep_var_flat = ep_var.reshape(n_stim_reps * n_reps, n_ch_reps)
    mu = ep_var_flat.mean(axis=0)
    sd = ep_var_flat.std(axis=0)
    ep_var_norm = (ep_var - mu[None, :, None]) / (sd[None, :, None] + 1e-8)
    
    print(f"Z-scores for shown repetitions (stim {stimulus_idx}, channel {channel_idx}):")
    for i in range(n_examples):
        rep = rep_offset + i
        z = ep_var_norm[stimulus_idx, channel_idx, rep]
        print(f"  Rep {rep}: z = {z:.2f}  |  max amp = {np.abs(data[stimulus_idx, channel_idx, rep, :]).max():.2f}")
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    # heatmap: reps x stimuli for this channel
    im = axes[0].imshow(ep_var_norm[:, channel_idx, :].T, aspect="auto", 
                        cmap="viridis", interpolation="nearest",
                        norm=mcolors.SymLogNorm(linthresh=1))
    axes[0].axvline(x=stimulus_idx, color="red", lw=1.5, label=f"Stimulus {stimulus_idx}")
    axes[0].axhline(y=rep_offset, color="orange", lw=1.5, ls="--")
    axes[0].axhline(y=rep_offset + n_examples - 1, color="orange", lw=1.5,
                    ls="--", label="Reps shown right")
    axes[0].set_xlabel("Stimulus index")
    axes[0].set_ylabel("Repetition index")
    axes[0].set_title(f"Z-scored variance — {subject} / channel {channel_idx} (test reps)")
    axes[0].legend(fontsize=8)
    plt.colorbar(im, ax=axes[0], label="Z-scored variance")
    
    # time courses for selected repetitions
    for i in range(n_examples):
        rep = rep_offset + i
        axes[1].plot(times_reps, data[stimulus_idx, channel_idx, rep, :],
                     alpha=0.7, label=f"Rep {rep}")
    
    # also plot the average
    axes[1].plot(times_reps, data[stimulus_idx, channel_idx, :, :].mean(axis=0),
                 color="black", lw=2.5, label="Mean", zorder=5)
    
    axes[1].set_xlabel("Time (s)")
    axes[1].set_ylabel("Amplitude")
    axes[1].set_title(f"Individual reps — {subject} / channel {channel_idx} / stim {stimulus_idx}")
    axes[1].legend(fontsize=8)
    
    plt.tight_layout()
    plt.show()

subj_slider_reps = widgets.Dropdown(options=SUBJECTS_EEG, value="sub-01",
    description="Subject:", style={"description_width": "initial"})
stim_slider_reps = widgets.IntSlider(value=0, min=0, max=n_stim_reps-1, step=1,
    description="Stimulus:", continuous_update=False,
    style={"description_width": "initial"})
channel_slider_reps = widgets.IntSlider(value=0, min=0, max=n_ch_reps-1, step=1,
    description="Channel:", continuous_update=False,
    style={"description_width": "initial"})
rep_slider_reps = widgets.IntSlider(value=0, min=0, max=n_reps-5, step=1,
    description="Rep offset:", continuous_update=False,
    style={"description_width": "initial"})

widgets.interact(plot_reps_detail,
                 subject=subj_slider_reps,
                 stimulus_idx=stim_slider_reps,
                 channel_idx=channel_slider_reps,
                 rep_offset=rep_slider_reps)

In [ ]:
import importlib
import utils
importlib.reload(utils)
from utils import safe_normalize
from scipy.spatial.distance import pdist
from scipy.stats import spearmanr

THRESHOLD = 2.5
PERCENTILE = 99.11
ROI_EEG = "occipital_parietal"
SUBJECTS_EEG = [f"sub-{i:02d}" for i in range(1, 11)]
N_CHANNELS = 17
GLOBAL_ARTIFACT_THRESHOLD = 14  # out of 17 channels

# Hardcoded bad channels identified through visual inspection
BAD_CHANNELS = {
    "sub-01": [2],   # O2 electrode — systematic spike artifacts
    "sub-04": [10],  # systematic drift artifacts
}

# Load test data for all subjects
eeg_test_raw_all = {}
eeg_test_ids_all = {}
eeg_nc_all_subj  = {}

with h5py.File(EEG_PATH, "r") as f:
    test_ids = f["test/stimulus_ids"][:]  # shared across subjects
    for subj in SUBJECTS_EEG:
        eeg_test_raw_all[subj] = f[f"test/neural_data/{subj}/{ROI_EEG}"][:]
        eeg_nc_all_subj[subj]  = f[f"noise_ceilings/{subj}/{ROI_EEG}"][:]

eeg_test_ids = test_ids
print(f"Test stimulus IDs shape: {eeg_test_ids.shape}")
for subj in SUBJECTS_EEG:
    print(f"  {subj}: {eeg_test_raw_all[subj].shape}")

In [ ]:
def clean_eeg_test(raw, subj, bad_channels_dict, threshold, percentile,
                   global_threshold, n_ch):
    """
    Apply two-level QC to averaged test EEG responses.
    
    Parameters
    ----------
    raw : np.ndarray (n_stim, n_ch, n_tp)
    subj : str
    bad_channels_dict : dict — hardcoded bad channels per subject
    threshold : float — z-score threshold
    percentile : float — amplitude percentile threshold
    global_threshold : int — min channels flagged to exclude entire stimulus row
    n_ch : int — total number of channels
    
    Returns
    -------
    cleaned : np.ndarray (n_stim, n_ch, n_tp) with NaN for excluded entries
    report : dict with QC statistics
    """
    cleaned = raw.copy().astype(np.float64)
    n_stim, n_ch_data, n_tp = raw.shape
    
    # Step 1: exclude hardcoded bad channels entirely
    bad_ch = bad_channels_dict.get(subj, [])
    for ch in bad_ch:
        cleaned[:, ch, :] = np.nan
    
    # Step 2: compute per-channel QC thresholds from test data
    ep_var = raw.var(axis=2)  # (n_stim, n_ch)
    mu  = ep_var.mean(axis=0)  # (n_ch,)
    sd  = ep_var.std(axis=0)   # (n_ch,)
    ep_var_norm = (ep_var - mu) / (sd + 1e-8)  # (n_stim, n_ch)
    
    max_abs = np.abs(raw).max(axis=2)  # (n_stim, n_ch)
    thresh_per_ch = np.percentile(max_abs, percentile, axis=0)  # (n_ch,)
    
    # combined per-channel mask
    ch_mask = (ep_var_norm > threshold) | (max_abs > thresh_per_ch[None, :])
    # (n_stim, n_ch) — True = flagged
    
    # don't flag already-excluded bad channels
    for ch in bad_ch:
        ch_mask[:, ch] = False
    
    # Step 3: stimulus-level exclusion — if >= global_threshold channels flagged
    n_flagged_per_stim = ch_mask.sum(axis=1)  # (n_stim,)
    global_bad_stim = n_flagged_per_stim >= global_threshold
    cleaned[global_bad_stim, :, :] = np.nan
    
    # Step 4: set remaining flagged channel-stimulus pairs to NaN
    for ch in range(n_ch_data):
        for stim in range(n_stim):
            if ch_mask[stim, ch] and not global_bad_stim[stim]:
                cleaned[stim, ch, :] = np.nan
    
    report = dict(
        subject=subj,
        bad_channels=bad_ch,
        n_global_bad_stim=global_bad_stim.sum(),
        n_ch_stim_pairs_flagged=ch_mask.sum(),
        pct_nan=np.isnan(cleaned).mean() * 100,
    )
    return cleaned, report

# Apply cleaning to all subjects
eeg_test_clean_all = {}
qc_reports = []

for subj in SUBJECTS_EEG:
    cleaned, report = clean_eeg_test(
        eeg_test_raw_all[subj], subj, BAD_CHANNELS,
        THRESHOLD, PERCENTILE, GLOBAL_ARTIFACT_THRESHOLD, N_CHANNELS
    )
    eeg_test_clean_all[subj] = cleaned
    qc_reports.append(report)
    print(f"{subj}: {report['n_global_bad_stim']} global bad stimuli, "
          f"{report['n_ch_stim_pairs_flagged']} ch-stim pairs flagged, "
          f"{report['pct_nan']:.2f}% NaN")

df_qc = pd.DataFrame(qc_reports)
print("\nQC summary:")
print(df_qc.to_string())

### Multi-subject EEG cleaning pipeline

#### Motivation
Having seen on sub-01 that single-channel artifacts can produce systematic-looking RSA 
features (the negative layer1-0 patch was driven by channel 2), we now apply a cleaning 
pipeline across all 10 subjects before recomputing the time-resolved RSA/CKA.

One design choice worth justifying upfront: all QC thresholds are computed from the 
test data directly rather than transferred from the training data. This may seem 
unusual, but it is the more principled choice here. Training responses are averages of 
4 repetitions per stimulus, while test responses are averages of 80 repetitions. The 
additional averaging produces systematically lower epoch variance in the test set, 
meaning training-derived thresholds would compare distributions of fundamentally 
different scales, potentially missing artifacts in high-amplitude channels or flagging 
clean responses as outliers. Computing thresholds from the test data itself avoids this 
mismatch. This does not introduce data leakage since the QC flags are used only to 
exclude stimulus rows from the response matrix, not to select stimuli for model fitting 
or evaluation.

This choice also connects to a broader observation about RSA sensitivity. In Section 
1.3 we saw that the variance-based noise ceiling estimator was fooled by consistent 
artifacts in channels 13 and 53, assigning them high reliability because the artifact 
pattern was reproducible across repetitions. The same logic applies here: a consistent 
artifact creates spurious pairwise stimulus similarities that inflate RSA without 
necessarily affecting the inner-product geometry that CKA measures. Whether CKA is 
more robust to this kind of inflation is one of the questions the cleaning comparison 
below will help answer.

#### Results
The pipeline has three components:

1. **Hardcoded bad-channel exclusion** for channels identified by visual inspection in 
   Section 1.2 (sub-01 channel 2, sub-04 channel 10).
2. **Per-channel statistical flagging** of stimulus rows whose epoch variance exceeds 
   2.5 standard deviations above the channel mean or whose maximum absolute amplitude 
   exceeds the 99.11th percentile of that channel's distribution.
3. **Global stimulus-level exclusion** of any stimulus where 14 or more of the 17 
   channels are simultaneously flagged.

Flagged entries are set to NaN and propagated through the downstream RSA/CKA 
computation. We produce three nested versions of the data — uncleaned, 
bad-channels-removed-only, and fully-cleaned — to compare what each cleaning step 
contributes.

### QC report

#### Motivation
Before recomputing RSA and CKA across all 10 subjects, we need to know how much data 
each subject loses to cleaning and whether the pipeline behaves consistently. The table 
below summarizes the QC output.

#### Results

| Subject | Bad channels | Global bad stimuli | Channel x stimulus flags | % NaN |
|---|---|---|---|---|
| sub-01 | [2]  | 0 | 75 | 8.09 |
| sub-02 | []   | 3 | 85 | 2.59 |
| sub-03 | []   | 1 | 60 | 1.76 |
| sub-04 | [10] | 3 | 78 | 8.21 |
| sub-05 | []   | 1 | 90 | 2.65 |
| sub-06 | []   | 1 | 95 | 2.79 |
| sub-07 | []   | 1 | 95 | 2.82 |
| sub-08 | []   | 0 | 81 | 2.38 |
| sub-09 | []   | 2 | 82 | 2.44 |
| sub-10 | []   | 1 | 95 | 2.79 |

Most subjects lose only 2–3% of cells to NaN, almost entirely from per-channel flagging 
with 0–3 stimuli excluded globally. Sub-01 and sub-04 sit higher at ~8% NaN, driven 
almost entirely by the one fully removed channel each — their remaining per-channel 
flags (75 and 78) are comparable to other subjects, which confirms their data is 
otherwise clean. The number of channel x stimulus flags (60–95) is small relative to 
the 200 x 17 = 3,400 total cells per subject, so the thresholds are conservative rather 
than aggressive. This is intentional: with only 200 test stimuli, over-pruning would 
hurt the reliability of RSA and CKA more than retaining slightly noisy data would. In 
practice the 8% NaN figure for sub-01 and sub-04 translates into a smaller actual 
stimulus loss, since the bad channels are already masked and only the per-stimulus and 
global flags reduce the stimulus count in the downstream RSA/CKA computation.

**Key takeaways**
- Only sub-01 and sub-04 have hardcoded bad channels; all other subjects lose at most 
  3% of cells
- The pipeline is conservative by design, prioritizing stimulus retention over 
  aggressive artifact removal
- Sub-01 and sub-04's elevated NaN rates are channel-driven, not stimulus-driven, 
  confirming their remaining data is clean

In [ ]:
# Load feature IDs
with h5py.File(FEAT_A_THINGS, "r") as f:
    feat_ids_A = f["ids"][:]
with h5py.File(FEAT_B_THINGS, "r") as f:
    feat_ids_B = f["ids"][:]

id_to_idx_A = {id_: i for i, id_ in enumerate(feat_ids_A)}
id_to_idx_B = {id_: i for i, id_ in enumerate(feat_ids_B)}

feat_idx_A = np.array([id_to_idx_A[x] for x in eeg_test_ids])
feat_idx_B = np.array([id_to_idx_B[x] for x in eeg_test_ids])

sort_A = np.argsort(feat_idx_A)
unsort_A = np.argsort(sort_A)
sort_B = np.argsort(feat_idx_B)
unsort_B = np.argsort(sort_B)

# Load features once — same for all subjects since stimulus IDs are shared
print("Loading features...")
eeg_feats_A = {}
eeg_feats_B = {}

with h5py.File(FEAT_A_THINGS, "r") as f:
    for layer in layers_A:
        eeg_feats_A[layer] = f[f"features/{layer}"][feat_idx_A[sort_A]][unsort_A]
        
with h5py.File(FEAT_B_THINGS, "r") as f:
    for layer in layers_B:
        eeg_feats_B[layer] = f[f"features/{layer}"][feat_idx_B[sort_B]][unsort_B]

print("Features loaded.")

In [ ]:
print("Precomputing model RDMs and Gram matrices...")
model_rdms_A = {}
model_K_A    = {}
for layer in layers_A:
    feats = safe_normalize(eeg_feats_A[layer])
    model_rdms_A[layer] = pdist(feats, metric="correlation")
    K = feats @ feats.T
    np.fill_diagonal(K, 0)
    model_K_A[layer] = K
    print(f"  ResNet152 {layer} done")

model_rdms_B = {}
model_K_B    = {}
for layer in layers_B:
    feats = safe_normalize(eeg_feats_B[layer])
    model_rdms_B[layer] = pdist(feats, metric="correlation")
    K = feats @ feats.T
    np.fill_diagonal(K, 0)
    model_K_B[layer] = K
    print(f"  Qwen3-VL {layer} done")

print("All model RDMs and Gram matrices precomputed.")

In [ ]:
def run_eeg_rsa_cka(eeg_data_dict, model_rdms_A, model_K_A,
                     model_rdms_B, model_K_B, label="", 
                     apply_bad_channels=True):
    results = []
    
    # Precompute normalized features once for all subjects
    norm_feats_A = {layer: safe_normalize(eeg_feats_A[layer]) for layer in layers_A}
    norm_feats_B = {layer: safe_normalize(eeg_feats_B[layer]) for layer in layers_B}
    
    for subj in SUBJECTS_EEG:
        print(f"\nProcessing {subj} ({label})...")
        neural_data = eeg_data_dict[subj]  # (200, n_ch, 80)
        
        # Use only good channels for validity check and computation
        bad_ch = BAD_CHANNELS.get(subj, []) if apply_bad_channels else []
        good_ch = [c for c in range(neural_data.shape[1]) if c not in bad_ch]
        neural_good = neural_data[:, good_ch, :]  # (200, n_good_ch, 80)
        
        # Global valid mask — stimuli with no NaN on good channels at any time point
        global_valid = ~np.isnan(neural_good).any(axis=(1, 2))  # (200,)
        n_valid = global_valid.sum()
        print(f"  Good channels: {len(good_ch)}/17, Valid stimuli: {n_valid}/200")
        
        if n_valid < 50:
            print(f"  Too few valid stimuli, skipping {subj}")
            continue
        
        for model_name, layers, model_rdms, model_K in [
            ("ResNet152", layers_A, model_rdms_A, model_K_A),
            ("Qwen3-VL",  layers_B, model_rdms_B, model_K_B),
        ]:
            norm_feats = norm_feats_A if model_name == "ResNet152" else norm_feats_B
            
            for layer in layers:
                K = model_K[layer]
                
                # Precompute once per subject per layer
                K_valid = K[np.ix_(global_valid, global_valid)]
                rdm_model_valid = pdist(norm_feats[layer][global_valid],
                                        metric="correlation")
                hsic_kk = (np.trace(K_valid @ K_valid) +
                           K_valid.sum()**2 / ((n_valid-1)*(n_valid-2))
                           - 2 * (K_valid @ K_valid).sum() / (n_valid-2)
                           ) / (n_valid * (n_valid-3))

                for t_idx in range(80):
                    neural_t = neural_good[global_valid, :, t_idx].astype(np.float64)

                    # RSA
                    rdm_neural = pdist(neural_t, metric="correlation")
                    rsa_score, _ = spearmanr(rdm_model_valid, rdm_neural)

                    # CKA
                    neural_norm = safe_normalize(neural_t)
                    L = neural_norm @ neural_norm.T
                    np.fill_diagonal(L, 0)
                    n = n_valid
                    KL = K_valid @ L
                    hsic_kl = (np.trace(KL) + K_valid.sum() * L.sum() / ((n-1)*(n-2))
                               - 2 * KL.sum() / (n-2)) / (n * (n-3))
                    hsic_ll = (np.trace(L @ L) + L.sum()**2 / ((n-1)*(n-2))
                               - 2 * (L @ L).sum() / (n-2)) / (n * (n-3))
                    denom = np.sqrt(max(hsic_kk, 0) * max(hsic_ll, 0)) + 1e-8
                    cka_score = hsic_kl / denom

                    results.append(dict(
                        subject=subj, model=model_name, layer=layer,
                        time=times[t_idx], rsa=float(rsa_score),
                        cka=float(cka_score)))

            print(f"  {model_name} done")
    
    return pd.DataFrame(results)

In [ ]:
import os
os.makedirs("results", exist_ok=True)

# Version 1: truly uncleaned — all 17 channels, all 200 stimuli
print("="*50)
print("Running UNCLEANED analysis...")
df_eeg_uncleaned = run_eeg_rsa_cka(
    eeg_test_raw_all, model_rdms_A, model_K_A,
    model_rdms_B, model_K_B, 
    label="uncleaned", apply_bad_channels=False)
df_eeg_uncleaned.to_pickle("results/df_eeg_rsa_cka_all_subjects_uncleaned.pkl")
print(f"Saved. Shape: {df_eeg_uncleaned.shape}")

# Version 2: bad channels removed only — no stimulus-level QC
print("="*50)
print("Running BAD CHANNELS REMOVED analysis...")
df_eeg_badch_removed = run_eeg_rsa_cka(
    eeg_test_raw_all, model_rdms_A, model_K_A,
    model_rdms_B, model_K_B,
    label="bad_channels_removed", apply_bad_channels=True)
df_eeg_badch_removed.to_pickle("results/df_eeg_rsa_cka_all_subjects_badch_removed.pkl")
print(f"Saved. Shape: {df_eeg_badch_removed.shape}")

# Version 3: fully cleaned — bad channels removed + stimulus-level QC
print("="*50)
print("Running FULLY CLEANED analysis...")
df_eeg_clean = run_eeg_rsa_cka(
    eeg_test_clean_all, model_rdms_A, model_K_A,
    model_rdms_B, model_K_B,
    label="cleaned", apply_bad_channels=True)
df_eeg_clean.to_pickle("results/df_eeg_rsa_cka_all_subjects_clean.pkl")
print(f"Saved. Shape: {df_eeg_clean.shape}")

In [ ]:
def plot_eeg_heatmap(df, title, filename):
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    
    for col, model_name in enumerate(["ResNet152", "Qwen3-VL"]):
        df_model = df[df["model"] == model_name]
        layer_order = layer_order_A if model_name == "ResNet152" else layer_order_B
        
        for row, metric in enumerate(["rsa", "cka"]):
            ax = axes[row, col]
            heatmap = np.zeros((len(layer_order), 80))
            
            for i, layer in enumerate(layer_order):
                df_layer = df_model[df_model["layer"] == layer].groupby("time")[metric].mean()
                heatmap[i] = df_layer.reindex(np.linspace(0.0, 0.8, 80)).values
            
            im = ax.imshow(heatmap, aspect="auto", origin="lower",
                           extent=[0, 0.8, -0.5, len(layer_order) - 0.5],
                           cmap="RdBu_r" if metric == "rsa" else "viridis",
                           vmin=-heatmap.max() if metric == "rsa" else 0,
                           vmax=heatmap.max())
            ax.set_yticks(range(len(layer_order)))
            ax.set_yticklabels(layer_order, fontsize=8)
            ax.set_xlabel("Time (s)")
            ax.set_ylabel("Layer")
            ax.set_title(f"{model_name} — {metric.upper()}")
            ax.axvline(0.0, color="k", lw=1, ls="--")
            fig.colorbar(im, ax=ax, shrink=0.85)
    
    fig.suptitle(title, fontsize=13)
    fig.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches="tight")
    plt.show()

for df, title, filename in [
    (df_eeg_uncleaned,
     f"Time-resolved RSA and CKA — {ROI_EEG} — uncleaned (mean 10 subjects)",
     "images/fig_eeg_rsa_cka_all_subjects_uncleaned.png"),
    (df_eeg_badch_removed,
     f"Time-resolved RSA and CKA — {ROI_EEG} — bad channels removed (mean 10 subjects)",
     "images/fig_eeg_rsa_cka_all_subjects_badch_removed.png"),
    (df_eeg_clean,
     f"Time-resolved RSA and CKA — {ROI_EEG} — fully cleaned (mean 10 subjects)",
     "images/fig_eeg_rsa_cka_all_subjects_clean.png"),
]:
    plot_eeg_heatmap(df, title, filename)

In [ ]:
for subj in SUBJECTS_EEG:
    for df, suffix in [
        (df_eeg_uncleaned, "uncleaned"),
        (df_eeg_badch_removed, "badch_removed"),
        (df_eeg_clean, "clean"),
    ]:
        df[df["subject"] == subj].to_pickle(
            f"results/df_eeg_rsa_cka_{subj}_{suffix}.pkl")

print("Per-subject results saved.")

In [ ]:
def plot_eeg_heatmap_single(df, subj, title, filename):
    df_subj = df[df["subject"] == subj]
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    
    for col, model_name in enumerate(["ResNet152", "Qwen3-VL"]):
        df_model = df_subj[df_subj["model"] == model_name]
        layer_order = layer_order_A if model_name == "ResNet152" else layer_order_B
        
        for row, metric in enumerate(["rsa", "cka"]):
            ax = axes[row, col]
            heatmap = np.zeros((len(layer_order), 80))
            
            for i, layer in enumerate(layer_order):
                df_layer = df_model[df_model["layer"] == layer].sort_values("time")
                heatmap[i] = df_layer[metric].values
            
            im = ax.imshow(heatmap, aspect="auto", origin="lower",
                           extent=[0, 0.8, -0.5, len(layer_order) - 0.5],
                           cmap="RdBu_r" if metric == "rsa" else "viridis",
                           vmin=-heatmap.max() if metric == "rsa" else 0,
                           vmax=heatmap.max())
            ax.set_yticks(range(len(layer_order)))
            ax.set_yticklabels(layer_order, fontsize=8)
            ax.set_xlabel("Time (s)")
            ax.set_ylabel("Layer")
            ax.set_title(f"{model_name} — {metric.upper()}")
            ax.axvline(0.0, color="k", lw=1, ls="--")
            fig.colorbar(im, ax=ax, shrink=0.85)
    
    fig.suptitle(title, fontsize=13)
    fig.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches="tight")
    plt.show()

for df, title, filename in [
    (df_eeg_uncleaned,
     f"Time-resolved RSA and CKA — sub-01 / {ROI_EEG} — uncleaned",
     "images/fig_eeg_rsa_cka_sub01_uncleaned.png"),
    (df_eeg_badch_removed,
     f"Time-resolved RSA and CKA — sub-01 / {ROI_EEG} — bad channels removed",
     "images/fig_eeg_rsa_cka_sub01_badch_removed.png"),
    (df_eeg_clean,
     f"Time-resolved RSA and CKA — sub-01 / {ROI_EEG} — fully cleaned",
     "images/fig_eeg_rsa_cka_sub01_clean.png"),
]:
    plot_eeg_heatmap_single(df, "sub-01", title, filename)

In [ ]:
def compute_differences(df_uncleaned, df_cleaned, subjects, layers_A, layers_B):
    """
    Compute absolute and relative differences between uncleaned and cleaned
    RSA/CKA results.
    
    Returns a dataframe with columns:
    subject, model, layer, time, 
    rsa_abs_diff, cka_abs_diff,
    rsa_rel_diff, cka_rel_diff
    """
    results = []
    
    for subj in subjects:
        df_u = df_uncleaned[df_uncleaned["subject"] == subj]
        df_c = df_cleaned[df_cleaned["subject"] == subj]
        
        for model_name, layer_order in [
            ("ResNet152", layers_A), ("Qwen3-VL", layers_B)
        ]:
            df_um = df_u[df_u["model"] == model_name]
            df_cm = df_c[df_c["model"] == model_name]
            
            for layer in layer_order:
                df_ul = df_um[df_um["layer"] == layer].sort_values("time")
                df_cl = df_cm[df_cm["layer"] == layer].sort_values("time")
                
                if len(df_ul) == 0 or len(df_cl) == 0:
                    continue
                
                rsa_u = df_ul["rsa"].values
                rsa_c = df_cl["rsa"].values
                cka_u = df_ul["cka"].values
                cka_c = df_cl["cka"].values
                times = df_ul["time"].values
                
                rsa_abs = rsa_u - rsa_c
                cka_abs = cka_u - cka_c
                rsa_rel = rsa_abs / (np.abs(rsa_u) + 1e-8)
                cka_rel = cka_abs / (np.abs(cka_u) + 1e-8)
                
                for i, t in enumerate(times):
                    results.append(dict(
                        subject=subj, model=model_name, layer=layer, time=t,
                        rsa_abs=float(rsa_abs[i]), cka_abs=float(cka_abs[i]),
                        rsa_rel=float(rsa_rel[i]), cka_rel=float(cka_rel[i]),
                        rsa_u=float(rsa_u[i]), cka_u=float(cka_u[i]),
                    ))
    
    return pd.DataFrame(results)

# Compute for sub-01
df_diff_sub01 = compute_differences(
    df_eeg_uncleaned, df_eeg_clean, ["sub-01"], layers_A, layers_B)

# Compute averaged across all subjects
df_diff_all = compute_differences(
    df_eeg_uncleaned, df_eeg_clean, SUBJECTS_EEG, layers_A, layers_B)

print(f"sub-01 diff shape: {df_diff_sub01.shape}")
print(f"all subjects diff shape: {df_diff_all.shape}")

In [ ]:
NEAR_ZERO_THRESH = 0.01

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, label, subjects, df_diff in [
    (axes[0], "sub-01", ["sub-01"], df_diff_sub01),
    (axes[1], "all subjects", SUBJECTS_EEG, df_diff_all),
]:
    data = []
    for model_name in ["ResNet152", "Qwen3-VL"]:
        df_m = df_diff[df_diff["model"] == model_name]
        
        # absolute
        rsa_abs_mean = df_m["rsa_abs"].abs().mean()
        cka_abs_mean = df_m["cka_abs"].abs().mean()
        
        # relative — mask near-zero
        mask = df_m["rsa_u"].abs() > NEAR_ZERO_THRESH
        rsa_rel_mean = df_m.loc[mask, "rsa_rel"].abs().mean()
        mask = df_m["cka_u"].abs() > NEAR_ZERO_THRESH
        cka_rel_mean = df_m.loc[mask, "cka_rel"].abs().mean()
        
        data.append([model_name, "RSA abs", rsa_abs_mean])
        data.append([model_name, "CKA abs", cka_abs_mean])
        data.append([model_name, "RSA rel", rsa_rel_mean])
        data.append([model_name, "CKA rel", cka_rel_mean])
    
    df_plot = pd.DataFrame(data, columns=["model", "metric", "value"])
    
    x = np.arange(4)
    width = 0.35
    resnet = df_plot[df_plot["model"] == "ResNet152"]["value"].values
    qwen   = df_plot[df_plot["model"] == "Qwen3-VL"]["value"].values
    
    ax.bar(x - width/2, resnet, width, label="ResNet152", color="steelblue", alpha=0.8)
    ax.bar(x + width/2, qwen,   width, label="Qwen3-VL",  color="darkorange", alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(["RSA\n(abs)", "CKA\n(abs)", "RSA\n(rel)", "CKA\n(rel)"])
    ax.set_ylabel("Mean change due to cleaning")
    ax.set_title(f"Overall cleaning effect — {label}")
    ax.legend()

fig.suptitle("How much does cleaning change RSA and CKA?", fontsize=13)
fig.tight_layout()
plt.savefig("images/fig_cleaning_effect_overall.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 8))
times_arr = np.linspace(0.0, 0.8, 80)

for col, (label, df_diff) in enumerate([
    ("sub-01", df_diff_sub01),
    ("all subjects mean", df_diff_all),
]):
    for row, metric_pair in enumerate([
        ("rsa_abs", "cka_abs", "Absolute difference"),
        ("rsa_rel", "cka_rel", "Relative difference"),
    ]):
        ax = axes[row, col]
        rsa_col, cka_col, ylabel = metric_pair
        
        for model_name, color in [("ResNet152", "steelblue"), ("Qwen3-VL", "darkorange")]:
            df_m = df_diff[df_diff["model"] == model_name]
            
            # average over layers and subjects
            rsa_t = df_m.groupby("time")[rsa_col].mean()
            cka_t = df_m.groupby("time")[cka_col].mean()
            
            ax.plot(rsa_t.index, rsa_t.values, color=color, lw=2,
                    label=f"{model_name} RSA")
            ax.plot(cka_t.index, cka_t.values, color=color, lw=2,
                    ls="--", label=f"{model_name} CKA")
        
        ax.axhline(0, color="k", lw=0.5, ls="--")
        ax.set_xlabel("Time (s)")
        ax.set_ylabel(ylabel)
        ax.set_title(f"{ylabel} over time — {label}")
        ax.legend(fontsize=8)

fig.suptitle("Time-resolved cleaning effect on RSA and CKA", fontsize=13)
fig.tight_layout()
plt.savefig("images/fig_cleaning_effect_time.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
NEAR_ZERO_THRESH = 0.05
CLIP_REL = 5.0

for df_diff, label, filesuffix in [
    (df_diff_sub01, "sub-01", "sub01"),
    (df_diff_all,   "all subjects mean", "all_subjects"),
]:
    fig, axes = plt.subplots(2, 4, figsize=(22, 8))
    
    for col, model_name in enumerate(["ResNet152", "Qwen3-VL"]):
        df_m = df_diff[df_diff["model"] == model_name]
        layer_order = layer_order_A if model_name == "ResNet152" else layer_order_B
        
        for row, (diff_col, title_suffix, cmap, use_mask) in enumerate([
            ("rsa_abs", "RSA absolute", "RdBu_r", False),
            ("cka_abs", "CKA absolute", "RdBu_r", False),
            ("rsa_rel", "RSA relative", "RdBu_r", True),
            ("cka_rel", "CKA relative", "RdBu_r", True),
        ]):
            ax = axes[row // 2, col * 2 + row % 2]
            heatmap = np.zeros((len(layer_order), 80))
            
            for i, layer in enumerate(layer_order):
                df_l = df_m[df_m["layer"] == layer].sort_values("time")
                
                # average across subjects if multiple subjects present
                if "subject" in df_l.columns and df_l["subject"].nunique() > 1:
                    df_l = df_l.groupby("time")[
                        [diff_col, "rsa_u", "cka_u"]].mean().reset_index()
                
                vals = df_l[diff_col].values
                
                if use_mask:
                    # mask near-zero uncleaned values
                    ref_col = "rsa_u" if "rsa" in diff_col else "cka_u"
                    near_zero = df_l[ref_col].abs().values < NEAR_ZERO_THRESH
                    vals = vals.copy()
                    vals[near_zero] = 0.0
                    # clip extreme relative differences
                    vals = np.clip(vals, -CLIP_REL, CLIP_REL)
                
                if len(vals) == 80:
                    heatmap[i] = vals
            
            # use percentile only on non-zero values to avoid outliers
            nonzero = heatmap[heatmap != 0]
            vmax = np.percentile(np.abs(nonzero), 95) if len(nonzero) > 0 else 1.0
            vmax = max(vmax, 1e-6)  # avoid zero vmax
            
            im = ax.imshow(heatmap, aspect="auto", origin="lower",
                           extent=[0, 0.8, -0.5, len(layer_order) - 0.5],
                           cmap=cmap, vmin=-vmax, vmax=vmax)
            ax.set_yticks(range(len(layer_order)))
            ax.set_yticklabels(layer_order, fontsize=7)
            ax.set_xlabel("Time (s)")
            ax.set_title(f"{model_name} — {title_suffix}")
            ax.axvline(0.0, color="k", lw=1, ls="--")
            fig.colorbar(im, ax=ax, shrink=0.8)
    
    fig.suptitle(f"Cleaning effect heatmaps — {label} (uncleaned minus cleaned)",
                 fontsize=13)
    fig.tight_layout()
    plt.savefig(f"images/fig_cleaning_effect_heatmap_{filesuffix}.png",
                dpi=150, bbox_inches="tight")
    plt.show()

### Sub-01: effect of cleaning on RSA and CKA

#### Motivation
Sub-01 had one of the most severely corrupted channels in the dataset (channel 2, 
40.81% of epochs flagged). Rather than treating this as a problem to ignore, we use 
it as an informative case study: by showing three versions of the analysis (uncleaned, 
bad-channel-removed, and fully-cleaned) we can isolate the effect of each cleaning 
step and test whether RSA and CKA respond differently to artifact removal.

#### Results

**Step 1: removing the bad channel.** The most striking effect is the disappearance 
of the apparent early RSA alignment for layer1-0 and layer2-0 in the 30–70ms window. 
In the uncleaned version this looked like genuine early visual cortex alignment; after 
removing channel 2 it vanishes entirely, confirming it was artifact-driven. The RSA 
scale also shifts substantially (from roughly ±0.1 to ±0.15), meaning the artifact 
was simultaneously inflating early alignment and compressing genuine alignment 
elsewhere. After removal, the genuine 100ms peak for intermediate and deeper layers 
becomes more clearly visible. Qwen3-VL's RSA is even more affected, with the scale 
jumping from ±0.1 to ±0.2, suggesting the artifact's pairwise similarity structure 
happened to correlate more with Qwen's RDM than with ResNet's.

CKA tells a different story: the early spurious signal visible in RSA for layer1-0 
and layer2-0 at 30–70ms does not appear in CKA in any version. The artifact created 
correlated pairwise distances that RSA detects, but not a systematic inner-product 
geometry that CKA requires. CKA scale does increase after channel removal (from 0–0.10 
to 0–0.14), but the overall pattern (peak at layer3-0 around 100–150ms) is 
preserved throughout, confirming this is genuine signal.

**Step 2: stimulus-level cleaning.** Going from bad-channel-removed to fully-cleaned 
produces smaller but visible changes. Sustained alignment after 200ms increases 
slightly for RSA, consistent with the second RSVP image response peaking around 
260–300ms being partially suppressed by stimulus-level artifacts. For CKA, the 
language model layers of Qwen3-VL show slightly stronger alignment after full cleaning. 
The core patterns established by step 1 are already stable at this point.

<div style="display: flex; gap: 10px; align-items: flex-start; margin-top: 15px;">
  <figure style="text-align: center; flex: 1;">
    <img src="images/fig_eeg_rsa_cka_sub01_uncleaned.png" style="width: 70%;"/>
    <figcaption>Sub-01 — uncleaned (all 17 channels, all 200 stimuli)</figcaption>
  </figure>
</div>

<div style="display: flex; gap: 10px; align-items: flex-start; margin-top: 15px;">
  <figure style="text-align: center; flex: 1;">
    <img src="images/fig_eeg_rsa_cka_sub01_badch_removed.png" style="width: 70%;"/>
    <figcaption>Sub-01 — bad channel removed (channel 2 excluded, all 200 stimuli)</figcaption>
  </figure>
</div>

<div style="display: flex; gap: 10px; align-items: flex-start; margin-top: 15px;">
  <figure style="text-align: center; flex: 1;">
    <img src="images/fig_eeg_rsa_cka_sub01_clean.png" style="width: 70%;"/>
    <figcaption>Sub-01 — fully cleaned (channel 2 excluded, stimulus-level QC applied)</figcaption>
  </figure>
</div>

The cleaning effect is quantified in the figures below. For sub-01, absolute RSA 
changes are modest (around 0.03–0.04) but relative changes are large (1.4x for 
ResNet152 RSA, 1.7x for Qwen3-VL RSA), reflecting the artifact's effect of 
suppressing genuine alignment while inflating spurious early signals. CKA relative 
changes are also substantial (1.6x for ResNet152) but in the opposite direction. Cleaning increased rather than decreased CKA, consistent with the artifact globally 
suppressing inner-product geometry rather than inflating it. The difference heatmap 
confirms this: red patches for RSA are concentrated in early layers at 0–200ms, while 
CKA differences are predominantly blue across all layer-time combinations.

<div style="display: flex; gap: 10px; align-items: flex-start; margin-top: 15px;">
  <figure style="text-align: center; flex: 1;">
    <img src="images/fig_cleaning_effect_overall.png" style="width: 60%;"/>
    <figcaption>Mean change due to cleaning (uncleaned vs fully cleaned), as absolute 
    and relative differences averaged over all layer x time combinations, for sub-01 
    and the group mean.</figcaption>
  </figure>
</div>

<div style="display: flex; gap: 10px; align-items: flex-start; margin-top: 15px;">
  <figure style="text-align: center; flex: 1;">
    <img src="images/fig_cleaning_effect_heatmap_sub01.png" style="width: 100%;"/>
    <figcaption>Cleaning effect heatmaps for sub-01 (uncleaned minus fully cleaned). 
    Red: uncleaned was higher. Blue: cleaning increased the value. Top row: absolute 
    differences. Bottom row: relative differences.</figcaption>
  </figure>
</div>

**Key takeaways**
- Removing channel 2 eliminates the spurious 30–70ms RSA signal and reveals the 
  genuine 100ms peak, confirming the earlier finding was artifact-driven
- CKA is robust to the artifact that fooled RSA: the spurious early signal never 
  appears in CKA across any of the three versions
- Stimulus-level cleaning adds smaller incremental improvements, mainly recovering 
  later-time alignment suppressed by global artifacts
- RSA changes are inflating then deflating; CKA changes are consistently in the 
  direction of increasing, reflecting fundamentally different sensitivity to artifacts

### Group average across 10 subjects

#### Motivation
The sub-01 analysis showed large cleaning effects for a single severely corrupted 
subject. The group average tests whether these artifacts meaningfully affect 
conclusions at the population level, and provides the clean multi-subject results 
that the Answer Box 2.1 conclusions are based on.

#### Results

The three group-average versions look remarkably similar. Averaging across 10 subjects 
with only 2 having problematic channels substantially dilutes the artifact effect. The 
main visible difference is in the 0–70ms window: the uncleaned version shows weak 
positive RSA there while the cleaned version shows near-zero values, which is 
biologically more plausible since the brain has not yet had time to process the 
stimulus at those latencies. The main peaks at approximately 70–80ms and 300ms are 
stable across all three versions, confirming they reflect genuine neural signal. The 
70–80ms peak is slightly earlier than the 100ms peak seen for sub-01, consistent with 
inter-subject latency variability noted in Section 1.2. The 300ms peak corresponds to 
the primary response to the second RSVP image (appearing at 200ms, feedforward 
latency ~60–100ms). CKA shows the same robustness as at the single-subject level — 
the peak at layer3-0 around 100–150ms is virtually identical across all three versions.

The group-level difference heatmap confirms the cleaning effect is small: around 
±0.010 for ResNet152 RSA compared to ±0.075 for sub-01 alone (roughly a 10x 
reduction, reflecting the dilution from averaging). The structure of differences 
follows the same pattern as sub-01: early layers show reduced RSA in the early time 
window after cleaning, and CKA differences are predominantly blue meaning cleaning 
slightly increased CKA values across the board.

<div style="display: flex; gap: 10px; align-items: flex-start; margin-top: 15px;">
  <figure style="text-align: center; flex: 1;">
    <img src="images/fig_eeg_rsa_cka_all_subjects_uncleaned.png" style="width: 70%;"/>
    <figcaption>Group average — uncleaned (mean across 10 subjects)</figcaption>
  </figure>
</div>

<div style="display: flex; gap: 10px; align-items: flex-start; margin-top: 15px;">
  <figure style="text-align: center; flex: 1;">
    <img src="images/fig_eeg_rsa_cka_all_subjects_badch_removed.png" style="width: 70%;"/>
    <figcaption>Group average — bad channels removed (mean across 10 subjects)</figcaption>
  </figure>
</div>

<div style="display: flex; gap: 10px; align-items: flex-start; margin-top: 15px;">
  <figure style="text-align: center; flex: 1;">
    <img src="images/fig_eeg_rsa_cka_all_subjects_clean.png" style="width: 70%;"/>
    <figcaption>Group average — fully cleaned (mean across 10 subjects)</figcaption>
  </figure>
</div>

<div style="display: flex; gap: 10px; align-items: flex-start; margin-top: 15px;">
  <figure style="text-align: center; flex: 1;">
    <img src="images/fig_cleaning_effect_heatmap_all_subjects.png" style="width: 100%;"/>
    <figcaption>Cleaning effect heatmaps for the group mean (uncleaned minus fully 
    cleaned). Absolute differences are approximately 10x smaller than for sub-01, 
    reflecting the dilution effect of averaging across 10 subjects.</figcaption>
  </figure>
</div>

**Key takeaways**
- Group-level results are robust here because only 2 of 10 subjects had severely
corrupted channels; had more subjects been as affected as sub-01, the artifact
contamination would not have been diluted by averaging
- The main peaks at 70–80ms and 300ms are stable across all versions, confirming 
  they reflect genuine neural signal
- CKA is virtually unchanged by cleaning at the group level, consistent with its 
  greater robustness to structured noise shown in the sub-01 analysis
- The fully cleaned version is used for all subsequent analyses and Answer Box 2.1 
  conclusions

### EEG temporal summary: early, mid, and late layers

#### Results

The line plot summarizes the temporal alignment for early, mid, and late layers of 
each model, averaged across all 10 cleaned subjects. CKA is considerably cleaner and 
easier to interpret than RSA, which remains noisy even after cleaning and averaging.

**RSA** shows an interesting temporal structure that we interpret cautiously. The 
data is cleaned and averaged across subjects, which increases our confidence, but 
single-subject artifacts could still influence the group average in subtle ways. Early 
layers of both models show a peak around 70ms, slightly earlier than the ~100ms peak 
of deeper layers, which is broadly consistent with the known V1 response latency in 
humans being slightly later than in macaque (40–60ms from Kravitz et al. 
<a href="#kravitz2013" title="Kravitz et al. (2013) — The ventral visual pathway">[11]</a>). 
After this early peak, early layers go negative plausibly reflecting suppression 
following the initial feedforward response. This pattern is visible for both ResNet152 
and Qwen3-VL, with ResNet152 showing consistently higher values, which is in line with 
all our previous findings. Later layers peak slightly later, around 100–170ms for 
ResNet152 and 120–140ms for Qwen3-VL, consistent with deeper layers capturing 
processing at a later stage. Mid layers behave intermediately: overlapping more with 
early layers for ResNet152 and more with late layers for Qwen3-VL.

**CKA** shows a clear peak for all layers around 100ms, with ResNet152 early layers 
reaching the highest values (~0.12). The temporal layer hierarchy visible in RSA is 
much less apparent in CKA: all layers increase around the same time, consistent with 
what we observed in the full heatmaps. After the initial peak, all layers decline 
gradually toward baseline. The second RSVP image appearing at 200ms produces a 
visible but smaller response in both RSA and CKA, consistent with the known 
attenuation of responses to successive images in rapid sequences.

<div style="display: flex; gap: 10px; align-items: flex-start; margin-top: 15px;">
  <figure style="text-align: center; flex: 1;">
    <img src="images/fig_eeg_rsa_cka_depth_summary.png" style="width: 100%;"/>
    <figcaption>Early, mid, and late layer alignment over time for both models, 
    averaged across 10 subjects on the fully cleaned occipital_parietal data. Left: 
    RSA. Right: CKA.</figcaption>
  </figure>
</div>

**Key takeaways**
- Early layers peak around 70ms in RSA, slightly before deeper layers (100–170ms 
  for ResNet152), tentatively consistent with feedforward visual processing latencies
- Both models show the same temporal ordering in RSA, with ResNet152 at higher 
  values throughout — consistent with all previous findings
- CKA shows a single shared peak around 100ms for all layers with no clear temporal 
  hierarchy, confirming the RSA/CKA disagreement seen in the full heatmaps
- The second RSVP image produces a visible but attenuated response in both metrics 
  around 200–300ms

In [ ]:
# Summary line plot: early / mid / late layers over time, both models.
# Complements the full heatmap with a more readable temporal summary.

times = np.linspace(0.0, 0.8, 80)

# Pick three representative depth indices for each model
picks_A = {
    "early": layer_order_A[0],   # layer1-0
    "mid":   layer_order_A[4],   # layer3-10
    "late":  layer_order_A[-1],  # layer4-1
}
picks_B = {
    "early": layer_order_B[0],   # visual-blocks-2
    "mid":   layer_order_B[4],   # visual-blocks-18
    "late":  layer_order_B[-1],  # language_model-layers-16
}

# Average df_eeg_clean across subjects first
df_eeg_grp = (df_eeg_clean
              .groupby(["model", "layer", "time"])[["rsa", "cka"]]
              .mean()
              .reset_index())

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharex=True)
colors   = {"ResNet152": "tab:orange", "Qwen3-VL": "tab:blue"}
styles   = {"early": "-", "mid": "--", "late": ":"}
depth_labels = {"early": "early", "mid": "mid", "late": "late"}

for metric, ax in zip(["rsa", "cka"], axes):
    for model_name, picks in [("ResNet152", picks_A), ("Qwen3-VL", picks_B)]:
        df_m = df_eeg_grp[df_eeg_grp["model"] == model_name]
        for depth, layer in picks.items():
            vals = df_m[df_m["layer"] == layer].sort_values("time")[metric].values
            ax.plot(times, vals,
                    color=colors[model_name],
                    linestyle=styles[depth],
                    lw=1.8,
                    label=f"{model_name} ({depth_labels[depth]})")
    ax.axvline(0.1, color="k", lw=1, ls=":", alpha=0.4, label="100 ms")
    ax.axhline(0,   color="k", lw=0.5, ls="--")
    ax.set_xlabel("Time after stimulus onset (s)")
    ax.set_ylabel(metric.upper())
    ax.set_title(f"EEG time-resolved — {metric.upper()} (mean across 10 subjects, fully cleaned)")
    ax.grid(alpha=0.25)

axes[0].legend(fontsize=8, loc="upper right", ncol=2)
fig.suptitle("Early / mid / late layer alignment over time — occipital_parietal",
             fontsize=12)
fig.tight_layout()
plt.savefig("images/fig_eeg_rsa_cka_depth_summary.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# new (from Anastasis)
# Direct head-to-head comparison: ResNet152 vs Qwen3-VL on representational metrics.
# TVSD: best score across layers, averaged across monkeys, per ROI.
# EEG : best score across layers and time on the fully-cleaned group-averaged data.
# This complements the deliverable "one direct comparison between the two models".

head2head_rows = []

# --- TVSD: best-over-layers per ROI, averaged across monkeys ---
for model_name in ["ResNet152", "Qwen3-VL"]:
    df_m = df_tvsd[df_tvsd["model"] == model_name]
    for roi in ["V1", "V4", "IT"]:
        df_roi = df_m[df_m["roi"] == roi]
        best_rsa = df_roi.groupby("layer")["rsa"].mean().max()
        best_cka = df_roi.groupby("layer")["cka"].mean().max()
        head2head_rows.append({
            "target": f"TVSD/{roi}",
            "model":  model_name,
            "best_RSA": float(best_rsa),
            "best_CKA": float(best_cka),
        })

# --- EEG: best-over-layers-and-time on cleaned group-averaged data ---
# df_eeg_clean has columns subject, model, layer, time, rsa, cka
# average over subjects first, then take max over (layer, time).
df_eeg_grp = (df_eeg_clean
              .groupby(["model", "layer", "time"])[["rsa", "cka"]]
              .mean()
              .reset_index())
for model_name in ["ResNet152", "Qwen3-VL"]:
    df_m = df_eeg_grp[df_eeg_grp["model"] == model_name]
    head2head_rows.append({
        "target":   "EEG/occipital_parietal",
        "model":    model_name,
        "best_RSA": float(df_m["rsa"].max()),
        "best_CKA": float(df_m["cka"].max()),
    })

df_head2head = pd.DataFrame(head2head_rows)
df_head2head.to_csv("results/head2head_resnet_vs_qwen.csv", index=False)

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
palette = {"ResNet152": "tab:orange", "Qwen3-VL": "tab:blue"}

for ax, metric in zip(axes, ["best_RSA", "best_CKA"]):
    sns.barplot(data=df_head2head, x="target", y=metric, hue="model",
                palette=palette, ax=ax)
    ax.set_title(metric.replace("_", " "))
    ax.set_xlabel("")
    ax.grid(axis="y", alpha=0.3)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=15, ha="right")

axes[0].legend(fontsize=9, loc="upper right")
axes[1].get_legend().remove()
fig.suptitle("ResNet152 vs Qwen3-VL — best representational alignment per target",
             fontsize=12)
fig.tight_layout()

os.makedirs("images", exist_ok=True)
plt.savefig("images/fig_resnet_vs_qwen_representational.png",
            dpi=150, bbox_inches="tight")
plt.show()

print("\nHead-to-head summary:")
print(df_head2head.round(3).to_string(index=False))

In [ ]:
# Direct head-to-head comparison: ResNet152 vs Qwen3-VL on representational metrics.
# TVSD: best score across layers, averaged across monkeys, per ROI.
# EEG : best score across layers and time on the fully-cleaned group-averaged data.
# NSD : best score across layers, averaged across 8 subjects, per ROI.
head2head_rows = []

# --- TVSD: best-over-layers per ROI, averaged across monkeys ---
for model_name in ["ResNet152", "Qwen3-VL"]:
    df_m = df_tvsd[df_tvsd["model"] == model_name]
    for roi in ["V1", "V4", "IT"]:
        df_roi = df_m[df_m["roi"] == roi]
        best_rsa = df_roi.groupby("layer")["rsa"].mean().max()
        best_cka = df_roi.groupby("layer")["cka"].mean().max()
        head2head_rows.append({
            "target": f"TVSD/{roi}",
            "model":  model_name,
            "best_RSA": float(best_rsa),
            "best_CKA": float(best_cka),
        })

# --- EEG: best-over-layers-and-time on cleaned group-averaged data ---
df_eeg_grp = (df_eeg_clean
              .groupby(["model", "layer", "time"])[["rsa", "cka"]]
              .mean()
              .reset_index())
for model_name in ["ResNet152", "Qwen3-VL"]:
    df_m = df_eeg_grp[df_eeg_grp["model"] == model_name]
    head2head_rows.append({
        "target":   "EEG/occipital_parietal",
        "model":    model_name,
        "best_RSA": float(df_m["rsa"].max()),
        "best_CKA": float(df_m["cka"].max()),
    })

# --- NSD: best-over-layers per ROI, averaged across 8 subjects ---
NSD_ROIS_SUBSET = ["V1v", "hV4", "EBA", "PPA"]   # representative subset for the figure
for model_name in ["ResNet152", "Qwen3-VL"]:
    df_m = df_nsd_all[df_nsd_all["model"] == model_name]
    for roi in NSD_ROIS:
        df_roi = df_m[df_m["roi"] == roi]
        best_rsa = df_roi.groupby("layer")["rsa"].mean().max()
        best_cka = df_roi.groupby("layer")["cka"].mean().max()
        head2head_rows.append({
            "target": f"NSD/{roi}",
            "model":  model_name,
            "best_RSA": float(best_rsa),
            "best_CKA": float(best_cka),
        })

df_head2head = pd.DataFrame(head2head_rows)
df_head2head.to_csv("results/head2head_resnet_vs_qwen.csv", index=False)

# --- Plot: three panels, one per dataset ---
fig, axes = plt.subplots(2, 3, figsize=(16, 7), sharey=False)

datasets = {
    "TVSD":    df_head2head[df_head2head["target"].str.startswith("TVSD")],
    "EEG":     df_head2head[df_head2head["target"].str.startswith("EEG")],
    "NSD":     df_head2head[df_head2head["target"].str.startswith("NSD") &
                            df_head2head["target"].str.split("/").str[1].isin(NSD_ROIS_SUBSET)],
}
palette = {"ResNet152": "tab:orange", "Qwen3-VL": "tab:blue"}

for col, (ds_name, df_ds) in enumerate(datasets.items()):
    for row, metric in enumerate(["best_RSA", "best_CKA"]):
        ax = axes[row, col]
        sns.barplot(data=df_ds, x="target", y=metric, hue="model",
                    palette=palette, ax=ax)
        ax.set_title(f"{ds_name} — {metric.replace('best_', '')}")
        ax.set_xlabel("")
        ax.grid(axis="y", alpha=0.3)
        ax.set_xticklabels(
            [t.get_text().split("/")[1] for t in ax.get_xticklabels()],
            rotation=20, ha="right"
        )
        if col == 0:
            ax.set_ylabel(metric.replace("_", " "))
        else:
            ax.set_ylabel("")
        if row == 0 and col == 0:
            ax.legend(fontsize=8, loc="upper right")
        else:
            legend = ax.get_legend()
            if legend:
                legend.remove()

fig.suptitle("ResNet152 vs Qwen3-VL — best representational alignment per dataset and target",
             fontsize=12)
fig.tight_layout()
os.makedirs("images", exist_ok=True)
plt.savefig("images/fig_resnet_vs_qwen_representational.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nHead-to-head summary (all targets):")
print(df_head2head.round(3).to_string(index=False))

In [ ]:
# Scatter plot: ResNet152 vs Qwen3-VL representational alignment
# One point per (dataset, ROI, metric) combination
# Points above diagonal: Qwen3-VL wins; below: ResNet152 wins

fig, axes = plt.subplots(1, 2, figsize=(8, 4))

metrics = ["best_RSA", "best_CKA"]
titles  = ["RSA", "CKA"]

# Color by dataset
dataset_colors = {
    "TVSD":  "steelblue",
    "EEG":   "darkorange",
    "NSD":   "forestgreen",
}

# Points to annotate - key representatives only
key_targets = {"V1", "V4", "IT", "V1v", "PPA", "EBA", "occipital_parietal"}

for ax, metric, title in zip(axes, metrics, titles):
    for ds_label, color in dataset_colors.items():
        df_ds = df_head2head[df_head2head["target"].str.startswith(ds_label)]
        resnet_vals = df_ds[df_ds["model"] == "ResNet152"][metric].values
        qwen_vals   = df_ds[df_ds["model"] == "Qwen3-VL"][metric].values
        targets     = df_ds[df_ds["model"] == "ResNet152"]["target"].str.split("/").str[1].values
        ax.scatter(resnet_vals, qwen_vals, color=color, label=ds_label,
           s=80, zorder=3, edgecolors="white", linewidths=0.5)
        for x, y, t in zip(resnet_vals, qwen_vals, targets):
            if t in key_targets:
                ax.annotate(t, (x, y), fontsize=7, ha="left",
            xytext=(4, 3), textcoords="offset points",
            bbox=dict(boxstyle="round,pad=0.1", fc="white", alpha=0.6, ec="none"))

    # Diagonal
    lims = [0, max(ax.get_xlim()[1], ax.get_ylim()[1])]
    ax.plot(lims, lims, "k--", lw=0.8, alpha=0.5, zorder=1)
    ax.set_xlim(0, lims[1])
    ax.set_ylim(0, lims[1])
    ax.set_xlabel("ResNet152")
    ax.set_ylabel("Qwen3-VL")
    ax.set_title(f"Best {title}")
    ax.grid(alpha=0.25)

axes[0].legend(fontsize=8, loc="upper left")
#fig.suptitle("ResNet152 vs Qwen3-VL — representational alignment per target",
#          fontsize=11)
fig.tight_layout()
os.makedirs("images", exist_ok=True)
plt.savefig("images/fig_resnet_vs_qwen_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

### Head-to-head: ResNet152 vs Qwen3-VL across all datasets

#### Motivation
Having examined each dataset separately, we now bring all three together in a single 
direct comparison. This lets us ask whether one model is uniformly better, or whether 
the advantage depends on the dataset and target, which has implications for what each 
model actually captures about visual processing.

#### Results

| Target | Model | best_RSA | best_CKA |
|---|---|---|---|
| TVSD/V1 | ResNet152 | 0.198 | 0.362 |
| TVSD/V1 | Qwen3-VL | 0.105 | 0.159 |
| TVSD/V4 | ResNet152 | 0.232 | 0.281 |
| TVSD/V4 | Qwen3-VL | 0.068 | 0.106 |
| TVSD/IT | ResNet152 | 0.223 | 0.252 |
| TVSD/IT | Qwen3-VL | 0.086 | 0.129 |
| EEG/occipital_parietal | ResNet152 | 0.141 | 0.150 |
| EEG/occipital_parietal | Qwen3-VL | 0.127 | 0.056 |
| NSD/V1v | ResNet152 | 0.223 | 0.307 |
| NSD/V1v | Qwen3-VL | 0.169 | 0.228 |
| NSD/V2v | ResNet152 | 0.197 | 0.262 |
| NSD/V2v | Qwen3-VL | 0.126 | 0.166 |
| NSD/V3v | ResNet152 | 0.148 | 0.197 |
| NSD/V3v | Qwen3-VL | 0.114 | 0.171 |
| NSD/hV4 | ResNet152 | 0.152 | 0.177 |
| NSD/hV4 | Qwen3-VL | 0.114 | 0.166 |
| NSD/FFA-1 | ResNet152 | 0.179 | 0.179 |
| NSD/FFA-1 | Qwen3-VL | 0.138 | 0.172 |
| NSD/VWFA-1 | ResNet152 | 0.184 | 0.156 |
| NSD/VWFA-1 | Qwen3-VL | 0.139 | 0.149 |
| NSD/PPA | ResNet152 | 0.243 | 0.245 |
| NSD/PPA | Qwen3-VL | 0.181 | 0.255 |
| NSD/OPA | ResNet152 | 0.217 | 0.197 |
| NSD/OPA | Qwen3-VL | 0.157 | 0.198 |
| NSD/EBA | ResNet152 | 0.223 | 0.226 |
| NSD/EBA | Qwen3-VL | 0.157 | 0.217 |

<div style="display: flex; gap: 10px; align-items: flex-start; margin-top: 15px;">
  <figure style="text-align: center; flex: 1;">
    <img src="images/fig_resnet_vs_qwen_representational.png" style="width: 100%;"/>
    <figcaption>Direct head-to-head comparison of ResNet152 and Qwen3-VL using best 
    RSA (top) and best CKA (bottom) across TVSD, EEG, and a representative NSD subset. 
    For TVSD each bar is the best score across layers averaged across both monkeys. For 
    EEG it is the best score across all layers and time points on the cleaned 
    group-averaged data. For NSD it is the best score across layers averaged across 8 
    subjects.</figcaption>
  </figure>
</div>

The picture that emerges is not "one model wins everywhere" but rather a clear 
dataset- and metric-dependent pattern. On TVSD, ResNet152 dominates across all ROIs 
and both metrics, with scores roughly twice those of Qwen3-VL which is consistent with the 
adversarially trained CNN being well-matched to single-unit macaque ventral cortex. On 
EEG, ResNet152 still leads on RSA (0.141 vs 0.127) but the gap on CKA is striking in 
the opposite direction: ResNet152 achieves 0.150 vs Qwen3-VL's 0.056, nearly a 3x 
difference. This is the largest relative CKA gap across all datasets, suggesting that 
ResNet's convolutional features produce a much cleaner inner-product geometry match to 
the occipital_parietal EEG signal than Qwen3-VL's representations.

NSD tells the most nuanced story. For early retinotopic areas (V1v, V2v, V3v, hV4), 
ResNet152 leads on both metrics, though Qwen3-VL is more competitive here than on 
TVSD, for example NSD/V1v CKA is 0.307 vs 0.228, a smaller relative gap than the 
TVSD/V1 CKA of 0.362 vs 0.159. For higher-level areas the picture shifts: on CKA, 
Qwen3-VL actually exceeds ResNet152 for PPA (0.255 vs 0.245) and comes very close for 
OPA (0.198 vs 0.197) and EBA (0.217 vs 0.226). This is consistent with the 
interpretation that Qwen3-VL's language model layers reorganize representations in a 
way that captures higher-order object and scene properties (precisely what PPA, OPA 
and EBA are selective for) while ResNet152's purely visual training gives it an 
advantage in early retinotopic areas.

**Key takeaways**
- ResNet152 wins overall, but the advantage is not uniform: it is largest on TVSD 
  and smallest on higher-level NSD areas
- Qwen3-VL matches or exceeds ResNet152 on CKA for PPA and OPA in NSD, the only 
  targets where the two models are genuinely competitive
- The EEG CKA gap is the largest relative difference across all comparisons, 
  suggesting Qwen3-VL's representations produce a poor inner-product geometry match 
  to scalp EEG despite reasonable RSA scores
- The two models capture complementary aspects of the visual hierarchy: ResNet152 
  is better matched to early visual areas and single-unit responses, Qwen3-VL to 
  higher-order scene and object regions via its language model layers

<div style="background:#fff4c2; border:1px solid #c89b1f; border-left:6px solid #9a6f00; padding:10px 12px; border-radius:6px; margin-top:8px; margin-bottom:4px; color:#241a00; line-height:1.45;"><strong style="color:#5c4300;">Answer box 2.1</strong><br>Do RSA and CKA tell the same story? Identify at least one case where they agree and one case where they disagree, and explain what that might mean.

We computed RSA and CKA between model layer features and neural responses across TVSD, 
THINGS-EEG2, and NSD for both ResNet152 and Qwen3-VL. The two metrics agree on the 
broad picture but diverge in scientifically informative ways.

<strong>Where they agree</strong>

Both metrics confirm that ResNet152 outperforms Qwen3-VL on early visual areas across 
all three datasets, and that model depth tracks the neural hierarchy for ResNet152: 
early layers align best with V1/V1v, deeper layers with IT/EBA/PPA. Both metrics also 
agree that Qwen3-VL's middle visual blocks show consistently weak alignment across all 
datasets and ROIs, and that the NSD hierarchy finding replicates cleanly across 8 
subjects. When both metrics point in the same direction, the conclusion is robust.

<strong>Where they disagree</strong>

The disagreements are more informative than the agreements.

The first concerns Qwen3-VL's language model layers. CKA shows a large jump in 
alignment for higher visual areas (IT, EBA, PPA) at the first language model layer (prominently in both TVSD and NSD) that is much less visible in RSA. This suggests 
the language model reorganizes the inner-product geometry of the representation to be 
more brain-like for higher areas, without changing the rank ordering of pairwise 
stimulus distances that RSA captures. Strikingly, on NSD Qwen3-VL actually exceeds 
ResNet152 on CKA for PPA (0.255 vs 0.245) and nearly matches it for OPA and EBA which is 
something RSA does not reveal. RSA and CKA are therefore sensitive to genuinely 
different aspects of representational structure.

The second concerns the EEG temporal hierarchy. RSA suggests early ResNet layers peak 
earlier (~70–80ms) and deeper layers peak later, consistent with feedforward visual 
processing. CKA shows no such hierarchy, concentrating alignment at layer3-0 around 
100–150ms regardless of depth. Crucially, we showed that the early RSA peak was 
partially artifact-driven in sub-01: removing the corrupted channel 2 substantially 
weakened it, while CKA was completely unaffected by the same artifact. This is the 
most practically important disagreement: RSA is sensitive to structured noise that 
creates spurious pairwise stimulus similarities, while CKA requires a systematic 
inner-product geometry that artifacts do not produce. The group-level results were 
robust because only 2 of 10 subjects had severe artifacts, but the single-subject 
analysis is a clear warning about trusting RSA when data quality is uncertain.

The third concerns geometric sensitivity. CKA shows a more dramatic early V1 
dominance in TVSD and NSD (V1 starts much higher than other ROIs and drops sharply) while RSA shows smaller differences between ROIs in early layers but resolves the 
finer hierarchy among higher-level areas more clearly in deeper layers, where CKA 
values converge. The two metrics are therefore complementary: CKA is more sensitive 
to large geometric reorganizations, RSA to fine-grained ordering differences.

<strong>Why do they diverge on EEG but agree on TVSD and NSD?</strong>

TVSD and NSD responses come from well-defined neural populations in specific visual 
areas, giving the response matrices a clean representational geometry that CKA can 
match to individual model layers. EEG responses in the occipital_parietal ROI 
aggregate activity across 16–17 channels, each capturing a mixture of sources. This 
mixing blurs the inner-product geometry in a way that CKA cannot match to any single 
layer, while RSA's rank-based requirement is more tolerant of this mixing, which also 
makes it more susceptible to artifacts in the same signal.

</div>

#### References

<a id="papale2025"></a>[1] P. Papale, F. Wang, M. W. Self, and P. R. Roelfsema, "An extensive dataset of spiking activity to reveal the syntax of the ventral stream," *Neuron*, vol. 113, no. 4, pp. 539–553, 2025.

<a id="gifford2022"></a>[2] A. T. Gifford, K. Dwivedi, G. Roig, and R. M. Cichy, "A large and rich EEG dataset for modeling human visual object recognition," *NeuroImage*, vol. 264, p. 119754, 2022.

<a id="allen2022"></a>[3] E. J. Allen, G. St-Yves, Y. Wu, J. L. Breedlove, J. S. Prince, L. T. Dowdle, M. Nau, B. Caron, F. Pestilli, I. Charest, et al., "A massive 7T fMRI dataset to bridge cognitive neuroscience and artificial intelligence," *Nature Neuroscience*, vol. 25, no. 1, pp. 116–126, 2022.

<a id="wong2020fast"></a>[4] E. Wong, L. Rice, and J. Z. Kolter, "Fast is better than free: Revisiting adversarial training," <em>arXiv preprint arXiv:2001.03994</em>, 2020.

<a id="bai2025"></a>[5] S. Bai, Y. Cai, R. Chen, K. Chen, X. Chen, Z. Cheng, L. Deng, W. Ding, C. Gao, C. Ge, et al., "Qwen3-VL technical report," *arXiv preprint arXiv:2511.21631*, 2025.

<a id="hebart2019"></a>[6] M. N. Hebart, A. H. Dickter, A. Kidder, W. Y. Kwok, A. Corriveau, C. Van Wicklin, and C. I. Baker, "THINGS: A database of 1,854 object concepts and more than 26,000 naturalistic object images," *PLOS ONE*, vol. 14, no. 10, p. e0223792, 2019.

<a id="lin2014"></a>[7] T.-Y. Lin, M. Maire, S. Belongie, J. Hays, P. Perona, D. Ramanan, P. Dollár, and C. L. Zitnick, "Microsoft COCO: Common objects in context," in *Proc. European Conference on Computer Vision (ECCV)*, pp. 740–755, 2014.

<a id="liao2002"></a>[8] C.-H. Liao, K. J. Worsley, J.-B. Poline, J. A. D. Aston, G. H. Duncan, and A. C. Evans, "Estimating the delay of the fMRI response," *NeuroImage*, vol. 16, no. 3, pp. 593–606, 2002.

<a id="vanbree2025"></a>[9] S. van Bree, M. Styrnal, and M. N. Hebart, "How much variance does your model explain? A clarifying note on the use of split-half reliability for computing noise ceilings," 2025.

<a id="yamins2014"></a>[10] D. L. K. Yamins, H. Hong, C. F. Cadieu, E. A. Solomon, 
D. Seibert, and J. J. DiCarlo, "Performance-optimized hierarchical models predict 
neural responses in higher visual cortex," *Proceedings of the National Academy of 
Sciences*, vol. 111, no. 23, pp. 8619–8624, 2014.

<a id="kravitz2013"></a>[11] D. J. Kravitz, K. S. Saleem, C. I. Baker, L. G. Ungerleider, 
and M. Mishkin, "The ventral visual pathway: an expanded neural framework for the 
processing of object quality," *Trends in Cognitive Sciences*, vol. 17, no. 1, 
pp. 26–49, 2013.

---

## 2.4 Predictive alignment: linear encoding models

## Linear encoding model

In the predictive part of the project, you will map model features to neural responses using a **linear encoding model with L2 regularization (ridge regression)**.

For a stimulus $\mathbf{x}$, let $\mathbf{z}_{\ell}(\mathbf{x})$ denote the feature vector extracted from model layer $\ell$. For a given subject $s$ and neural target $r$ (for example an ROI, a group of voxels, or a set of channels / time points), the predicted neural response is

$$
\widehat{\mathbf{y}}_{r,s}(\mathbf{x})
=
W_{r,s}\,\mathbf{z}_{\ell}(\mathbf{x}) + \mathbf{b}_{r,s},
$$

where:
- $\mathbf{z}_{\ell}(\mathbf{x}) \in \mathbb{R}^{d}$ is the model feature vector,
- $\widehat{\mathbf{y}}_{r,s}(\mathbf{x}) \in \mathbb{R}^{p}$ is the predicted neural response,
- $W_{r,s} \in \mathbb{R}^{p \times d}$ is the learned linear mapping,
- $\mathbf{b}_{r,s} \in \mathbb{R}^{p}$ is a bias term.

We fit the mapping on the training split using ridge regression:

$$
\min_{W_{r,s},\,\mathbf{b}_{r,s}}
\sum_{\mathbf{x}\in\mathcal{D}_{\mathrm{train}}}
\left\|
\mathbf{y}_{r,s}(\mathbf{x}) - \widehat{\mathbf{y}}_{r,s}(\mathbf{x})
\right\|_2^2
\;+\;
\alpha \left\|W_{r,s}\right\|_F^2.
$$

Here, $\mathbf{y}_{r,s}(\mathbf{x})$ is the measured neural response, and $\alpha$ controls the strength of L2 regularization. Larger $\alpha$ penalizes large weights more strongly, which can improve generalization when the feature dimension is high. You should select $\alpha$ using only the training data, for example with a validation split or cross-validation, and then evaluate the final model on the held-out test set.

<div style="background:#eef8f4; border-left:4px solid #5b9a7a; padding:8px 12px; border-radius:6px; font-weight:700; color:#285943;">Select the required targets</div>

Use the following targets:

- **TVSD:** all ROIs
- **EEG2:** `occipital_parietal`
- **NSD:** `V1v`, `V2v`, `V3v`, `hV4`, `FFA-1`, `VWFA-1`, `PPA`, `OPA`, `EBA`

You may explore additional ROIs if you wish.

<div style="background:#fff4c2; border:1px solid #c89b1f; border-left:6px solid #9a6f00; padding:10px 12px; border-radius:6px; margin-top:8px; margin-bottom:4px; color:#241a00; line-height:1.45;"><strong style="color:#5c4300;">Answer box 2.2</strong><br>Briefly explain why these targets are scientifically interesting. Are they chosen mainly for reliability, interpretability, or both?
<p></p>

**TVSD (V1, V4, IT)**: contains the macaque ventral stream from low-level edge detection to high-level object identity. This helps to test whether there is layer–area mapping, e.g. do early model layers align with V1 and late layers with IT.

**EEG2 (occipital_parietal)**: contains channels above visual cortex with millisecond temporal resolution. This enables us to do questions about the temporal structure, e.g. which model layers match the brain at which time points. We can see the unfolding of visual representations from early sensory components (100 ms) to category-selective signatures (150–200 ms).

**NSD ROIs**: combine early retinotopic areas (V1v, V2v, V3v, hV4) with category-selective high-level regions (FFA-1, VWFA-1, PPA, OPA, EBA). This helps us to test how the two models (ResNet and Qwen) could differ in how they map with semantically-defined regions, since multimodal language supervision induces categorical structure that pure visual training does not.

These targets are chosen for both reliability and interpretability. Reliability is ensured because each target comes from preprocessing pipelines that maximize SNR and filters out unreliable units, channels, or voxels. Interpretability is ensured because each target has a well-characterized computational role: TVSD's V1/V4/IT span the canonical macaque ventral hierarchy, enabling tests of layer–area correspondence, EEG2's occipital_parietal grouping isolates visual cortex with high temporal resolution, enabling tests of layer–time correspondence, and NSD's ROI panel combines early retinotopic areas (V1v, V2v, V3v, hV4) with category-selective regions (FFA-1, VWFA-1, PPA, OPA, EBA), enabling tests of whether model representations capture both low-level features and high-level semantic categories which is a particularly diagnostic comparison between the ResNet and the Qwen models.
</div>

<div style="background:#eef5fb; border-left:4px solid #4c78a8; padding:8px 12px; border-radius:6px; font-weight:700; color:#26445e;">What you must do</div>

For each dataset, target, model, and candidate layer:

- fit a **linear encoding model**,
- select hyperparameters without using the test split,
- evaluate on the test split.

Use iterative solvers (e.g. SGD, Adam) when needed to avoid memory issues, since `sklearn` Ridge might cause OOM.

<div style="background:#f3f6fa; border-left:4px solid #7a93ac; padding:8px 12px; border-radius:6px; font-weight:700; color:#32475b;">Required deliverables</div>

You must include all of the following:

1. **A clearly defined train/validation/test procedure** that does not use the test set for model selection.
2. **Linear encoding model results** for all required datasets and targets.
3. **The following predictive metrics:** Pearson correlation, noise-corrected Pearson correlation, explained variance, and noise-corrected explained variance.
4. **The following hybrid representational metrics on predicted responses:** encoding-RSA and encoding-CKA.
5. **Layer-wise plots** showing performance across candidate layers.
6. **One best-layer summary table** for the required targets.
7. **One comparison between the two models** using predictive results.
8. **One short written interpretation** in Answer box 2.3.

### Model training, validation, and test set prediction

<div style="background:#fff5f5; border:1px solid #f5c6c6; border-left:6px solid #cc3333; padding:10px 12px; border-radius:6px; margin-top:8px; margin-bottom:4px; color:#4a0000; line-height:1.45;">

<strong style="color:#cc3333;">Important Note</strong>

The training, validation and test set prediction was run in Google Colab because of the extra compute.
In order to save time (by a magnitude of 10) we performed batch ridge regression over the layers, instead of looping them, utilizing this way the power of GPUs.

<strong>Requirements:</strong>
<ul>
<li>RAM: >100GB</li>
<li>VRAM (GPU): >81GB (NSD requires 81GB of VRAM)</li>
<li>Disk: >150GB</li>
</ul>

Our training strategy:
<ul>
<li>Used a A100 GPU (80GB) for training on the EEG and TVSD datasets (~1-2 hours for both Qwen and ResNet for both datasets)</li>
<li>Used a G4 GPU (95GB) for training on the NSD dataset (~1.5 hours for both models)</li>
</ul>

After training we stored the best model (W and b), best alpha which are stored in the <code>./training_results/..._results.pkl</code> files, the encoded($\hat{Y}$) values of the validation set which are used for training the extended model (part 3) and are stored in the <code>./predictions/..._val_predictions.pkl</code>, and finally the encoded test set stored in the <code>./predictions/..._predictions.pkl</code> files. We have not uploaded the model weights in this folder because of their size (but they are available if needed).

<strong>Executing the notebook</strong><br>
If you want to load the predictions instead of re-training the model, you must execute all the cells except the ones with <code># Train</code> in their title (the last 4 in this subsection).

</div>

This subsection trains the linear (ridge) encoding model end-to-end:

1. **Train**: fit one model per layer on the training split, with all layers fit in parallel on GPU using a batched ridge formulation.
2. **Validate**: hold out 20% of the training split (fixed seed) and use it to pick the best `α` per layer from a small grid.
3. **Predict**: apply the best per-layer model to the test split.

In [ ]:
# Setup: paths, imports, global constants

predictions_dir = './predictions/'
results_dir     = './training_results/'
os.makedirs(predictions_dir, exist_ok=True)
os.makedirs(results_dir,     exist_ok=True)

SUBJECTS = {
    'tvsd': ['monkeyF', 'monkeyN'],
    'eeg2': [f'sub-{i:02d}' for i in range(1, 11)],
    'nsd':  [f'subj0{i}'   for i in range(1, 9)],
}

NSD_ROIS = ['V1v', 'V2v', 'V3v', 'hV4',
            'FFA-1', 'VWFA-1', 'PPA', 'OPA', 'EBA']

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# Dataset loaders. TVSD/EEG2 mirror the loaders used in the analysis below
# (kept here so this section runs standalone); NSD is added for completeness.

def load_tvsd():
    out = defaultdict(dict)
    with h5py.File(TVSD_PATH, 'r') as f:
        train_ids = f['train/stimulus_ids'][:]
        test_ids  = f['test/stimulus_ids'][:]
        out['stimulus_ids'] = {'train': train_ids, 'test': test_ids}
        for m in ['monkeyF', 'monkeyN']:
            for r in ['V1', 'V4', 'IT']:
                train = f[f'train/neural_data/{m}/{r}'][:].astype(np.float32, copy=False)
                test  = f[f'test/neural_data/{m}/{r}'][:].astype(np.float32, copy=False)
                nc = f[f'noise_ceilings/{m}/{r}'][:].astype(np.float32) / 100.0
                out[m][r] = {'train': train, 'test': test, 'noise_ceiling': nc}
    return out


def load_eeg2():
    out = {}
    with h5py.File(EEG_PATH, 'r') as f:
        for s in [f'sub-{i:02d}' for i in range(1, 11)]:
            train = f[f'train/neural_data/{s}/occipital_parietal'][:].astype(np.float32, copy=False)
            test  = f[f'test/neural_data/{s}/occipital_parietal'][:].astype(np.float32, copy=False)
            try:
                train_ids = f[f'train/stimulus_ids/{s}'][:]
                test_ids  = f[f'test/stimulus_ids/{s}'][:]
            except KeyError:
                train_ids = f['train/stimulus_ids'][:]
                test_ids  = f['test/stimulus_ids'][:]
            nc_path = f'noise_ceilings/{s}/occipital_parietal'
            nc = f[nc_path][:].astype(np.float32) / 100.0 if nc_path in f else None
            out[s] = {'train': train, 'test': test, 'noise_ceiling': nc,
                      'train_ids': train_ids, 'test_ids': test_ids}
    return out


def load_nsd(skip_missing_rois=True):
    out = defaultdict(dict)
    with h5py.File(NSD_PATH, 'r') as f:
        for s in [f'subj{i:02d}' for i in range(1, 9)]:
            train_ids = f[f'train/stimulus_ids/{s}'][:]
            test_ids  = f[f'test/stimulus_ids/{s}'][:]
            out[s]['stimulus_ids'] = {'train': train_ids, 'test': test_ids}
            for r in NSD_ROIS:
                k_train = f'train/neural_data/{s}/{r}'
                k_test  = f'test/neural_data/{s}/{r}'
                if k_train not in f or k_test not in f:
                    if skip_missing_rois:
                        continue
                    raise KeyError(f'[NSD] {s}/{r} missing in HDF5.')
                train = f[k_train][:].astype(np.float32, copy=False)
                test  = f[k_test][:].astype(np.float32, copy=False)
                nc_path = f'noise_ceilings/{s}/{r}'
                nc = f[nc_path][:].astype(np.float32) / 100.0 if nc_path in f else None
                out[s][r] = {'train': train, 'test': test, 'noise_ceiling': nc}
    return out

In [ ]:
# Helpers: feature loading, target stacking, batched ridge fit, test prediction.
# `get_layer_keys` and `sort_by_depth` were already defined in section 2.3.

def get_features_batch(train_ids, test_ids, model, layers, dataset_name):
    """Return features for all `layers` as (L, n_train, d) and (L, n_test, d)."""
    fname = 'nsd_stimuli.h5' if dataset_name == 'nsd' else 'things_stimuli.h5'
    with h5py.File(FEAT_DIR + '/' + model + fname, 'r') as f:
        feat_ids  = f['ids'][:]
        id_to_idx = {sid: i for i, sid in enumerate(feat_ids)}
        train_idx = np.array([id_to_idx[sid] for sid in train_ids])
        test_idx  = np.array([id_to_idx[sid] for sid in test_ids])

        all_idx    = np.concatenate([train_idx, test_idx])
        order      = np.argsort(all_idx)
        sorted_idx = all_idx[order]
        unsort     = np.argsort(order)

        per_layer = []
        for layer in layers:
            data = f[layer][sorted_idx, :][unsort]   # restore train/test order
            per_layer.append(data)
        data = np.stack(per_layer, axis=0)           # (L, n_total, d)

    n_train = len(train_idx)
    return (data[:, :n_train].astype(np.float32),
            data[:, n_train:].astype(np.float32))


def get_targets(dataset, dataset_name, subject):
    """Return Y_train, Y_test, roi_slices.

    For TVSD/NSD, ROIs are concatenated along the unit axis and `roi_slices`
    maps ROI -> (start, stop). For EEG2, targets are flattened to
    (n_stim, n_chan*n_time) and roi_slices is None.
    """
    if dataset_name == 'eeg2':
        Y_train = dataset[subject]['train'].reshape(dataset[subject]['train'].shape[0], -1)
        Y_test  = dataset[subject]['test'].reshape(dataset[subject]['test'].shape[0], -1)
        return Y_train, Y_test, None

    rois = ['V1', 'V4', 'IT'] if dataset_name == 'tvsd' \
           else [r for r in NSD_ROIS if r in dataset[subject]]

    Y_tr_parts, Y_te_parts, roi_slices, off = [], [], {}, 0
    for r in rois:
        tr = dataset[subject][r]['train']
        te = dataset[subject][r]['test']
        Y_tr_parts.append(tr); Y_te_parts.append(te)
        roi_slices[r] = (off, off + tr.shape[1])
        off += tr.shape[1]
    return np.hstack(Y_tr_parts), np.hstack(Y_te_parts), roi_slices


def fit_encoding_model_batch(X_train, Y_train, params=None):
    """Fit one ridge model per layer in parallel on GPU.

    X_train : (L, n, d)
    Y_train : (n, p)

    Sweeps `alphas`, picks the best per layer using validation MSE on a
    fixed 80/20 split of the training data. Returns (W, b, best_alphas).
    """
    params   = params or {}
    n_epochs = params.get('n_epochs', 50)
    lr       = params.get('lr', 1e-3)
    alphas   = params.get('alphas', [1e-1, 1.0, 10.0])
    device   = params.get('device', DEVICE)

    L, n, d = X_train.shape
    p = Y_train.shape[1]

    # Fixed-seed 80/20 split, shared across layers
    rng = np.random.default_rng(0)
    perm = rng.permutation(n)
    n_val = int(0.2 * n)
    val_idx, tr_idx = perm[:n_val], perm[n_val:]

    X_tr  = torch.tensor(X_train[:, tr_idx],  dtype=torch.float32, device=device)
    Y_tr  = torch.tensor(Y_train[tr_idx],     dtype=torch.float32, device=device)
    X_val = torch.tensor(X_train[:, val_idx], dtype=torch.float32, device=device)
    Y_val = torch.tensor(Y_train[val_idx],    dtype=torch.float32, device=device)

    # Center features and targets to absorb the bias term (training stats only)
    x_mean = X_tr.mean(dim=1, keepdim=True)   # (L, 1, d)
    y_mean = Y_tr.mean(dim=0, keepdim=True)   # (1, p)
    X_tr.sub_(x_mean);  X_val.sub_(x_mean)
    Y_tr.sub_(y_mean);  Y_val.sub_(y_mean)

    best_W         = torch.zeros(L, p, d, device=device)
    best_val_mse   = torch.full((L,), float('inf'), device=device)
    best_alpha_idx = torch.zeros(L, dtype=torch.long, device=device)

    for ai, alpha in enumerate(alphas):
        W = nn.Parameter(torch.zeros(L, p, d, device=device))
        opt = torch.optim.Adam([W], lr=lr)
        for _ in range(n_epochs):
            opt.zero_grad(set_to_none=True)
            Yhat = torch.einsum('lpd,lnd->lnp', W, X_tr)
            sse = ((Yhat - Y_tr.unsqueeze(0)) ** 2).sum(dim=(1, 2))
            l2  = (W ** 2).sum(dim=(1, 2))
            (sse + alpha * l2).sum().backward()
            opt.step()

        with torch.no_grad():
            Yhat_v  = torch.einsum('lpd,lnd->lnp', W, X_val)
            val_mse = ((Yhat_v - Y_val.unsqueeze(0)) ** 2).mean(dim=(1, 2))

        improved = (val_mse < best_val_mse).nonzero(as_tuple=True)[0]
        if improved.numel() > 0:
            best_W[improved]         = W.detach()[improved]
            best_val_mse[improved]   = val_mse[improved]
            best_alpha_idx[improved] = ai

        print(f'  alpha={alpha:.0e}: ' +
              ' '.join(f'L{l}={val_mse[l].item():.3f}' for l in range(L)))
        del W, opt, Yhat, Yhat_v, val_mse
        torch.cuda.empty_cache()

    b = y_mean - torch.einsum('lpd,ld->lp', best_W, x_mean.squeeze(1))
    best_alphas = [alphas[i.item()] for i in best_alpha_idx]
    for l, a in enumerate(best_alphas):
        print(f'  layer {l}: best alpha={a:.0e}   val_mse={best_val_mse[l].item():.4f}')
    return best_W, b, best_alphas


def predict_test(W, b, X_test, device=None):
    """Apply (L, p, d) weights and (L, p) biases to (L, n_test, d) features."""
    device = device or DEVICE
    with torch.no_grad():
        Wt = W.to(device).float()
        bt = b.to(device).float()
        Xt = torch.as_tensor(X_test, dtype=torch.float32, device=device)
        Yhat = (torch.einsum('lpd,lnd->lnp', Wt, Xt) + bt.unsqueeze(1)).cpu().numpy()
        del Wt, bt, Xt
        torch.cuda.empty_cache()
    return Yhat

In [ ]:
# Train + validate + predict + save for one (model, dataset) combination.
# Replaces the six near-duplicate blocks in the BLCI notebook.

def get_layer_keys(feature_path):
    """All layer datasets under /features/, prefixed."""
    with h5py.File(feature_path, 'r') as f:
        return [f'features/{k}' for k in f['features']]

def sort_by_depth(layers):
    """Sort by integers embedded in layer names (depth-ordered)."""
    return sorted(layers, key=lambda n: tuple(int(x) for x in re.findall(r'\d+', n)))

def align_features(feature_path, layer_key, neural_ids):
    """Load layer features in the row-order of `neural_ids`."""
    with h5py.File(feature_path, 'r') as f:
        feat_ids = f['ids'][:]
        id_to_idx = {fid: i for i, fid in enumerate(feat_ids)}
        feat_idx = np.array([id_to_idx[x] for x in neural_ids])
        order = np.argsort(feat_idx)
        feats = f[layer_key][feat_idx[order], :][np.argsort(order)]
    return feats.astype(np.float32)

def layerwise_rsa_cka(neural_resp, neural_ids, feature_path, layers):
    """Per-layer RSA and CKA between aligned features and neural responses."""
    rsa_s = np.zeros(len(layers))
    cka_s = np.zeros(len(layers))
    for i, layer in enumerate(layers):
        feats = align_features(feature_path, layer, neural_ids)
        rsa_s[i] = rsa(feats, neural_resp)
        cka_s[i] = cka(feats, neural_resp)
    return rsa_s, cka_s

def run_training(model_dir, model_short, dataset_name, dataset_obj=None,
                 fit_params=None):
    """Train, validate, predict, and persist results for one (model, dataset).

    Outputs are written locally:
      ./training_results/{model_short}_{dataset_name}_results.pkl   (W, b, alpha)
      ./predictions/{model_short}_{dataset_name}_predictions.pkl    (test Yhat)

    Returns the in-memory (results, predictions) dicts as well.
    """
    fit_params = fit_params or {}

    if dataset_obj is None:
        dataset_obj = {'tvsd': load_tvsd, 'eeg2': load_eeg2,
                       'nsd':  load_nsd}[dataset_name]()

    feat_file = FEAT_DIR + '/' + model_dir + ('nsd_stimuli.h5' if dataset_name == 'nsd'
                                            else 'things_stimuli.h5')
    layers = sort_by_depth(get_layer_keys(feat_file))

    results, predictions = {}, {}

    for subject in SUBJECTS[dataset_name]:
        if dataset_name == 'tvsd':
            train_ids = dataset_obj['stimulus_ids']['train']
            test_ids  = dataset_obj['stimulus_ids']['test']
        elif dataset_name == 'eeg2':
            train_ids = dataset_obj[subject]['train_ids']
            test_ids  = dataset_obj[subject]['test_ids']
        else:  # 'nsd'
            train_ids = dataset_obj[subject]['stimulus_ids']['train']
            test_ids  = dataset_obj[subject]['stimulus_ids']['test']

        print(f'[{dataset_name}/{model_short}] {subject}: loading features...')
        X_train, X_test = get_features_batch(train_ids, test_ids, model_dir,
                                             layers, dataset_name)
        print(f'[{dataset_name}/{model_short}] {subject}: stacking targets...')
        Y_train, Y_test, roi_slices = get_targets(dataset_obj, dataset_name, subject)
        print(f'  X_train {X_train.shape} | Y_train {Y_train.shape}')

        print(f'[{dataset_name}/{model_short}] {subject}: fitting all layers...')
        W, b, best_alphas = fit_encoding_model_batch(X_train, Y_train, fit_params)

        print(f'[{dataset_name}/{model_short}] {subject}: predicting on test...')
        Yhat = predict_test(W, b, X_test)

        for li, layer in enumerate(layers):
            key = (dataset_name, model_dir, layer, subject)
            results[key]     = {'W': W[li].cpu(), 'b': b[li].cpu(),
                                'alpha': best_alphas[li]}
            predictions[key] = Yhat[li].astype(np.float16)

        if roi_slices:
            results[(dataset_name, subject, 'roi_slices')]     = roi_slices
            predictions[(dataset_name, subject, 'roi_slices')] = roi_slices

        # Free GPU + RAM between subjects
        del X_train, X_test, Y_train, Y_test, W, b, Yhat
        if subject in dataset_obj:
            del dataset_obj[subject]
        gc.collect(); torch.cuda.empty_cache()
        print('-' * 60)

    res_path  = os.path.join(results_dir,     f'{model_short}_{dataset_name}_results.pkl')
    pred_path = os.path.join(predictions_dir, f'{model_short}_{dataset_name}_predictions.pkl')
    with open(res_path,  'wb') as f: pickle.dump(results,     f)
    with open(pred_path, 'wb') as f: pickle.dump(predictions, f)
    print(f'Saved {res_path}')
    print(f'Saved {pred_path}')
    return results, predictions

In [ ]:
# Train: TVSD (ResNet then Qwen)
for model_dir, model_short in [(resnet, 'resnet'), (qwen, 'qwen')]:
    # Fresh dataset per run — run_training drops subjects to free RAM as it goes
    run_training(model_dir, model_short, 'tvsd', dataset_obj=load_tvsd())
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Train: EEG2 (ResNet then Qwen)
for model_dir, model_short in [(resnet, 'resnet'), (qwen, 'qwen')]:
    run_training(model_dir, model_short, 'eeg2', dataset_obj=load_eeg2())
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Train: NSD (optional — heaviest run)
# Skip this cell if you only need TVSD/EEG2 results that the analysis below loads.
for model_dir, model_short in [(resnet, 'resnet'), (qwen, 'qwen')]:
    run_training(model_dir, model_short, 'nsd', dataset_obj=load_nsd())
gc.collect(); torch.cuda.empty_cache()

### Analysis

In [ ]:
def build_roi_slices(dataset, dataset_name, subject):
    if dataset_name == 'tvsd':
        rois = list(dataset[subject].keys())
    elif dataset_name == 'nsd':
        rois = [r for r in NSD_ROIS if r in dataset[subject]]
    else:
        return {}
    roi_slices, off = {}, 0
    for r in rois:
        n = dataset[subject][r]['train'].shape[1]
        roi_slices[r] = (off, off + n)
        off += n
    return roi_slices
    
def get_test_targets_and_nc(dataset, dataset_name, subject, roi_slices):
    if dataset_name == 'eeg2':
        Y  = dataset[subject]['test']
        nc = dataset[subject]['noise_ceiling']                 # already in [0,1] from load_eeg2
        n, C, T = Y.shape
        return Y.reshape(n, C * T).astype(np.float32), nc.reshape(-1).astype(np.float32)

    if not roi_slices:
        raise ValueError(f"empty roi_slices for {dataset_name}/{subject}")

    Yparts, ncparts = [], []
    for r in roi_slices.keys():
        Yparts.append(dataset[subject][r]['test'])
        ncparts.append(dataset[subject][r]['noise_ceiling'])    # already in [0,1] from load_tvsd
    return (np.hstack(Yparts).astype(np.float32),
            np.concatenate(ncparts).astype(np.float32))
def pearsonr_per_target(Yhat, Y, eps=1e-12):
    Yc = Y    - Y.mean(0, keepdims=True)
    Hc = Yhat - Yhat.mean(0, keepdims=True)
    return (Yc * Hc).sum(0) / (np.sqrt((Yc**2).sum(0) * (Hc**2).sum(0)) + eps)

def ev_per_target(Yhat, Y, eps=1e-12):
    ss_res = ((Y - Yhat)**2).sum(0)
    ss_tot = ((Y - Y.mean(0, keepdims=True))**2).sum(0) + eps
    return 1.0 - ss_res / ss_tot

def _slice_bounds(sl):
    """Accepts either a slice object or a (lo, hi) tuple."""
    if isinstance(sl, slice): return sl.start, sl.stop
    return sl[0], sl[1]

In [ ]:
def evaluate_predictions(predictions, dataset, dataset_name, layers,
                         eeg_nc_thresh=0.1):
    rows = []
    subjects = sorted({k[3] for k in predictions if len(k) == 4})

    for subject in subjects:
        roi_slices = predictions.get((dataset_name, subject, 'roi_slices'))
        if not roi_slices:
            roi_slices = build_roi_slices(dataset, dataset_name, subject)

        Y_test, nc = get_test_targets_and_nc(dataset, dataset_name, subject, roi_slices)

        if dataset_name == 'eeg2':
            keep = nc >= eeg_nc_thresh
            agg  = lambda v: float(v[keep].mean()) if keep.any() else np.nan
        else:
            agg  = lambda v: float(np.nanmean(v))

        for li, layer in enumerate(layers):
            hits = [k for k in predictions if len(k)==4 and k[2]==layer and k[3]==subject]
            if not hits:
                continue
            model = hits[0][1]
            Yhat  = predictions[hits[0]].astype(np.float32)

            r     = pearsonr_per_target(Yhat, Y_test)
            ev    = ev_per_target(Yhat, Y_test)
            r_nc  = r  / np.sqrt(np.clip(nc, 1e-6, None))
            ev_nc = ev / np.clip(nc, 1e-6, None)

            base = dict(dataset=dataset_name, model=model, subject=subject,
                        layer=layer, layer_idx=li)

            rows.append({**base, 'roi': 'all',
                'pearsonr':              agg(r),
                'pearsonr_nc':           agg(r_nc),
                'explained_variance':    agg(ev),
                'explained_variance_nc': agg(ev_nc),
                'encoding_RSA':          rsa(Yhat, Y_test),
                'encoding_CKA':          cka(Yhat, Y_test),
            })

            if dataset_name in ('tvsd', 'nsd'):
                for r_name, sl in roi_slices.items():
                    lo, hi = _slice_bounds(sl)
                    rows.append({**base, 'roi': r_name,
                        'pearsonr':              float(np.nanmean(r[lo:hi])),
                        'pearsonr_nc':           float(np.nanmean(r_nc[lo:hi])),
                        'explained_variance':    float(np.nanmean(ev[lo:hi])),
                        'explained_variance_nc': float(np.nanmean(ev_nc[lo:hi])),
                        'encoding_RSA':          rsa(Yhat[:, lo:hi], Y_test[:, lo:hi]),
                        'encoding_CKA':          cka(Yhat[:, lo:hi], Y_test[:, lo:hi]),
                    })
    return pd.DataFrame(rows)

In [ ]:
# Get the predictions
preds_dir = './predictions/'

def load_predictions(model_short, dataset_short):
    fname = f'{model_short}_{dataset_short}_predictions.pkl'
    with open(os.path.join(preds_dir, fname), 'rb') as f:
        preds = pickle.load(f)
    layers = []
    for k in preds:
        if len(k) == 4 and k[2] not in layers:
            layers.append(k[2])
    print(f'  {fname:40s}  →  {len(layers):2d} layers')
    return preds, layers

print('Loaded prediction artefacts:')
prediction_sets = {
    (model_short, dataset_short): load_predictions(model_short, dataset_short)
    for model_short  in ['resnet', 'qwen']
    for dataset_short in ['tvsd', 'eeg2', 'nsd']
}

resnet_tvsd_predictions, resnet_tvsd_layers = prediction_sets[('resnet', 'tvsd')]
qwen_tvsd_predictions,   qwen_tvsd_layers   = prediction_sets[('qwen',   'tvsd')]
resnet_eeg_predictions,  resnet_eeg_layers  = prediction_sets[('resnet', 'eeg2')]
qwen_eeg_predictions,    qwen_eeg_layers    = prediction_sets[('qwen',   'eeg2')]
resnet_eeg_predictions,  resnet_eeg_layers  = prediction_sets[('resnet', 'nsd')]
qwen_eeg_predictions,    qwen_eeg_layers    = prediction_sets[('qwen',   'nsd')]

In [ ]:
# Get the results
dataset_tvsd = load_tvsd()
dataset_eeg  = load_eeg2()
dataset_nsd = load_nsd()
dataset_objs = {'tvsd': dataset_tvsd, 'eeg2': dataset_eeg, 'nsd': dataset_nsd}

model_full_name = {'resnet': resnet, 'qwen': qwen}

all_dfs = []
for (model_short, dataset_short), (preds, layers) in prediction_sets.items():
    df = evaluate_predictions(preds, dataset_objs[dataset_short],
                              dataset_short, layers)
    df['model'] = model_full_name[model_short]
    all_dfs.append(df)

results = pd.concat(all_dfs, ignore_index=True)
results['model_short'] = np.where(
    results['model'].str.contains('resnet', case=False), 'resnet', 'qwen')

metrics = ['pearsonr', 'pearsonr_nc',
           'explained_variance', 'explained_variance_nc',
           'encoding_RSA', 'encoding_CKA']

results.to_csv('./tables/2_4_results.csv', index=False)
results.head()

In [ ]:
def plot_layerwise(df, dataset, metrics_to_plot=('pearsonr_nc', 'explained_variance_nc',
                                                  'encoding_RSA', 'encoding_CKA')):
    sub = df[df['dataset'] == dataset]
    rois = sorted(sub['roi'].unique())

    # single-ROI case: arrange metrics in a 2×2 grid instead of a vertical strip
    if len(rois) == 1:
        roi = rois[0]
        n = len(metrics_to_plot)
        ncols = 2
        nrows = (n + 1) // 2
        fig, axes = plt.subplots(nrows, ncols, figsize=(4.5*ncols, 3.0*nrows), squeeze=False)
        for i, m in enumerate(metrics_to_plot):
            ax = axes[i // ncols, i % ncols]
            d = sub[sub['roi'] == roi]
            g = d.groupby(['model_short', 'layer_idx'])[m].agg(['mean', 'sem']).reset_index()
            for model, gm in g.groupby('model_short'):
                gm = gm.sort_values('layer_idx')
                ax.plot(gm['layer_idx'], gm['mean'], marker='o', ms=3, label=model)
                ax.fill_between(gm['layer_idx'],
                                gm['mean'] - gm['sem'].fillna(0),
                                gm['mean'] + gm['sem'].fillna(0), alpha=0.2)
            ax.set_title(m); ax.set_xlabel('layer idx')
            if i == 0: ax.legend(fontsize=8)
        # hide any unused subplot
        for j in range(n, nrows*ncols):
            axes[j // ncols, j % ncols].axis('off')
        fig.suptitle(f'{dataset} — layer-wise alignment ({roi})', y=1.02)
        fig.tight_layout()
        fig.savefig(f'./plots/2_4_layerwise_plot_{dataset}', bbox_inches='tight')
        plt.show()
        return

    # multi-ROI case (TVSD, NSD): metrics × ROIs grid
    fig, axes = plt.subplots(len(metrics_to_plot), len(rois),
                             figsize=(3.2*len(rois), 2.6*len(metrics_to_plot)),
                             squeeze=False, sharex=True)
    for i, m in enumerate(metrics_to_plot):
        for j, roi in enumerate(rois):
            ax = axes[i, j]
            d = sub[sub['roi'] == roi]
            g = d.groupby(['model_short', 'layer_idx'])[m].agg(['mean', 'sem']).reset_index()
            for model, gm in g.groupby('model_short'):
                gm = gm.sort_values('layer_idx')
                ax.plot(gm['layer_idx'], gm['mean'], marker='o', ms=3, label=model)
                ax.fill_between(gm['layer_idx'],
                                gm['mean'] - gm['sem'].fillna(0),
                                gm['mean'] + gm['sem'].fillna(0), alpha=0.2)
            if i == 0: ax.set_title(roi)
            if j == 0: ax.set_ylabel(m)
            if i == len(metrics_to_plot)-1: ax.set_xlabel('layer idx')
            if i == 0 and j == 0: ax.legend(fontsize=8)
    fig.suptitle(f'{dataset} — layer-wise alignment', y=1.01)
    fig.tight_layout()
    fig.savefig(f'./plots/2_4_layerwise_plot_{dataset}', bbox_inches='tight')
    plt.show()

for ds in results['dataset'].unique():
    plot_layerwise(results, ds)

In [ ]:
layer_avg = (results
             .groupby(['dataset', 'model_short', 'roi', 'layer_idx', 'layer'],
                      as_index=False)[metrics]
             .mean())
best_idx   = layer_avg.groupby(['dataset', 'model_short', 'roi'])['pearsonr_nc'].idxmax()
best_layer = (layer_avg.loc[best_idx]
                       .sort_values(['dataset', 'roi', 'model_short'])
                       .reset_index(drop=True))
best_layer.to_csv('./tables/2_4_best_layer.csv')
best_layer

In [ ]:
for metric in ['pearsonr_nc', 'explained_variance_nc']:
    g = sns.catplot(data=best_layer, x='roi', y=metric, hue='model_short',
                    col='dataset', kind='bar', sharex=False, sharey=False,
                    height=3.5, aspect=1.3,
                    palette={'resnet': 'tab:orange', 'qwen': 'tab:blue'})
    g.set_titles('{col_name}'); g.set_xticklabels(rotation=30)
    g.fig.suptitle(f'Best-layer {metric}', y=1.03)
    g.fig.savefig(f'./plots/2_4_best_{metric}.png', dpi=200, bbox_inches='tight')
    plt.show()
comp = (best_layer
        .pivot_table(index=['dataset', 'roi'], columns='model_short',
                     values=['pearsonr_nc', 'explained_variance_nc',
                             'encoding_RSA', 'encoding_CKA'])
        .round(3))
comp.to_csv('./tables/2_4_comp.csv')
comp

### Deliverables

**Linear encoding model results with predictive metrics and hyprid representational metrics**

In [ ]:
# Read the results from the table
pd.read_csv('./tables/2_4_results.csv')

**Layer-wise plots showing performance across candidate layers**

![Alt Text](./plots/2_4_layerwise_plot_tvsd.png)
![Alt Text](./plots/2_4_layerwise_plot_eeg2.png)
![Alt Text](./plots/2_4_layerwise_plot_nsd.png)

**One best-layer summary table for the required targets**

In [ ]:
# Read summary table
pd.read_csv('./tables/2_4_best_layer.csv')

**Summary table**

| dataset | model_short | roi | layer_idx | layer | pearsonr | pearsonr_nc | explained_variance | explained_variance_nc | encoding_RSA | encoding_CKA |
|---------|-------------|-------|-----------|-------------------------------------|----------|-------------|--------------------|-----------------------|--------------|--------------|
| eeg2    | qwen        | all   | 2         | features/visual-blocks-6            | 0.302291 | 0.426140    | 0.023294           | -0.195473             | 0.245456     | 0.217888     |
| eeg2    | resnet      | all   | 8         | features/layer3-30                  | 0.361624 | 0.524416    | 0.047009           | -0.097487             | 0.274165     | 0.254705     |
| nsd     | qwen        | EBA   | 1         | features/language_model-layers-3    | 0.303451 | 0.479989    | 0.004401           | -0.278088             | 0.538863     | 0.494480     |
| nsd     | resnet      | EBA   | 8         | features/layer3-30                  | 0.394561 | 0.660172    | 0.138684           | 0.332639              | 0.524586     | 0.474260     |
| nsd     | qwen        | FFA-1 | 3         | features/language_model-layers-8    | 0.295351 | 0.491413    | 0.043380           | -0.111888             | 0.453220     | 0.466971     |
| nsd     | resnet      | FFA-1 | 8         | features/layer3-30                  | 0.378635 | 0.667539    | 0.122203           | 0.327438              | 0.475465     | 0.446728     |
| nsd     | qwen        | OPA   | 3         | features/language_model-layers-8    | 0.276982 | 0.478835    | -0.004938          | -0.202821             | 0.319009     | 0.289766     |
| nsd     | resnet      | OPA   | 8         | features/layer3-30                  | 0.381373 | 0.685457    | 0.123803           | 0.354586              | 0.380680     | 0.322676     |
| nsd     | qwen        | PPA   | 1         | features/language_model-layers-3    | 0.327038 | 0.551976    | -0.071610          | -0.566016             | 0.337196     | 0.399137     |
| nsd     | resnet      | PPA   | 8         | features/layer3-30                  | 0.433525 | 0.761193    | 0.168501           | 0.459092              | 0.451122     | 0.486668     |
| nsd     | qwen        | V1v   | 2         | features/visual-blocks-6            | 0.398278 | 0.607226    | 0.135364           | 0.169204              | 0.332480     | 0.420179     |
| nsd     | resnet      | V1v   | 3         | features/layer3-5                   | 0.434094 | 0.687225    | 0.164295           | 0.359961              | 0.365662     | 0.460867     |
| nsd     | qwen        | V2v   | 2         | features/visual-blocks-6            | 0.364413 | 0.564056    | 0.106170           | 0.117768              | 0.291913     | 0.375825     |
| nsd     | resnet      | V2v   | 6         | features/layer3-20                  | 0.418768 | 0.673383    | 0.156730           | 0.355163              | 0.359849     | 0.433672     |
| nsd     | qwen        | V3v   | 2         | features/visual-blocks-6            | 0.309723 | 0.502601    | 0.074716           | 0.068809              | 0.249889     | 0.291979     |
| nsd     | resnet      | V3v   | 6         | features/layer3-20                  | 0.387358 | 0.657740    | 0.128989           | 0.328898              | 0.315659     | 0.352664     |
| nsd     | qwen        | VWFA-1| 1         | features/language_model-layers-3    | 0.237141 | 0.414909    | -0.072714          | -0.499428             | 0.380967     | 0.261599     |
| nsd     | resnet      | VWFA-1| 8         | features/layer3-30                  | 0.326285 | 0.599849    | 0.092933           | 0.268093              | 0.400031     | 0.270744     |
| nsd     | qwen        | all   | 3         | features/language_model-layers-8    | 0.298085 | 0.485299    | 0.033135           | -0.123442             | 0.590516     | 0.462408     |
| nsd     | resnet      | all   | 8         | features/layer3-30                  | 0.390159 | 0.668065    | 0.132893           | 0.338422              | 0.581147     | 0.465064     |
| nsd     | qwen        | hV4   | 3         | features/language_model-layers-8    | 0.279629 | 0.475295    | 0.019686           | -0.129207             | 0.296902     | 0.289644     |
| nsd     | resnet      | hV4   | 6         | features/layer3-20                  | 0.359760 | 0.638993    | 0.112438           | 0.313755              | 0.306610     | 0.284169     |
| tvsd    | qwen        | IT    | 2         | features/visual-blocks-6            | 0.515872 | 0.528296    | -0.373477          | -0.457581             | 0.376385     | 0.470699     |
| tvsd    | resnet      | IT    | 6         | features/layer3-20                  | 0.650472 | 0.668110    | 0.190403           | 0.181840              | 0.548764     | 0.688830     |
| tvsd    | qwen        | V1    | 0         | features/visual-blocks-2            | 0.637874 | 0.649884    | -0.465482          | -0.541282             | 0.338686     | 0.591888     |
| tvsd    | resnet      | V1    | 1         | features/layer2-0                   | 0.749583 | 0.764437    | 0.303700           | 0.296705              | 0.688230     | 0.832022     |
| tvsd    | qwen        | V4    | 2         | features/visual-blocks-6            | 0.579484 | 0.589393    | -0.183207          | -0.236603             | 0.479921     | 0.639731     |
| tvsd    | resnet      | V4    | 2         | features/layer3-0                   | 0.690108 | 0.703142    | 0.326615           | 0.323692              | 0.697896     | 0.788300     |
| tvsd    | qwen        | all   | 2         | features/visual-blocks-6            | 0.564281 | 0.575535    | -0.459604          | -0.536023             | 0.411115     | 0.507364     |
| tvsd    | resnet      | all   | 2         | features/layer3-0                   | 0.687509 | 0.702009    | 0.297031           | 0.292226              | 0.722240     | 0.834868     |

**One comparison between the two models using predictive results**

In [ ]:
pd.read_csv('./tables/2_4_comp.csv')

**Comparison of models using a table**

| dataset | roi    | CKA (qwen) | CKA (resnet) | RSA (qwen) | RSA (resnet) | EV_nc (qwen) | EV_nc (resnet) | pearsonr_nc (qwen) | pearsonr_nc (resnet) |
|---------|--------|------------|--------------|------------|--------------|--------------|----------------|--------------------|----------------------|
| eeg2    | all    | 0.218      | 0.255        | 0.245      | 0.274        | -0.195       | -0.097         | 0.426              | 0.524                |
| nsd     | EBA    | 0.494      | 0.474        | 0.539      | 0.525        | -0.278       | 0.333          | 0.480              | 0.660                |
| nsd     | FFA-1  | 0.467      | 0.447        | 0.453      | 0.475        | -0.112       | 0.327          | 0.491              | 0.668                |
| nsd     | OPA    | 0.290      | 0.323        | 0.319      | 0.381        | -0.203       | 0.355          | 0.479              | 0.685                |
| nsd     | PPA    | 0.399      | 0.487        | 0.337      | 0.451        | -0.566       | 0.459          | 0.552              | 0.761                |
| nsd     | V1v    | 0.420      | 0.461        | 0.332      | 0.366        | 0.169        | 0.360          | 0.607              | 0.687                |
| nsd     | V2v    | 0.376      | 0.434        | 0.292      | 0.360        | 0.118        | 0.355          | 0.564              | 0.673                |
| nsd     | V3v    | 0.292      | 0.353        | 0.250      | 0.316        | 0.069        | 0.329          | 0.503              | 0.658                |
| nsd     | VWFA-1 | 0.262      | 0.271        | 0.381      | 0.400        | -0.499       | 0.268          | 0.415              | 0.600                |
| nsd     | all    | 0.462      | 0.465        | 0.591      | 0.581        | -0.123       | 0.338          | 0.485              | 0.668                |
| nsd     | hV4    | 0.290      | 0.284        | 0.297      | 0.307        | -0.129       | 0.314          | 0.475              | 0.639                |
| tvsd    | IT     | 0.471      | 0.689        | 0.376      | 0.549        | -0.458       | 0.182          | 0.528              | 0.668                |
| tvsd    | V1     | 0.592      | 0.832        | 0.339      | 0.688        | -0.541       | 0.297          | 0.650              | 0.764                |
| tvsd    | V4     | 0.640      | 0.788        | 0.480      | 0.698        | -0.237       | 0.324          | 0.589              | 0.703                |
| tvsd    | all    | 0.507      | 0.835        | 0.411      | 0.722        | -0.536       | 0.292          | 0.576              | 0.702                |

**Comparison using plots**
![Alt Text](./plots/2_4_best_pearsonr_nc.png)
![Alt Text](./plots/2_4_best_explained_variance_nc.png)

<div style="background:#fff4c2; border:1px solid #c89b1f; border-left:6px solid #9a6f00; padding:10px 12px; border-radius:6px; margin-top:8px; margin-bottom:4px; color:#241a00; line-height:1.45;"><strong style="color:#5c4300;">Answer box 2.3</strong><br>Which model and which layer perform best for each dataset? Summarize the main trends in a short paragraph.
<p></p>

**Best model and best layer per dataset**

| Dataset | ROI    | Best model       | Best layer        | layer_idx | r_NC      | encoding-RSA | encoding-CKA |
|---------|--------|------------------|-------------------|-----------|-----------|--------------|--------------|
| **TVSD** | V1    | **ResNet-152**  | `layer2-0`        | 1         | **0.764** | 0.688        | 0.832        |
| **TVSD** | V4    | **ResNet-152**  | `layer3-0`        | 2         | **0.703** | 0.698        | 0.788        |
| **TVSD** | IT    | **ResNet-152**  | `layer3-20`       | 6         | **0.668** | 0.549        | 0.689        |
| **TVSD** | all   | **ResNet-152**  | `layer3-0`        | 2         | **0.702** | 0.722        | **0.835**    |
| **NSD**  | V1v   | **ResNet-152**  | `layer3-5`        | 3         | **0.687** | 0.366        | 0.461        |
| **NSD**  | V2v   | **ResNet-152**  | `layer3-20`       | 6         | **0.673** | 0.360        | 0.434        |
| **NSD**  | V3v   | **ResNet-152**  | `layer3-20`       | 6         | **0.658** | 0.316        | 0.353        |
| **NSD**  | hV4   | **ResNet-152**  | `layer3-20`       | 6         | **0.639** | 0.307        | 0.284        |
| **NSD**  | EBA   | **ResNet-152**  | `layer3-30`       | 8         | **0.660** | 0.525        | 0.474        |
| **NSD**  | FFA-1 | **ResNet-152**  | `layer3-30`       | 8         | **0.668** | 0.475        | 0.447        |
| **NSD**  | OPA   | **ResNet-152**  | `layer3-30`       | 8         | **0.685** | 0.381        | 0.323        |
| **NSD**  | PPA   | **ResNet-152**  | `layer3-30`       | 8         | **0.761** | 0.451        | **0.487**    |
| **NSD**  | VWFA-1| **ResNet-152**  | `layer3-30`       | 8         | **0.600** | 0.400        | 0.271        |
| **NSD**  | all   | **ResNet-152**  | `layer3-30`       | 8         | **0.668** | 0.581        | 0.465        |
| **EEG2** | all   | **ResNet-152**  | `layer3-30`       | 8         | **0.524** | 0.274        | 0.255        |

**Best model and best layer per dataset**

ResNet wins on every dataset and every ROI. On macaque electrophysiology (TVSD), ResNet's best-aligned layers reproduce the canonical ventral-stream hierarchy: V1 ↔ layer2-0 (idx 1, r_NC = 0.764), V4 ↔ layer3-0 (idx 2, r_NC = 0.703), IT ↔ layer3-20 (idx 6, r_NC = 0.668). The layer-wise curves show the right hierarchy signature: ResNet alignment for V1 decays smoothly with depth, while for IT it rises with depth and plateaus in the deep layer3 blocks — the canonical early-layer ↔ early-area / late-layer ↔ late-area pattern. Qwen oscillates block-to-block and trails ResNet by ~0.13 in r_NC and ~0.3 in encoding-CKA on TVSD. On human EEG (EEG2), ResNet's layer3-30 (idx 8) is best (r_NC = 0.524), beating Qwen's best block visual-blocks-6 (idx 2, r_NC = 0.426); ResNet also wins on every other EEG metric (RSA, CKA, encoding-RSA, encoding-CKA). On NSD, ResNet wins across all 9 ROIs, and the best-layer index broadly tracks ROI position in the ventral stream: early visual ROIs (V1v, V2v, V3v, hV4) peak at layer3-5–layer3-20 (idx 3–6), while category-selective high-level regions (FFA-1, EBA, OPA, PPA, VWFA-1) all peak at the deepest layer3-30 (idx 8). The strongly negative explained_variance_nc values for Qwen — visible across all of Image 5 — reflect raw EV falling below the mean-predictor baseline and being amplified by noise-ceiling division, rather than a genuine representational collapse, so the rankings rely on r_NC, encoding-RSA, and encoding-CKA, which all agree. Qwen's EEG curve also zig-zags sharply while ResNet rises smoothly — the oscillation likely reflects alternating attention/MLP sub-blocks in the transformer rather than a clean representational hierarchy.

</div>

---

## 2.5 Compare predictive and representational metrics

<div style="background:#eef5fb; border-left:4px solid #4c78a8; padding:8px 12px; border-radius:6px; font-weight:700; color:#26445e;">What you must do</div>

Compare the ranking of models and layers according to:

- Pearson correlation,
- explained variance,
- RSA,
- CKA.
- encoding-RSA/ encoding-CKA

Discuss whether the same layers are favored by all metrics.

<div style="background:#f3f6fa; border-left:4px solid #7a93ac; padding:8px 12px; border-radius:6px; font-weight:700; color:#32475b;">Required deliverables</div>

You must include all of the following:

1. **One figure comparing layer or model rankings across metrics**
2. **One concrete example where two metrics agree**
3. **One concrete example where two metrics disagree**
4. **One short written interpretation** in Answer box 2.4.

In [ ]:
results = pd.read_csv('./tables/2_4_results.csv')
model_paths = {
    'qwen':   FEAT_DIR + '/' + qwen,
    'resnet': FEAT_DIR + '/' + resnet,
}

def tvsd_test_targets(dataset_tvsd, subject):
    """Per-ROI test responses + concatenated 'all'."""
    rois = ['V1', 'V4', 'IT']
    parts = {r: dataset_tvsd[subject][r]['test'].astype(np.float32) for r in rois}
    parts['all'] = np.hstack([parts[r] for r in rois])
    return parts

def eeg2_test_targets(dataset_eeg, subject):
    """Flatten (n, C, T) -> (n, C*T) to match the 2.4 encoding target."""
    Y = dataset_eeg[subject]['test'].astype(np.float32)
    n, C, T = Y.shape
    return {'all': Y.reshape(n, C * T)}

def nsd_test_targets(dataset_nsd, subject):
    """Per-ROI test responses + concatenated 'all'."""
    rois = ['V1v', 'V2v', 'V3v', 'hV4', 'FFA-1', 'VWFA-1', 'PPA', 'OPA', 'EBA']
    parts = {r: dataset_nsd[subject][r]['test'].astype(np.float32) for r in rois}
    parts['all'] = np.hstack([parts[r] for r in rois])
    return parts

raw_rows = []
for dataset_short in ['nsd', 'eeg2', 'tvsd']:
    print(f'Starting {dataset_short}')
    for model_short in ['qwen', 'resnet']:
        fp = model_paths[model_short] + ('nsd_stimuli.h5' if dataset_short == 'nsd' else 'things_stimuli.h5')
        sub = results[(results['dataset']     == dataset_short) &
                      (results['model_short'] == model_short)]

        layers_here = (sub[['layer', 'layer_idx']]
                       .drop_duplicates()
                       .sort_values('layer_idx')
                       .reset_index(drop=True))
        subjects = sorted(sub['subject'].unique())

        for _, lrow in layers_here.iterrows():
            layer, li = lrow['layer'], int(lrow['layer_idx'])
            print(f'Exploring layer {layer} of model {model_short} in dataset {dataset_short}')
            X_cache = {}                       # cache features per unique ids

            for s in subjects:
                if dataset_short == 'tvsd':
                    test_ids = dataset_tvsd['stimulus_ids']['test']
                    targets  = tvsd_test_targets(dataset_tvsd, s)
                elif dataset_short == 'nsd':
                    test_ids = dataset_nsd[s]['stimulus_ids']['test']
                    targets = nsd_test_targets(dataset_nsd, s)
                else:
                    test_ids = dataset_eeg[s]['test_ids']
                    targets  = eeg2_test_targets(dataset_eeg, s)

                key = test_ids.tobytes()
                if key not in X_cache:
                    X_cache[key] = align_features(fp, layer, test_ids)
                X = X_cache[key]

                for roi, Y in targets.items():
                    raw_rows.append(dict(
                        dataset=dataset_short, model_short=model_short,
                        subject=s, roi=roi,
                        layer=layer, layer_idx=li,
                        RSA=float(rsa(X, Y)),
                        CKA=float(cka(X, Y)),
                    ))

raw_rsa_cka = pd.DataFrame(raw_rows)

# Merge raw RSA/CKA with the predictive + encoding metrics from 2.4
key_cols = ['dataset', 'model_short', 'subject', 'roi', 'layer', 'layer_idx']
metrics_long = results.merge(raw_rsa_cka, on=key_cols, how='left')

# Sanity: nothing should be missing after the merge
missing = metrics_long[metrics_long[['RSA', 'CKA']].isna().any(axis=1)]
assert len(missing) == 0, f'{len(missing)} rows missing raw RSA/CKA'

# Tidy column order: keys, predictive metrics, representational metrics
ordered = key_cols + ['model'] + [
    'pearsonr', 'pearsonr_nc',
    'explained_variance', 'explained_variance_nc',
    'RSA', 'CKA',
    'encoding_RSA', 'encoding_CKA',
]
metrics_long = metrics_long[[c for c in ordered if c in metrics_long.columns]]

metrics_long.to_csv('./tables/2_5_all_metrics.csv', index=False)
metrics_long.head()

In [ ]:
metric_cols = ['pearsonr_nc', 'explained_variance_nc',
               'RSA', 'CKA', 'encoding_RSA', 'encoding_CKA']

layer_avg = (metrics_long
             .groupby(['dataset', 'model_short', 'roi', 'layer_idx'],
                      as_index=False)[metric_cols].mean())

rho_rows = []
for (ds, mdl, roi), g in layer_avg.groupby(['dataset', 'model_short', 'roi']):
    g = g.sort_values('layer_idx')
    if len(g) < 3:               # Spearman needs ≥ 3 layers to be meaningful
        continue
    for m1, m2 in combinations(metric_cols, 2):
        rho_rows.append({
            'dataset': ds,
            'cell'   : f'{ds}/{mdl}/{roi}',
            'm1': m1, 'm2': m2,
            'rho': spearmanr(g[m1], g[m2]).statistic,
        })
rho_df = pd.DataFrame(rho_rows)

def mean_rho_matrix(df_subset, metrics):
    M = pd.DataFrame(np.eye(len(metrics)), index=metrics, columns=metrics)
    if len(df_subset) == 0:
        return M
    for (m1, m2), v in df_subset.groupby(['m1', 'm2'])['rho'].mean().items():
        M.loc[m1, m2] = M.loc[m2, m1] = v
    return M

datasets = sorted(rho_df['dataset'].unique())
panels   = []
for ds in datasets:
    sub = rho_df[rho_df['dataset'] == ds]
    panels.append((f'{ds}  (n={sub["cell"].nunique()} cells)',
                   mean_rho_matrix(sub, metric_cols)))
panels.append((f'pooled  (n={rho_df["cell"].nunique()} cells)',
               mean_rho_matrix(rho_df, metric_cols)))

n_panels = len(panels)
fig, axes = plt.subplots(
    1, n_panels,
    figsize=(4.6 * n_panels + 0.6, 4.8),
    gridspec_kw={'width_ratios': [1] * n_panels},
)
if n_panels == 1:
    axes = [axes]

for i, (ax, (title, M)) in enumerate(zip(axes, panels)):
    show_y    = (i == 0)                    # y-tick labels only on first panel
    show_cbar = (i == n_panels - 1)         # colorbar only on last panel
    sns.heatmap(
        M, annot=True, fmt='.2f', cmap='RdBu_r',
        vmin=-1, vmax=1, square=True,
        cbar=show_cbar,
        cbar_kws={'label': 'mean Spearman ρ'} if show_cbar else None,
        yticklabels=metric_cols if show_y else False,
        xticklabels=metric_cols,
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha='right')
    if show_y:
        ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

fig.suptitle('Layer-ranking agreement across metrics, per dataset and pooled',
             y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig('./plots/2_5_metric_ranking_comparison.png',
            dpi=200, bbox_inches='tight')
plt.show()

![Alt Text](./plots/2_5_metric_ranking_comparison.png)

**How the heatmap is computed.** For each `(dataset, model, ROI)` cell we have one layer-curve per metric (10 layers × 6 metrics, averaged across subjects). Within each cell we compute the Spearman ρ between every pair of metrics' layer rankings — so each cell contributes one ρ per metric pair. The heatmap entry at `(metric_i, metric_j)` is then the **mean ρ across all cells** for that dataset (or pooled across all 10 cells for the rightmost panel). High values mean the two metrics rank the layers in the same order; values near zero mean independent rankings; negative values mean the metrics rank layers oppositely.

**Concrete example where two metrics agree.** `pearsonr_nc` and `encoding_RSA` rank the layers near-identically (pooled Spearman ρ = 0.88; TVSD ρ = 0.93).

**Concrete example where two metrics disagree.** `pearsonr_nc` and raw `RSA` flip sign across modalities: TVSD ρ = +0.37 but EEG2 ρ = −0.50.

<div style="background:#fff4c2; border:1px solid #c89b1f; border-left:6px solid #9a6f00; padding:10px 12px; border-radius:6px; margin-top:8px; margin-bottom:4px; color:#241a00; line-height:1.45;"><strong style="color:#5c4300;">Answer box 2.4</strong><br>Does a model that is representationally similar to the brain also predict neural responses well? Use at least one example from your results.
<p></p>
Only weakly, and the answer depends on the modality. The raw representational metrics (RSA, CKA), computed directly between unweighted model features and neural responses, correlate only modestly with predictive accuracy on TVSD and NSD, and the correlation collapses to roughly zero on EEG2. So the layer whose 30k-dimensional features are geometrically closest to the brain is not generally the layer that best predicts neural responses — especially for high-dimensional, noise-dominated EEG targets.
A plausible reason is that the ridge encoder can selectively re-weight feature dimensions and target dimensions to exploit high-SNR channel–time combinations, while raw RSA and CKA see all dimensions equally. On TVSD and NSD the targets are already aggregated (mean spike rate per unit; voxel betas), so their raw geometry is less noise-dominated and tracks predictability better. On EEG, where the target is flattened (channels × time) and most cells are noise, raw similarity stops being a useful proxy.
<br>

The encoder-derived metrics (encoding_RSA, encoding_CKA) track pearsonr_nc very tightly across all datasets, but this is expected rather than informative: all three are summaries of the same fitted predictions Ŷ on the same test stimuli, so they cannot disagree much. The strong agreement is best read as a sanity check on the encoding pipeline rather than as evidence that representational similarity predicts predictive accuracy.

Concrete agreement: on NSD, raw RSA and raw CKA rank the layers similarly — two independently-defined geometric metrics tell the same story about which layers are most brain-like.

Concrete disagreement: raw CKA vs. pearsonr_nc is moderately positive on TVSD but essentially zero on EEG2 — the same pair of metrics gives different answers depending on modality.

<div

---

## 2.6 Relate layer hierarchy to brain hierarchy

<div style="background:#eef5fb; border-left:4px solid #4c78a8; padding:8px 12px; border-radius:6px; font-weight:700; color:#26445e;">What you must do</div>

Test whether deeper layers align better with higher-level neural targets.

- Does TVSD IT align with deeper layers than V1?
- Do higher-level NSD regions prefer later layers?
- For EEG, are particular time windows associated with later layers?

<div style="background:#f3f6fa; border-left:4px solid #7a93ac; padding:8px 12px; border-radius:6px; font-weight:700; color:#32475b;">Required deliverables</div>

You must include one of the following analyses:

1. **A heatmap of layer × ROI**
2. **A ranked-layer plot by ROI**
3. **A time-resolved EEG layer comparison**

You must also include a short written conclusion in Answer box 2.5 stating whether the results support a hierarchy correspondence.

In [ ]:
ROI = ['V1', 'V4', 'IT']
fig, axes = plt.subplots(1, 2, figsize=(9, 5))
for ax, model in zip(axes, ['resnet', 'qwen']):
    pv = (metrics_long.query("dataset == 'tvsd' and roi in @ROI and model_short == @model")
          .groupby(['layer_idx', 'layer', 'roi'])['pearsonr_nc'].mean()
          .unstack('roi')[ROI].sort_index(level='layer_idx'))
    sns.heatmap(pv.values, xticklabels=ROI,
                yticklabels=[l.replace('features/', '') for _, l in pv.index],
                cmap='viridis', annot=True, fmt='.2f', ax=ax,
                cbar_kws={'label': 'pearsonr_nc'})
    ax.set_title(model); ax.set_xlabel('ROI'); ax.set_ylabel('layer')
fig.tight_layout()
fig.savefig('./plots/2_6_tvsd_layer_roi_heatmap.png', dpi=200, bbox_inches='tight')
plt.show()

<div style="background:#fff4c2; border:1px solid #c89b1f; border-left:6px solid #9a6f00; padding:10px 12px; border-radius:6px; margin-top:8px; margin-bottom:4px; color:#241a00; line-height:1.45;"><strong style="color:#5c4300;">Answer box 2.5</strong><br>Is there evidence for a correspondence between model depth and neural hierarchy? State your conclusion clearly and support it with results.
<p></p>

![Alt Text](./plots/2_6_tvsd_layer_roi_heatmap.png)

On TVSD, ResNet's best-aligned layer index increases monotonically across
the ventral stream — `layer2-0 → layer3-0 → layer3-15`. The
heatmap shows a clean crossover: shallow layers favour V1 by a wide
margin while deep layers reverse the ordering. This reproduces the
CNN and ventral-stream correspondence, with IT peaking
towards the deeper layer.

For Qwen the correspondence is only partial. V1 still separates cleanly,
but V4 and IT both peak at the same mid-block and the V4, IT distinction is lost.
Also, the heatmap alternates row-by-row because visual-blocks outperform every
interleaved `language_model-layers` row at every depth, and visual-block alignment
itself decays after idx 2. The transformer's late blocks therefore drift *away*
from primate ventral cortex toward a multimodal/semantic abstraction that no longer
matches single-unit geometry.

And thus the hierarchy correspondence holds cleanly for ResNet on TVSD and breaks
down for Qwen beyond V1.
</div>

---

## 2.7 Compare the two feature extractors

<div style="background:#eef5fb; border-left:4px solid #4c78a8; padding:8px 12px; border-radius:6px; font-weight:700; color:#26445e;">What you must do</div>

Compare **Qwen3-VL-2B-Instruct** and **Adv-ResNet152** across datasets, ROIs, layers, and metrics.

<div style="background:#f3f6fa; border-left:4px solid #7a93ac; padding:8px 12px; border-radius:6px; font-weight:700; color:#32475b;">Required deliverables</div>

You must include all of the following:

1. **One summary figure comparing Qwen3-VL-2B-Instruct and Adv-ResNet152**
2. **One table of best scores across datasets and targets**
3. **One short written interpretation** in Answer box 2.6.

In [ ]:
metric_cols = ['pearsonr_nc', 'explained_variance_nc',
               'RSA', 'CKA',
               'encoding_RSA', 'encoding_CKA']

best = (metrics_long
        .groupby(['dataset', 'model_short', 'roi', 'layer_idx'],
                 as_index=False)[metric_cols].mean()
        .groupby(['dataset', 'model_short', 'roi'],
                 as_index=False)[metric_cols].max())
best['target'] = best['dataset'] + '/' + best['roi']

DATASET_ORDER = ['tvsd', 'eeg2', 'nsd']
ROI_PREFERRED = {
    'tvsd': ['V1', 'V4', 'IT', 'all'],
    'eeg2': ['occipital', 'parietal', 'temporal',
             'frontal', 'central', 'occipital_parietal',
             'whole_brain', 'all'],
    # Common NSD individual ROI ordering, early → late visual.
    # Unknown ROIs fall through to alphabetical.
    'nsd' : ['V1v', 'V1d', 'V1', 'V2v', 'V2d', 'V2', 'V3v', 'V3d', 'V3',
             'hV4', 'V4', 'VO1', 'VO2', 'PHC1', 'PHC2',
             'LO1', 'LO2', 'TO1', 'TO2',
             'OFA', 'FFA-1', 'FFA-2', 'mTL-faces', 'aTL-faces',
             'EBA', 'FBA-1', 'FBA-2', 'mTL-bodies',
             'OPA', 'PPA', 'RSC',
             'OWFA', 'VWFA-1', 'VWFA-2', 'mfs-words', 'mTL-words',
             'early', 'midventral', 'midlateral', 'midparietal',
             'ventral', 'lateral', 'parietal', 'all'],
}

def order_rois(dataset, available_rois):
    pref = ROI_PREFERRED.get(dataset, [])
    in_pref      = [r for r in pref if r in available_rois]
    not_in_pref  = sorted(r for r in available_rois if r not in pref)
    return in_pref + not_in_pref

target_order = []
for ds in DATASET_ORDER:
    rois = best.loc[best['dataset'] == ds, 'roi'].unique().tolist()
    if not rois:
        continue
    for r in order_rois(ds, rois):
        target_order.append(f'{ds}/{r}')
# Append any datasets not in DATASET_ORDER (defensive)
for ds in best['dataset'].unique():
    if ds not in DATASET_ORDER:
        for r in sorted(best.loc[best['dataset'] == ds, 'roi'].unique()):
            target_order.append(f'{ds}/{r}')

n_targets = len(target_order)
fig_w     = max(14, 0.55 * n_targets * 2)   # widen if many ROIs
fig, axes = plt.subplots(2, 3, figsize=(fig_w, 7.5), sharex=True)

for ax, m in zip(axes.flat, metric_cols):
    sns.barplot(data=best, x='target', y=m, hue='model_short',
                order=target_order,
                palette={'qwen': 'tab:blue', 'resnet': 'tab:orange'},
                ax=ax)
    ax.set_title(m)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=60)
    for lbl in ax.get_xticklabels():
        lbl.set_ha('right')
    ax.grid(axis='y', alpha=0.3)
    if ax is not axes[0, 0]:
        leg = ax.get_legend()
        if leg is not None:
            leg.remove()

fig.suptitle('Qwen vs ResNet — best-layer scores across datasets and ROIs',
             y=1.00)
plt.tight_layout()
plt.savefig('./plots/2_7_qwen_vs_resnet_summary.png',
            dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# 2.7 — table of best-layer scores per (dataset, roi) for each model
table = (best.pivot_table(index=['dataset', 'roi'],
                          columns='model_short',
                          values=metric_cols)[metric_cols]
              .reindex(pd.MultiIndex.from_tuples([t.split('/') for t in target_order],
                                                 names=['dataset', 'roi']))
              .round(3))
table.to_csv('./tables/2_7_best_scores.csv')
table

**Summary figure comparing Qwen3-VL-2B-Instruct and Adv-ResNet152**
![Alt Text](./plots/2_7_qwen_vs_resnet_summary.png)

**Table of best scores across datasets and targets**
| dataset | roi    | pearsonr_nc (qwen) | pearsonr_nc (resnet) | explained_variance_nc (qwen) | explained_variance_nc (resnet) | RSA (qwen) | RSA (resnet) | CKA (qwen) | CKA (resnet) | encoding_RSA (qwen) | encoding_RSA (resnet) | encoding_CKA (qwen) | encoding_CKA (resnet) |
|---------|--------|-------------------:|---------------------:|-----------------------------:|-------------------------------:|-----------:|-------------:|-----------:|-------------:|--------------------:|----------------------:|--------------------:|----------------------:|
| tvsd    | V1     | 0.650 | 0.764 | −0.541 |  0.379 | 0.105 | 0.198 | 0.159 | 0.362 | 0.339 | 0.688 | 0.592 | 0.843 |
| tvsd    | V4     | 0.589 | 0.703 | −0.237 |  0.324 | 0.068 | 0.232 | 0.106 | 0.281 | 0.480 | 0.698 | 0.640 | 0.788 |
| tvsd    | IT     | 0.528 | 0.668 | −0.458 |  0.364 | 0.086 | 0.223 | 0.129 | 0.252 | 0.376 | 0.621 | 0.532 | 0.739 |
| tvsd    | all    | 0.576 | 0.702 | −0.536 |  0.292 | 0.121 | 0.282 | 0.149 | 0.360 | 0.412 | 0.722 | 0.551 | 0.838 |
| eeg2    | all    | 0.426 | 0.524 | −0.195 | −0.096 | 0.074 | 0.155 | 0.077 | 0.140 | 0.245 | 0.278 | 0.218 | 0.255 |
| nsd     | V1v    | 0.607 | 0.687 |  0.170 |  0.391 | 0.169 | 0.223 | 0.227 | 0.311 | 0.332 | 0.366 | 0.420 | 0.461 |
| nsd     | V2v    | 0.564 | 0.673 |  0.118 |  0.355 | 0.126 | 0.197 | 0.174 | 0.277 | 0.331 | 0.360 | 0.395 | 0.434 |
| nsd     | V3v    | 0.503 | 0.658 |  0.069 |  0.329 | 0.114 | 0.148 | 0.162 | 0.192 | 0.307 | 0.316 | 0.340 | 0.353 |
| nsd     | hV4    | 0.475 | 0.639 | −0.007 |  0.317 | 0.114 | 0.152 | 0.153 | 0.159 | 0.297 | 0.307 | 0.290 | 0.284 |
| nsd     | FFA-1  | 0.491 | 0.668 | −0.112 |  0.359 | 0.138 | 0.179 | 0.151 | 0.165 | 0.453 | 0.508 | 0.467 | 0.458 |
| nsd     | EBA    | 0.480 | 0.660 | −0.033 |  0.367 | 0.157 | 0.223 | 0.194 | 0.210 | 0.539 | 0.553 | 0.494 | 0.493 |
| nsd     | OPA    | 0.479 | 0.685 |  0.038 |  0.382 | 0.157 | 0.217 | 0.175 | 0.174 | 0.319 | 0.381 | 0.290 | 0.324 |
| nsd     | PPA    | 0.552 | 0.761 |  0.120 |  0.489 | 0.181 | 0.243 | 0.240 | 0.234 | 0.346 | 0.457 | 0.440 | 0.489 |
| nsd     | VWFA-1 | 0.415 | 0.600 | −0.131 |  0.286 | 0.139 | 0.184 | 0.130 | 0.140 | 0.381 | 0.416 | 0.262 | 0.271 |
| nsd     | all    | 0.485 | 0.668 |  0.016 |  0.361 | 0.207 | 0.278 | 0.267 | 0.289 | 0.591 | 0.597 | 0.462 | 0.470 |

<div style="background:#fff4c2; border:1px solid #c89b1f; border-left:6px solid #9a6f00; padding:10px 12px; border-radius:6px; margin-top:8px; margin-bottom:4px; color:#241a00; line-height:1.45;"><strong style="color:#5c4300;">Answer box 2.6</strong><br>Does the vision-language model provide a clear advantage over the CNN? Is that advantage consistent across modalities and targets?

ResNet152 outperforms Qwen3-VL across essentially every dataset, ROI, and metric — there is no setting where Qwen clearly wins. The advantage is therefore consistent in direction, but its size depends strongly on the modality and the type of metric.

The gap is largest on TVSD: Qwen's noise-corrected explained variance is strongly negative on every ROI, meaning its predictions are worse than predicting the mean response, while ResNet sits solidly positive. The gap shrinks substantially on NSD and shrinks further on EEG2, where both models are weak in absolute terms and the ridge encoder produces only a modest difference between them.

The kind of metric also matters. The Qwen–ResNet gap is largest on the raw representational metrics (RSA, CKA) and smaller on the encoder-derived ones — the ridge encoder partly compensates for Qwen's worse feature geometry by re-weighting dimensions, but the compensation is incomplete and predictive accuracy still favours ResNet.

The most interesting exception is on category-selective NSD ROIs (FFA-1, EBA, PPA), where Qwen comes much closer to ResNet and even ties on encoding_CKA for EBA and FFA-1. This is consistent with the layer-wise pattern from 2.3: Qwen's language-model layers carry semantic information that helps for category-selective regions but contributes little to early retinotopic cortex, where ResNet's convolutional hierarchy is a better match. So while the headline conclusion is "ResNet is better everywhere," the size of the gap reveals where each model's inductive bias pays off.

<div>

---

# 3. Open-Ended Research

So far you have explored a simple encoding model with a linear readout from a single layer per subject/ROI. In this section, you will extend the baseline pipeline in one clearly defined direction. The goal is to explore a meaningful extension that goes beyond the standard linear readout and to evaluate whether it provides a practically meaningful improvement. Depth is more important than breadth: a focused experiment is better than a broad but shallow exploration.

Possible directions include:

- readouts shared across ROIs,
- readouts shared across subjects,
- readouts shared across modalities,
- combining multiple layers,
- low-rank readouts,
- nonlinear readouts,
- temporal readouts for EEG,
- attention-based readouts,
- cross-subject pooling.

## What you must include

1. **Question**  
   What are you testing?

2. **Motivation**  
   Why is this extension interesting?

3. **Method**  
   What did you change relative to the linear baseline?

4. **Comparison**  
   How does it compare to the baseline?

5. **Interpretation**  
   Did it help, and why might that be?

<div style="background:#f3f6fa; border-left:4px solid #7a93ac; padding:8px 12px; border-radius:6px; font-weight:700; color:#32475b;">Required deliverables</div>

You must include all of the following:

1. **A clearly stated hypothesis**
2. **A short motivation for the extension**
3. **A clear description of the new method**
4. **One direct comparison against the linear baseline**
5. **At least one figure or one table summarizing the comparison**
6. **A short discussion of whether the extension helped in a practically meaningful way**

## Combining Model Layers via Per-Voxel / Per-Unit Stacking: Open-Ended Extension on NSD (human fMRI) and TVSD (macaque electrophysiology)

### Motivation


Section 2 establishes a baseline where each neural target is predicted **using a single model layer—the “best-layer per ROI” approach**. While simple, this assumption is **likely too restrictive**. Different layers can encode complementary information, and even within a single ROI, individual targets (e.g., voxels in V4) may align with different levels of the processing hierarchy—some resembling earlier visual areas (like V1), others closer to higher-level regions (like IT).

To address this, Section 3 introduces a more flexible approach: instead of selecting a single layer, we learn a softmax-weighted mixture α(p)∈R
10 for each target (voxel or unit). This allows every target to combine information across all 10 layers of the model.

This formulation brings three key advantages. First, it tests whether combining layers improves performance over the single-best-layer baseline. Second, the learned mixture weights are directly interpretable: they reveal each target’s “preferred depth” in the model. Third, in the case of Qwen—where half the layers belong to a visual encoder and the other half to a language model—the mixture weights can show whether category-selective regions (such as VWFA-1 for word processing) rely on language-related representations in a way that purely visual architectures like ResNet cannot capture.

### Why two datasets

We evaluate this approach on two complementary datasets:

- The **NSD (human fMRI)** dataset is the primary benchmark, offering a rich set of ROIs spanning early visual to higher-level category-selective regions. This makes it particularly suitable for probing differences between visual and language representations in models like Qwen.

- The **TVSD (macaque electrophysiology)** dataset serves as a robustness check. Previous results (Section 2.6) showed a clear hierarchical progression from V1 to V4 to IT, with a perfect rank correlation ($\rho = +1.00$). Applying the same stacking method here allows us to test whether per-target mixtures recover an even sharper hierarchy at the level of individual neurons.

### Method in one paragraph


For each combination of dataset, model, and subject, we start with 10 pre-trained linear encoders—one per model layer—each predicting neural responses to a set of stimuli. We then learn a softmax-constrained mixture α(p)∈R10\alpha(p) \in \mathbb{R}^{10}α(p)∈R10, ensuring non-negative weights that sum to one, to combine these predictions into a single stacked output per target.

The mixture weights are optimized on a validation set (held out during encoder training) by minimizing mean squared error. These learned weights are then applied to the test predictions for final evaluation.

We compare three approaches:

- The single-best-layer baseline (one layer per ROI),

- Per-ROI stacking (one mixture per ROI),

- Per-target stacking (one mixture per voxel or unit).

---
<div style="background:#fff5f5; border:1px solid #f5c6c6; border-left:6px solid #cc3333; padding:10px 12px; border-radius:6px; margin-top:8px; margin-bottom:4px; color:#4a0000; line-height:1.45;">
<strong style="color:#cc3333;">Note:</strong> The stacking analysis (Section 3) was run on Google Colab due to computational issues.
</div>

### 3.1 Setup

In [ ]:
# Mount Drive and import dependencies
sns.set_theme(style='whitegrid', context='notebook')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)

In [ ]:
# Paths and constants
data_dir         = '/shared/NX-414/data/'
features_dir     = '/shared/NX-414/extracted_features/'
predictions_dir  = './predictions/'
results_dir      = './training_results'
alphas_dir       = os.path.join(results_dir, 'alphas')
figures_dir      = os.path.join(results_dir, 'figures')
for d in [results_dir, alphas_dir, figures_dir]:
    os.makedirs(d, exist_ok=True)

# Dataset files
nsd_file  = 'nsd_func1pt8mm_individualROIs.h5'
tvsd_file = 'tvsd.h5'

# Model directories
qwen   = 'Qwen3-VL-2B-Instruct/'
resnet = 'adv_resnet152_imagenet_full_ffgsm_eps-1_alpha-125-ep10_seed-0/'

# NSD ROIs in hierarchical order: early/retinotopic → category-selective
NSD_ROIS     = ['V1v', 'V2v', 'V3v', 'hV4',
                'FFA-1', 'VWFA-1', 'PPA', 'OPA', 'EBA']
NSD_SUBJECTS = [f'subj0{i}' for i in range(1, 9)]

# TVSD ROIs in hierarchical order: V1 → V4 → IT
TVSD_ROIS     = ['V1', 'V4', 'IT']
TVSD_SUBJECTS = ['monkeyF', 'monkeyN']

# Models being compared
MODELS = [('resnet', resnet), ('qwen', qwen)]

# Per-dataset: which feature file to use (NSD uses nsd_stimuli, TVSD uses things_stimuli)
DATASET_FEATURE_FILE = {'nsd': 'nsd_stimuli.h5', 'tvsd': 'things_stimuli.h5'}

---
### 3.2 Representational metrics (reused from Section 2)

In [ ]:
class RepresentationalSimilarityAnalysis:
    """
    RSA: builds an RDM for each input and correlates their upper triangles.
    Defaults: correlation distance + Spearman correlation (rank-based).
    """
    def __init__(self,
                 dissimilarity: Literal['correlation', 'euclidean', 'cosine'] = 'correlation',
                 similarity_metric: Literal['pearson', 'spearman'] = 'spearman'):
        self.dissimilarity = dissimilarity
        self.similarity_metric = similarity_metric

    def __call__(self, X, Y): return self.forward(X, Y)

    def forward(self, X, Y):
        X = X.reshape(X.shape[0], -1)
        Y = Y.reshape(Y.shape[0], -1)
        return self._compare_rdms(self._compute_rdm(X), self._compute_rdm(Y))

    def _compute_rdm(self, X):
        if self.dissimilarity == 'correlation':
            corr = np.corrcoef(X); np.clip(corr, -1.0, 1.0, out=corr)
            rdm = 1.0 - corr
        elif self.dissimilarity == 'cosine':
            sim = cosine_similarity(X); np.clip(sim, -1.0, 1.0, out=sim)
            rdm = 1.0 - sim
        elif self.dissimilarity == 'euclidean':
            rdm = euclidean_distances(X)
        else:
            raise ValueError(self.dissimilarity)
        np.fill_diagonal(rdm, 0.0)
        return 0.5 * (rdm + rdm.T)

    def _compare_rdms(self, rdm1, rdm2):
        iu = np.triu_indices(rdm1.shape[0], k=1)
        v1, v2 = rdm1[iu], rdm2[iu]
        r, _ = (spearmanr(v1, v2) if self.similarity_metric == 'spearman'
                else pearsonr(v1, v2))
        return float(r)


class CenteredKernelAlignment:
    """Unbiased linear CKA via the U-statistic HSIC estimator."""
    def __init__(self, eps=1e-8, dtype=np.float64):
        self.eps, self.dtype = eps, dtype

    def __call__(self, X, Y): return self.forward(X, Y)

    def forward(self, X, Y):
        X = np.asarray(X, dtype=self.dtype).reshape(X.shape[0], -1)
        Y = np.asarray(Y, dtype=self.dtype).reshape(Y.shape[0], -1)
        if X.shape[0] != Y.shape[0]:
            raise ValueError(f'shape mismatch: {X.shape[0]} vs {Y.shape[0]}')
        return self._cka(X, Y)

    def _hsic(self, X, Y):
        n = X.shape[0]
        K = X @ X.T; L = Y @ Y.T
        np.fill_diagonal(K, 0.); np.fill_diagonal(L, 0.)
        sum_K, sum_L = K.sum(), L.sum()
        cross = float(K.sum(1) @ L.sum(1))
        return (float((K * L).sum())
                + (sum_K * sum_L) / ((n - 1) * (n - 2))
                - 2.0 * cross / (n - 2)) / (n * (n - 3))

    def _cka(self, X, Y):
        h_xy = self._hsic(X, Y)
        h_xx = self._hsic(X, X)
        h_yy = self._hsic(Y, Y)
        return float(h_xy / np.sqrt(max(h_xx * h_yy, self.eps)))


rsa_metric = RepresentationalSimilarityAnalysis(
    dissimilarity='correlation', similarity_metric='spearman')
cka_metric = CenteredKernelAlignment()

---
### 3.3 Helpers - layers, dataset loader, ROI slices, predictive metrics

In [ ]:
def get_layer_keys(feature_path):
    """Return all layer dataset names under '/features/...'."""
    with h5py.File(feature_path, 'r') as f:
        return [f'features/{k}' for k in f['features']]

def sort_by_depth(layers):
    """Sort layer keys by the integers embedded in their names (depth-ordered)."""
    return sorted(layers, key=lambda n: tuple(int(x) for x in re.findall(r'\d+', n)))

def get_layers_for(model_dir, dataset_name):
    """Sorted list of layer names for a given (model, dataset)."""
    feat_file = DATASET_FEATURE_FILE[dataset_name]
    return sort_by_depth(get_layer_keys(features_dir + model_dir + feat_file))

In [ ]:
def load_nsd(skip_missing_rois=True):
    """Load NSD train/test responses + per-voxel noise ceilings, restricted to NSD_ROIS."""
    out = defaultdict(dict)
    with h5py.File(data_dir + nsd_file, 'r') as f:
        for s in NSD_SUBJECTS:
            train_ids = f[f'train/stimulus_ids/{s}'][:]
            test_ids  = f[f'test/stimulus_ids/{s}'][:]
            out[s]['stimulus_ids'] = {'train': train_ids, 'test': test_ids}
            for r in NSD_ROIS:
                k_train = f'train/neural_data/{s}/{r}'
                k_test  = f'test/neural_data/{s}/{r}'
                if k_train not in f or k_test not in f:
                    if skip_missing_rois:
                        continue
                    raise KeyError(f'[NSD] {s}/{r} missing.')
                train = f[k_train][:].astype(np.float32, copy=False)
                test  = f[k_test ][:].astype(np.float32, copy=False)
                nc_path = f'noise_ceilings/{s}/{r}'
                nc = (f[nc_path][:].astype(np.float32) / 100.0
                      if nc_path in f
                      else np.ones(train.shape[1], dtype=np.float32))
                out[s][r] = {'train': train, 'test': test, 'noise_ceiling': nc}
    return out


def load_tvsd():
    """Load TVSD train/test responses + per-unit noise ceilings.

    TVSD has a single shared train/test stimulus set across both monkeys,
    so we mirror the same stimulus_ids under each subject for code uniformity.
    """
    out = defaultdict(dict)
    with h5py.File(data_dir + tvsd_file, 'r') as f:
        train_ids = f['train/stimulus_ids'][:]
        test_ids  = f['test/stimulus_ids'][:]
        for m in TVSD_SUBJECTS:
            out[m]['stimulus_ids'] = {'train': train_ids, 'test': test_ids}
            for r in TVSD_ROIS:
                train = f[f'train/neural_data/{m}/{r}'][:].astype(np.float32, copy=False)
                test  = f[f'test/neural_data/{m}/{r}' ][:].astype(np.float32, copy=False)
                nc_path = f'noise_ceilings/{m}/{r}'
                nc = (f[nc_path][:].astype(np.float32) / 100.0
                      if nc_path in f
                      else np.ones(train.shape[1], dtype=np.float32))
                out[m][r] = {'train': train, 'test': test, 'noise_ceiling': nc}
    return out


def build_roi_slices(dataset, subject, roi_list):
    """Return {roi: (start, end)} mapping target columns to ROIs for one subject."""
    roi_slices, off = {}, 0
    for r in roi_list:
        if r in dataset[subject]:
            n_r = dataset[subject][r]['test'].shape[1]
            roi_slices[r] = (off, off + n_r)
            off += n_r
    return roi_slices

In [ ]:
# Per-target predictive metrics (reused from Section 2)
def pearsonr_per_target(Yhat, Y, eps=1e-12):
    Yc = Y    - Y.mean(0, keepdims=True)
    Hc = Yhat - Yhat.mean(0, keepdims=True)
    return (Yc * Hc).sum(0) / (np.sqrt((Yc**2).sum(0) * (Hc**2).sum(0)) + eps)

def ev_per_target(Yhat, Y, eps=1e-12):
    ss_res = ((Y - Yhat)**2).sum(0)
    ss_tot = ((Y - Y.mean(0, keepdims=True))**2).sum(0) + eps
    return 1.0 - ss_res / ss_tot

---
### 3.4 Stacking core

Four functions:

- **`load_subject_data`** — assembles `(P_val, Y_val, P_test, Y_test, nc, roi_slices)` for one (dataset, model, subject).
- **`fit_stacking_softmax`** — learns mixing weights $\alpha(p)$ on val MSE; supports `per_target` and `per_roi` modes.
- **`evaluate_stacked`** — computes Pearson, EV, encoding-RSA, encoding-CKA per ROI on test predictions.
- **`best_single_layer_baseline`** — picks each ROI's best layer on val (Section-2-style baseline) and evaluates on test.

In [ ]:
def load_subject_data(model_dir, subject, dataset_name,
                      preds_test, preds_val, dataset, layer_order, roi_list):
    """Stack per-layer val/test predictions and pull aligned targets."""
    P_val_layers = [preds_val [(dataset_name, model_dir, l, subject)].astype(np.float32)
                    for l in layer_order]
    P_test_layers= [preds_test[(dataset_name, model_dir, l, subject)].astype(np.float32)
                    for l in layer_order]
    P_val  = np.stack(P_val_layers,  axis=0)   # (L, n_val,  p)
    P_test = np.stack(P_test_layers, axis=0)   # (L, n_test, p)

    Y_val = preds_val[(dataset_name, subject, 'Y_val')].astype(np.float32)

    roi_slices = build_roi_slices(dataset, subject, roi_list)

    Y_test_parts = [dataset[subject][r]['test']          for r in roi_slices]
    nc_parts     = [dataset[subject][r]['noise_ceiling'] for r in roi_slices]
    Y_test = np.hstack(Y_test_parts).astype(np.float32)
    nc     = np.concatenate(nc_parts).astype(np.float32)

    return dict(P_val=P_val, P_test=P_test,
                Y_val=Y_val, Y_test=Y_test,
                nc=nc, roi_slices=roi_slices)


def fit_stacking_softmax(P_val, Y_val, P_test,
                         mode='per_target', roi_slices=None,
                         lr=1e-2, n_epochs=300, l2=0.0,
                         device=DEVICE, verbose=False):
    """
    Per-target or per-ROI softmax stacking.

    Returns
    -------
    alpha      : (L, p)        softmax mixing weights, broadcast to all targets
    Y_test_hat : (n_test, p)   stacked test predictions
    losses     : list[float]   training-loss trace
    """
    L, n_val, p = P_val.shape

    P_val_t  = torch.as_tensor(P_val,  device=device, dtype=torch.float32)
    Y_val_t  = torch.as_tensor(Y_val,  device=device, dtype=torch.float32)
    P_test_t = torch.as_tensor(P_test, device=device, dtype=torch.float32)

    if mode == 'per_target':
        theta = nn.Parameter(torch.zeros(L, p, device=device))
        roi_idx = None
    elif mode == 'per_roi':
        roi_names = list(roi_slices.keys())
        theta = nn.Parameter(torch.zeros(L, len(roi_names), device=device))
        roi_idx = torch.zeros(p, dtype=torch.long, device=device)
        for ri, r in enumerate(roi_names):
            s, e = roi_slices[r]
            roi_idx[s:e] = ri
    else:
        raise ValueError(mode)

    opt = torch.optim.Adam([theta], lr=lr)
    losses = []

    for epoch in range(n_epochs):
        opt.zero_grad()
        if mode == 'per_target':
            alpha = torch.softmax(theta, dim=0)
        else:
            alpha = torch.softmax(theta, dim=0)[:, roi_idx]
        Y_hat = torch.einsum('lp,lnp->np', alpha, P_val_t)
        loss = ((Y_hat - Y_val_t) ** 2).mean()
        if l2 > 0:
            loss = loss + l2 * (theta ** 2).mean()
        loss.backward(); opt.step()
        losses.append(loss.item())
        if verbose and (epoch + 1) % 50 == 0:
            print(f'  epoch {epoch+1:3d}: loss={loss.item():.5f}')

    with torch.no_grad():
        if mode == 'per_target':
            alpha_final = torch.softmax(theta, dim=0)
        else:
            alpha_final = torch.softmax(theta, dim=0)[:, roi_idx]
        Y_test_hat = torch.einsum('lp,lnp->np', alpha_final, P_test_t)

    return alpha_final.cpu().numpy(), Y_test_hat.cpu().numpy(), losses


def evaluate_stacked(Y_test_hat, Y_test, nc, roi_slices,
                     rsa_metric, cka_metric):
    """Per-ROI test metrics: pearsonr, ev, noise-corrected versions, encoding RSA/CKA."""
    r  = pearsonr_per_target(Y_test_hat, Y_test)
    ev = ev_per_target(Y_test_hat, Y_test)
    nc_safe = np.clip(nc, 1e-6, None)
    r_nc   = r  / np.sqrt(nc_safe)
    ev_nc  = ev / nc_safe

    rows = []
    for roi, (s, e) in roi_slices.items():
        rows.append(dict(
            roi=roi, n_targets=e - s,
            pearsonr              = float(np.nanmean(r [s:e])),
            pearsonr_nc           = float(np.nanmean(r_nc [s:e])),
            explained_variance    = float(np.nanmean(ev[s:e])),
            explained_variance_nc = float(np.nanmean(ev_nc[s:e])),
            encoding_RSA          = float(rsa_metric(Y_test_hat[:, s:e], Y_test[:, s:e])),
            encoding_CKA          = float(cka_metric(Y_test_hat[:, s:e], Y_test[:, s:e])),
        ))
    return pd.DataFrame(rows)


def best_single_layer_baseline(P_val, Y_val, P_test, Y_test, nc, roi_slices,
                                rsa_metric, cka_metric):
    """Section-2-style baseline: pick the best layer per ROI on val, evaluate on test."""
    L = P_val.shape[0]
    nc_safe = np.clip(nc, 1e-6, None)

    rows = []
    for roi, (s, e) in roi_slices.items():
        # pick best layer for this ROI on val
        val_scores = []
        for l in range(L):
            r_val = pearsonr_per_target(P_val[l, :, s:e], Y_val[:, s:e])
            r_val_nc = r_val / np.sqrt(nc_safe[s:e])
            val_scores.append(np.nanmean(r_val_nc))
        best_l = int(np.argmax(val_scores))

        # evaluate the chosen layer on test
        Yh = P_test[best_l, :, s:e]
        Yt = Y_test[:, s:e]
        r  = pearsonr_per_target(Yh, Yt)
        ev = ev_per_target(Yh, Yt)
        nc_roi = nc_safe[s:e]
        rows.append(dict(
            roi=roi, n_targets=e - s, best_layer=best_l,
            pearsonr              = float(np.nanmean(r)),
            pearsonr_nc           = float(np.nanmean(r / np.sqrt(nc_roi))),
            explained_variance    = float(np.nanmean(ev)),
            explained_variance_nc = float(np.nanmean(ev / nc_roi)),
            encoding_RSA          = float(rsa_metric(Yh, Yt)),
            encoding_CKA          = float(cka_metric(Yh, Yt)),
        ))
    return pd.DataFrame(rows)

---
### 3.5 Fit stacking on both datasets

The full sweep is **(NSD: 8 subjects + TVSD: 2 subjects) × 2 models × 3 methods**. Each per-target stacking fit takes a few seconds on GPU; the whole loop runs in a couple of minutes.

Outputs saved per (dataset, model, subject):
- `results_df`: per-ROI test metrics for all three methods.
- `alphas_dir/{dataset}_{model}_{subject}_alpha.npy`: the (L, p) per-target mixture matrix.

In [ ]:
# Dataset registry
DATASETS = {
    'nsd':  dict(rois=NSD_ROIS,  subjects=NSD_SUBJECTS,  loader=load_nsd),
    'tvsd': dict(rois=TVSD_ROIS, subjects=TVSD_SUBJECTS, loader=load_tvsd),
}

# Load all prediction pickles up-front
print('Loading prediction pickles...')
preds_cache = {}  # {(model, dataset, split) -> dict}
for ds_name in DATASETS:
    for model_name, _ in MODELS:
        test_path = predictions_dir + f'{model_name}_{ds_name}_predictions.pkl'
        val_path  = predictions_dir + f'{model_name}_{ds_name}_val_predictions.pkl'
        if not (os.path.exists(test_path) and os.path.exists(val_path)):
            print(f'  WARNING: missing pickle(s) for {model_name}/{ds_name} — skipping')
            continue
        with open(test_path, 'rb') as f: preds_cache[(model_name, ds_name, 'test')] = pickle.load(f)
        with open(val_path,  'rb') as f: preds_cache[(model_name, ds_name, 'val')]  = pickle.load(f)
        print(f'  loaded {model_name}/{ds_name} (test + val)')

# Load datasets
loaded_datasets = {}
for ds_name, cfg in DATASETS.items():
    print(f'\nLoading {ds_name} dataset...')
    loaded_datasets[ds_name] = cfg['loader']()
    print(f'  loaded {len(loaded_datasets[ds_name])} subjects')

In [ ]:
# Run the full fitting sweep across both datasets
all_results = []

for ds_name, cfg in DATASETS.items():
    print(f'\n========== {ds_name.upper()} ==========')
    dataset = loaded_datasets[ds_name]

    for model_name, model_dir in MODELS:
        if (model_name, ds_name, 'test') not in preds_cache:
            print(f'  [skip] no preds for {model_name}/{ds_name}')
            continue

        p_test = preds_cache[(model_name, ds_name, 'test')]
        p_val  = preds_cache[(model_name, ds_name, 'val')]
        layers = get_layers_for(model_dir, ds_name)
        L = len(layers)
        print(f'\n  --- {model_name}: {L} layers ---')

        for subject in tqdm(cfg['subjects'], desc=f'{ds_name}/{model_name}'):
            try:
                data = load_subject_data(model_dir, subject, ds_name,
                                          p_test, p_val, dataset, layers, cfg['rois'])
            except KeyError as e:
                print(f'    [skip {subject}]: {e}')
                continue

            # 1) single-best-layer baseline
            df_best = best_single_layer_baseline(
                data['P_val'], data['Y_val'], data['P_test'], data['Y_test'],
                data['nc'], data['roi_slices'], rsa_metric, cka_metric)
            df_best['method'] = 'best_single'

            # 2) per-ROI stacking
            _, Y_hat_roi, _ = fit_stacking_softmax(
                data['P_val'], data['Y_val'], data['P_test'],
                mode='per_roi', roi_slices=data['roi_slices'])
            df_roi = evaluate_stacked(Y_hat_roi, data['Y_test'], data['nc'],
                                       data['roi_slices'], rsa_metric, cka_metric)
            df_roi['method'] = 'per_roi'

            # 3) per-target stacking — also save alpha for interpretation figures
            alpha, Y_hat_pt, _ = fit_stacking_softmax(
                data['P_val'], data['Y_val'], data['P_test'],
                mode='per_target', roi_slices=data['roi_slices'])
            df_pt = evaluate_stacked(Y_hat_pt, data['Y_test'], data['nc'],
                                      data['roi_slices'], rsa_metric, cka_metric)
            df_pt['method'] = 'per_target'
            np.save(os.path.join(alphas_dir,
                                  f'{ds_name}_{model_name}_{subject}_alpha.npy'),
                    alpha)

            for df in [df_best, df_roi, df_pt]:
                df['model']   = model_name
                df['subject'] = subject
                df['dataset'] = ds_name
                all_results.append(df)

results_df = pd.concat(all_results, ignore_index=True)
results_df.to_csv(os.path.join(results_dir, 'stacking_results.csv'), index=False)
print(f'\nSaved {len(results_df)} rows to stacking_results.csv')
results_df.head()

---
### 3.6 Sanity checks

Three diagnostics before trusting the figures:

1. **Loss converges** — stacking optimization is well-behaved.
2. **Stacking ≥ single-best on test** — the learned mixture must at minimum match what we could get by picking one layer.
3. **α reflects the underlying val signal** — voxel/unit-level argmax layer in the val-correlation ranking should align with α's argmax.

In [ ]:
# 6.1 Loss convergence — uses NSD subj01 ResNet as a probe
print('Loss convergence on NSD subj01 ResNet (per-target):')

layers_resnet_nsd = get_layers_for(resnet, 'nsd')
data_check = load_subject_data(
    resnet, 'subj01', 'nsd',
    preds_cache[('resnet', 'nsd', 'test')],
    preds_cache[('resnet', 'nsd', 'val')],
    loaded_datasets['nsd'], layers_resnet_nsd, NSD_ROIS)

_, _, losses = fit_stacking_softmax(
    data_check['P_val'], data_check['Y_val'], data_check['P_test'],
    mode='per_target', roi_slices=data_check['roi_slices'],
    n_epochs=400, verbose=False)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(losses, lw=1.8, color='steelblue')
ax.set(xlabel='epoch', ylabel='val MSE',
       title='Stacking loss — NSD subj01, ResNet, per-target')
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f'  initial loss : {losses[0]:.5f}')
print(f'  final loss   : {losses[-1]:.5f}')
print(f'  reduction    : {100*(losses[0]-losses[-1])/losses[0]:.1f}%')
print(f'  Δ in last 50 : {losses[-50] - losses[-1]:.6f}  (smaller = converged)')

The stacking weights are optimized via gradient descent on the validation MSE. 
The loss curve below confirms convergence within 400 epochs, with a 4.6% 
reduction in validation MSE and a negligible change of 0.0003 in the final 50 
epochs.

<div style="display: flex; gap: 10px; align-items: flex-start; margin-top: 15px;">
  <figure style="text-align: center; flex: 1;">
    <img src="images/stacking_loss_nds_subj_1.png" style="width: 80%;"/>
    <figcaption>Stacking loss convergence — NSD subj01, ResNet152, per-target 
    mode. Validation MSE decreases from 0.392 to 0.374 over 400 epochs.</figcaption>
  </figure>
</div>

In [ ]:
# 6.2 Methods comparison — stacking should ≥ best-single on average
print('Mean pearsonr_nc averaged across all ROIs and subjects:\n')
summary = (results_df
           .groupby(['dataset', 'model', 'method'])['pearsonr_nc']
           .agg(['mean', 'std'])
           .round(4))
print(summary)

# Per (dataset, model, subject, ROI): does stacking beat best_single?
wide = (results_df
        .pivot_table(index=['dataset', 'model', 'subject', 'roi'],
                     columns='method', values='pearsonr_nc')
        .reset_index())
wide['per_target_beats_best'] = wide['per_target'] > wide['best_single']
wide['per_roi_beats_best']    = wide['per_roi']    > wide['best_single']

**Mean pearsonr_nc averaged across all ROIs and subjects:**
| dataset | model  | method      | mean   | std    |
|---------|--------|-------------|--------|--------|
| nsd     | qwen   | best_single | 0.5294 | 0.1068 |
| nsd     | qwen   | per_roi     | 0.6564 | 0.0970 |
| nsd     | qwen   | per_target  | 0.6732 | 0.0912 |
| nsd     | resnet | best_single | 0.6707 | 0.0720 |
| nsd     | resnet | per_roi     | 0.6662 | 0.0723 |
| nsd     | resnet | per_target  | 0.6651 | 0.0728 |
| tvsd    | qwen   | best_single | 0.5892 | 0.0682 |
| tvsd    | qwen   | per_roi     | 0.6480 | 0.0474 |
| tvsd    | qwen   | per_target  | 0.6634 | 0.0441 |
| tvsd    | resnet | best_single | 0.7151 | 0.0488 |
| tvsd    | resnet | per_roi     | 0.7688 | 0.0401 |
| tvsd    | resnet | per_target  | 0.7849 | 0.0354 |

Per-target stacking outperformed the single-best-layer baseline in three of four (dataset, model) cells, with the largest gain on NSD-Qwen and consistent gains on both TVSD cells. The single null cell, NSD-ResNet, reflects a property of the underlying alignment rather than a method failure: ResNet's late layers are individually so dominant for fMRI voxels that the optimal mixture collapses onto a single layer, which best_single already finds. Decomposing the gain shows that most of the improvement comes from going single → mixture (the per_roi step), with per-voxel customization adding a smaller but consistent bonus — meaning stacking primarily exploits layer complementarity at the ROI level, with modest within-ROI heterogeneity on top. A model-level contrast also emerges: ResNet's individual layers are stronger predictors than Qwen's, but more redundant; Qwen's are weaker but more complementary, so stacking lets Qwen close the gap with ResNet on NSD entirely. The improvements are practically meaningful — gains of this magnitude are substantial on a metric that typically ranges 0.4–0.8 in this domain, and they come at zero retraining cost.

---
### 3.7 Aggregate α across subjects

For each (dataset, model), build:

- `alpha_by_roi[ROI] → (n_subjects, L)`: per-ROI mean α for each subject.
- `depth_by_roi[ROI] → 1D array of voxel-level depths pooled across subjects.

The "preferred depth" is $\text{depth}(p) = \sum_l l \cdot \alpha_l(p)$ — center of mass in layer-index space.

In [ ]:
def aggregate_alpha_by_roi(model_name, dataset_name, dataset, roi_list):
    """Aggregate α across all subjects of one (model, dataset)."""
    alpha_by_roi = {r: [] for r in roi_list}
    depth_by_roi = {r: [] for r in roi_list}

    subjects = DATASETS[dataset_name]['subjects']
    for subject in subjects:
        path = os.path.join(alphas_dir,
                             f'{dataset_name}_{model_name}_{subject}_alpha.npy')
        if not os.path.exists(path):
            continue
        alpha = np.load(path)                       # (L, p)
        L = alpha.shape[0]
        layer_idx = np.arange(L).reshape(-1, 1)     # (L, 1)
        depth_per_target = (alpha * layer_idx).sum(axis=0)  # (p,)

        roi_slices = build_roi_slices(dataset, subject, roi_list)
        for r, (s, e) in roi_slices.items():
            alpha_by_roi[r].append(alpha[:, s:e].mean(axis=1))   # (L,)
            depth_by_roi[r].append(depth_per_target[s:e])

    alpha_by_roi = {r: np.stack(v, axis=0) for r, v in alpha_by_roi.items() if v}
    depth_by_roi = {r: np.concatenate(v)   for r, v in depth_by_roi.items() if v}
    return alpha_by_roi, depth_by_roi


# Build aggregates for both datasets and both models
agg = {}  # agg[(dataset, model)] = (alpha_by_roi, depth_by_roi)
for ds_name in DATASETS:
    for model_name, _ in MODELS:
        if (model_name, ds_name, 'test') not in preds_cache:
            continue
        a, d = aggregate_alpha_by_roi(model_name, ds_name,
                                       loaded_datasets[ds_name],
                                       DATASETS[ds_name]['rois'])
        agg[(ds_name, model_name)] = (a, d)

# Quick numerical preview: NSD ResNet
if ('nsd', 'resnet') in agg:
    print('Mean α (averaged across subjects) — NSD / ResNet')
    a, _ = agg[('nsd', 'resnet')]
    L = next(iter(a.values())).shape[1]
    print('ROI       ' + ' '.join(f'L{l:>4d}' for l in range(L)))
    for r in NSD_ROIS:
        if r in a:
            m = a[r].mean(axis=0)
            print(f'{r:<8} ' + ' '.join(f'{x:5.3f}' for x in m))

---
### 3.8 Figure A — α heatmap per dataset and model

Four panels: NSD/ResNet, NSD/Qwen, TVSD/ResNet, TVSD/Qwen. Rows are ROIs in hierarchical order; columns are layers ordered by depth.

If the brain–model hierarchy correspondence holds, expect a diagonal-like band: early-visual ROIs light up at low layers, high-level ROIs at high layers. TVSD should show a cleaner gradient than NSD given its single-unit resolution.

In [ ]:
# Identify Qwen visual-vs-LM layers (same for both datasets — same model)
qwen_layers = get_layers_for(qwen, 'nsd')
qwen_kinds  = ['V' if 'visual' in l.lower()
               else 'L' if 'language' in l.lower()
               else '?'
               for l in qwen_layers]

def plot_alpha_heatmap(alpha_by_roi, roi_list, title, ax, layer_kinds=None):
    rois = [r for r in roi_list if r in alpha_by_roi]
    if not rois:
        ax.set_title(f'{title}  (no data)'); ax.axis('off'); return
    M = np.stack([alpha_by_roi[r].mean(axis=0) for r in rois], axis=0)   # (n_roi, L)
    L = M.shape[1]
    im = ax.imshow(M, aspect='auto', cmap='viridis', vmin=0, vmax=M.max())
    ax.set_yticks(range(len(rois))); ax.set_yticklabels(rois)
    if layer_kinds is None:
        ax.set_xticks(range(L)); ax.set_xticklabels([f'L{l}' for l in range(L)])
    else:
        ax.set_xticks(range(L))
        ax.set_xticklabels([f'L{l}\n[{k}]' for l, k in enumerate(layer_kinds)],
                            fontsize=9)
    ax.set_xlabel('layer (depth →)')
    ax.set_title(title, fontsize=11)
    plt.colorbar(im, ax=ax, fraction=0.04, pad=0.02, label='mean α')


fig, axes = plt.subplots(2, 2, figsize=(15, 9), constrained_layout=True)
panels = [
    ('nsd',  'resnet', NSD_ROIS,  'NSD — ResNet',  None,        axes[0, 0]),
    ('nsd',  'qwen',   NSD_ROIS,  'NSD — Qwen',    qwen_kinds,  axes[0, 1]),
    ('tvsd', 'resnet', TVSD_ROIS, 'TVSD — ResNet', None,        axes[1, 0]),
    ('tvsd', 'qwen',   TVSD_ROIS, 'TVSD — Qwen',   qwen_kinds,  axes[1, 1]),
]
for ds, mdl, rois, title, kinds, ax in panels:
    if (ds, mdl) in agg:
        a, _ = agg[(ds, mdl)]
        plot_alpha_heatmap(a, rois, title, ax, layer_kinds=kinds)
    else:
        ax.set_title(f'{title}  (skipped)'); ax.axis('off')

fig.suptitle('Per-target stacking — mean α(p) by ROI and layer', y=1.02, fontsize=13)
plt.savefig(os.path.join(figures_dir, 'alpha_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

#### Interpretation

The heatmaps below show the mean stacking weights $\alpha$ per ROI and layer, 
revealing which layers each brain region draws from.

<div style="display: flex; gap: 10px; align-items: flex-start; margin-top: 15px;">
  <figure style="text-align: center; flex: 1;">
    <img src="images/alpha_heatmap.png" style="width: 100%;"/>
    <figcaption>Mean stacking weights per ROI and layer for all four 
    model-dataset combinations. Brighter cells indicate higher contribution 
    of that layer to predicting the corresponding ROI. V/L labels on Qwen 
    layers denote visual encoder (V) and language model (L) layers.</figcaption>
  </figure>
</div>

**TVSD-ResNet (bottom-left)** shows the cleanest result: bright cells move diagonally from L0–L1 (V1) to L2 (V4) to L9 (IT), recovering the classical V1→V4→IT hierarchy at single-unit resolution.

**TVSD-Qwen (bottom-right)** concentrates mass almost entirely on visual-encoder layers (L0, L2), with negligible weight on language-model layers — consistent with macaque ventral cortex engaging only Qwen's visual representations.

**NSD-ResNet (top-left)** preserves V1v's early-layer signature but compresses the rest of the hierarchy: every ROI from V2v onward collapses heavily onto L9. This monolithic structure explains why stacking added no gain for NSD-ResNet — one layer already dominates.

**NSD-Qwen (top-right)** spreads α across many layers, with no single column dominating. Visual layers (especially L2 and L4) contribute everywhere, and language-model layers contribute modestly but consistently across both early-visual and category-selective ROIs. This genuine layer complementarity explains NSD-Qwen's large stacking gain.

Two cross-panel patterns are notable: (1) TVSD recovers a sharper hierarchy than NSD, consistent with single-unit responses preserving specificity that voxel averaging blurs; (2) Qwen's LM layers contribute on NSD but not on TVSD, suggesting fMRI BOLD aligns with semantic representations in a way single-unit data does not.

---
### 3.9 Figure B — preferred-depth distributions per ROI

Violin plot of $\text{depth}(p) = \sum_l l \cdot \alpha_l(p)$ pooled across all subjects, per ROI.

- **Median shift across ROIs** → hierarchical correspondence at the population level.
- **Width of each violin** → narrow means functionally homogeneous ROI; wide means within-ROI heterogeneity that ROI-level analyses miss.

In [ ]:
def plot_depth_violins(depth_by_roi, roi_list, title, ax):
    rois = [r for r in roi_list if r in depth_by_roi and len(depth_by_roi[r]) > 0]
    if not rois:
        ax.set_title(f'{title}  (no data)'); ax.axis('off'); return
    data = [depth_by_roi[r] for r in rois]
    parts = ax.violinplot(data, showmeans=True, showmedians=False)
    for pc in parts['bodies']:
        pc.set_facecolor('steelblue'); pc.set_alpha(0.6)
    ax.set_xticks(range(1, len(rois) + 1)); ax.set_xticklabels(rois, rotation=30)
    ax.set_ylabel('preferred depth = Σ l · α_l(p)')
    ax.set_title(title, fontsize=11); ax.grid(alpha=0.3)


fig, axes = plt.subplots(2, 2, figsize=(14, 8), constrained_layout=True)
panels = [
    ('nsd',  'resnet', NSD_ROIS,  'NSD — ResNet',  axes[0, 0]),
    ('nsd',  'qwen',   NSD_ROIS,  'NSD — Qwen',    axes[0, 1]),
    ('tvsd', 'resnet', TVSD_ROIS, 'TVSD — ResNet', axes[1, 0]),
    ('tvsd', 'qwen',   TVSD_ROIS, 'TVSD — Qwen',   axes[1, 1]),
]
for ds, mdl, rois, title, ax in panels:
    if (ds, mdl) in agg:
        _, d = agg[(ds, mdl)]
        plot_depth_violins(d, rois, title, ax)
    else:
        ax.set_title(f'{title}  (skipped)'); ax.axis('off')

fig.suptitle('Per-target preferred-depth distributions (pooled across subjects)',
             y=1.02, fontsize=13)
plt.savefig(os.path.join(figures_dir, 'depth_violins.png'), dpi=150, bbox_inches='tight')
plt.show()

#### Interpretation

The violin plots below show the distribution of preferred depth 
$\bar{d}(p) = \sum_l l \cdot \alpha_l(p)$ per ROI, quantifying how deep in 
the network each target draws its representations.

<div style="display: flex; gap: 10px; align-items: flex-start; margin-top: 15px;">
  <figure style="text-align: center; flex: 1;">
    <img src="images/per_target_preferred_depth.png" style="width: 100%;"/>
    <figcaption>Preferred depth distributions per ROI and model-dataset 
    combination, pooled across subjects. Each violin shows the distribution 
    of $\bar{d}(p) = \sum_l l \cdot \alpha_l(p)$ across targets within 
    that ROI.</figcaption>
  </figure>
</div>

**TVSD-ResNet (bottom-left)** shows the clearest gradient: median preferred depth shifts monotonically from V1 (~3) → V4 (~4.5) → IT (~6), with violins that are well-separated and relatively narrow. Single-unit recordings recover a sharp, hierarchical layer-depth correspondence.

**NSD-ResNet (top-left)** shows a steep but compressed gradient: V1v sits cleanly at depth ~3, but V2v→EBA all crowd between depths ~5 and ~8 with overlapping distributions. Mid-tier and high-tier ROIs aren't well separated by depth — confirming the heatmap's "everything collapses onto late layers" pattern.

**NSD-Qwen (top-right)** shows a smoother, more uniform gradient than NSD-ResNet: medians shift gently from V1v (~3) to EBA (~4.5). Violins are fairly wide and increasingly overlapping along the hierarchy, but the progression itself is cleaner than NSD-ResNet's. Qwen's distributed alignment differentiates ROIs more gracefully than ResNet's monolithic alignment.

**TVSD-Qwen (bottom-right)** shows essentially flat distributions: V1, V4, IT all sit near depth ~2.5 with no meaningful shift. Macaque ventral cortex doesn't engage Qwen's deeper layers, and the hierarchical signal is absent.

Two cross-panel patterns: (1) **violin width reflects within-ROI heterogeneity** — TVSD violins are narrower than NSD violins, consistent with single-unit responses being more functionally homogeneous than fMRI voxels that average over hundreds of thousands of neurons; (2) **only TVSD-ResNet recovers the textbook hierarchical separation cleanly** — the other three panels show progressively weaker or absent gradients depending on modality and architecture.

---
### 3.10 Figure C — methods comparison (best-single vs per-ROI vs per-target)

Bar chart of test `pearsonr_nc` per ROI, per method, mean ± SEM across subjects. Does stacking beat the Section 2 baseline?

In [ ]:
def plot_baseline_delta(results_df, dataset_name, model_name, roi_list, ax):
    """Bar chart of (per-target − best-single) per ROI, mean ± SEM across subjects."""
    sub = results_df[(results_df['dataset'] == dataset_name) &
                     (results_df['model']   == model_name)].copy()
    if sub.empty:
        ax.set_title(f'{dataset_name} — {model_name}  (no data)')
        ax.axis('off'); return

    wide = (sub.pivot_table(index=['subject', 'roi'],
                             columns='method', values='pearsonr_nc')
              .reset_index())
    wide['delta'] = wide['per_target'] - wide['best_single']

    rois = [r for r in roi_list if r in wide['roi'].unique()]
    agg = (wide.groupby('roi')['delta']
                .agg(['mean', 'sem'])
                .reindex(rois))

    colors = ['#5b9a7a' if m >= 0 else '#c64545' for m in agg['mean']]
    x = np.arange(len(rois))
    ax.bar(x, agg['mean'], yerr=agg['sem'],
           color=colors, capsize=3, edgecolor='white', linewidth=0.5)
    ax.axhline(0, color='black', lw=1)
    ax.set_xticks(x); ax.set_xticklabels(rois, rotation=30)
    ax.set_ylabel('Δ pearsonr_nc')
    ax.set_title(f'{dataset_name.upper()} — {model_name}', fontsize=11)
    ax.grid(alpha=0.3, axis='y')


fig, axes = plt.subplots(2, 2, figsize=(13, 7.5), constrained_layout=True)
panels = [
    ('nsd',  'resnet', NSD_ROIS,  axes[0, 0]),
    ('nsd',  'qwen',   NSD_ROIS,  axes[0, 1]),
    ('tvsd', 'resnet', TVSD_ROIS, axes[1, 0]),
    ('tvsd', 'qwen',   TVSD_ROIS, axes[1, 1]),
]
for ds, mdl, rois, ax in panels:
    plot_baseline_delta(results_df, ds, mdl, rois, ax)

fig.suptitle('Stacking improvement over single-best-layer baseline\n'
             '(per-target stacking − best_single), mean ± SEM across subjects',
             y=1.04, fontsize=13)
plt.savefig(os.path.join(figures_dir, 'baseline_delta.png'),
            dpi=150, bbox_inches='tight')
plt.show()

---

# Final Discussion

End the notebook with a short final discussion.

<div style="background:#eef5fb; border-left:4px solid #4c78a8; padding:8px 12px; border-radius:6px; font-weight:700; color:#26445e;">What you must address</div>

- Which dataset appeared noisiest?
- Which neural targets were most reliable?
- Which model aligned best overall?
- Which metrics were most consistent with each other?
- What was the main limitation of your analysis?
- What would you try next with more time?

<div style="background:#fff4c2; border:1px solid #c89b1f; border-left:6px solid #9a6f00; padding:10px 12px; border-radius:6px; margin-top:8px; margin-bottom:4px; color:#241a00; line-height:1.45;"><strong style="color:#5c4300;">Answer box final</strong><br>Write a concise final conclusion of 1–2 paragraphs summarizing your main findings and their limitations.</div>

Signal quality dominated alignment quality across modalities: TVSD (aggregated MUA) was cleanest with a best r_NC of 0.764 on V1, NSD was intermediate and ROI-dependent — peaking at 0.761 in PPA but dropping below 0.03 in motor and auditory cortex — and THINGS-EEG2 was noisiest, with the best encoder reaching only r_NC = 0.524 and negative noise-corrected explained variance throughout. Adversarially trained ResNet-152 outperformed Qwen3-VL-2B on essentially every (dataset, ROI, metric) cell of the baseline and reproduced the canonical V1→V4→IT correspondence on TVSD; Qwen showed this hierarchy only partially. The encoder-derived metrics (pearsonr_nc, encoding-RSA, encoding-CKA) tracked each other tightly as expected, raw RSA and raw CKA agreed on layer rankings, but raw representational similarity decoupled from predictive accuracy on EEG — a useful reminder that geometric closeness and encoder-recoverable information are not the same thing once targets are noise-dominated.
As for the limitations, the baseline restricted each target to a single best layer with a flat linear readout. Section 3's per-target stacking partially addressed this and produced practically meaningful gains of ≈0.07–0.14 in r_NC on TVSD and NSD-Qwen, but the gain vanished on NSD-ResNet because one late layer already dominated — plausibly an artifact of the shared 30k random projection homogenizing layer differences. The extension was also limited to NSD and TVSD, leaving EEG's rich temporal structure unexploited, and only two models, two monkeys, and no untrained or ViT baseline were compared. With more time, the natural next steps are extending stacking to EEG with a joint temporal readout, comparing the softmax mixture against a PCA-based multi-layer combination — concatenating all layer features and reducing them jointly with PCA before ridge fitting, which mixes layers along data-driven directions rather than through per-target convex weights — replacing random projections with learned compression, and trying low-rank or nonlinear readouts to estimate how much of the remaining gap to the noise ceiling is recoverable.

---

# Report Content

Your **2-page PDF report** should tell a clear and coherent story. It does **not** need to reproduce every notebook result.

It can include:

1. **A brief dataset overview**
2. **An exploratory figure from Section 1**
3. **The EEG noise ceiling comparison**
4. **The NSD reliability visualization**
5. **One or two key brain–model alignment results from Section 2**


Rather, you should primarily focus on the open-ended extension you designed, describing:
- the motivation for your extension,
- the methods you implemented,
- the results you obtained,
- and the scientific insights you gained from it.


The report should emphasize interpretation, not just figures. Since the notebook is the main technical deliverable, the report should act as a **compressed scientific summary** of your most important findings rather than a figure dump.

---

# Detailed Grading Rubric

The project is graded out of **100 points** as follows:

- **Section 1: Inspection, Visualization, and Noise Ceiling Estimates — 20 points**
- **Section 2: Brain–Model Alignment — 20 points**
- **Section 3: Open-Ended Research — 30 points**
- **Report — 30 points**

**Section 0 is required but not graded separately.** It is treated as setup and reproducibility infrastructure for the rest of the notebook.

## Section 1 — 20 points

### 1.1 Dataset inspection — 3 points
- 1 pt: TVSD structure is correctly inspected and explained.
- 1 pt: EEG2 structure is correctly inspected and explained.
- 1 pt: NSD structure is correctly inspected and explained.

### 1.2 EEG visualization — 4 points
- 1 pt: example EEG time-course plot is present and readable.
- 1 pt: channel × time heatmap is present and readable.
- 1 pt: provided EEG noise ceiling visualization is present and readable.
- 1 pt: written interpretation identifies informative time windows or channel groups.

### 1.3 EEG noise ceiling estimation — 7 points
- 2 pts: variance-based estimator is implemented correctly.
- 2 pts: split-half estimator is implemented correctly.
- 1 pt: required summary visualizations are included.
- 1 pt: comparison to stored EEG noise ceilings is shown clearly.
- 1 pt: Answer box 1.3 interprets similarities and differences between estimators.

### 1.4 Statistical comparison of EEG noise ceilings — 3 points
- 1 pt: quantitative comparison table is present.
- 1 pt: statistical test or formal comparison is appropriate and correctly interpreted.
- 1 pt: final conclusion is clearly justified.

### 1.5 NSD reliability visualization — 3 points
- 1 pt: ncsnr is correctly converted and visualized on cortex.
- 1 pt: parcel overlay or parcel-wise summary is included.
- 1 pt: Answer box 1.5 correctly interprets reliable and unreliable regions.

## Section 2 — 20 points

### 2.1 RSA implementation — 3 points
- 1 pt: RDM computation is correct.
- 1 pt: RDM comparison is correct.
- 1 pt: implementation is used properly in later analyses.

### 2.2 Unbiased linear CKA implementation — 3 points
- 2 pts: unbiased linear CKA is implemented correctly.
- 1 pt: implementation is used properly in later analyses.

### 2.3 Representational analyses across layers, models, and targets — 4 points
- 1 pt: layer-wise RSA results are reported clearly.
- 1 pt: layer-wise CKA results are reported clearly.
- 1 pt: a model comparison is included.
- 1 pt: ROI-wise or time-resolved analysis is included and interpreted.

### 2.4 Predictive alignment with linear encoding models — 6 points
- 1 pt: required targets are selected and described correctly.
- 2 pts: train/validation/test procedure and ridge fitting are correct.
- 1 pt: required predictive metrics are reported correctly.
- 1 pt: encoding-RSA and encoding-CKA are reported correctly.
- 1 pt: best-layer summary and model comparison are included.

### 2.5 Compare predictive and representational metrics — 2 points
- 1 pt: ranking comparison figure is present and informative.
- 1 pt: agreement and disagreement between metrics are discussed clearly.

### 2.6 Layer hierarchy vs brain hierarchy — 1 point
- 1 pt: at least one hierarchy analysis is included and interpreted correctly.

### 2.7 Compare the two feature extractors — 1 point
- 1 pt: final comparison between Qwen3-VL and Adv-ResNet is clear and supported by results.

## Section 3 — 30 points

### Research question and motivation — 5 points
- 2 pts: research question is clear and focused.
- 3 pts: motivation is scientifically sensible and well connected to the baseline project.

### Method and implementation — 10 points
- 4 pts: the extension is described clearly.
- 4 pts: the method is implemented correctly.
- 2 pts: the design remains focused and technically appropriate for the project scope.

### Baseline comparison and evaluation — 10 points
- 4 pts: the comparison to the linear baseline is fair.
- 3 pts: at least one figure or table communicates the comparison clearly.
- 3 pts: evaluation supports the stated conclusion.

### Interpretation and limitations — 5 points
- 3 pts: the student explains whether the method helped in a practically meaningful way.
- 2 pts: limitations or caveats are acknowledged.

## Report — 30 points

### Structure and clarity — 6 points
- clear organization, readable flow, and concise scientific writing.

### Selection of results — 6 points
- the report focuses on the strongest and most relevant results rather than trying to include everything.

### Methodological correctness — 6 points
- metrics, comparisons, and claims are described accurately.

### Interpretation and synthesis — 6 points
- the report explains what the results mean and ties them back to the project goals.

### Figure quality and presentation — 6 points
- figures are readable, labeled, well-chosen, and integrated into the narrative.

## Important grading note

A submission that is technically correct but poorly interpreted will lose points. A submission with good intuition but missing required analyses will also lose points. The strongest submissions will be both **correct** and **scientifically well explained**.

---

# Final Checklist Before Submission

Before submitting, make sure that:

- group information is filled in,
- the notebook runs from top to bottom,
- all notebook outputs are cleared,
- figures have readable titles and labels,
- written answers are included in the answer boxes,
- the zip archive name follows the required format,
- no large unnecessary files are included.

---

# References

Use the references below when you need scientific context for the datasets, models, and analysis methods.

## Datasets

- Papale et al. (2025) — *An extensive dataset of spiking activity to reveal the syntax of the ventral stream*
- Gifford et al. (2022) — *A large and rich EEG dataset for modeling human visual object recognition*
- Allen et al. (2022) — *A massive 7T fMRI dataset to bridge cognitive neuroscience and artificial intelligence*
- van Bree, Styrnal, and Hebart (2025) — *How Much Variance Does Your Model Explain? A Clarifying Note On The Use Of Split-Half Reliability For Computing Noise Ceilings*

## Models

- Wong et al. (2020) — *Fast is better than free: Revisiting adversarial training*
- He et al. (2016) — *Deep Residual Learning for Image Recognition*
- Bai et al. (2025) — *Qwen3-VL Technical Report*

## Alignment and encoding

- Conwell et al. (2024) — *A large-scale examination of inductive biases shaping high-level visual representation in brains and machines*
- Gokce and Schrimpf (2025) — *Scaling Laws for Task-Optimized Models of the Primate Visual Ventral Stream*

Use these references selectively. You are not expected to read everything in full.